In [1]:
# ===========================================================================================
# CELL -2: COMPLETE PACKAGE INSTALLATION FOR INDICTRANS2 TATN
# ===========================================================================================

import sys
import subprocess

print("=" * 80)
print("STEP 1: CLEANING CONFLICTING PACKAGES")
print("=" * 80)

# Step 1: Clean uninstall (avoid conflicts)
packages_to_remove = [
    "transformers",
    "tokenizers", 
    "sentence-transformers",
    "huggingface-hub"
]

for pkg in packages_to_remove:
    print(f"Uninstalling {pkg}...", end=" ")
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", pkg],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False
        )
        print("✓")
    except Exception:
        print("(skipped)")

print("\n" + "=" * 80)
print("STEP 2: INSTALLING REQUIRED PACKAGES")
print("=" * 80)

# Step 2: Install required packages in correct order
packages_to_install = [
    ("transformers==4.46.0", "Transformers 4.46.0"),  # CHANGED: Official IndicTrans2 compatible version
    ("sentencepiece", "SentencePiece"),  # ADDED: Required for IndicTrans2 tokenizer
    ("protobuf", "Protobuf"),  # ADDED: Required for SentencePiece
    ("IndicTransToolkit", "IndicTransToolkit (latest)"),
    ("sacrebleu", "SacreBLEU"),  # CHANGED: Removed version constraint for flexibility
    ("sacremoses", "SacreMoses"),
]

installed_successfully = []
failed_packages = []

for package, display_name in packages_to_install:
    print(f"\nInstalling {display_name}...")
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            capture_output=True,
            text=True,
            check=True
        )
        print(f"  ✓ {display_name} installed successfully")
        installed_successfully.append(display_name)
    except subprocess.CalledProcessError as e:
        print(f"  ✗ Failed to install {display_name}")
        if e.stderr:
            print(f"  Error: {e.stderr[:200]}")
        failed_packages.append(display_name)

print("\n" + "=" * 80)
print("STEP 3: VERIFICATION")
print("=" * 80)

# Step 3: Verify all imports
verification_results = {}

print("\nChecking installed packages...")

# Check transformers (CRITICAL FOR INDICTRANS2)
try:
    import transformers
    import tokenizers
    print(f"✅ transformers: {transformers.__version__}")
    print(f"✅ tokenizers: {tokenizers.__version__}")
    
    # ADDED: Verify IndicTrans2 model classes are available
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    print(f"✅ AutoModelForSeq2SeqLM: Available")
    print(f"✅ AutoTokenizer: Available")
    verification_results['transformers'] = True
except ImportError as e:
    print(f"✗ transformers import failed: {e}")
    verification_results['transformers'] = False

# ADDED: Check SentencePiece (required for IndicTrans2 tokenizer)
try:
    import sentencepiece
    print(f"✅ sentencepiece: Available")
    verification_results['sentencepiece'] = True
except ImportError as e:
    print(f"✗ sentencepiece import failed: {e}")
    verification_results['sentencepiece'] = False

# Check IndicTransToolkit (CRITICAL FOR PREPROCESSING)
try:
    from IndicTransToolkit import IndicProcessor
    print(f"✅ IndicTransToolkit: Available")
    
    # Test IndicProcessor initialization
    test_processor = IndicProcessor(inference=True)
    print(f"✅ IndicProcessor: Initialized successfully")
    
    # ADDED: Test preprocessing functionality
    test_input = ["আমি ভালো আছি"]
    try:
        test_output = test_processor.preprocess_batch(
            test_input,
            src_lang="ben_Beng",
            tgt_lang="eng_Latn"
        )
        if test_output and len(test_output) > 0:
            print(f"✅ Preprocessing test: PASSED")
            print(f"   Sample output: {test_output[0][:60]}...")
        verification_results['IndicTransToolkit'] = True
    except Exception as e:
        print(f"⚠  Preprocessing test failed: {e}")
        verification_results['IndicTransToolkit'] = True  # Still mark as available
        
except ImportError as e:
    print(f"✗ IndicTransToolkit import failed: {e}")
    verification_results['IndicTransToolkit'] = False
except Exception as e:
    print(f"✗ IndicProcessor initialization failed: {e}")
    verification_results['IndicTransToolkit'] = False

# Check sacrebleu
try:
    import sacrebleu
    print(f"✅ sacrebleu: {sacrebleu.__version__}")
    verification_results['sacrebleu'] = True
except ImportError as e:
    print(f"✗ sacrebleu import failed: {e}")
    verification_results['sacrebleu'] = False

# Check sacremoses
try:
    import sacremoses
    print(f"✅ sacremoses: Available")
    verification_results['sacremoses'] = True
except ImportError as e:
    print(f"✗ sacremoses import failed: {e}")
    verification_results['sacremoses'] = False

print("\n" + "=" * 80)
print("INSTALLATION SUMMARY")
print("=" * 80)

total_packages = len(packages_to_install)
successful_verifications = sum(verification_results.values())

print(f"\nPackages installed: {len(installed_successfully)}/{total_packages}")
print(f"Verifications passed: {successful_verifications}/{len(verification_results)}")

if failed_packages:
    print(f"\n⚠ Failed packages: {', '.join(failed_packages)}")

if successful_verifications == len(verification_results):
    print("\n✅ ALL PACKAGES INSTALLED AND VERIFIED")
    print("=" * 80)
    print("🚀 Ready to proceed:")
    print("   1. Run Cell -1 (Hugging Face Authentication)")
    print("   2. Run Cell 0 (Configuration - IndicTrans2 settings)")
    print("   3. Run Cell 1 (Tokenizer utilities + IndicProcessor)")
    print("   4. Run Cell 2-7 (Model setup and training)")
    print("=" * 80)
    print("\n📋 INDICTRANS2 COMPONENTS VERIFIED:")
    print("   ✓ AutoModelForSeq2SeqLM (model loading)")
    print("   ✓ AutoTokenizer (tokenization)")
    print("   ✓ IndicProcessor (Bengali → Devanagari preprocessing)")
    print("   ✓ SentencePiece (subword tokenization)")
    print("=" * 80)
else:
    print("\n⚠ SOME PACKAGES FAILED")
    print("=" * 80)
    print("Troubleshooting steps:")
    print("1. Check internet connectivity")
    print("2. Restart kernel and re-run this cell")
    print("3. If IndicTransToolkit fails, try:")
    print("   !pip install git+https://github.com/VarunGumma/IndicTransToolkit.git")
    print("4. If transformers version conflicts occur:")
    print("   !pip install --upgrade transformers==4.46.0")
    print("=" * 80)
    
    # Raise error if critical packages failed
    critical_failed = []
    if not verification_results.get('transformers'):
        critical_failed.append('transformers')
    if not verification_results.get('IndicTransToolkit'):
        critical_failed.append('IndicTransToolkit')
    if not verification_results.get('sentencepiece'):
        critical_failed.append('sentencepiece')
    
    if critical_failed:
        raise RuntimeError(
            f"Critical packages failed to install: {', '.join(critical_failed)}. "
            "Please check the errors above and retry."
        )

print("\n[READY] You can now run the next cells ✓")


STEP 1: CLEANING CONFLICTING PACKAGES
Uninstalling transformers... ✓
Uninstalling tokenizers... ✓
Uninstalling sentence-transformers... ✓
Uninstalling huggingface-hub... ✓

STEP 2: INSTALLING REQUIRED PACKAGES

Installing Transformers 4.46.0...
  ✓ Transformers 4.46.0 installed successfully

Installing SentencePiece...
  ✓ SentencePiece installed successfully

Installing Protobuf...
  ✓ Protobuf installed successfully

Installing IndicTransToolkit (latest)...
  ✓ IndicTransToolkit (latest) installed successfully

Installing SacreBLEU...
  ✓ SacreBLEU installed successfully

Installing SacreMoses...
  ✓ SacreMoses installed successfully

STEP 3: VERIFICATION

Checking installed packages...
✅ transformers: 4.46.0
✅ tokenizers: 0.20.3
✅ AutoModelForSeq2SeqLM: Available
✅ AutoTokenizer: Available
✅ sentencepiece: Available


2026-03-03 08:15:55.057207: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772525755.453195      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772525755.564976      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772525756.475415      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772525756.475451      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772525756.475454      55 computation_placer.cc:177] computation placer alr

✅ IndicTransToolkit: Available
✅ IndicProcessor: Initialized successfully
✅ Preprocessing test: PASSED
   Sample output: ben_Beng eng_Latn आमि भालो आछि...
✅ sacrebleu: 2.6.0
✅ sacremoses: Available

INSTALLATION SUMMARY

Packages installed: 6/6
Verifications passed: 5/5

✅ ALL PACKAGES INSTALLED AND VERIFIED
🚀 Ready to proceed:
   1. Run Cell -1 (Hugging Face Authentication)
   2. Run Cell 0 (Configuration - IndicTrans2 settings)
   3. Run Cell 1 (Tokenizer utilities + IndicProcessor)
   4. Run Cell 2-7 (Model setup and training)

📋 INDICTRANS2 COMPONENTS VERIFIED:
   ✓ AutoModelForSeq2SeqLM (model loading)
   ✓ AutoTokenizer (tokenization)
   ✓ IndicProcessor (Bengali → Devanagari preprocessing)
   ✓ SentencePiece (subword tokenization)

[READY] You can now run the next cells ✓


In [2]:
# ===========================================================================================
# CELL -1: HUGGING FACE AUTHENTICATION FOR INDICTRANS2
# ===========================================================================================

import os
import sys

print("=" * 80)
print("HUGGING FACE AUTHENTICATION FOR INDICTRANS2")
print("=" * 80)

HF_TOKEN = None
AUTH_SUCCESS = False

print("\n[METHOD 1] Trying Kaggle Secrets...")
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HUGGINGFACE_TOKEN")
    print("  ✓ Token retrieved from Kaggle Secrets")
except Exception as e:
    print(f"  ✗ Kaggle Secrets not available: {type(e).__name__}")

if not HF_TOKEN:
    print("\n[METHOD 2] Trying Environment Variables...")
    HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
    if HF_TOKEN:
        print("  ✓ Token found in environment")
    else:
        print("  ✗ No token in environment")

if not HF_TOKEN:
    print("\n" + "=" * 80)
    print("❌ NO TOKEN FOUND - SETUP REQUIRED")
    print("=" * 80)
    print("\n📋 SETUP INSTRUCTIONS:")
    print("\n1. CREATE NEW TOKEN:")
    print("   → https://huggingface.co/settings/tokens")
    print("   → Click 'New token'")
    print("   → Name: kaggle_indictrans2")
    print("   → Type: Read")
    print("   → Generate and COPY it")
    print("\n2. ADD TO KAGGLE SECRETS:")
    print("   → https://www.kaggle.com/settings")
    print("   → Go to 'Secrets' tab")
    print("   → Click 'Add a new secret'")
    print("   → Label: HUGGINGFACE_TOKEN")
    print("   → Value: Paste your token")
    print("   → Click 'Add'")
    print("\n3. ENABLE IN THIS NOTEBOOK:")
    print("   → Notebook settings (right sidebar)")
    print("   → Secrets section")
    print("   → Toggle ON: HUGGINGFACE_TOKEN")
    print("   → Also enable: Internet")
    print("\n4. RESTART KERNEL:")
    print("   → Kernel → Restart")
    print("   → Re-run this cell")
    print("\n" + "=" * 80)
    raise RuntimeError("Hugging Face token required - follow setup instructions above")

print("\n[STEP 2] Installing/Updating huggingface_hub...")
try:
    from huggingface_hub import login, HfFolder
    print("  ✓ huggingface_hub already installed")
except ImportError:
    print("  Installing huggingface_hub...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import login, HfFolder
    print("  ✓ Installed huggingface_hub")

print("\n[STEP 3] Authenticating with Hugging Face...")
try:
    login(token=HF_TOKEN, add_to_git_credential=False)
    
    stored_token = HfFolder.get_token()
    if stored_token:
        print("  ✓ Authentication successful")
        AUTH_SUCCESS = True
    else:
        print("  ✗ Token not stored properly")
        AUTH_SUCCESS = False
        
except Exception as e:
    print(f"  ✗ Authentication failed: {e}")
    AUTH_SUCCESS = False

if AUTH_SUCCESS:
    print("\n[STEP 4] Verifying IndicTrans2 access...")
    
    # ─── TO (200M distilled ~800MB each, fits T4 easily) ───
    INDICTRANS2_MODELS = [
        "ai4bharat/indictrans2-en-indic-dist-200M",   # English → Bengali
        "ai4bharat/indictrans2-indic-en-dist-200M"    # Bengali → English
    ]
    
    access_granted = []
    access_denied = []
    
    for model_name in INDICTRANS2_MODELS:
        try:
            print(f"  Testing {model_name.split('/')[-1]}...")
            
            from transformers import AutoTokenizer
            test_tokenizer = AutoTokenizer.from_pretrained(
                model_name,
                trust_remote_code=True,
                token=HF_TOKEN
            )
            
            vocab_size = len(test_tokenizer)
            print(f"    ✓ Access granted (vocab: {vocab_size})")
            access_granted.append(model_name)
            
            del test_tokenizer
            
        except Exception as e:
            error_msg = str(e)
            print(f"    ✗ Access denied: {type(e).__name__}")
            access_denied.append((model_name, error_msg))
    
    if len(access_granted) == len(INDICTRANS2_MODELS):
        print("\n" + "=" * 80)
        print("✅ AUTHENTICATION COMPLETE")
        print("=" * 80)
        print("Status: Ready to run Cell 0 → Cell 11")
        print(f"Models accessible: {len(access_granted)}/{len(INDICTRANS2_MODELS)}")
        for model in access_granted:
            print(f"  ✓ {model}")
        print("=" * 80 + "\n")
        
    else:
        print("\n" + "=" * 80)
        print("❌ ACCESS NOT GRANTED YET")
        print("=" * 80)
        print("\n🔓 REQUEST ACCESS (Required - One Time):")
        print("\n1. Visit these URLs and click 'Request Access':")
        for model, _ in access_denied:
            print(f"   → https://huggingface.co/{model}")
        print("\n2. Access is usually granted INSTANTLY")
        print("\n3. Wait 30 seconds, then re-run this cell")
        print("\n" + "=" * 80)
        
        AUTH_SUCCESS = False

if not AUTH_SUCCESS:
    print("\n" + "=" * 80)
    print("❌ SETUP INCOMPLETE")
    print("=" * 80)
    print("Cannot proceed without authentication")
    print("Follow instructions above to complete setup")
    print("=" * 80 + "\n")
    raise RuntimeError("Authentication required")

print("\n✅ Cell -1 complete. Proceed to Cell 0.\n")


HUGGING FACE AUTHENTICATION FOR INDICTRANS2

[METHOD 1] Trying Kaggle Secrets...
  ✓ Token retrieved from Kaggle Secrets

[STEP 2] Installing/Updating huggingface_hub...
  ✓ huggingface_hub already installed

[STEP 3] Authenticating with Hugging Face...
  ✓ Authentication successful

[STEP 4] Verifying IndicTrans2 access...
  Testing indictrans2-en-indic-dist-200M...


tokenizer_config.json:   0%|          | 0.00/1.11k [00:00<?, ?B/s]

tokenization_indictrans.py:   0%|          | 0.00/8.04k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json:   0%|          | 0.00/645k [00:00<?, ?B/s]

...

dict.SRC.json:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

dict.TGT.json:   0%|          | 0.00/645k [00:00<?, ?B/s]

model.SRC:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

model.TGT:   0%|          | 0.00/759k [00:00<?, ?B/s]

    ✓ Access granted (vocab: 122706)

✅ AUTHENTICATION COMPLETE
Status: Ready to run Cell 0 → Cell 11
Models accessible: 2/2
  ✓ ai4bharat/indictrans2-en-indic-dist-200M
  ✓ ai4bharat/indictrans2-indic-en-dist-200M


✅ Cell -1 complete. Proceed to Cell 0.



In [3]:
# ==============================================================================
# CELL 0: DUAL-PATH TATN CONFIGURATION - INDICTRANS2 VERSION (FIXED)
# ==============================================================================
import os
import sys
import math
import random
import re
import unicodedata
import time
import threading
from pathlib import Path
from collections import deque, defaultdict
from typing import List, Dict, Tuple, Optional, Union, Set, Any
from types import SimpleNamespace

import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import warnings
import gc

os.environ['CUDA_LAUNCH_BLOCKING'] = '0'

try:
    import pandas as pd
    _HAS_PANDAS = True
except ImportError:
    _HAS_PANDAS = False

try:
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    _HAS_INDICTRANS2 = True
except Exception:
    AutoModelForSeq2SeqLM = None
    AutoTokenizer = None
    _HAS_INDICTRANS2 = False

try:
    from IndicTransToolkit.processor import IndicProcessor
    _HAS_INDIC_PROCESSOR = True
except ImportError:
    IndicProcessor = None
    _HAS_INDIC_PROCESSOR = False

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Force single GPU — DataParallel DISABLED
NUM_GPUS = 1
USE_MULTI_GPU = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Cell 0] Single GPU Mode (DataParallel DISABLED)")
print(f"[Cell 0] Device: {DEVICE}")


DATASET_CSV_PATH = os.environ.get(
    "DATASET_PATH",
    "/kaggle/input/datasets/manaskumarmanna/bpcc-data/bpcc_bn_en_400k.csv"
)

def _safe_int(value, default: int, name: str, min_val: int = 1) -> int:
    try:
        result = int(value)
        if result < min_val:
            return default
        return result
    except:
        return default

def _safe_float(value, default: float, name: str, min_val: float = 0.0) -> float:
    try:
        result = float(value)
        if result < min_val:
            return default
        return result
    except:
        return default

BATCH_SIZE           = _safe_int(2, 2, "BATCH_SIZE", min_val=1)
NUM_SAMPLES          = _safe_int(60000, 60000, "NUM_SAMPLES", min_val=1)
MAX_LENGTH           = _safe_int(128, 128, "MAX_LENGTH", min_val=8)

LR_NMT               = _safe_float(3e-5, 3e-5, "LR_NMT", min_val=1e-7)
LR_TRG               = _safe_float(1e-5, 1e-5, "LR_TRG", min_val=1e-7)
LR_PHI               = _safe_float(1e-4, 1e-4, "LR_PHI", min_val=1e-7)

EPOCHS               = _safe_int(2, 2, "EPOCHS", min_val=1)
GRAD_CLIP_NORM       = _safe_float(1.0, 1.0, "GRAD_CLIP_NORM", min_val=0.1)
USE_AMP              = True
PRINT_INTERVAL       = _safe_int(200, 200, "PRINT_INTERVAL", min_val=1)
SEED                 = _safe_int(42, 42, "SEED", min_val=0)
ACCUMULATION_STEPS   = _safe_int(4, 4, "ACCUMULATION_STEPS", min_val=1)

MC_DROPOUT_PASSES    = _safe_int(3, 3, "MC_DROPOUT_PASSES", min_val=1)
TRG_EVIDENCE_K       = _safe_int(3, 3, "TRG_EVIDENCE_K", min_val=1)
MAX_SILVER_BUFFER    = _safe_int(50, 50, "MAX_SILVER_BUFFER", min_val=1)

NUM_WORKERS          = _safe_int(1, 1, "NUM_WORKERS", min_val=0)
PIN_MEMORY           = True
PREFETCH_FACTOR      = _safe_int(1, 1, "PREFETCH_FACTOR", min_val=1)
GRADIENT_CHECKPOINTING = True

ENABLE_OOM_PROTECTION            = True
OOM_BATCH_SIZE_REDUCTION         = True
OOM_EMERGENCY_CLEANUP_FREQUENCY  = _safe_int(50, 50, "OOM_EMERGENCY_CLEANUP_FREQUENCY", min_val=1)
OOM_MEMORY_THRESHOLD             = _safe_float(0.90, 0.90, "OOM_MEMORY_THRESHOLD", min_val=0.5)

ENABLE_VALIDATION       = True
VALIDATION_FREQUENCY    = _safe_int(200, 200, "VALIDATION_FREQUENCY", min_val=1)
VALIDATION_SAMPLES      = _safe_int(50, 50, "VALIDATION_SAMPLES", min_val=1)
VALIDATION_MAX_LENGTH   = _safe_int(32, 32, "VALIDATION_MAX_LENGTH", min_val=8)

DEBUG_PATH1      = True
DEBUG_PATH2      = True
DEBUG_LOSSES     = True
DEBUG_GRADIENTS  = True
DEBUG_MEMORY     = True
DEBUG_CLUSTERS   = True
DEBUG_FREQUENCY  = _safe_int(100, 100, "DEBUG_FREQUENCY", min_val=1)
DEBUG_DISCOVERY  = False
DEBUG_TIMING     = True
DEBUG_VERBOSE    = False

DSCD_BUFFER_SIZE               = _safe_int(500, 500, "DSCD_BUFFER_SIZE", min_val=1)
DSCD_MAX_PROTOS                = _safe_int(10, 10, "DSCD_MAX_PROTOS", min_val=1)
DSCD_N_MIN                     = _safe_int(2, 2, "DSCD_N_MIN", min_val=1)
DSCD_DISPERSION_THRESHOLD      = _safe_float(0.15, 0.15, "DSCD_DISPERSION_THRESHOLD", min_val=0.0)
DSCD_NEWSENSE_LAMBDA           = _safe_float(1.2, 1.2, "DSCD_NEWSENSE_LAMBDA", min_val=0.1)
DSCD_EMBED_DIM                 = _safe_int(512, 512, "DSCD_EMBED_DIM", min_val=64)
DSCD_TEMPERATURE               = _safe_float(0.7, 0.7, "DSCD_TEMPERATURE", min_val=0.01)
DSCD_DROPOUT                   = _safe_float(0.1, 0.1, "DSCD_DROPOUT", min_val=0.0)
DSCD_AUGMENT_SCALE             = _safe_float(0.05, 0.05, "DSCD_AUGMENT_SCALE", min_val=0.0)
DSCD_ENABLE_TRAINING_CLUSTERING = True
DSCD_WARMUP_SAMPLES            = _safe_int(500, 500, "DSCD_WARMUP_SAMPLES", min_val=0)
DSCD_MIN_LETTERS               = _safe_int(3, 3, "DSCD_MIN_LETTERS", min_val=2)
DSCD_MIN_LETTER_FRACTION       = _safe_float(0.6, 0.6, "DSCD_MIN_LETTER_FRACTION", min_val=0.0)
DSCD_ENABLE_BUFFER_CLEANUP     = True

PERIODIC_DISCOVERY_FREQUENCY   = _safe_int(100, 100, "PERIODIC_DISCOVERY_FREQUENCY", min_val=1)
MAX_TOKENS_PER_DISCOVERY       = _safe_int(100, 100, "MAX_TOKENS_PER_DISCOVERY", min_val=1)

ENABLE_PATH1_TRAINING   = True
ENABLE_PATH2_TRAINING   = True
PATH1_START_STEP        = _safe_int(100, 100, "PATH1_START_STEP", min_val=0)

ENABLE_ASBN_TRAINING    = True
ENABLE_ASBN_INFERENCE   = True
ENABLE_TRG_TRAINING     = True
ENABLE_TRG_INFERENCE    = True
USE_DUAL_PATH_TRAINING  = True

CLUSTERING_TIMEOUT         = _safe_int(3, 3, "CLUSTERING_TIMEOUT", min_val=1)
MEMORY_CLEANUP_FREQUENCY   = _safe_int(50, 50, "MEMORY_CLEANUP_FREQUENCY", min_val=1)
VALIDATION_CHECK_INTERVAL  = _safe_int(200, 200, "VALIDATION_CHECK_INTERVAL", min_val=1)
VERBOSE_LOGGING            = False

CHECKPOINT_DIR               = "/kaggle/working/"
CHECKPOINT_SAVE_AFTER_TRAINING = True
CHECKPOINT_FILENAME          = "tatn_final.pt"
CHECKPOINT_INTERVAL          = 99999999
SAVE_CHECKPOINT_FREQUENCY    = _safe_int(500, 500, "SAVE_CHECKPOINT_FREQUENCY", min_val=1)
EMERGENCY_CHECKPOINT_ON_OOM  = True

SAVE_REPLAY_BUFFER    = False
LOAD_REPLAY_BUFFER    = False
REPLAY_BUFFER_SIZE    = _safe_int(10000, 10000, "REPLAY_BUFFER_SIZE", min_val=0)
RESUME_FROM_CHECKPOINT = False
CHECKPOINT_PATH       = ""

TAU_LOW    = _safe_float(0.15, 0.15, "TAU_LOW", min_val=0.0)
TAU_HIGH   = _safe_float(0.85, 0.85, "TAU_HIGH", min_val=0.0)
TAU_ACCEPT = _safe_float(0.70, 0.70, "TAU_ACCEPT", min_val=0.0)

TRG_MAX_GEN_LEN              = _safe_int(12, 12, "TRG_MAX_GEN_LEN", min_val=1)
TRG_GEN_EMBED                = _safe_int(64, 64, "TRG_GEN_EMBED", min_val=8)
TRG_GEN_HID                  = _safe_int(64, 64, "TRG_GEN_HID", min_val=8)
TRG_SPAN_THRESHOLD           = _safe_float(0.20, 0.20, "TRG_SPAN_THRESHOLD", min_val=0.0)
TRG_UNCERTAINTY_THRESHOLD    = _safe_float(0.15, 0.15, "TRG_UNCERTAINTY_THRESHOLD", min_val=0.0)
TRG_TEMPERATURE              = _safe_float(1.0, 1.0, "TRG_TEMPERATURE", min_val=0.01)
MAX_EXPLANATIONS_PER_SENTENCE = _safe_int(10, 10, "MAX_EXPLANATIONS_PER_SENTENCE", min_val=1)

SPAN_THRESHOLD         = _safe_float(0.20, 0.20, "SPAN_THRESHOLD", min_val=0.0)
UNCERTAINTY_THRESHOLD  = _safe_float(0.15, 0.15, "UNCERTAINTY_THRESHOLD", min_val=0.0)

ASBN_MAX_BUFFER_SIZE = _safe_int(50, 50, "ASBN_MAX_BUFFER_SIZE", min_val=1)
ASBN_HIDDEN_DIM      = _safe_int(512, 512, "ASBN_HIDDEN_DIM", min_val=8)
ASBN_LAMBDA          = _safe_float(0.05, 0.05, "ASBN_LAMBDA", min_val=0.0)
ASBN_DROPOUT         = _safe_float(0.1, 0.1, "ASBN_DROPOUT", min_val=0.0)
WARMUP_STEPS         = _safe_int(4000, 4000, "WARMUP_STEPS", min_val=1)

LAMBDA_PATH1      = _safe_float(0.1, 0.1, "LAMBDA_PATH1", min_val=0.0)
LAMBDA_PATH2      = _safe_float(1.0, 1.0, "LAMBDA_PATH2", min_val=0.0)
LAMBDA_ASBN       = _safe_float(0.0, 0.0, "LAMBDA_ASBN", min_val=0.0)
LAMBDA_DSCD       = _safe_float(0.15, 0.15, "LAMBDA_DSCD", min_val=0.0)
LAMBDA_TRG        = _safe_float(0.15, 0.15, "LAMBDA_TRG", min_val=0.0)
LAMBDA_TOKEN      = _safe_float(0.0, 0.0, "LAMBDA_TOKEN", min_val=0.0)
LAMBDA_CONFIDENCE = _safe_float(0.0, 0.0, "LAMBDA_CONFIDENCE", min_val=0.0)
LAMBDA_LENGTH     = _safe_float(0.0, 0.0, "LAMBDA_LENGTH", min_val=0.0)

LABEL_SMOOTHING   = _safe_float(0.1, 0.1, "LABEL_SMOOTHING", min_val=0.0)
RDROP_ALPHA       = _safe_float(5.0, 5.0, "RDROP_ALPHA", min_val=0.0)
USE_RDROP         = False
WEIGHT_DECAY      = _safe_float(0.01, 0.01, "WEIGHT_DECAY", min_val=0.0)

TRAIN_DOMAIN    = 0
TEST_DOMAIN     = 1
USE_DOMAIN_LABELS = True

GRL_ALPHA_START    = _safe_float(0.1, 0.1, "GRL_ALPHA_START", min_val=0.0)
GRL_ALPHA_END      = _safe_float(1.0, 1.0, "GRL_ALPHA_END", min_val=0.0)
GRL_ALPHA_SCHEDULE = "linear"
GRL_ALPHA_STEPS    = _safe_int(500, 500, "GRL_ALPHA_STEPS", min_val=1)

MODEL_NAME         = "ai4bharat/indictrans2-indic-en-dist-200M"
SOURCE_LANGUAGE    = "ben_Beng"
TARGET_LANGUAGE    = "eng_Latn"
RESIZE_MODEL_EMBEDDINGS = True

FREEZE_ENCODER         = False
FREEZE_FIRST_N_LAYERS  = 4

EVAL_BATCH_SIZE     = _safe_int(4, 4, "EVAL_BATCH_SIZE", min_val=1)
EVAL_NUM_BEAMS      = _safe_int(5, 5, "EVAL_NUM_BEAMS", min_val=1)
EVAL_LENGTH_PENALTY = _safe_float(1.0, 1.0, "EVAL_LENGTH_PENALTY", min_val=0.0)
LABEL_SMOOTHING_EPS = LABEL_SMOOTHING

HOMOGRAPH_REFERENCE_LIST_BN: Set[str] = {
    "কল", "কাল", "পাতা", "ফল", "বার", "হার", "তারা",
    "পড়া", "দেখা", "চলা", "ধরা", "অর্থ", "শব্দ", "মুখ",
    "তোলা", "বাঁচা", "মারা", "উত্তর", "পাত্র", "বেলা", "গান",
    "নাম", "বল", "চাল", "কলা", "ধারা", "পত্র", "রাগ", "রস",
    "তীর", "জমা", "মান", "দাবি", "আসন", "সাড়া", "বসা", "পদ",
    "অংশ", "মোড়", "ঘর", "মন", "ব্যাংক"
}

HOMOGRAPH_WATCHLIST_BN: Set[str] = set()
HOMOGRAPH_WATCHLIST: Set[str] = set()
USE_WATCHLIST_PRIORITIZATION = False
WATCHLIST_ONLY_FOR_TRG = False

def normalize_bengali(t: str) -> str:
    if not t:
        return ""
    t = unicodedata.normalize("NFKC", t)
    t = t.replace("▁", "").replace("##", "").strip()
    return t

def normalize_english(t: str) -> str:
    if not t:
        return ""
    t = unicodedata.normalize("NFKC", t).lower().strip()
    return t

def empty_cuda_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

def safe_cuda_synchronize() -> None:
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass

def monitor_gpu_usage() -> None:
    if torch.cuda.is_available():
        visible_gpus = torch.cuda.device_count()
        print(f"\n[GPU MONITOR] Checking {visible_gpus} GPU(s):")
        for i in range(visible_gpus):
            try:
                mem_alloc = torch.cuda.memory_allocated(i) / (1024 ** 3)
                mem_reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)
                print(f"  GPU {i}: {mem_alloc:.2f}GB allocated / {mem_reserved:.2f}GB reserved")
            except Exception:
                pass

def get_checkpoint_path() -> str:
    return os.path.join(CHECKPOINT_DIR, CHECKPOINT_FILENAME)

def should_save_checkpoint(global_step: int, epoch: int, is_final: bool = False) -> bool:
    if is_final and CHECKPOINT_SAVE_AFTER_TRAINING:
        return True
    return False

class FunctionTimeoutError(Exception):
    pass

def with_timeout(seconds: int):
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = [FunctionTimeoutError("Function timed out")]
            def target():
                try:
                    result[0] = func(*args, **kwargs)
                except Exception as e:
                    result[0] = e
            thread = threading.Thread(target=target, daemon=True)
            thread.start()
            thread.join(timeout=seconds)
            if thread.is_alive():
                return None
            if isinstance(result[0], Exception):
                if isinstance(result[0], FunctionTimeoutError):
                    return None
                raise result[0]
            return result[0]
        return wrapper
    return decorator

def get_special_tokens(tokenizer) -> Set[str]:
    try:
        s = set(getattr(tokenizer, "all_special_tokens", []))
    except Exception:
        s = {"<pad>", "</s>", "<s>", "<unk>"}
    s.update({SOURCE_LANGUAGE, TARGET_LANGUAGE})
    return s

_token_validation_cache: Dict[Tuple[str, str], bool] = {}
_cache_lock = threading.Lock()
_cache_max_size = 5000

def is_valid_token(
    token,
    special_tokens: Optional[Set[str]] = None,
    tokenizer=None,
    language: str = "bn",
) -> bool:
    token = "" if token is None else str(token)
    cache_key = (token, language)
    with _cache_lock:
        if cache_key in _token_validation_cache:
            return _token_validation_cache[cache_key]
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    if special_tokens and token in special_tokens:
        result = False
    else:
        if len(clean) < 2:
            result = False
        else:
            has_bengali_chars = any("\u0980" <= c <= "\u09FF" for c in clean)
            if not has_bengali_chars:
                result = False
            else:
                bengali_count = sum(1 for c in clean if "\u0980" <= c <= "\u09FF")
                alphanum_count = sum(1 for c in clean if c.isalnum())
                if alphanum_count == 0:
                    result = False
                else:
                    result = (bengali_count / alphanum_count) >= 0.5
    with _cache_lock:
        if len(_token_validation_cache) < _cache_max_size:
            _token_validation_cache[cache_key] = result
    return result

class DiscoveryTimer:
    def __init__(self):
        self.discovery_times: List[float] = []
        self.discovery_steps: List[int] = []

    def record(self, step: int, duration: float) -> None:
        self.discovery_times.append(duration)
        self.discovery_steps.append(step)

    def get_stats(self) -> Dict[str, float]:
        if not self.discovery_times:
            return {"count": 0, "total": 0.0, "avg": 0.0, "max": 0.0}
        total = sum(self.discovery_times)
        return {
            "count": len(self.discovery_times),
            "total": total,
            "avg": total / len(self.discovery_times),
            "max": max(self.discovery_times),
        }

_discovery_timer = DiscoveryTimer()
discoverytimer = _discovery_timer

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if hasattr(torch, "set_float32_matmul_precision"):
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

try:
    effective_batch = BATCH_SIZE * ACCUMULATION_STEPS
    if USE_MULTI_GPU and NUM_GPUS > 0:
        effective_batch *= NUM_GPUS
except Exception:
    effective_batch = BATCH_SIZE

print("\n" + "=" * 80)
print("DUAL-PATH TATN CONFIGURATION - INDICTRANS2 VERSION")
print("=" * 80)
print(f"User: {os.getenv('KAGGLE_USERNAME', os.getenv('USER', 'manas0003'))}")
print(f"Multi-GPU: {'ENABLED' if USE_MULTI_GPU else 'DISABLED'} ({NUM_GPUS} GPUs)")
print(f"Dataset: {DATASET_CSV_PATH}")
print(f"Samples: {NUM_SAMPLES:,} | Batch: {BATCH_SIZE} | Accum: {ACCUMULATION_STEPS}")
print(f"Effective batch: {effective_batch}")
print(f"Max length: {MAX_LENGTH} | Epochs: {EPOCHS}")
print()
print("✅ OOM PROTECTION:")
print(f"  - Enabled: {ENABLE_OOM_PROTECTION}")
print(f"  - Cleanup frequency: {OOM_EMERGENCY_CLEANUP_FREQUENCY} steps")
print(f"  - Memory threshold: {OOM_MEMORY_THRESHOLD * 100:.0f}%")
print(f"  - Gradient checkpointing: {GRADIENT_CHECKPOINTING}")
print()
print("✅ VALIDATION:")
print(f"  - Enabled: {ENABLE_VALIDATION}")
print(f"  - Frequency: {VALIDATION_FREQUENCY} steps")
print(f"  - Samples: {VALIDATION_SAMPLES}")
print()
print("✅ DEBUG FLAGS:")
print(f"  - Path1: {DEBUG_PATH1}")
print(f"  - Path2: {DEBUG_PATH2}")
print(f"  - Losses: {DEBUG_LOSSES}")
print(f"  - Gradients: {DEBUG_GRADIENTS}")
print(f"  - Memory: {DEBUG_MEMORY}")
print(f"  - Clusters: {DEBUG_CLUSTERS}")
print(f"  - Frequency: {DEBUG_FREQUENCY} steps")
print()
print("✅ PATH 1 (TATN - Homograph Detection):")
print(f"  - Word-level tokenization")
print(f"  - Start step: {PATH1_START_STEP}")
print(f"  - DSCD discovery freq: {PERIODIC_DISCOVERY_FREQUENCY} steps")
print(f"  - DSCD warmup: {DSCD_WARMUP_SAMPLES} samples")
print(f"  - DSCD buffer size: {DSCD_BUFFER_SIZE}")
print(f"  - DSCD max protos: {DSCD_MAX_PROTOS}")
print(f"  - DSCD N_min: {DSCD_N_MIN}")
print(f"  - DSCD dispersion: {DSCD_DISPERSION_THRESHOLD}")
print(f"  - DSCD embed_dim: {DSCD_EMBED_DIM}")
print(f"  - ASBN: {'DISABLED' if not ENABLE_ASBN_TRAINING else 'ENABLED'}")
print(f"  - ASBN hidden_dim: {ASBN_HIDDEN_DIM}")
print(f"  - TRG: {'ENABLED' if ENABLE_TRG_TRAINING else 'DISABLED'}")
print(f"  - LAMBDA_PATH1: {LAMBDA_PATH1}")
print(f"  - LAMBDA_DSCD: {LAMBDA_DSCD}")
print(f"  - LAMBDA_TRG: {LAMBDA_TRG}")
print()
print("✅ PATH 2 (IndicTrans2 - Translation):")
print(f"  - Model: {MODEL_NAME}")
print(f"  - Source language: {SOURCE_LANGUAGE}")
print(f"  - Target language: {TARGET_LANGUAGE}")
print(f"  - Subword tokenization (SentencePiece)")
print(f"  - LAMBDA_PATH2: {LAMBDA_PATH2}")
print(f"  - Learning rate: {LR_NMT}")
print(f"  - Label smoothing: {LABEL_SMOOTHING}")
print(f"  - R-Drop alpha: {RDROP_ALPHA}")
print(f"  - Weight decay: {WEIGHT_DECAY}")
print(f"  - Warmup steps: {WARMUP_STEPS}")
print(f"  - Frozen layers: First {FREEZE_FIRST_N_LAYERS} encoder/decoder")
print(f"  - Resize embeddings: {'YES' if RESIZE_MODEL_EMBEDDINGS else 'NO'}")
print()
print("✅ TOXIC PENALTIES REMOVED:")
print(f"  - LAMBDA_TOKEN: 0.25 → {LAMBDA_TOKEN} (DISABLED)")
print(f"  - LAMBDA_CONFIDENCE: 0.15 → {LAMBDA_CONFIDENCE} (DISABLED)")
print(f"  - LAMBDA_LENGTH: 0.05 → {LAMBDA_LENGTH} (DISABLED)")
print(f"  - LAMBDA_ASBN: 0.3 → {LAMBDA_ASBN} (DISABLED)")
print()
print("📊 DSCD CONFIGURATION:")
print(f"  - Discovery frequency: {PERIODIC_DISCOVERY_FREQUENCY} steps")
print(f"  - Dispersion threshold: {DSCD_DISPERSION_THRESHOLD}")
print(f"  - Buffer size: {DSCD_BUFFER_SIZE}")
print(f"  - Max prototypes: {DSCD_MAX_PROTOS}")
print(f"  - N_min: {DSCD_N_MIN}")
print(f"  - New sense lambda: {DSCD_NEWSENSE_LAMBDA}")
print(f"  - Buffer cleanup: {DSCD_ENABLE_BUFFER_CLEANUP}")
print()
print("📊 TRG CONFIGURATION:")
print(f"  - SPAN_THRESHOLD: {SPAN_THRESHOLD}")
print(f"  - UNCERTAINTY_THRESHOLD: {UNCERTAINTY_THRESHOLD}")
print(f"  - TRG_SPAN_THRESHOLD: {TRG_SPAN_THRESHOLD}")
print(f"  - TRG_UNCERTAINTY_THRESHOLD: {TRG_UNCERTAINTY_THRESHOLD}")
print(f"  - TAU_LOW: {TAU_LOW}")
print()
print("📊 EVALUATION SETTINGS:")
print(f"  - Batch size: {EVAL_BATCH_SIZE}")
print(f"  - Num beams: {EVAL_NUM_BEAMS}")
print(f"  - Length penalty: {EVAL_LENGTH_PENALTY}")
print()
monitor_gpu_usage()

print("\n" + "=" * 80)
print("Cell 0: Dual-Path Configuration loaded - Ready for training")
print("=" * 80)

print("\n[Cell 0] Loading IndicTrans2 tokenizer...")

def _safe_get_vocab(tokenizer):
    try:
        return tokenizer.get_vocab()
    except Exception:
        try:
            return dict(getattr(tokenizer, "vocab", {}))
        except Exception:
            return None

def validate_language_token(tokenizer, lang_code: str, role: str):
    vocab = _safe_get_vocab(tokenizer)
    vocab_size = None
    try:
        vocab_size = len(tokenizer)
    except Exception:
        try:
            vocab_size = len(vocab) if vocab is not None else None
        except Exception:
            vocab_size = None

    lang_id = None
    mapping_source = None

    if hasattr(tokenizer, "lang_code_to_id"):
        try:
            mapping_source = "lang_code_to_id"
            lang_id = tokenizer.lang_code_to_id.get(lang_code)
        except Exception:
            lang_id = None

    if lang_id is None:
        try:
            mapping_source = "convert_tokens_to_ids"
            lang_id = tokenizer.convert_tokens_to_ids(lang_code)
            if hasattr(tokenizer, "unk_token_id") and lang_id == getattr(tokenizer, "unk_token_id"):
                lang_id = None
        except Exception:
            lang_id = None

    if lang_id is None and vocab is not None:
        candidates = [
            lang_code,
            f"<{lang_code}>",
            f"</s>{lang_code}",
            f">>{lang_code}<<",
            f"▁{lang_code}"
        ]
        for cand in candidates:
            if cand in vocab:
                lang_id = vocab.get(cand)
                mapping_source = f"vocab_key:{cand}"
                break

    if lang_id is None:
        print(f"\n[Tokenizer Validation] {role} language code '{lang_code}' NOT FOUND in tokenizer.")
        print("  Suggested actions:")
        print("   - Ensure the tokenizer/model supports explicit language codes (e.g., mBART-style).")
        print("   - If the tokenizer requires SentencePiece models, ensure sentencepiece is installed and the tokenizer was built with the correct vocab files.")
        print(f"   - Alternatively add the language code as a token: tokenizer.add_tokens(['{lang_code}']) and then call model.resize_token_embeddings(len(tokenizer)) after model is loaded.")
        return None, mapping_source, vocab_size

    try:
        if vocab_size is not None and (lang_id >= vocab_size):
            print(f"\n[Tokenizer Validation] {role} language code '{lang_code}' has token id {lang_id} which is >= tokenizer vocab size ({vocab_size}).")
            print("  Suggested actions:")
            print("   - The tokenizer appears to reference a token id beyond the tokenizer vocab. You may need to resize embeddings after adding tokens.")
            print("   - If you added tokens, call tokenizer.add_tokens([...]) then call model.resize_token_embeddings(len(tokenizer)) after loading the model.")
            return lang_id, mapping_source, vocab_size
    except Exception:
        pass

    print(f"\n[Tokenizer Validation] {role} language code '{lang_code}' resolved to token id: {lang_id} (mapping: {mapping_source})")
    return lang_id, mapping_source, vocab_size

try:
    tokenizer_indic = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True
    )

    try:
        tokenizer_indic.src_lang = SOURCE_LANGUAGE
    except Exception:
        pass
    try:
        tokenizer_indic.tgt_lang = TARGET_LANGUAGE
    except Exception:
        pass

    src_id, src_map, src_vocab_size = validate_language_token(tokenizer_indic, SOURCE_LANGUAGE, "SOURCE")
    tgt_id, tgt_map, tgt_vocab_size = validate_language_token(tokenizer_indic, TARGET_LANGUAGE, "TARGET")

    try:
        vocab_size = len(tokenizer_indic)
    except Exception:
        vocab = _safe_get_vocab(tokenizer_indic)
        vocab_size = len(vocab) if vocab is not None else None

    print(f"  ✓ Tokenizer loaded: {MODEL_NAME}")
    print(f"  ✓ Vocabulary size: {vocab_size}")

    if src_id is not None:
        print(f"  ✓ src_lang property: {getattr(tokenizer_indic, 'src_lang', SOURCE_LANGUAGE)} (token id: {src_id})")
    else:
        print(f"  ⚠ src_lang property set to: {getattr(tokenizer_indic, 'src_lang', SOURCE_LANGUAGE)} (token NOT found in tokenizer)")

    if tgt_id is not None:
        print(f"  ✓ tgt_lang property: {getattr(tokenizer_indic, 'tgt_lang', TARGET_LANGUAGE)} (token id: {tgt_id})")
    else:
        print(f"  ⚠ tgt_lang property set to: {getattr(tokenizer_indic, 'tgt_lang', TARGET_LANGUAGE)} (token NOT found in tokenizer)")

    if src_id is None or tgt_id is None:
        guidance_msgs = []
        guidance_msgs.append("\n[TOKENIZER LANG-CODE VALIDATION FAILURE] One or both language tokens could not be resolved.")
        guidance_msgs.append("Actions you can take (pick one or more as appropriate):")
        guidance_msgs.append("  1) Use a tokenizer/model that includes language codes (e.g., mBART-style models) or ensure sentencepiece is available and the tokenizer was built with the correct sentencepiece model.")
        guidance_msgs.append(f"  2) Add the language tokens to the tokenizer: tokenizer.add_tokens(['{SOURCE_LANGUAGE}', '{TARGET_LANGUAGE}']) and then after loading the model call model.resize_token_embeddings(len(tokenizer)).")
        guidance_msgs.append("  3) If you already added tokens but ids are out-of-range, ensure you call model.resize_token_embeddings(len(tokenizer)) immediately after model creation.")
        guidance_msgs.append("  4) If you are not using explicit language token ids, set tokenizer.src_lang/tgt_lang to the appropriate strings and ensure downstream code inserts language tokens correctly.")
        guidance_msgs.append("  5) Verify the Transformer version and SentencePiece installation.")
        print("\n".join(guidance_msgs))
        raise RuntimeError("Tokenizer language-code validation failed. See messages above for corrective actions.")

    print(f"  ✓ Languages configured: {SOURCE_LANGUAGE} → {TARGET_LANGUAGE}")
    print("[Cell 0] ✅ Tokenizer ready for pipeline\n")

except Exception as e:
    print(f"\n❌ TOKENIZER LOADING FAILED: {e}")
    print("\nTROUBLESHOOTING:")
    print("1. Verify environment authentication and permissions.")
    print("2. Check internet connection or cached model availability.")
    print(f"3. Model name: {MODEL_NAME}")
    print("4. Ensure 'sentencepiece' package is installed if the tokenizer uses SentencePiece.")
    raise RuntimeError(f"Failed to load or validate tokenizer: {e}")


[Cell 0] Single GPU Mode (DataParallel DISABLED)
[Cell 0] Device: cuda

DUAL-PATH TATN CONFIGURATION - INDICTRANS2 VERSION
User: manas0003
Multi-GPU: DISABLED (1 GPUs)
Dataset: /kaggle/input/datasets/manaskumarmanna/bpcc-data/bpcc_bn_en_400k.csv
Samples: 60,000 | Batch: 2 | Accum: 4
Effective batch: 8
Max length: 128 | Epochs: 2

✅ OOM PROTECTION:
  - Enabled: True
  - Cleanup frequency: 50 steps
  - Memory threshold: 90%
  - Gradient checkpointing: True

✅ VALIDATION:
  - Enabled: True
  - Frequency: 200 steps
  - Samples: 50

✅ DEBUG FLAGS:
  - Path1: True
  - Path2: True
  - Losses: True
  - Gradients: True
  - Memory: True
  - Clusters: True
  - Frequency: 100 steps

✅ PATH 1 (TATN - Homograph Detection):
  - Word-level tokenization
  - Start step: 100
  - DSCD discovery freq: 100 steps
  - DSCD warmup: 500 samples
  - DSCD buffer size: 500
  - DSCD max protos: 10
  - DSCD N_min: 2
  - DSCD dispersion: 0.15
  - DSCD embed_dim: 512
  - ASBN: ENABLED
  - ASBN hidden_dim: 512
  - TRG:

In [4]:
# ===========================================================================================
# CELL 1: DUAL-PATH TOKENIZER UTILITIES + TRAINING LOSSES (INDICTRANS2) - FIXED
# ===========================================================================================

import threading
from typing import Tuple, List, Dict, Optional, Set, Union
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import re

try:
    from IndicTransToolkit.processor import IndicProcessor
    _HAS_INDIC_PROCESSOR = True
except Exception:
    IndicProcessor = None
    _HAS_INDIC_PROCESSOR = False
    print("[WARN] IndicTransToolkit not found. Install with: pip install IndicTransToolkit")

# Defensive defaults for globals that are defined in Cell 0
try:
    if isinstance(MAX_LENGTH, (int, float)) and MAX_LENGTH > 0:
        SAFE_OFFSET_MAX_LEN = max(8, int(MAX_LENGTH))
    else:
        SAFE_OFFSET_MAX_LEN = 48
except Exception:
    SAFE_OFFSET_MAX_LEN = 48

try:
    _SOURCE_LANG = SOURCE_LANGUAGE
except Exception:
    _SOURCE_LANG = "ben_Beng"

try:
    _TARGET_LANG = TARGET_LANGUAGE
except Exception:
    _TARGET_LANG = "eng_Latn"

try:
    _DEBUG_VERBOSE = bool(DEBUG_VERBOSE)
except Exception:
    _DEBUG_VERBOSE = False

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except Exception:
    _DEBUG_DISCOVERY = False

try:
    _DSCD_MIN_LETTERS = int(DSCD_MIN_LETTERS)
except Exception:
    _DSCD_MIN_LETTERS = 3

try:
    _DSCD_MIN_LETTER_FRACTION = float(DSCD_MIN_LETTER_FRACTION)
except Exception:
    _DSCD_MIN_LETTER_FRACTION = 0.6

try:
    _LABEL_SMOOTHING = float(LABEL_SMOOTHING)
except Exception:
    _LABEL_SMOOTHING = 0.1

try:
    _RDROP_ALPHA = float(RDROP_ALPHA)
except Exception:
    _RDROP_ALPHA = 5.0

_SPECIAL_TOKENS_CACHE: Dict[str, Set[str]] = {}
_SPECIAL_TOKENS_LOCK = threading.Lock()
_LANGUAGE_WARNING_COUNT = 0
_MAX_LANGUAGE_WARNINGS = 3
_VOCAB_SIZE_CACHE: Dict[str, int] = {}

# Bengali word-level tokenizer used for Path 1 (word-level)
class BengaliWordTokenizer:
    def __init__(self, vocab_size: int = 50000):
        self.vocab_size = int(vocab_size) if isinstance(vocab_size, int) else 50000
        self.word_to_id: Dict[str, int] = {"<pad>": 0, "<unk>": 1, "<s>": 2, "</s>": 3}
        self.id_to_word: Dict[int, str] = {0: "<pad>", 1: "<unk>", 2: "<s>", 3: "</s>"}
        self.next_id = 4
        self._lock = threading.Lock()

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.bos_token = "<s>"
        self.eos_token = "</s>"
        self.pad_token_id = 0
        self.unk_token_id = 1
        self.bos_token_id = 2
        self.eos_token_id = 3

        self.bengali_pattern = re.compile(r'[\u0980-\u09FF]+')
        self.punct_pattern = re.compile(r'[।॥,.;:!?"\'\-\(\)\[\]{}]')

    def tokenize(self, text: str) -> List[str]:
        if not text or not isinstance(text, str):
            return []
        text = text.strip()
        if not text:
            return []
        tokens = re.findall(r'[\u0980-\u09FF]+|[a-zA-Z]+|[0-9]+|[।॥]|[,.;:!?"\'\-\(\)\[\]{}]', text)
        words = [t.strip() for t in tokens if t and t.strip()]
        return words

    def encode(
        self,
        text: Union[str, List[str]],
        add_special_tokens: bool = True,
        max_length: Optional[int] = None,
        padding: bool = False,
        truncation: bool = False,
        return_tensors: Optional[str] = None,
    ) -> Dict[str, Union[List[int], torch.Tensor]]:
        if isinstance(text, str):
            texts = [text]
        else:
            texts = list(text)

        all_input_ids = []
        all_attention_masks = []

        for txt in texts:
            words = self.tokenize(txt)
            with self._lock:
                ids = []
                for word in words:
                    if word not in self.word_to_id:
                        if self.next_id < self.vocab_size:
                            self.word_to_id[word] = self.next_id
                            self.id_to_word[self.next_id] = word
                            self.next_id += 1
                            ids.append(self.word_to_id[word])
                        else:
                            ids.append(self.unk_token_id)
                    else:
                        ids.append(self.word_to_id[word])

            if add_special_tokens:
                ids = [self.bos_token_id] + ids + [self.eos_token_id]

            if truncation and max_length:
                ids = ids[:max_length]

            attention_mask = [1] * len(ids)
            all_input_ids.append(ids)
            all_attention_masks.append(attention_mask)

        if padding and max_length:
            for i in range(len(all_input_ids)):
                if len(all_input_ids[i]) < max_length:
                    pad_len = max_length - len(all_input_ids[i])
                    all_input_ids[i] = all_input_ids[i] + [self.pad_token_id] * pad_len
                    all_attention_masks[i] = all_attention_masks[i] + [0] * pad_len

        if return_tensors == "pt":
            max_len = max(len(ids) for ids in all_input_ids) if all_input_ids else 0
            for i in range(len(all_input_ids)):
                if len(all_input_ids[i]) < max_len:
                    pad_len = max_len - len(all_input_ids[i])
                    all_input_ids[i] = all_input_ids[i] + [self.pad_token_id] * pad_len
                    all_attention_masks[i] = all_attention_masks[i] + [0] * pad_len

            return {
                "input_ids": torch.tensor(all_input_ids, dtype=torch.long),
                "attention_mask": torch.tensor(all_attention_masks, dtype=torch.long),
            }

        if len(all_input_ids) == 1:
            return {
                "input_ids": all_input_ids[0],
                "attention_mask": all_attention_masks[0],
            }

        return {
            "input_ids": all_input_ids,
            "attention_mask": all_attention_masks,
        }

    def decode(self, token_ids: Union[List[int], torch.Tensor], skip_special_tokens: bool = True) -> str:
        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.tolist()

        words = []
        for tid in token_ids:
            if tid in self.id_to_word:
                word = self.id_to_word[tid]
                if skip_special_tokens and word in {"<pad>", "<unk>", "<s>", "</s>"}:
                    continue
                words.append(word)
        return " ".join(words)

    def convert_ids_to_tokens(self, ids: Union[List[int], torch.Tensor]) -> List[str]:
        if isinstance(ids, torch.Tensor):
            ids = ids.tolist()
        return [self.id_to_word.get(tid, self.unk_token) for tid in ids]

    def convert_tokens_to_ids(self, tokens: Union[str, List[str]]) -> Union[int, List[int]]:
        if isinstance(tokens, str):
            return self.word_to_id.get(tokens, self.unk_token_id)
        return [self.word_to_id.get(tok, self.unk_token_id) for tok in tokens]

    def __call__(self, text: Union[str, List[str]], **kwargs):
        return self.encode(text, **kwargs)

    def __len__(self):
        return len(self.word_to_id)


def initialize_indic_processor(inference: bool = True):
    if not _HAS_INDIC_PROCESSOR:
        return None
    try:
        ip = IndicProcessor(inference=inference)
        return ip
    except Exception as e:
        if _DEBUG_VERBOSE:
            print(f"[WARN] Failed to initialize IndicProcessor: {e}")
        return None


def preprocess_for_indictrans2(
    sentences: Union[str, List[str]],
    src_lang: str,
    tgt_lang: str,
    indic_processor=None
) -> List[str]:
    if isinstance(sentences, str):
        sentences = [sentences]
    processed = []
    for sentence in sentences:
        sentence_clean = (sentence or "").strip()
        if not sentence_clean:
            processed.append(f"{src_lang} {tgt_lang} ")
            continue
        tagged_sentence = f"{src_lang} {tgt_lang} {sentence_clean}"
        processed.append(tagged_sentence)
    return processed


def preprocess_single_word_for_indictrans2(
    word: str,
    src_lang: str,
    tgt_lang: str,
    indic_processor=None
) -> str:
    if not word or not isinstance(word, str):
        return f"{src_lang} {tgt_lang} "
    word_clean = word.strip()
    if not word_clean:
        return f"{src_lang} {tgt_lang} "
    tagged_word = f"{src_lang} {tgt_lang} {word_clean}"
    return tagged_word


def postprocess_for_indictrans2(
    translations: Union[str, List[str]],
    tgt_lang: str,
    indic_processor=None
) -> Union[str, List[str]]:
    was_string = isinstance(translations, str)
    if was_string:
        translations = [translations]
    cleaned = []
    for trans in translations:
        trans_clean = (trans or "").strip()
        if trans_clean.startswith(tgt_lang):
            trans_clean = trans_clean[len(tgt_lang):].strip()
        for lang_code in ["ben_Beng", "eng_Latn", "hin_Deva"]:
            if trans_clean.startswith(lang_code):
                trans_clean = trans_clean[len(lang_code):].strip()
                break
        trans_clean = re.sub(r'\s+([.,!?;:])', r'\1', trans_clean)
        trans_clean = re.sub(r'\s+', ' ', trans_clean)
        trans_clean = trans_clean.strip()
        cleaned.append(trans_clean)
    return cleaned[0] if was_string else cleaned


def create_decoder_labels(
    input_ids: torch.Tensor,
    pad_token_id: int,
    ignore_index: int = -100
) -> torch.Tensor:
    labels = input_ids.clone()
    labels[labels == pad_token_id] = ignore_index
    return labels


class LabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes: int, smoothing: float = 0.1, ignore_index: int = -100):
        super().__init__()
        self.num_classes = int(num_classes)
        self.smoothing = float(smoothing)
        self.ignore_index = int(ignore_index)
        self.confidence = 1.0 - self.smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if logits.dim() == 3:
            logits = logits.reshape(-1, logits.size(-1))
        if targets.dim() == 2:
            targets = targets.reshape(-1)

        mask = targets != self.ignore_index
        targets = targets.masked_select(mask)
        logits = logits[mask]

        if targets.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        log_probs = F.log_softmax(logits, dim=-1)
        nll_loss = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_probs.mean(dim=-1)
        loss = self.confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()


class RDropLoss(nn.Module):
    def __init__(self, alpha: float = 5.0):
        super().__init__()
        self.alpha = float(alpha)

    def forward(
        self,
        logits1: torch.Tensor,
        logits2: torch.Tensor,
        targets: torch.Tensor,
        ignore_index: int = -100
    ) -> torch.Tensor:
        if logits1.dim() == 3:
            logits1 = logits1.reshape(-1, logits1.size(-1))
        if logits2.dim() == 3:
            logits2 = logits2.reshape(-1, logits2.size(-1))
        if targets.dim() == 2:
            targets = targets.reshape(-1)

        mask = targets != ignore_index
        logits1 = logits1[mask]
        logits2 = logits2[mask]

        if logits1.numel() == 0:
            return torch.tensor(0.0, device=logits1.device, requires_grad=True)

        p1 = F.log_softmax(logits1, dim=-1)
        p2 = F.log_softmax(logits2, dim=-1)

        p1_probs = F.softmax(logits1, dim=-1)
        p2_probs = F.softmax(logits2, dim=-1)

        kl_1_2 = F.kl_div(p1, p2_probs, reduction='batchmean', log_target=False)
        kl_2_1 = F.kl_div(p2, p1_probs, reduction='batchmean', log_target=False)

        kl_loss = (kl_1_2 + kl_2_1) / 2.0
        return self.alpha * kl_loss


def _special_token_cache_key(tokenizer) -> str:
    name = getattr(tokenizer, "name_or_path", None) or getattr(tokenizer, "name", None)
    if not name:
        name = "unknown_tokenizer"
    vocab = None
    try:
        if hasattr(tokenizer, "vocab_size"):
            vocab = int(getattr(tokenizer, "vocab_size"))
        elif hasattr(tokenizer, "__len__"):
            vocab = len(tokenizer)
        elif hasattr(tokenizer, "get_vocab"):
            vocab = len(tokenizer.get_vocab())
    except Exception:
        vocab = None
    return f"{name}__vocab={vocab}"


def get_tokenizer_vocab_size(tokenizer) -> int:
    cache_key = _special_token_cache_key(tokenizer)
    if cache_key in _VOCAB_SIZE_CACHE:
        return _VOCAB_SIZE_CACHE[cache_key]

    vocab_size = 30000
    try:
        if tokenizer is None:
            vocab_size = 30000
        elif hasattr(tokenizer, "__len__"):
            vocab_size = len(tokenizer)
        elif hasattr(tokenizer, "vocab_size"):
            vocab_size = int(getattr(tokenizer, "vocab_size"))
        elif hasattr(tokenizer, "get_vocab"):
            vocab = tokenizer.get_vocab()
            if isinstance(vocab, dict):
                vocab_size = len(vocab)
    except Exception:
        vocab_size = 30000

    _VOCAB_SIZE_CACHE[cache_key] = vocab_size
    return vocab_size


def get_tokenizer_special_tokens(tokenizer) -> Set[str]:
    cache_key = _special_token_cache_key(tokenizer)
    with _SPECIAL_TOKENS_LOCK:
        if cache_key in _SPECIAL_TOKENS_CACHE:
            return set(_SPECIAL_TOKENS_CACHE[cache_key])

        special_tokens: Set[str] = set()
        try:
            if tokenizer is None:
                special_tokens = {"</s>", "<pad>", "<s>", "<unk>", _SOURCE_LANG, _TARGET_LANG}
            else:
                if hasattr(tokenizer, "all_special_tokens"):
                    try:
                        result = getattr(tokenizer, "all_special_tokens")
                        if isinstance(result, (list, tuple, set)):
                            special_tokens.update(x for x in result if x)
                    except Exception:
                        pass
                if hasattr(tokenizer, "additional_special_tokens"):
                    try:
                        result = getattr(tokenizer, "additional_special_tokens")
                        if isinstance(result, (list, tuple, set)):
                            special_tokens.update(x for x in result if x)
                    except Exception:
                        pass
                for attr in ("pad_token", "unk_token", "bos_token", "eos_token",
                             "cls_token", "sep_token", "mask_token"):
                    if hasattr(tokenizer, attr):
                        try:
                            tok = getattr(tokenizer, attr)
                            if tok:
                                special_tokens.add(tok)
                        except Exception:
                            pass
                try:
                    stm = getattr(tokenizer, "special_tokens_map", None) or getattr(tokenizer, "special_tokens_map_extended", None)
                    if isinstance(stm, dict):
                        for v in stm.values():
                            if isinstance(v, str) and v:
                                special_tokens.add(v)
                except Exception:
                    pass

                # Always include language tokens and common markers
                special_tokens.update({
                    _SOURCE_LANG, _TARGET_LANG,
                    "</s>", "<pad>", "<s>", "<unk>",
                    "[PAD]", "[EOS]", "[UNK]", "[CLS]", "[SEP]", "[MASK]",
                })

                # Try to filter tokens to those present in vocab when possible
                try:
                    if hasattr(tokenizer, "get_vocab"):
                        vocab = tokenizer.get_vocab()
                        if isinstance(vocab, dict):
                            special_tokens = {tok for tok in special_tokens if tok in vocab or tok in {"</s>", "<pad>", "<s>", "<unk>"}}
                except Exception:
                    pass
        except Exception:
            special_tokens = {"</s>", "<pad>", "<s>", "<unk>", _SOURCE_LANG, _TARGET_LANG}

        _SPECIAL_TOKENS_CACHE[cache_key] = set(special_tokens)
        return set(special_tokens)


def get_cached_special_tokens(tokenizer) -> Set[str]:
    return get_tokenizer_special_tokens(tokenizer)


def _normalize_offset_mapping_for_batchencoding(enc):
    try:
        if isinstance(enc, dict) and "offset_mapping" in enc and enc["offset_mapping"] is not None:
            off = enc["offset_mapping"]
            try:
                if hasattr(off, "tolist"):
                    arr = off.tolist()
                    if isinstance(arr, list) and len(arr) > 0 and isinstance(arr[0], list):
                        enc["offset_mapping"] = [(x[0], x[1]) if (isinstance(x, (list, tuple)) and len(x) >= 2) else (None, None) for x in arr[0]]
                        return enc
                if isinstance(off, (list, tuple)):
                    if len(off) > 0 and isinstance(off[0], (list, tuple)):
                        enc["offset_mapping"] = [(x[0], x[1]) if (isinstance(x, (list, tuple)) and len(x) >= 2) else (None, None) for x in off[0]]
                        return enc
            except Exception:
                pass
    except Exception:
        pass

    try:
        data = getattr(enc, "data", None)
        if data and isinstance(data, dict) and "offset_mapping" in data and data["offset_mapping"] is not None:
            om = data["offset_mapping"]
            if isinstance(om, (list, tuple)) and len(om) > 0 and isinstance(om[0], (list, tuple)):
                enc["offset_mapping"] = [(x[0], x[1]) if (isinstance(x, (list, tuple)) and len(x) >= 2) else (None, None) for x in om[0]]
                return enc
    except Exception:
        pass

    try:
        seq_len = 0
        if isinstance(enc, dict) and "input_ids" in enc:
            input_ids = enc["input_ids"]
            if hasattr(input_ids, "shape") and len(input_ids.shape) > 0:
                seq_len = int(input_ids.shape[-1])
            elif isinstance(input_ids, (list, tuple)) and len(input_ids) > 0 and isinstance(input_ids[0], (list, tuple)):
                seq_len = len(input_ids[0])
        enc["offset_mapping"] = [(None, None)] * seq_len
    except Exception:
        enc["offset_mapping"] = []
    return enc


def safe_offsets_tokenize(
    tokenizer,
    text: str,
    max_length: Optional[int] = None,
    include_special_tokens: bool = False,
    preprocessed_text: Optional[str] = None,
) -> dict:
    if max_length is None:
        max_length = SAFE_OFFSET_MAX_LEN
    eff_max = int(max_length)

    try:
        if not isinstance(text, str):
            text = "" if text is None else str(text)
    except Exception:
        if _DEBUG_VERBOSE:
            print("[WARN] Failed to convert input to string, using empty string")
        text = ""

    text_to_tokenize = preprocessed_text if preprocessed_text is not None else text
    if _DEBUG_DISCOVERY and preprocessed_text is not None:
        print(f"[CELL1-TOKENIZE] Using preprocessed text: {text_to_tokenize[:50]}")

    char_limit = min(eff_max * 30, 8000)
    sample_text = text_to_tokenize[:char_limit]

    is_fast = bool(getattr(tokenizer, "is_fast", False))
    vocab_size = get_tokenizer_vocab_size(tokenizer) or 30000

    tokenize_kwargs = {
        "return_tensors": "pt",
        "truncation": True,
        "padding": False,
        "max_length": eff_max,
        "add_special_tokens": include_special_tokens,
    }

    if is_fast:
        try:
            tokenize_kwargs["return_offsets_mapping"] = True
            enc = tokenizer(sample_text, **tokenize_kwargs)
            enc = _normalize_offset_mapping_for_batchencoding(enc)
            if "input_ids" in enc and isinstance(enc["input_ids"], torch.Tensor) and vocab_size and vocab_size > 0:
                enc["input_ids"] = torch.clamp(enc["input_ids"], 0, max(0, vocab_size - 1))
            return enc
        except Exception as e:
            if _DEBUG_VERBOSE:
                print(f"[CELL1-TOKENIZE] Fast tokenization failed: {e}")

    try:
        enc = tokenizer(sample_text, **tokenize_kwargs)
        if "input_ids" in enc and isinstance(enc["input_ids"], torch.Tensor) and vocab_size and vocab_size > 0:
            enc["input_ids"] = torch.clamp(enc["input_ids"], 0, max(0, vocab_size - 1))
    except Exception as e:
        if _DEBUG_VERBOSE:
            print(f"[WARN] Tokenization failed: {e}, returning empty encoding")
        pad_id = getattr(tokenizer, "pad_token_id", 0) if tokenizer is not None else 0
        enc = {
            "input_ids": torch.tensor([[pad_id]], dtype=torch.long),
            "attention_mask": torch.tensor([[1]], dtype=torch.long),
        }
        enc = _normalize_offset_mapping_for_batchencoding(enc)
        return enc

    try:
        input_ids = None
        try:
            input_ids = enc["input_ids"][0].tolist()
        except Exception:
            if hasattr(enc, "data") and "input_ids" in enc.data:
                input_ids = enc.data["input_ids"][0]

        tokens: List[str] = []
        if input_ids is not None:
            try:
                tokens = tokenizer.convert_ids_to_tokens(input_ids)
            except Exception:
                tokens = []

        offsets_list: List[Tuple[Optional[int], Optional[int]]] = []
        src = text
        cur_pos = 0
        for tok in tokens:
            token_text = (tok or "").replace("▁", "").replace("##", "").replace("Ġ", "").strip()
            if not token_text:
                offsets_list.append((None, None))
                continue
            idx = src.find(token_text, cur_pos)
            if idx == -1:
                idx = src.lower().find(token_text.lower(), cur_pos)
            if idx == -1:
                offsets_list.append((None, None))
            else:
                start = int(idx)
                end = int(idx + len(token_text))
                offsets_list.append((start, end))
                cur_pos = end

        enc["offset_mapping"] = offsets_list
        enc = _normalize_offset_mapping_for_batchencoding(enc)
        return enc
    except Exception:
        enc = _normalize_offset_mapping_for_batchencoding(enc)
        return enc


def reconstruct_word_spans(
    tokenizer,
    text: str,
    max_length: Optional[int] = None,
    preprocessed_text: Optional[str] = None,
) -> Tuple[Dict[int, str], List[str]]:
    global _LANGUAGE_WARNING_COUNT

    if max_length is None:
        max_length = SAFE_OFFSET_MAX_LEN
    eff_max = int(max_length)

    if not isinstance(text, str) or len(text.strip()) == 0:
        return {}, []

    has_bengali = any("\u0980" <= c <= "\u09FF" for c in text)
    has_english = any("a" <= c.lower() <= "z" for c in text)

    if _DEBUG_VERBOSE and _DEBUG_DISCOVERY:
        bengali_pct = (sum(1 for c in text if "\u0980" <= c <= "\u09FF") / max(1, len(text)) * 100.0)
        print(f"[TOKENIZER] Text sample: {text[:50]}")
        print(f"[TOKENIZER] Bengali: {has_bengali} ({bengali_pct:.1f}%), English: {has_english}")

    if not has_bengali and has_english and _LANGUAGE_WARNING_COUNT < _MAX_LANGUAGE_WARNINGS:
        if _DEBUG_DISCOVERY:
            print("[TOKENIZER WARNING] Text appears to be ENGLISH, not BENGALI")
            print(f"  Sample: {text[:80]}")
        _LANGUAGE_WARNING_COUNT += 1
        if _LANGUAGE_WARNING_COUNT == _MAX_LANGUAGE_WARNINGS:
            print("[TOKENIZER] Suppressing further language warnings")

    char_limit = min(eff_max * 30, 8000)
    text = text[:char_limit]
    text_len = len(text)

    special_tokens = get_tokenizer_special_tokens(tokenizer)
    vocab_size = get_tokenizer_vocab_size(tokenizer) or 30000

    try:
        current_lang = _SOURCE_LANG
    except Exception:
        current_lang = _SOURCE_LANG

    try:
        encoded = safe_offsets_tokenize(tokenizer, text, max_length=eff_max, include_special_tokens=False, preprocessed_text=preprocessed_text)
    except Exception:
        return {}, []

    offsets = encoded.get("offset_mapping", [])
    try:
        input_ids = encoded["input_ids"][0].tolist()
        input_ids = [min(max(0, int(tid)), max(0, vocab_size - 1)) for tid in input_ids]
    except Exception:
        input_ids = []
    try:
        tokens = []
        if input_ids:
            try:
                tokens = tokenizer.convert_ids_to_tokens(input_ids)
            except Exception:
                tokens = []
    except Exception:
        tokens = []

    if isinstance(offsets, list) and len(offsets) > 0 and all(isinstance(x, tuple) for x in offsets):
        offsets_list = offsets
    elif isinstance(offsets, list) and len(offsets) > 0 and isinstance(offsets[0], (list, tuple)):
        offsets_list = [(x[0], x[1]) if (isinstance(x, (list, tuple)) and len(x) >= 2) else (None, None) for x in offsets[0]]
    else:
        offsets_list = [(None, None)] * len(tokens)

    token_word_map: Dict[int, str] = {}
    words: List[str] = []

    used_any_offset = any(isinstance(o, tuple) and o[0] is not None and o[1] is not None for o in offsets_list)
    if used_any_offset:
        word_start = None
        word_end = None
        current_accumulated_word = ""

        for idx, (off, tok) in enumerate(zip(offsets_list, tokens)):
            try:
                off_start = int(off[0]) if off[0] is not None else None
                off_end = int(off[1]) if off[1] is not None else None
            except Exception:
                off_start, off_end = None, None

            if off_start is not None and off_end is not None:
                if off_start < 0 or off_end < 0:
                    if _DEBUG_VERBOSE:
                        print(f"[WARN] Negative offset detected: ({off_start}, {off_end}), skipping")
                    off_start, off_end = None, None
                else:
                    off_start = max(0, min(off_start, text_len))
                    off_end = max(off_start, min(off_end, text_len))

            if off_start is None or off_end is None:
                if current_accumulated_word:
                    token_word_map[idx] = current_accumulated_word
                if word_start is not None and word_end is not None:
                    try:
                        wtext = text[word_start:word_end].strip()
                        if wtext:
                            words.append(wtext)
                    except Exception:
                        pass
                word_start = None
                word_end = None
                continue

            if tok in special_tokens:
                continue

            if word_start is None:
                word_start = off_start
                word_end = off_end
            else:
                if off_start > word_end:
                    try:
                        wtext = text[word_start:word_end].strip()
                        if wtext:
                            words.append(wtext)
                    except Exception:
                        pass
                    word_start = off_start
                    word_end = off_end
                else:
                    word_end = max(word_end, off_end)

            try:
                current_word = text[word_start:word_end].strip()
                if current_word:
                    token_word_map[idx] = current_word
                    current_accumulated_word = current_word
            except Exception:
                pass

        if word_start is not None and word_end is not None:
            try:
                wtext = text[word_start:word_end].strip()
                if wtext:
                    words.append(wtext)
            except Exception:
                pass

        if token_word_map:
            words = [w for w in words if isinstance(w, str) and w.strip()]
            return token_word_map, words

    token_word_map = {}
    assembled: List[str] = []
    current_parts: List[str] = []
    running_word = ""
    max_word_len = 100

    for i, tok in enumerate(tokens):
        if tok in special_tokens:
            continue

        clean = (tok or "").replace("▁", "").replace("Ġ", "").replace("##", "").strip()
        if not clean:
            continue

        if tok.startswith("▁") or tok.startswith("Ġ"):
            if current_parts:
                word = "".join(current_parts)
                if len(word) <= max_word_len:
                    assembled.append(word)
            current_parts = [clean]
            running_word = clean
        else:
            current_parts.append(clean)
            running_word = "".join(current_parts)
            if len(running_word) > max_word_len:
                if current_parts[:-1]:
                    word = "".join(current_parts[:-1])
                    assembled.append(word)
                current_parts = [clean]
                running_word = clean

        if running_word:
            token_word_map[i] = running_word

    if current_parts:
        word = "".join(current_parts)
        if len(word) <= max_word_len:
            assembled.append(word)

    if token_word_map:
        words = [w for w in assembled if w and w.strip()]
        return token_word_map, words

    try:
        words_from_markers: List[str] = []
        current_word_parts: List[str] = []

        for tok in tokens:
            if tok in special_tokens:
                continue
            clean = (tok or "").replace("▁", "").replace("Ġ", "").replace("##", "").strip()
            if not clean:
                continue
            if tok.startswith("▁") or tok.startswith("Ġ"):
                if current_word_parts:
                    words_from_markers.append("".join(current_word_parts))
                current_word_parts = [clean]
            else:
                current_word_parts.append(clean)

        if current_word_parts:
            words_from_markers.append("".join(current_word_parts))

        if words_from_markers:
            word_list = words_from_markers
        else:
            word_list = [w for w in text.split() if w.strip()]

        token_word_map = {}
        if tokens and word_list:
            word_idx = 0
            for i, tok in enumerate(tokens):
                clean = (tok or "").replace("▁", "").replace("Ġ", "").replace("##", "").strip()
                if not clean or tok in special_tokens:
                    continue
                if word_idx < len(word_list):
                    current_word = word_list[word_idx]
                    if clean in current_word or current_word.startswith(clean):
                        token_word_map[i] = current_word
                    else:
                        word_idx = min(word_idx + 1, len(word_list) - 1)
                        token_word_map[i] = word_list[word_idx]
                else:
                    if word_list:
                        token_word_map[i] = word_list[-1]

        return token_word_map, word_list
    except Exception:
        return {}, []


_token_validation_cache: Dict[Tuple[str, str], bool] = {}
_cache_lock = threading.Lock()
_cache_max_size = 10000

def is_valid_token(
    token,
    special_tokens: Optional[Set[str]] = None,
    tokenizer=None,
    language: str = "bn",
) -> bool:
    token = "" if token is None else str(token)
    cache_key = (token, language)
    with _cache_lock:
        if cache_key in _token_validation_cache:
            return _token_validation_cache[cache_key]

    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()

    if special_tokens and token in special_tokens:
        result = False
    else:
        if len(clean) < _DSCD_MIN_LETTERS:
            result = False
        else:
            has_bengali_chars = any("\u0980" <= c <= "\u09FF" for c in clean)
            if not has_bengali_chars:
                result = False
            else:
                bengali_count = sum(1 for c in clean if "\u0980" <= c <= "\u09FF")
                alphanum_count = sum(1 for c in clean if c.isalnum())
                if alphanum_count == 0:
                    result = False
                else:
                    result = (bengali_count / alphanum_count) >= _DSCD_MIN_LETTER_FRACTION

    with _cache_lock:
        if len(_token_validation_cache) < _cache_max_size:
            _token_validation_cache[cache_key] = result
    return result


def should_track_token(
    token: str,
    special_tokens: Optional[Set[str]] = None,
    tokenizer=None,
    language: str = "bn",
) -> bool:
    return is_valid_token(token, special_tokens, tokenizer, language)


def validate_tokenizer_vocab(tokenizer, expected_vocab_size: Optional[int] = None) -> bool:
    actual_vocab_size = get_tokenizer_vocab_size(tokenizer)
    print(f"[TOKENIZER-VALIDATION] Actual vocab size: {actual_vocab_size}")

    if expected_vocab_size is not None:
        if actual_vocab_size != expected_vocab_size:
            print(f"[TOKENIZER-VALIDATION] ❌ MISMATCH: Expected {expected_vocab_size}, got {actual_vocab_size}")
            return False
        else:
            print(f"[TOKENIZER-VALIDATION] ✅ Vocab size matches: {actual_vocab_size}")
            return True

    src_token_str = _SOURCE_LANG
    tgt_token_str = _TARGET_LANG

    try:
        src_id = None
        tgt_id = None
        if tokenizer is not None:
            try:
                src_id = tokenizer.convert_tokens_to_ids(src_token_str)
            except Exception:
                try:
                    src_id = tokenizer.convert_tokens_to_ids([src_token_str])[0]
                except Exception:
                    src_id = None
            try:
                tgt_id = tokenizer.convert_tokens_to_ids(tgt_token_str)
            except Exception:
                try:
                    tgt_id = tokenizer.convert_tokens_to_ids([tgt_token_str])[0]
                except Exception:
                    tgt_id = None

        print(f"[TOKENIZER-VALIDATION] Language codes:")
        print(f"  {src_token_str} → {src_id}")
        print(f"  {tgt_token_str} → {tgt_id}")

        if src_id is None or tgt_id is None:
            print(f"[TOKENIZER-VALIDATION] ❌ One or both language tokens are missing from tokenizer.")
            return False

        if actual_vocab_size and (src_id >= actual_vocab_size or tgt_id >= actual_vocab_size):
            print(f"[TOKENIZER-VALIDATION] ❌ Language token IDs exceed vocab size!")
            return False

        print(f"[TOKENIZER-VALIDATION] ✅ Language codes valid")
        return True

    except Exception as e:
        print(f"[TOKENIZER-VALIDATION] ❌ Language code validation failed: {e}")
        return False


def test_tokenizer_utilities_quick(tokenizer=None) -> bool:
    sample_bn = "কাল আমি বাজারে যাব।"
    sample_en = "Tomorrow I will go to the market."

    print("\n" + "=" * 60)
    print("TOKENIZER UTILITIES TEST (INDICTRANS2)")
    print("=" * 60)

    try:
        if tokenizer is None:
            print("No tokenizer provided: skipping test")
            return True

        print("\n[TEST 0] Vocabulary validation:")
        validate_tokenizer_vocab(tokenizer)

        print("\n[TEST 1] Bengali text processing:")
        print(f"  Input: {sample_bn}")
        enc_bn = safe_offsets_tokenize(tokenizer, sample_bn, max_length=32, include_special_tokens=False)
        enc_len = int(enc_bn["input_ids"].shape[-1]) if isinstance(enc_bn, dict) and "input_ids" in enc_bn else "N/A"
        print(f"  Encoded length: {enc_len}")
        offsets_bn = enc_bn.get("offset_mapping") or []
        print(f"  Offsets (first 5): {offsets_bn[:5]}")

        token_map_bn, words_bn = reconstruct_word_spans(tokenizer, sample_bn, max_length=32)
        print(f"  Reconstructed words: {words_bn}")
        print(f"  Token map sample: {dict(list(token_map_bn.items())[:3])}")

        has_bengali_words = any(any("\u0980" <= c <= "\u09FF" for c in w) for w in words_bn)
        print(f"  Contains Bengali words: {has_bengali_words}")

        print("\n[TEST 2] English text processing (should show warning):")
        print(f"  Input: {sample_en}")
        token_map_en, words_en = reconstruct_word_spans(tokenizer, sample_en, max_length=32)
        print(f"  Reconstructed words: {words_en}")

        has_english_words = any(any("a" <= c.lower() <= "z" for c in w) for w in words_en)
        print(f"  Contains English words: {has_english_words}")

        print("\n[TEST 3] Token validation:")
        special_tokens = get_tokenizer_special_tokens(tokenizer)
        test_tokens = ["কাল", "▁আমি", "</s>", "##ing", "a"]
        for tok in test_tokens:
            valid = is_valid_token(tok, special_tokens, tokenizer, "bn")
            print(f"  '{tok}': {'valid' if valid else 'invalid'}")

        print("\n[TEST 4] Preprocessing test:")
        if _HAS_INDIC_PROCESSOR:
            test_batch = preprocess_for_indictrans2([sample_bn], src_lang=_SOURCE_LANG, tgt_lang=_TARGET_LANG, indic_processor=None)
            print(f"  ✅ Preprocessing successful: {test_batch[0][:70]}...")
        else:
            print("  ❌ IndicTransToolkit not installed")

        print("\n[TEST 5] Single word preprocessing test:")
        test_word = "কাল"
        processed_word = preprocess_single_word_for_indictrans2(test_word, src_lang=_SOURCE_LANG, tgt_lang=_TARGET_LANG, indic_processor=None)
        print(f"  Input word: {test_word}")
        print(f"  Processed: {processed_word}")

        if has_bengali_words and not any("a" <= c.lower() <= "z" for c in "".join(words_bn)):
            print("\nTest PASSED: Bengali processing works correctly")
            return True
        else:
            print("\nTest WARNING: Check language detection logic")
            return False

    except Exception as e:
        print(f"\nTest FAILED: {repr(e)}")
        import traceback
        traceback.print_exc()
        return False
    finally:
        print("=" * 60 + "\n")


# Aliases exported for other cells
safeoffsetstokenize = safe_offsets_tokenize
reconstructwordspans = reconstruct_word_spans
gettokenizerspecialtokens = get_tokenizer_special_tokens
getcachedspecialtokens = get_cached_special_tokens
isvalidtoken = is_valid_token
shouldtracktoken = should_track_token
gettokenizervocabsize = get_tokenizer_vocab_size
validatetokenizervocab = validate_tokenizer_vocab
initializeindicprocessor = initialize_indic_processor
preprocessforindictrans2 = preprocess_for_indictrans2
preprocesssinglewordforindictrans2 = preprocess_single_word_for_indictrans2
postprocessforindictrans2 = postprocess_for_indictrans2
createdecoderlabels = create_decoder_labels

try:
    indic_processor_instance = initialize_indic_processor(inference=True)
except Exception:
    indic_processor_instance = None

print("=" * 80)
print("Cell 1: DUAL-PATH Tokenizer Utilities + Training Losses - READY (INDICTRANS2)")
print("=" * 80)
print("VERIFICATION:")
print(f"  ✅ DSCD_MIN_LETTERS: {_DSCD_MIN_LETTERS}")
print(f"  ✅ DSCD_MIN_LETTER_FRACTION: {_DSCD_MIN_LETTER_FRACTION}")
print(f"  ✅ Token validation cache size: {_cache_max_size}")
print(f"  ✅ Source language: {_SOURCE_LANG}")
print(f"  ✅ Target language: {_TARGET_LANG}")
print(f"  ✅ Label Smoothing: {_LABEL_SMOOTHING}")
print(f"  ✅ R-Drop alpha: {_RDROP_ALPHA}")
print(f"  ✅ IndicProcessor available: {_HAS_INDIC_PROCESSOR}")
print(f"  ✅ Global instance created: {indic_processor_instance is not None}")
print("\nDUAL-PATH COMPONENTS:")
print("  ✅ BengaliWordTokenizer - Path 1 (word-level)")
print("  ✅ IndicTrans2 utilities - Path 2 (subword)")
print("  ✅ preprocess_single_word_for_indictrans2() - NEW FUNCTION ADDED")
print("  ✅ Clean text preprocessing - FIXED for Path 2")
print("  ✅ LabelSmoothingLoss - Path 2 training")
print("  ✅ RDropLoss - Path 2 regularization")
print("  ✅ create_decoder_labels - Path 2 label creation")
print("=" * 80 + "\n")

Cell 1: DUAL-PATH Tokenizer Utilities + Training Losses - READY (INDICTRANS2)
VERIFICATION:
  ✅ DSCD_MIN_LETTERS: 3
  ✅ DSCD_MIN_LETTER_FRACTION: 0.6
  ✅ Token validation cache size: 10000
  ✅ Source language: ben_Beng
  ✅ Target language: eng_Latn
  ✅ Label Smoothing: 0.1
  ✅ R-Drop alpha: 5.0
  ✅ IndicProcessor available: True
  ✅ Global instance created: True

DUAL-PATH COMPONENTS:
  ✅ BengaliWordTokenizer - Path 1 (word-level)
  ✅ IndicTrans2 utilities - Path 2 (subword)
  ✅ preprocess_single_word_for_indictrans2() - NEW FUNCTION ADDED
  ✅ Clean text preprocessing - FIXED for Path 2
  ✅ LabelSmoothingLoss - Path 2 training
  ✅ RDropLoss - Path 2 regularization
  ✅ create_decoder_labels - Path 2 label creation



In [5]:
# ==============================================================================
# CELL 2: DUAL-PATH DATA LOADING - WORD + SUBWORD TOKENIZATION (INDICTRANS2)
# ==============================================================================
from typing import Optional, List, Tuple, Dict, Any
from collections import defaultdict
import os
import time
import random
import traceback
import re

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, get_worker_info
from tqdm import tqdm

try:
    import pandas as pd
    _HAS_PANDAS = True
except Exception:
    pd = None
    _HAS_PANDAS = False
    print("[CELL2] WARNING: pandas not available; CSV loading will fail!")

try:
    from datasets import load_dataset
    _HAS_DATASETS = True
except Exception:
    load_dataset = None
    _HAS_DATASETS = False

_VERBOSE_LOGGING = bool(globals().get("VERBOSE_LOGGING", False))
_DEBUG_VERBOSE = bool(globals().get("DEBUG_VERBOSE", False))
_DEBUG_DISCOVERY = bool(globals().get("DEBUG_DISCOVERY", False))

DEBUG_CELL2 = bool(_VERBOSE_LOGGING) or bool(_DEBUG_VERBOSE) or bool(_DEBUG_DISCOVERY)
DEBUG_LIMIT = 10
_cell2_dbg_counts: Dict[str, int] = defaultdict(int)


def cell2_dbg(key: str, msg: str, limit: int = DEBUG_LIMIT) -> None:
    if not DEBUG_CELL2:
        return
    _cell2_dbg_counts[key] += 1
    if _cell2_dbg_counts[key] <= limit:
        print(f"[CELL2-DBG] {msg}")


_NUM_SAMPLES = int(globals().get("NUM_SAMPLES", 50000))
_MAX_LENGTH = int(globals().get("MAX_LENGTH", 64))

_SOURCE_LANG = str(globals().get("SOURCE_LANGUAGE", "ben_Beng"))
_TARGET_LANG = str(globals().get("TARGET_LANGUAGE", "eng_Latn"))

_NUM_GPUS = int(globals().get("NUM_GPUS", torch.cuda.device_count() if torch.cuda.is_available() else 0))
_USE_MULTI_GPU = bool(globals().get("USE_MULTI_GPU", _NUM_GPUS > 1))

_NUM_WORKERS = int(globals().get("NUM_WORKERS", 0))
_PIN_MEMORY = bool(globals().get("PIN_MEMORY", False))
_PREFETCH_FACTOR = int(globals().get("PREFETCH_FACTOR", 2))

_DATASET_CSV_PATH = str(globals().get("DATASET_CSV_PATH", "/kaggle/input/datasets/manaskumarmanna/bpcc-data/bpcc_bn_en_400k.csv"))

_TRAIN_DOMAIN = int(globals().get("TRAIN_DOMAIN", 0))
_TEST_DOMAIN = int(globals().get("TEST_DOMAIN", 1))
_USE_DOMAIN_LABELS = bool(globals().get("USE_DOMAIN_LABELS", True))

_has_normalize = callable(globals().get("normalize_bengali", None)) and callable(globals().get("normalize_english", None))
_has_reconstruct_word_spans = callable(globals().get("reconstruct_word_spans", None))
_has_safe_offsets_tokenize = callable(globals().get("safe_offsets_tokenize", None))
_has_preprocess_indictrans2 = callable(globals().get("preprocess_for_indictrans2", None))

if not _has_normalize:
    print("[CELL2] WARNING: normalize_bengali/normalize_english not found; using simple .strip()")
if not _has_preprocess_indictrans2:
    print("[CELL2] WARNING: preprocess_for_indictrans2 not found; using plain text")

_BENGALI_CHAR_RE = re.compile(r"[\u0980-\u09FF]")
_BENGALI_PUNCT_SET = set(["।", "॥"])
_COMMON_PUNCT_SET = set([".", ",", ";", ":", "!", "?", '"', "'", "-", "(", ")", "[", "]", "{", "}"])


def is_bengali_text(s: str) -> bool:
    if s is None:
        return False
    if not isinstance(s, str) or not s:
        return False
    return bool(_BENGALI_CHAR_RE.search(s))


def separate_bengali_punctuation(text: str, language: str = "bn") -> str:
    if not text or not isinstance(text, str):
        return ""
    text = text.strip()
    if language in ["bn", "hi", "te", "ta", "ml", "mr", "gu", "pa"]:
        text = re.sub(r"([।॥])", r" \1 ", text)
    text = re.sub(r"([,;:!?()\[\]{}])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_and_normalize_text(text: str, language: str = "bn") -> str:
    if not text or not isinstance(text, str):
        return ""
    text = text.strip()
    if not text:
        return ""
    text = separate_bengali_punctuation(text, language)
    if _has_normalize:
        try:
            if language == "bn":
                return globals()["normalize_bengali"](text)
            else:
                return globals()["normalize_english"](text)
        except Exception:
            return text.strip()
    else:
        if language == "en":
            return text.lower().strip()
        return text.strip()


def is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET:
        return True
    if clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all((c in _BENGALI_PUNCT_SET) or (c in _COMMON_PUNCT_SET) for c in clean)


def _dataloader_worker_init_fn(worker_id: int) -> None:
    worker_info = get_worker_info()
    dataset = worker_info.dataset if worker_info is not None else None

    try:
        if dataset is not None and hasattr(dataset, "_tokenizer_name_or_path") and dataset._tokenizer_name_or_path:
            try:
                from transformers import AutoTokenizer
                tk_name = dataset._tokenizer_name_or_path
                tk = AutoTokenizer.from_pretrained(tk_name, trust_remote_code=True)
                try:
                    tk.src_lang = _SOURCE_LANG
                    tk.tgt_lang = _TARGET_LANG
                except Exception:
                    pass
                dataset.tokenizer = tk
                dataset.is_fast = getattr(tk, "is_fast", False)
                dataset.vocab_size = len(tk) if hasattr(tk, "__len__") else getattr(tk, "vocab_size", 0)
            except Exception as e:
                cell2_dbg("worker_tokenizer_reload", f"Worker {worker_id} tokenizer reload failed: {e}")
                dataset.tokenizer = None
                dataset.is_fast = False
                dataset.vocab_size = int(globals().get("DEFAULT_VOCAB_SIZE", 128112))
    except Exception:
        pass

    try:
        base = int(os.environ.get("PYTHONHASHSEED", "0"))
        seed = (base ^ (worker_id + 1) ^ int(time.time())) & 0xFFFFFFFF
        random.seed(seed)
        np.random.seed(seed % (2**31 - 1))
        torch.manual_seed(seed % (2**31 - 1))
    except Exception:
        pass


def load_and_preprocess_optimized(
    num_samples: Optional[int] = None,
    split: str = "train",
) -> List[Tuple[str, str]]:
    if num_samples is None:
        num_samples = _NUM_SAMPLES
    if num_samples <= 0:
        raise ValueError("num_samples must be positive")

    print(f"[CELL2] Loading up to {num_samples} samples from local CSV: {_DATASET_CSV_PATH}")

    if not _HAS_PANDAS:
        print("[CELL2] ERROR: pandas not available; cannot load CSV!")
        print("[CELL2] Using fallback dataset for debugging.")
        return _get_fallback_dataset()

    if not os.path.exists(_DATASET_CSV_PATH):
        print(f"[CELL2] ERROR: CSV file not found at: {_DATASET_CSV_PATH}")
        print("[CELL2] Using fallback dataset for debugging.")
        return _get_fallback_dataset()

    try:
        print("[CELL2] Reading CSV file...")
        df = pd.read_csv(_DATASET_CSV_PATH)

        if df is None or df.empty:
            print("[CELL2] ERROR: CSV file is empty")
            return _get_fallback_dataset()

        if "src" not in df.columns or "tgt" not in df.columns:
            print(f"[CELL2] ERROR: CSV missing required columns. Found columns: {list(df.columns)}")
            print("[CELL2] Expected format: src (Bengali), tgt (English) OR src (English), tgt (Bengali)")
            return _get_fallback_dataset()

        sample_size = min(10, len(df))
        sample_rows = df.head(sample_size)

        src_bengali_count = sum(1 for s in sample_rows["src"] if is_bengali_text(str(s)))
        tgt_bengali_count = sum(1 for s in sample_rows["tgt"] if is_bengali_text(str(s)))

        src_is_bengali = src_bengali_count > sample_size * 0.5
        tgt_is_bengali = tgt_bengali_count > sample_size * 0.5

        if (not src_is_bengali) and tgt_is_bengali:
            print("[CELL2] Detected src=English, tgt=Bengali: Swapping columns for bn→en task.")
            df = df.rename(columns={"src": "_temp_tgt", "tgt": "_temp_src"})
            df = df.rename(columns={"_temp_src": "src", "_temp_tgt": "tgt"})

            sample_rows = df.head(sample_size)
            src_bengali_count = sum(1 for s in sample_rows["src"] if is_bengali_text(str(s)))
            src_is_bengali = src_bengali_count > sample_size * 0.5

            if not src_is_bengali:
                print("[CELL2] ERROR: Swap failed, src is still not Bengali.")
                return _get_fallback_dataset()
            else:
                print("[CELL2] Swap successful: src=Bengali, tgt=English")
        elif not src_is_bengali:
            print("[CELL2] WARNING: src column does not appear to be Bengali. Proceeding but output may be incorrect.")

        df = df.head(num_samples)
        print(f"[CELL2] Processing {len(df)} rows from CSV...")

        pairs: List[Tuple[str, str]] = []
        skipped = 0

        for row_tuple in tqdm(df.itertuples(index=False), total=len(df), desc="Loading dataset"):
            try:
                src_val = getattr(row_tuple, "src", None)
                tgt_val = getattr(row_tuple, "tgt", None)

                if pd.isna(src_val) or pd.isna(tgt_val):
                    skipped += 1
                    cell2_dbg("nan_value", "NaN value detected")
                    continue

                bn = str(src_val).strip()
                en = str(tgt_val).strip()

                if not bn or not en:
                    skipped += 1
                    cell2_dbg("empty_field", "Empty src/tgt field")
                    continue

                if not is_bengali_text(bn):
                    skipped += 1
                    cell2_dbg("not_bengali_src", "src field not Bengali")
                    continue

                if not re.search(r"[a-zA-Z]", en):
                    skipped += 1
                    cell2_dbg("not_english_tgt", "tgt field not English")
                    continue

                max_words = max(20, _MAX_LENGTH // 2)
                if len(bn.split()) > max_words or len(en.split()) > max_words:
                    skipped += 1
                    cell2_dbg("too_long", "Text too long")
                    continue

                bn_norm = clean_and_normalize_text(bn, language="bn")
                en_norm = clean_and_normalize_text(en, language="en")

                if not bn_norm or not en_norm:
                    skipped += 1
                    cell2_dbg("empty_after_norm", "Empty after normalization")
                    continue

                pairs.append((bn_norm, en_norm))
            except Exception as e:
                skipped += 1
                cell2_dbg("row_exception", f"Row load exception: {type(e).__name__}")
                continue

        print(f"[CELL2] Loaded {len(pairs)} pairs from CSV, skipped {skipped} rows")

        if len(pairs) == 0:
            print("[CELL2] ERROR: No valid pairs loaded from CSV!")
            print("[CELL2] Check that src column contains Bengali and tgt column contains English.")
            return _get_fallback_dataset()

        return pairs

    except getattr(pd, "errors", Exception).__dict__.get("EmptyDataError", Exception):
        print(f"[CELL2] ERROR: CSV file is empty: {_DATASET_CSV_PATH}")
        return _get_fallback_dataset()
    except Exception as e:
        print(f"[CELL2] ERROR loading CSV: {type(e).__name__}: {str(e)}")
        try:
            traceback.print_exc()
        except Exception:
            pass
        print("[CELL2] Using fallback dataset")
        return _get_fallback_dataset()


def _get_fallback_dataset() -> List[Tuple[str, str]]:
    print("[CELL2] Using fallback dataset (50 unique samples)")
    fallback_pairs = [
        ("আমি কল বন্ধ করেছি", "i turned off the tap"),
        ("সে আমাকে পরে কল করবে", "he will call me later"),
        ("আমরা প্রতিদিন তাজা ফল খাই", "we eat fresh fruits every day"),
        ("তার কঠোর পরিশ্রমের ভালো ফল হয়েছে", "his hard work has brought good results"),
        ("গাছে নতুন পাতাগুলো গজিয়েছে", "new leaves have sprouted on the tree"),
        ("আমি বইয়ের পাতা উল্টাচ্ছি", "i am turning the pages of the book"),
        ("কাল আমি বাজারে গিয়েছিলাম", "yesterday i went to the market"),
        ("কাল আমি তোমার সাথে দেখা করব", "tomorrow i will meet you"),
        ("তারা আকাশে উজ্জ্বল", "the stars are bright in the sky"),
        ("তারা বাড়িতে নেই", "they are not at home"),
        ("ব্যাংক নদীর ধারে ভেঙে গেছে", "the bank by the river has collapsed"),
        ("আমি ব্যাংকে টাকা জমা দিয়েছি", "i deposited money in the bank"),
        ("বার বার চেষ্টা করতে হবে", "you have to try again and again"),
        ("আমি বার খুলে ভিতরে ঢুকলাম", "i opened the bar and entered"),
        ("তার মাথা ব্যথা করছে", "his head is hurting"),
        ("আমি মাথা নেড়ে সম্মতি দিলাম", "i nodded my head in agreement"),
        ("সে হার মেনে নিয়েছে", "he accepted defeat"),
        ("আমি গলায় সোনার হার পরেছি", "i am wearing a gold necklace"),
        ("পানি খুব ঠান্ডা", "the water is very cold"),
        ("আমি পানি খাচ্ছি", "i am drinking water"),
        ("দল খেলায় জিতেছে", "the team won the game"),
        ("আমি মাটি দল দিয়ে ফেললাম", "i trampled the soil"),
        ("বাজার থেকে সবজি কিনলাম", "i bought vegetables from the market"),
        ("বাজার অনেক ভিড় ছিল", "the market was very crowded"),
        ("তার নাম আহমেদ", "his name is ahmed"),
        ("নাম না করে কাজ করো", "work without making a name"),
        ("কথা বলা বন্ধ করো", "stop talking"),
        ("তার কথা শুনে ভালো লাগল", "i felt good hearing his words"),
        ("বই পড়তে ভালো লাগে", "i like reading books"),
        ("আমি একটি নতুন বই কিনেছি", "i bought a new book"),
        ("ঘর পরিষ্কার করা হয়েছে", "the house has been cleaned"),
        ("আমি ঘরে বসে আছি", "i am sitting at home"),
        ("মন ভালো নেই", "my mind is not good"),
        ("আমার মন চায় বেড়াতে যেতে", "my mind wants to go for a walk"),
        ("হাত ধুয়ে নাও", "wash your hands"),
        ("আমি তার হাত ধরলাম", "i held his hand"),
        ("দিন কেটে যাচ্ছে", "the day is passing by"),
        ("আজ কি দিন", "what day is today"),
        ("রাত হয়ে এসেছে", "night has come"),
        ("আমি রাত জেগে পড়েছি", "i studied staying up at night"),
        ("জল খুব গরম", "the water is very hot"),
        ("আমি জল দিয়ে গাছ সিঞ্চন করেছি", "i watered the plants"),
        ("বাড়ি যাচ্ছি", "i am going home"),
        ("আমার বাড়ি ঢাকায়", "my house is in dhaka"),
        ("পার্কে অনেক মানুষ", "there are many people in the park"),
        ("আমি প্রতিদিন পার্কে হাঁটি", "i walk in the park every day"),
        ("নদী বইছে", "the river is flowing"),
        ("আমি নদীর ধারে দাঁড়িয়ে আছি", "i am standing by the river"),
        ("বন খুব সুন্দর", "the forest is very beautiful"),
        ("আমি বন দেখতে গিয়েছিলাম", "i went to see the forest"),
    ]

    processed_pairs: List[Tuple[str, str]] = []
    for bn, en in fallback_pairs:
        bn_clean = clean_and_normalize_text(bn, "bn")
        en_clean = clean_and_normalize_text(en, "en")
        if bn_clean and en_clean:
            processed_pairs.append((bn_clean, en_clean))

    return processed_pairs


class DualPathDataset(Dataset):
    def __init__(
        self,
        pairs: List[Tuple[str, str]],
        tokenizer: Any = None,
        max_length: Optional[int] = None,
        split: str = "train",
    ):
        if max_length is None:
            max_length = _MAX_LENGTH

        self.max_length = int(max_length)
        self.tokenizer = tokenizer
        self.split = split

        self.vocab_size = len(tokenizer) if tokenizer is not None and hasattr(tokenizer, "__len__") else int(globals().get("DEFAULT_VOCAB_SIZE", 128112))
        print(f"[CELL2] Dataset vocab size: {self.vocab_size}")

        try:
            self._tokenizer_name_or_path = getattr(tokenizer, "name_or_path", None) or globals().get("MODEL_NAME", None)
        except Exception:
            self._tokenizer_name_or_path = None

        try:
            self.is_fast = getattr(self.tokenizer, "is_fast", False)
        except Exception:
            self.is_fast = False

        self.pairs: List[Tuple[str, str]] = []
        invalid = 0

        for i, p in enumerate(pairs):
            try:
                if not isinstance(p, (list, tuple)) or len(p) != 2:
                    invalid += 1
                    cell2_dbg("init_badpair", f"Bad pair structure at idx={i}")
                    continue
                src, tgt = p
                if not isinstance(src, str) or not isinstance(tgt, str):
                    invalid += 1
                    cell2_dbg("init_badtype", f"Non-string src/tgt at idx={i}")
                    continue
                if not src or not tgt:
                    invalid += 1
                    cell2_dbg("init_empty", f"Empty src/tgt field at idx={i}")
                    continue
                if len(src) > self.max_length * 20 or len(tgt) > self.max_length * 20:
                    invalid += 1
                    cell2_dbg("init_long", f"Extremely long text at idx={i}")
                    continue
                self.pairs.append((src, tgt))
            except Exception as e:
                invalid += 1
                cell2_dbg("init_exc", f"Init pair exception idx={i}: {type(e).__name__}")

        print(f"[CELL2] Dataset initialized: {len(self.pairs)} valid pairs, {invalid} invalid, split={self.split}")

        try:
            self.special_tokens = set(getattr(self.tokenizer, "all_special_tokens", [])) if self.tokenizer is not None else set()
        except Exception:
            self.special_tokens = {
                _SOURCE_LANG,
                _TARGET_LANG,
                "</s>",
                "<pad>",
                "<s>",
                "<unk>",
            }

    def __getstate__(self):
        state = self.__dict__.copy()
        state["tokenizer"] = None
        state["_tokenizer_name_or_path"] = getattr(self, "_tokenizer_name_or_path", None)
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        self.tokenizer = None
        self.is_fast = False

    def __len__(self) -> int:
        return len(self.pairs)

    def _encode_src(self, src_text: str):
        src_text = src_text if isinstance(src_text, str) else str(src_text)

        try:
            if self.tokenizer is None:
                self.tokenizer = globals().get("tokenizer_indic", None)
                self.is_fast = getattr(self.tokenizer, "is_fast", False) if self.tokenizer is not None else False
                self.vocab_size = len(self.tokenizer) if self.tokenizer is not None and hasattr(self.tokenizer, "__len__") else int(globals().get("DEFAULT_VOCAB_SIZE", 128112))

            if self.tokenizer is None:
                raise RuntimeError("Tokenizer not available")

            preprocessed_text = src_text
            if _has_preprocess_indictrans2:
                try:
                    indic_processor = globals().get("indic_processor_instance", None)
                    if indic_processor is None:
                        try:
                            from IndicTransToolkit import IndicProcessor
                            indic_processor = IndicProcessor(inference=True)
                            globals()["indic_processor_instance"] = indic_processor
                        except Exception:
                            indic_processor = None

                    preprocessed_batch = globals()["preprocess_for_indictrans2"]([src_text], src_lang=_SOURCE_LANG, tgt_lang=_TARGET_LANG, indic_processor=indic_processor)
                    preprocessed_text = preprocessed_batch[0] if preprocessed_batch else src_text
                except Exception as e:
                    cell2_dbg("preprocess_fail", f"Preprocessing failed: {e}")
                    preprocessed_text = src_text

            try:
                self.tokenizer.src_lang = _SOURCE_LANG
            except Exception:
                pass

            if _has_safe_offsets_tokenize:
                enc = globals()["safe_offsets_tokenize"](self.tokenizer, src_text, max_length=self.max_length, include_special_tokens=True, preprocessed_text=preprocessed_text)
                try:
                    if isinstance(enc.get("input_ids"), torch.Tensor):
                        input_ids = enc["input_ids"].squeeze(0) if enc["input_ids"].dim() > 1 else enc["input_ids"]
                    else:
                        raw = enc.get("input_ids", None)
                        if isinstance(raw, list) and raw and isinstance(raw[0], list):
                            input_ids = torch.tensor(raw[0], dtype=torch.long)
                        elif isinstance(raw, list):
                            input_ids = torch.tensor(raw, dtype=torch.long)
                        else:
                            input_ids = torch.tensor([1], dtype=torch.long)
                except Exception:
                    input_ids = torch.tensor([1], dtype=torch.long)

                attention_mask = enc.get("attention_mask", None)
                if attention_mask is None:
                    attention_mask = torch.ones_like(input_ids)
                elif isinstance(attention_mask, list):
                    attention_mask = torch.tensor(attention_mask[0] if attention_mask and isinstance(attention_mask[0], list) else attention_mask, dtype=torch.long)
                elif isinstance(attention_mask, torch.Tensor):
                    attention_mask = attention_mask.squeeze(0) if attention_mask.dim() > 1 else attention_mask

                try:
                    ids_list = input_ids.tolist()
                    tokens = []
                    try:
                        tokens = self.tokenizer.convert_ids_to_tokens(ids_list)
                    except Exception:
                        tokens = []
                except Exception:
                    tokens = []
            else:
                enc = self.tokenizer(preprocessed_text, max_length=self.max_length, padding=False, truncation=True, return_tensors="pt", add_special_tokens=True)
                input_ids = enc["input_ids"].squeeze(0).long()
                attention_mask = enc.get("attention_mask", torch.ones_like(input_ids)).squeeze(0).long()
                try:
                    tokens = self.tokenizer.convert_ids_to_tokens(input_ids.tolist())
                except Exception:
                    tokens = []

            try:
                vocab_cap = self.vocab_size
            except Exception:
                vocab_cap = 128112
            if vocab_cap and vocab_cap > 0:
                input_ids = torch.clamp(input_ids, 0, max(0, vocab_cap - 1)).long()
            else:
                input_ids = input_ids.long()

            attention_mask = attention_mask.long()

            token_word_map: Dict[int, str] = {}

            if _has_reconstruct_word_spans:
                try:
                    wm, _words = globals()["reconstruct_word_spans"](self.tokenizer, src_text, max_length=self.max_length, preprocessed_text=preprocessed_text)
                    if isinstance(wm, dict) and wm:
                        token_word_map = wm
                except Exception as e:
                    cell2_dbg("wm_exc", f"reconstruct_word_spans failed: {e}")

            if not token_word_map and tokens:
                try:
                    current_word: List[str] = []
                    for idx, tok in enumerate(tokens):
                        if isinstance(tok, str) and tok not in self.special_tokens:
                            if is_punctuation_only(tok):
                                continue
                            clean = tok.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
                            if clean:
                                if tok.startswith("▁") or tok.startswith("Ġ"):
                                    current_word = [clean]
                                else:
                                    current_word.append(clean)
                                word_str = "".join(current_word)
                                if not is_punctuation_only(word_str):
                                    token_word_map[idx] = word_str
                except Exception as e:
                    cell2_dbg("fallback_wm", f"Fallback word map failed: {e}")

            return input_ids, attention_mask, tokens, token_word_map

        except Exception as e:
            cell2_dbg("encode_src_exc", f"Encoding source failed: {type(e).__name__}")
            pad_id = getattr(self.tokenizer, "pad_token_id", 1) if self.tokenizer is not None else 1
            input_ids = torch.full((self.max_length,), int(pad_id), dtype=torch.long)
            attention_mask = torch.zeros(self.max_length, dtype=torch.long)
            return input_ids, attention_mask, [], {}

    def _encode_tgt(self, tgt_text: str):
        tgt_text = tgt_text if isinstance(tgt_text, str) else str(tgt_text)

        try:
            if self.tokenizer is None:
                self.tokenizer = globals().get("tokenizer_indic", None)
                self.vocab_size = len(self.tokenizer) if self.tokenizer is not None and hasattr(self.tokenizer, "__len__") else int(globals().get("DEFAULT_VOCAB_SIZE", 128112))

            if self.tokenizer is None:
                raise RuntimeError("Tokenizer not available")

            tgt_text_clean = clean_and_normalize_text(tgt_text, language="en")
            if not tgt_text_clean:
                tgt_text_clean = tgt_text

            decoder_vocab_size = int(globals().get("DECODER_VOCAB_SIZE", 32296))

            # Ensure tokenizer is in target/decoder mode if available
            try:
                self.tokenizer.tgt_lang = _TARGET_LANG
            except Exception:
                pass

            use_target_ctx = getattr(self.tokenizer, "as_target_tokenizer", None)
            labels_tensor = None

            try:
                if callable(use_target_ctx):
                    with self.tokenizer.as_target_tokenizer():
                        enc = self.tokenizer(tgt_text_clean, max_length=self.max_length, padding=False, truncation=True, return_tensors="pt", add_special_tokens=True)
                else:
                    enc = self.tokenizer(tgt_text_clean, max_length=self.max_length, padding=False, truncation=True, return_tensors="pt", add_special_tokens=True)

                labels_tensor = enc["input_ids"].squeeze(0).long()
            except Exception:
                unk_id = getattr(self.tokenizer, "unk_token_id", None)
                pad_id = getattr(self.tokenizer, "pad_token_id", None) or 1
                safe_id = unk_id if (unk_id is not None) else min(max(1, pad_id - 1), decoder_vocab_size - 1)
                labels_tensor = torch.tensor([safe_id], dtype=torch.long)

            if not isinstance(labels_tensor, torch.Tensor):
                labels_tensor = torch.tensor([getattr(self.tokenizer, "unk_token_id", 1)], dtype=torch.long)

            labels_tensor = labels_tensor.view(-1).long()
            labels_tensor = torch.clamp(labels_tensor, 0, decoder_vocab_size - 1)

            pad_id = getattr(self.tokenizer, "pad_token_id", 1) if self.tokenizer is not None else 1

            labels_with_padding = torch.full((self.max_length,), int(pad_id), dtype=torch.long)
            L = min(int(labels_tensor.size(0)), int(self.max_length))
            if L > 0:
                labels_with_padding[:L] = labels_tensor[:L]

            labels = labels_with_padding
            labels[labels == int(pad_id)] = -100

            # Double-check bounds after masking
            labels = torch.where(labels == -100, labels, torch.clamp(labels, 0, max(0, decoder_vocab_size - 1)))

            if _DEBUG_DISCOVERY:
                valid_after_mask = int((labels != -100).sum().item())
                if valid_after_mask == 0:
                    print(f"[CELL2-TGT] ⚠️ No valid labels generated for: '{tgt_text_clean[:60]}'")

            return labels

        except Exception:
            pad_id = getattr(self.tokenizer, "pad_token_id", 1) if self.tokenizer is not None else 1
            decoder_vocab_size = int(globals().get("DECODER_VOCAB_SIZE", 32296))
            safe_id = min(max(1, pad_id - 1), decoder_vocab_size - 1)

            try:
                temp = torch.full((self.max_length,), int(pad_id), dtype=torch.long)
                temp[0] = int(safe_id)
                temp[temp == int(pad_id)] = -100
                return temp
            except Exception:
                return torch.full((self.max_length,), -100, dtype=torch.long)

    def _make_safe_sample(self, reason: str = "fallback") -> Dict[str, Any]:
        try:
            src = "আমি"
            tgt = "i"
            input_ids, attention_mask, tokens, token_word_map = self._encode_src(src)
            labels = self._encode_tgt(tgt)

            if self.split in ("train", "training"):
                domain_label = _TRAIN_DOMAIN
            elif self.split in ("val", "validation", "test"):
                domain_label = _TEST_DOMAIN
            else:
                domain_label = random.randint(_TRAIN_DOMAIN, _TEST_DOMAIN)

            return {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
                "token_word_map": token_word_map,
                "src_text": src,
                "tokens": tokens,
                "domain_label": domain_label,
            }
        except Exception:
            pad_id = 1
            if self.split in ("train", "training"):
                domain_label = _TRAIN_DOMAIN
            elif self.split in ("val", "validation", "test"):
                domain_label = _TEST_DOMAIN
            else:
                domain_label = random.randint(_TRAIN_DOMAIN, _TEST_DOMAIN)
            return {
                "input_ids": torch.full((self.max_length,), int(pad_id), dtype=torch.long),
                "attention_mask": torch.zeros(self.max_length, dtype=torch.long),
                "labels": torch.full((self.max_length,), -100, dtype=torch.long),
                "token_word_map": {},
                "src_text": "",
                "tokens": [],
                "domain_label": domain_label,
            }

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        try:
            if idx < 0 or idx >= len(self.pairs):
                cell2_dbg("getitem_oob", f"Index out of range idx={idx}")
                return self._make_safe_sample("oob")

            src, tgt = self.pairs[idx]

            if not isinstance(src, str) or not isinstance(tgt, str):
                cell2_dbg("getitem_bad_types", f"Bad types at idx={idx}")
                return self._make_safe_sample("bad_types")

            input_ids, attention_mask, tokens, token_word_map = self._encode_src(src)
            labels = self._encode_tgt(tgt)

            if labels is None or not isinstance(labels, torch.Tensor) or int((labels != -100).sum().item()) == 0:
                cell2_dbg("getitem_bad_labels", f"idx={idx} produced invalid labels")
                return self._make_safe_sample("bad_tgt")

            if self.split in ("train", "training"):
                domain_label = _TRAIN_DOMAIN
            elif self.split in ("val", "validation", "test"):
                domain_label = _TEST_DOMAIN
            else:
                domain_label = idx % 2

            return {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
                "token_word_map": token_word_map,
                "src_text": src,
                "tokens": tokens,
                "domain_label": domain_label,
            }
        except Exception as e:
            cell2_dbg("getitem_exc", f"Unhandled __getitem__ exception idx={idx}: {type(e).__name__}")
            return self._make_safe_sample("unhandled")


def _infer_pad_id_from_sample(sample: Dict[str, Any], default_pad_id: int = 1) -> int:
    try:
        tk = globals().get("tokenizer_indic", None)
        if tk is not None:
            pad = getattr(tk, "pad_token_id", None)
            if pad is not None:
                return int(pad)
    except Exception:
        cell2_dbg("infer_pad_exc", "infer pad id failed")
    return int(default_pad_id)


def _pad_or_truncate_array(tensor: torch.Tensor, length: int, pad_value: int) -> torch.Tensor:
    if tensor is None:
        return torch.full((length,), int(pad_value), dtype=torch.long)
    t = tensor.view(-1).long()
    L = int(t.size(0))
    if L == length:
        return t
    if L < length:
        pad = torch.full((length - L,), int(pad_value), dtype=t.dtype)
        return torch.cat([t, pad], dim=0)
    return t[:length]


def safe_collate(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    valid = [
        b
        for b in batch
        if isinstance(b, dict)
        and "input_ids" in b
        and isinstance(b["input_ids"], torch.Tensor)
    ]

    default_domain = _TRAIN_DOMAIN

    if not valid:
        pad = _infer_pad_id_from_sample({}, default_pad_id=1)
        return {
            "input_ids": torch.full((1, _MAX_LENGTH), int(pad), dtype=torch.long),
            "attention_mask": torch.zeros(1, _MAX_LENGTH, dtype=torch.long),
            "labels": torch.full((1, _MAX_LENGTH), -100, dtype=torch.long),
            "token_word_map": [{}],
            "src_texts": [""],
            "tokens": [[]],
            "domain_labels": torch.tensor([default_domain], dtype=torch.long),
        }

    pad_id = _infer_pad_id_from_sample(valid[0], default_pad_id=1)

    raw_inputs: List[torch.Tensor] = []
    raw_masks: List[torch.Tensor] = []
    raw_labs: List[torch.Tensor] = []
    twmaps: List[Dict[int, str]] = []
    srcs: List[str] = []
    toks: List[List[str]] = []
    domains: List[int] = []

    for s in valid:
        try:
            in_ids = s["input_ids"]
            att = s.get("attention_mask", None)
            lab = s.get("labels", None)
            domain = s.get("domain_label", default_domain)

            if not isinstance(domain, int):
                try:
                    domain = int(domain)
                except Exception:
                    domain = default_domain

            if att is None or not isinstance(att, torch.Tensor):
                att = (in_ids != int(pad_id)).long()
            else:
                try:
                    att = att.view(-1).long()
                except Exception:
                    att = (in_ids != int(pad_id)).long()

            try:
                in_ids = in_ids.view(-1).long()
            except Exception:
                in_ids = in_ids.flatten().long()

            if lab is None or not isinstance(lab, torch.Tensor):
                lab = torch.full((in_ids.size(0),), -100, dtype=torch.long)
            else:
                try:
                    lab = lab.view(-1).long()
                except Exception:
                    lab = lab.flatten().long()

            in_ids = _pad_or_truncate_array(in_ids, _MAX_LENGTH, int(pad_id))
            att = _pad_or_truncate_array(att, _MAX_LENGTH, 0)

            if lab.numel() != _MAX_LENGTH:
                lab = _pad_or_truncate_array(lab, _MAX_LENGTH, int(pad_id))
                lab[lab == int(pad_id)] = -100

            raw_inputs.append(in_ids)
            raw_masks.append(att)
            raw_labs.append(lab)

            twmaps.append(s.get("token_word_map", {}) if isinstance(s.get("token_word_map", {}), dict) else {})
            srcs.append(str(s.get("src_text", "")))
            toks.append(s.get("tokens", []) if isinstance(s.get("tokens", []), list) else [])
            domains.append(domain)
        except Exception:
            continue

    if not raw_inputs:
        pad = int(pad_id)
        return {
            "input_ids": torch.full((1, _MAX_LENGTH), pad, dtype=torch.long),
            "attention_mask": torch.zeros(1, _MAX_LENGTH, dtype=torch.long),
            "labels": torch.full((1, _MAX_LENGTH), -100, dtype=torch.long),
            "token_word_map": [{}],
            "src_texts": [""],
            "tokens": [[]],
            "domain_labels": torch.tensor([default_domain], dtype=torch.long),
        }

    input_ids = torch.stack(raw_inputs, dim=0).long()
    attention_mask = torch.stack(raw_masks, dim=0).long()
    labels = torch.stack(raw_labs, dim=0).long()
    domain_labels = torch.tensor(domains, dtype=torch.long)

    # Final validation and clamping of labels to decoder vocab
    decoder_vocab_size = int(globals().get("DECODER_VOCAB_SIZE", 32296))
    valid_mask = labels != -100
    if valid_mask.any():
        labels[valid_mask] = torch.clamp(labels[valid_mask], 0, max(0, decoder_vocab_size - 1))

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "token_word_map": twmaps,
        "src_texts": srcs,
        "tokens": toks,
        "domain_labels": domain_labels,
    }


def create_optimized_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    num_workers: Optional[int] = None,
    pin_memory: Optional[bool] = None,
    prefetch_factor: Optional[int] = None,
    drop_last: bool = False,
) -> DataLoader:
    if num_workers is None:
        num_workers = int(_NUM_WORKERS)
    if pin_memory is None:
        pin_memory = bool(_PIN_MEMORY) or bool(torch.cuda.is_available())
    if prefetch_factor is None:
        prefetch_factor = int(_PREFETCH_FACTOR)

    dl_kwargs: Dict[str, Any] = {
        "dataset": dataset,
        "batch_size": int(batch_size),
        "shuffle": bool(shuffle),
        "num_workers": int(num_workers),
        "pin_memory": bool(pin_memory),
        "collate_fn": safe_collate,
        "drop_last": bool(drop_last),
    }

    if int(num_workers) > 0:
        dl_kwargs["worker_init_fn"] = _dataloader_worker_init_fn
        try:
            dl_kwargs["persistent_workers"] = True
        except Exception:
            pass
        try:
            dl_kwargs["prefetch_factor"] = int(prefetch_factor)
        except Exception:
            pass

    return DataLoader(**dl_kwargs)


MemoryEfficientDataset = DualPathDataset

print("[Cell 2] DualPathDataset / MemoryEfficientDataset + safe_collate + create_optimized_dataloader loaded successfully")

[Cell 2] DualPathDataset / MemoryEfficientDataset + safe_collate + create_optimized_dataloader loaded successfully


In [6]:
# ==============================================================================
# CELL 3: DSCD MODULE - WORD-LEVEL HOMOGRAPH DISAMBIGUATION - FIXED
# ==============================================================================
import threading
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
from collections import deque
import unicodedata
from typing import Optional, Dict, List, Any, Set, Tuple

PRINT_INTERVAL = int(globals().get("DEBUG_FREQUENCY", 200))

# clustering backend flags
try:
    from scipy.cluster.hierarchy import linkage, fcluster
    from scipy.spatial.distance import pdist as scipy_pdist
    HAS_CLUSTERING = True
except Exception:
    linkage = None
    fcluster = None
    scipy_pdist = None
    HAS_CLUSTERING = False

try:
    from sklearn.cluster import KMeans
    HAS_KMEANS = True
except Exception:
    KMeans = None
    HAS_KMEANS = False

# DSCD configuration with safe defaults
try:
    DSCD_MAX_PROTOS = int(globals().get("DSCD_MAX_PROTOS", 3))
    DSCD_BUFFER_SIZE = int(globals().get("DSCD_BUFFER_SIZE", 20))
    DSCD_N_MIN = int(globals().get("DSCD_N_MIN", 3))
    DSCD_DISPERSION_THRESHOLD = float(globals().get("DSCD_DISPERSION_THRESHOLD", 0.25))
    DSCD_NEWSENSE_LAMBDA = float(globals().get("DSCD_NEWSENSE_LAMBDA", 1.2))
    VERBOSE_LOGGING = bool(globals().get("VERBOSE_LOGGING", False))
    DSCD_ENABLE_TRAINING_CLUSTERING = bool(globals().get("DSCD_ENABLE_TRAINING_CLUSTERING", True))
    DSCD_MIN_LETTERS = int(globals().get("DSCD_MIN_LETTERS", 3))
    DSCD_MIN_LETTER_FRACTION = float(globals().get("DSCD_MIN_LETTER_FRACTION", 0.6))
except Exception:
    DSCD_MAX_PROTOS = 3
    DSCD_BUFFER_SIZE = 20
    DSCD_N_MIN = 3
    DSCD_DISPERSION_THRESHOLD = 0.25
    DSCD_NEWSENSE_LAMBDA = 1.2
    VERBOSE_LOGGING = False
    DSCD_ENABLE_TRAINING_CLUSTERING = True
    DSCD_MIN_LETTERS = 3
    DSCD_MIN_LETTER_FRACTION = 0.6

try:
    DEBUG_DISCOVERY = bool(globals().get("DEBUG_DISCOVERY", False))
except Exception:
    DEBUG_DISCOVERY = False

try:
    MAX_TOKENS_PER_DISCOVERY = int(globals().get("MAX_TOKENS_PER_DISCOVERY", 150))
except Exception:
    MAX_TOKENS_PER_DISCOVERY = 150

try:
    HOMOGRAPH_REFERENCE_LIST_BN = set(globals().get("HOMOGRAPH_REFERENCE_LIST_BN", []))
    if not HOMOGRAPH_REFERENCE_LIST_BN:
        raise Exception("empty")
except Exception:
    HOMOGRAPH_REFERENCE_LIST_BN = {
        'কল', 'কাল', 'পাতা', 'ফল', 'বার', 'হার', 'তারা',
        'পড়া', 'দেখা', 'চলা', 'ধরা', 'অর্থ', 'শব্দ', 'মুখ',
        'তোলা', 'বাঁচা', 'মারা', 'উত্তর', 'পাত্র', 'বেলা', 'গান',
        'নাম', 'বল', 'চাল', 'কলা', 'ধারা', 'পত্র', 'রাগ', 'রস',
        'তীর', 'জমা', 'মান', 'দাবি', 'আসন', 'সাড়া', 'বসা', 'পদ',
        'অংশ', 'মোড়', 'ঘর', 'মন', 'ব্যাংক'
    }

DSCD_MAX_CLUSTERING_POINTS = int(globals().get("DSCD_MAX_CLUSTERING_POINTS", 500))

BENGALI_PUNCT_SET = set(['।', '॥'])
COMMON_PUNCT_SET = set(['.', ',', '!', '?', ';', ':', '-', '—', '"', "'", '(', ')', '[', ']', '{', '}'])
PUNCT_SET = BENGALI_PUNCT_SET | COMMON_PUNCT_SET


def is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    if not clean:
        return False
    if clean in BENGALI_PUNCT_SET or clean in COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in PUNCT_SET for c in clean)


def clean_token_for_dscd(token: str) -> str:
    if not token or not isinstance(token, str):
        return ""
    cleaned = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    for punct in list(PUNCT_SET):
        cleaned = cleaned.replace(punct, "")
    cleaned = cleaned.lower()
    cleaned = ' '.join(cleaned.split())
    return cleaned


def normalize_token_key(token: str) -> str:
    return clean_token_for_dscd(token)


def is_word_token(token: str, min_letters: int = 2, min_letter_fraction: float = 0.6) -> bool:
    if not token or not isinstance(token, str):
        return False
    token = token.strip()
    if not token:
        return False
    letters = 0
    total = 0
    for ch in token:
        if unicodedata.category(ch).startswith('L'):
            letters += 1
        if not ch.isspace():
            total += 1
    if total == 0:
        return False
    if letters < min_letters:
        return False
    if (letters / total) < min_letter_fraction:
        return False
    return True


def is_indic_subword_fragment(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    token = token.strip()
    if not token:
        return False

    only_vowel_marks = True
    only_combining_marks = True
    has_virama = False
    letter_count = 0

    virama_chars = {
        '\u094D', '\u09CD', '\u0A4D', '\u0ACD', '\u0B4D', '\u0BCD', '\u0C4D', '\u0CCD', '\u0D4D'
    }

    for ch in token:
        cat = unicodedata.category(ch)
        if cat.startswith('L'):
            letter_count += 1
            only_vowel_marks = False
            only_combining_marks = False
        if cat not in ('Mn', 'Mc'):
            only_combining_marks = False
        if ch in virama_chars:
            has_virama = True

    if only_vowel_marks or only_combining_marks:
        return True
    if has_virama and len(token) <= 2:
        return True
    if letter_count == 0:
        return True

    vowel_modifier_ranges = [
        ('\u093E', '\u094C'),
        ('\u09BE', '\u09CC'),
        ('\u0ABE', '\u0ACC'),
        ('\u0BBE', '\u0BCC'),
        ('\u0C3E', '\u0C4C'),
        ('\u0CBE', '\u0CCC'),
    ]

    modifier_count = 0
    for ch in token:
        for start, end in vowel_modifier_ranges:
            if start <= ch <= end:
                modifier_count += 1
                break

    if modifier_count > 0 and modifier_count == len(token):
        return True
    if len(token) <= 2 and modifier_count > 0:
        return True

    return False


class MemoryEfficientPrototypeStore:
    def __init__(self, embed_dim, max_protos: Optional[int] = None):
        if max_protos is None:
            max_protos = DSCD_MAX_PROTOS
        self.embed_dim = int(embed_dim)
        self.max_protos = int(max_protos)
        self.centroids: List[torch.Tensor] = []
        self.counts: List[int] = []
        self.creation_time: List[float] = []
        self.distances: List[float] = []
        self.mu = 0.0
        self.tau = 1e-6
        self.alpha = 0.1
        self.labels: Optional[torch.Tensor] = None

    def add_prototype(self, vector: torch.Tensor, current_time: Optional[float] = None, count: int = 1) -> None:
        if current_time is None:
            current_time = time.time()
        v = vector.detach().cpu().clone().float()
        if len(self.centroids) < self.max_protos:
            self.centroids.append(v)
            self.counts.append(int(count))
            self.creation_time.append(float(current_time))
        else:
            min_idx = int(np.argmin(self.counts)) if len(self.counts) > 0 else 0
            self.centroids[min_idx] = v
            self.counts[min_idx] = int(count)
            self.creation_time[min_idx] = float(current_time)

    def update_prototype(self, idx: int, vector: torch.Tensor, eta: float = 0.05, assignment_distance: Optional[float] = None) -> None:
        if idx < 0 or idx >= len(self.centroids):
            self.add_prototype(vector, time.time(), count=1)
            return
        old_centroid = self.centroids[idx]
        new_vector = vector.detach().cpu().float()
        self.centroids[idx] = (1.0 - eta) * old_centroid + eta * new_vector
        self.counts[idx] = int(self.counts[idx]) + 1
        if assignment_distance is not None:
            self.update_rolling_stats(float(assignment_distance))

    def update_rolling_stats(self, d: float) -> None:
        if not self.distances:
            self.mu = float(d)
            self.tau = max(1e-6, float(d) * 0.1)
            self.distances = [float(d)]
            return
        prev_mu = self.mu
        self.mu = (1 - self.alpha) * self.mu + self.alpha * float(d)
        self.tau = (1 - self.alpha) * self.tau + self.alpha * abs(float(d) - prev_mu)
        self.distances.append(float(d))
        if len(self.distances) > 50:
            self.distances.pop(0)

    def get_adaptive_threshold(self, lam: float = 1.0) -> float:
        return float(self.mu + lam * max(self.tau, 1e-4))

    def size(self) -> int:
        return len(self.centroids)

    def ensure_consistency(self) -> None:
        n = len(self.centroids)
        if len(self.counts) != n:
            self.counts = self.counts[:n] if len(self.counts) > n else self.counts + [1] * (n - len(self.counts))
        if len(self.creation_time) != n:
            self.creation_time = self.creation_time[:n] if len(self.creation_time) > n else self.creation_time + [time.time()] * (n - len(self.creation_time))


class MemoryEfficientDSCDOnline(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        tokenizer=None,
        buffer_size: Optional[int] = None,
        max_protos: Optional[int] = None,
        n_min: Optional[int] = None,
        dispersion_threshold: Optional[float] = None,
        language: str = "bn",
        enable_training_clustering: Optional[bool] = None,
        max_clustering_points: Optional[int] = None,
        max_candidates_per_step: int = 2,
        dscd_min_letters: int = 3,
        dscd_min_letter_fraction: float = 0.6,
    ):
        super().__init__()
        if buffer_size is None:
            buffer_size = DSCD_BUFFER_SIZE
        if max_protos is None:
            max_protos = DSCD_MAX_PROTOS
        if n_min is None:
            n_min = DSCD_N_MIN
        if dispersion_threshold is None:
            dispersion_threshold = DSCD_DISPERSION_THRESHOLD
        if max_clustering_points is None:
            max_clustering_points = DSCD_MAX_CLUSTERING_POINTS
        if enable_training_clustering is None:
            enable_training_clustering = DSCD_ENABLE_TRAINING_CLUSTERING

        self.embed_dim = int(embed_dim)
        self.buffer_size = int(buffer_size)
        self.max_protos = int(max_protos)
        self.n_min = int(n_min)
        self.dispersion_threshold = float(dispersion_threshold)
        self.language = language
        self.tokenizer = tokenizer
        self.dscd_min_letters = int(dscd_min_letters)
        self.dscd_min_letter_fraction = float(dscd_min_letter_fraction)

        try:
            if tokenizer is not None and callable(globals().get("get_tokenizer_special_tokens", None)):
                self.special_tokens = globals()["get_tokenizer_special_tokens"](tokenizer)
            else:
                self.special_tokens = set(getattr(tokenizer, 'all_special_tokens', [])) if tokenizer is not None else set()
        except Exception:
            self.special_tokens = set()

        self.dscd_allowed_tokens: Set[str] = set()
        self.dscd_ignored_tokens: Set[str] = set()
        self.dscd_cache_max_size = int(globals().get("DSCD_CACHE_MAX_SIZE", 10000))

        self.prototype_stores: Dict[str, MemoryEfficientPrototypeStore] = {}
        self.buffers: Dict[str, deque] = {}
        self.discovered_log: List[Dict[str, Any]] = []
        self.discovered_homographs: Set[str] = set()

        self.last_periodic_check = 0
        self.cleanup_counter = 0

        self.dispersion_cache: Dict[str, float] = {}
        self.dispersion_last_updated: Dict[str, float] = {}
        self.dispersion_lock = threading.Lock()
        self.clustering_lock = threading.Lock()
        self.buffer_lock = threading.Lock()

        from collections import deque as thread_deque
        self.active_threads = thread_deque(maxlen=100)
        self.thread_lock = threading.Lock()

        self.last_cluster_time: Dict[str, float] = {}
        self.cluster_cooldown_seconds = float(globals().get("CLUSTER_COOLDOWN_SECONDS", 5.0))

        self.enable_training_clustering = bool(enable_training_clustering)
        self.discovery_count = 0
        self.discovery_times: List[float] = []
        self.clustered_tokens: Set[str] = set()

        self.cluster_stats: Dict[str, Dict[str, Any]] = {}

        self.span_head = nn.Sequential(
            nn.Linear(self.embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

        self.sigma_net = nn.Sequential(
            nn.Linear(self.embed_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(16, 1),
        )

        self.gate_w = nn.Parameter(torch.tensor(1.0))
        self.gate_b = nn.Parameter(torch.tensor(0.4))
        self.gamma = nn.Parameter(torch.tensor(0.3))

        self.max_clustering_points = int(max_clustering_points)
        self.max_candidates_per_step = int(max_candidates_per_step)

        try:
            self.homograph_reference_list = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST_BN)
        except Exception:
            self.homograph_reference_list = set()

    def state_dict(self, destination=None, prefix='', keep_vars=False):
        state = super().state_dict(destination, prefix, keep_vars)

        plain_stores = {}
        try:
            with self.clustering_lock:
                for token, store in self.prototype_stores.items():
                    centroids_serial = []
                    for c in getattr(store, 'centroids', []):
                        try:
                            if isinstance(c, torch.Tensor):
                                centroids_serial.append(c.cpu().numpy())
                            else:
                                centroids_serial.append(np.asarray(c, dtype=np.float32))
                        except Exception:
                            try:
                                centroids_serial.append(np.asarray(c))
                            except Exception:
                                centroids_serial.append(None)
                    plain_stores[token] = {
                        'centroids': centroids_serial,
                        'counts': list(getattr(store, 'counts', [])),
                        'creation_time': list(getattr(store, 'creation_time', [])),
                        'mu': float(getattr(store, 'mu', 0.0)),
                        'tau': float(getattr(store, 'tau', 0.0)),
                        'size': int(store.size()) if hasattr(store, 'size') else 0,
                    }
        except Exception:
            plain_stores = {}

        state[prefix + 'prototype_stores_data'] = plain_stores
        state[prefix + 'prototype_stores'] = {k: {
            'centroids_count': len(v.get('centroids', [])) if isinstance(v, dict) else 0,
            'counts': v.get('counts', []) if isinstance(v, dict) else [],
        } for k, v in plain_stores.items()}
        state[prefix + 'discovered_homographs'] = list(self.discovered_homographs)
        return state

    def load_state_dict(self, state_dict, strict=True):
        # Accept both new and legacy formats for prototype stores
        plain_stores = {}
        discovered = []
        if 'prototype_stores_data' in state_dict:
            plain_stores = state_dict.get('prototype_stores_data', {}) or {}
        elif 'prototype_stores' in state_dict:
            legacy = state_dict.get('prototype_stores', {}) or {}
            for token, info in legacy.items():
                centroids = info.get('centroids', info.get('centroids_list', []))
                counts = info.get('counts', info.get('counts_list', []))
                plain_stores[token] = {
                    'centroids': centroids,
                    'counts': counts,
                    'creation_time': info.get('creation_time', []),
                    'mu': info.get('mu', 0.0),
                    'tau': info.get('tau', 0.0),
                    'size': len(centroids) if centroids else 0,
                }
        discovered = state_dict.get('discovered_homographs', state_dict.get('discovered', [])) or []

        # Load model params first
        super().load_state_dict(state_dict, strict=strict)

        # Reconstruct prototype stores
        reconstructed: Dict[str, MemoryEfficientPrototypeStore] = {}
        for token, store_dict in plain_stores.items():
            try:
                store = MemoryEfficientPrototypeStore(embed_dim=self.embed_dim, max_protos=self.max_protos)
                centroids_data = store_dict.get('centroids', [])
                store.centroids = []
                for c in centroids_data:
                    try:
                        if isinstance(c, torch.Tensor):
                            store.centroids.append(c.cpu())
                        elif isinstance(c, np.ndarray):
                            store.centroids.append(torch.from_numpy(c).float())
                        elif c is None:
                            continue
                        else:
                            store.centroids.append(torch.tensor(c, dtype=torch.float32))
                    except Exception:
                        try:
                            store.centroids.append(torch.tensor(np.asarray(c, dtype=np.float32)))
                        except Exception:
                            continue
                store.counts = list(store_dict.get('counts', []))
                store.creation_time = list(store_dict.get('creation_time', []))
                store.mu = float(store_dict.get('mu', 0.0))
                store.tau = float(store_dict.get('tau', 0.0))
                store.ensure_consistency()
                reconstructed[token] = store
            except Exception:
                continue

        with self.clustering_lock:
            self.prototype_stores = reconstructed
            self.discovered_homographs = set(discovered)

        if DEBUG_DISCOVERY:
            try:
                total_protos = sum(s.size() for s in self.prototype_stores.values())
                print(f"[DSCD] Loaded {len(self.prototype_stores)} tokens, {total_protos} prototypes")
            except Exception:
                pass

    @staticmethod
    def clean_token(token):
        return clean_token_for_dscd(str(token))

    def is_valid_multi_sense(self, token):
        with self.clustering_lock:
            if token not in self.prototype_stores:
                return False
            store = self.prototype_stores[token]
            total_occurrences = sum(store.counts) if hasattr(store, 'counts') else 0
            min_per_proto = min(store.counts) if hasattr(store, 'counts') and store.counts else 0
            return store.size() >= 2 and total_occurrences >= 10 and min_per_proto >= 2

    def is_multi_sense_store(self, store: MemoryEfficientPrototypeStore) -> bool:
        k = store.size()
        if k < 2:
            return False

        counts = store.counts if store.counts else [1] * k
        strong = sum(1 for c in counts if c >= max(2, self.n_min // 2))
        if strong < 2:
            return False

        try:
            cents = []
            for c in store.centroids:
                if isinstance(c, torch.Tensor):
                    cents.append(c.cpu().numpy())
                else:
                    cents.append(np.asarray(c, dtype=np.float32))

            if len(cents) < 2:
                return False

            cents = np.stack(cents, axis=0)
            dists = np.linalg.norm(cents[:, None, :] - cents[None, :, :], axis=-1)
            tri = dists[np.triu_indices(len(cents), k=1)]
            if tri.size == 0:
                return False

            min_dist = float(tri.min())
            base = max(store.tau, 1e-3)
            return min_dist > base * float(globals().get("DSCD_NEWSENSE_LAMBDA", DSCD_NEWSENSE_LAMBDA))
        except Exception:
            return True

    def discover_homographs_for_tokens(
        self,
        token_names: List[str],
        min_cluster_samples: int,
        dispersion_threshold: float,
        global_step: int,
    ) -> int:
        discovered_in_run: List[str] = []

        for token in token_names:
            try:
                if is_punctuation_only(token):
                    continue

                success = self.cluster_buffer_to_prototypes_hierarchical(token)

                if success:
                    with self.clustering_lock:
                        store = self.prototype_stores.get(token)
                        if store and store.size() >= 2:
                            clean_token = normalize_token_key(token)
                            self.discovered_homographs.add(clean_token)
                            discovered_in_run.append(clean_token)
            except Exception:
                continue

        try:
            self.discovered_log.append({
                'timestamp': time.time(),
                'global_step': global_step,
                'candidates_processed': len(token_names),
                'discovered_count': len(discovered_in_run),
                'homographs': discovered_in_run,
                'total_discovered': len(self.discovered_homographs),
            })
        except Exception:
            pass

        return len(discovered_in_run)

    def discover_homographs(
        self,
        min_cluster_samples: Optional[int] = None,
        dispersion_threshold: Optional[float] = None,
        max_candidates: int = 500,
    ) -> int:
        if min_cluster_samples is None:
            min_cluster_samples = self.n_min
        if dispersion_threshold is None:
            dispersion_threshold = self.dispersion_threshold

        candidates: List[Tuple[str, float, int, float]] = []

        with self.buffer_lock:
            for token, buffer in self.buffers.items():
                if is_punctuation_only(token):
                    continue

                buffer_size = len(buffer)
                if buffer_size >= max(min_cluster_samples + 2, 10):
                    clean_token = clean_token_for_dscd(token)

                    if clean_token in HOMOGRAPH_REFERENCE_LIST_BN:
                        dispersion = max(self.get_dispersion(token), dispersion_threshold * 1.15)
                    else:
                        dispersion = self.get_dispersion(token)

                    if dispersion >= dispersion_threshold:
                        rank_score = dispersion * buffer_size
                        candidates.append((token, rank_score, buffer_size, dispersion))

        if not candidates:
            return 0

        candidates.sort(key=lambda x: x[1], reverse=True)
        candidates = candidates[:max_candidates]

        discovered: List[str] = []

        for token, score, buf_size, disp in candidates:
            try:
                with self.clustering_lock:
                    success = self.cluster_buffer_to_prototypes_hierarchical(token)
                    if success:
                        store = self.prototype_stores.get(token)
                        if store and store.size() >= 2:
                            clean_token = normalize_token_key(token)
                            self.discovered_homographs.add(clean_token)
                            discovered.append(clean_token)
            except Exception:
                continue

        try:
            self.discovered_log.append({
                'timestamp': time.time(),
                'candidates': len(candidates),
                'discovered': len(discovered),
                'homographs': discovered[:20],
            })
        except Exception:
            pass

        return len(discovered)

    def get_dispersion(self, token_type: str) -> float:
        with self.dispersion_lock:
            if token_type in self.dispersion_cache:
                try:
                    last_update = self.dispersion_last_updated.get(token_type, 0.0)
                    if time.time() - last_update < 3600:
                        return self.dispersion_cache[token_type]
                except Exception:
                    pass

        with self.buffer_lock:
            if token_type not in self.buffers:
                return 0.0

            buf_len = len(self.buffers[token_type])
            if buf_len < 2:
                return 0.05 if buf_len == 1 else 0.0

            try:
                embeddings: List[np.ndarray] = []
                for emb in self.buffers[token_type]:
                    try:
                        if isinstance(emb, torch.Tensor):
                            embeddings.append(emb.cpu().numpy())
                        else:
                            embeddings.append(np.asarray(emb, dtype=np.float32))
                    except Exception:
                        continue

                if len(embeddings) < 2:
                    return 0.05 if len(embeddings) == 1 else 0.0

                embeddings_np = np.stack(embeddings, axis=0)
                centroid = embeddings_np.mean(axis=0)
                distances = np.linalg.norm(embeddings_np - centroid[None, :], axis=1)
                dispersion = float(distances.std())

                with self.dispersion_lock:
                    self.dispersion_cache[token_type] = dispersion
                    self.dispersion_last_updated[token_type] = time.time()

                return dispersion
            except Exception:
                return 0.0

    def validate_prototypes(
        self,
        homograph_list: Optional[List[str]] = None,
        cluster_missing: bool = True,
    ) -> Dict[str, Any]:
        if homograph_list is None:
            try:
                homograph_list = list(HOMOGRAPH_REFERENCE_LIST_BN)
            except Exception:
                homograph_list = ['কল', 'পাতা', 'ফল', 'মান']

        print("=" * 80)
        print("DSCD-VALIDATION: Prototype Quality Check")
        print("=" * 80)

        validation_results: Dict[str, Any] = {
            'total_tokens': 0,
            'total_prototypes': 0,
            'multi_sense_tokens': 0,
            'homographs_found': 0,
            'homographs_missing': [],
            'avg_prototypes_per_token': 0.0,
            'avg_samples_per_prototype': 0.0,
            'quality_score': 0.0,
        }

        with self.clustering_lock:
            validation_results['total_tokens'] = len(self.prototype_stores)
            total_samples = 0
            for token, store in self.prototype_stores.items():
                num_protos = len(store.centroids)
                validation_results['total_prototypes'] += num_protos
                if self.is_multi_sense_store(store):
                    validation_results['multi_sense_tokens'] += 1
                try:
                    total_samples += sum(store.counts)
                except Exception:
                    pass

        if validation_results['total_tokens'] > 0:
            validation_results['avg_prototypes_per_token'] = validation_results['total_prototypes'] / validation_results['total_tokens']

        if validation_results['total_prototypes'] > 0:
            validation_results['avg_samples_per_prototype'] = total_samples / validation_results['total_prototypes']

        print("VALIDATION: Reference Homograph Coverage")
        print("-" * 80)

        missing_tokens_to_cluster: List[str] = []

        for homograph in homograph_list:
            clean_h = clean_token_for_dscd(homograph)
            found = False
            found_key = None
            found_protos = 0

            with self.clustering_lock:
                for key in self.prototype_stores.keys():
                    clean_key = clean_token_for_dscd(str(key))
                    if clean_key == clean_h:
                        found = True
                        found_key = key
                        found_protos = len(self.prototype_stores[key].centroids)
                        break

            if found and self.is_multi_sense_store(self.prototype_stores[found_key]):
                validation_results['homographs_found'] += 1
                try:
                    counts = self.prototype_stores[found_key].counts
                    print(f"  ✓ {homograph} - {found_protos} prototypes (counts={counts})")
                except Exception:
                    print(f"  ✓ {homograph} - {found_protos} prototypes")
            elif found and found_protos == 1:
                validation_results['homographs_missing'].append(homograph)
                print(f"  ⚠ {homograph} - Only 1 prototype")
                if cluster_missing:
                    missing_tokens_to_cluster.append(found_key)
            else:
                validation_results['homographs_missing'].append(homograph)
                print(f"  ✗ {homograph} - NOT FOUND")
                if cluster_missing:
                    with self.buffer_lock:
                        for buf_key in self.buffers.keys():
                            clean_buf_key = clean_token_for_dscd(str(buf_key))
                            if clean_buf_key == clean_h:
                                if len(self.buffers[buf_key]) >= max(self.n_min + 2, 10):
                                    missing_tokens_to_cluster.append(buf_key)
                                break

        if cluster_missing and missing_tokens_to_cluster:
            for token in missing_tokens_to_cluster:
                try:
                    with self.clustering_lock:
                        self.cluster_buffer_to_prototypes_hierarchical(token)
                        if token in self.prototype_stores and self.is_multi_sense_store(self.prototype_stores[token]):
                            print(f"  ✓ Successfully clustered: {token}")
                except Exception as e:
                    print(f"  ✗ Failed to cluster {token}: {e}")

        homograph_coverage = validation_results['homographs_found'] / len(homograph_list) if homograph_list else 0.0
        multi_sense_ratio = validation_results['multi_sense_tokens'] / validation_results['total_tokens'] if validation_results['total_tokens'] > 0 else 0.0
        validation_results['quality_score'] = (homograph_coverage * 0.6) + (multi_sense_ratio * 0.4)

        print("-" * 80)
        print("VALIDATION: Summary")
        print(f"  - Total tokens: {validation_results['total_tokens']}")
        print(f"  - Total prototypes: {validation_results['total_prototypes']}")
        print(f"  - Multi-sense tokens: {validation_results['multi_sense_tokens']}")
        print(f"  - Reference found: {validation_results['homographs_found']}/{len(homograph_list)}")
        print(f"  - Quality Score: {validation_results['quality_score']*100:.2f}%")
        print("=" * 80)

        return validation_results

    def should_track_token(self, token_text: str) -> bool:
        if not token_text or not isinstance(token_text, str):
            return False

        if len(self.dscd_allowed_tokens) > self.dscd_cache_max_size:
            self.dscd_allowed_tokens.clear()
        if len(self.dscd_ignored_tokens) > self.dscd_cache_max_size:
            self.dscd_ignored_tokens.clear()

        if token_text in self.dscd_allowed_tokens:
            return True
        if token_text in self.dscd_ignored_tokens:
            return False

        if not getattr(self, 'training', False):
            with self.clustering_lock:
                if token_text in self.prototype_stores:
                    self.dscd_allowed_tokens.add(token_text)
                    return True
                clean = clean_token_for_dscd(token_text)
                if clean and clean in self.prototype_stores:
                    self.dscd_allowed_tokens.add(token_text)
                    return True

        if token_text in self.special_tokens:
            self.dscd_ignored_tokens.add(token_text)
            return False

        if is_punctuation_only(token_text):
            self.dscd_ignored_tokens.add(token_text)
            return False

        clean = clean_token_for_dscd(token_text)
        if not clean:
            self.dscd_ignored_tokens.add(token_text)
            return False

        if len(clean) < self.dscd_min_letters:
            self.dscd_ignored_tokens.add(token_text)
            return False

        if not any(c.isalpha() for c in clean):
            self.dscd_ignored_tokens.add(token_text)
            return False

        if clean.isdigit():
            self.dscd_ignored_tokens.add(token_text)
            return False

        try:
            indic_range_1 = any('\u0900' <= c <= '\u0DFF' for c in clean)
            indic_range_2 = any('\u0980' <= c <= '\u09FF' for c in clean)
            has_indic = indic_range_1 or indic_range_2

            if has_indic:
                if len(clean) >= self.dscd_min_letters:
                    self.dscd_allowed_tokens.add(token_text)
                    return True
                else:
                    self.dscd_ignored_tokens.add(token_text)
                    return False
        except Exception:
            pass

        if is_word_token(
            clean,
            min_letters=self.dscd_min_letters,
            min_letter_fraction=self.dscd_min_letter_fraction,
        ):
            self.dscd_allowed_tokens.add(token_text)
            return True

        self.dscd_ignored_tokens.add(token_text)
        return False

    def canonical_token_key(
        self,
        raw_token: str,
        token_word_map: Optional[Dict[int, Optional[str]]],
        idx: int,
    ) -> Optional[str]:
        canonical: Optional[str] = None

        try:
            if token_word_map and isinstance(token_word_map, dict) and idx in token_word_map and token_word_map[idx]:
                word = str(token_word_map[idx]).strip()
                canonical = clean_token_for_dscd(word)
                if canonical and len(canonical) >= self.dscd_min_letters:
                    indic_range_1 = any('\u0900' <= c <= '\u0DFF' for c in canonical)
                    indic_range_2 = any('\u0980' <= c <= '\u09FF' for c in canonical)
                    has_indic = indic_range_1 or indic_range_2
                    if has_indic:
                        if not is_indic_subword_fragment(canonical):
                            return canonical
        except Exception:
            pass

        canonical = clean_token_for_dscd(raw_token)

        if not canonical or len(canonical) < self.dscd_min_letters:
            return None

        indic_range_1 = any('\u0900' <= c <= '\u0DFF' for c in canonical)
        indic_range_2 = any('\u0980' <= c <= '\u09FF' for c in canonical)
        has_indic = indic_range_1 or indic_range_2
        if not has_indic:
            return None

        if is_indic_subword_fragment(canonical):
            return None

        return canonical

    def cleanup_threads(self) -> None:
        try:
            with self.thread_lock:
                alive = [th for th in list(self.active_threads) if th.is_alive()]
                self.active_threads.clear()
                self.active_threads.extend(alive)
        except Exception:
            pass

    def cleanup_memory(self) -> None:
        try:
            for token_type, buffer in list(self.buffers.items()):
                if len(buffer) > int(self.buffer_size * 1.5):
                    while len(buffer) > self.buffer_size:
                        buffer.popleft()

            try:
                now = time.time()
                expired = [k for k, v in self.dispersion_last_updated.items() if now - v > 3600]
                for k in expired:
                    self.dispersion_cache.pop(k, None)
                    self.dispersion_last_updated.pop(k, None)
            except Exception:
                pass

            if gc.isenabled():
                gc.collect()
        except Exception:
            pass

    def forward(
        self,
        token_embeddings=None,
        token_types=None,
        train_mode: bool = True,
        token_word_map=None,
        h_all=None,
        input_ids=None,
        attention_mask=None,
    ):
        if token_embeddings is None and h_all is not None:
            token_embeddings = h_all

        if token_embeddings is None:
            raise ValueError("MemoryEfficientDSCDOnline.forward requires token_embeddings or h_all")

        if input_ids is not None and token_types is None:
            batch_size, seq_len = input_ids.shape
            token_types = []
            for b in range(batch_size):
                if self.tokenizer is not None:
                    try:
                        token_types.append(
                            self.tokenizer.convert_ids_to_tokens(input_ids[b].tolist())
                        )
                    except Exception:
                        token_types.append([f"tok{i}" for i in range(seq_len)])
                else:
                    token_types.append([f"tok{i}" for i in range(seq_len)])

        self.cleanup_counter += 1
        if self.cleanup_counter % 50 == 0:
            self.cleanup_counter = 0
            self.cleanup_memory()
            self.cleanup_threads()

        device = token_embeddings.device
        batch_size = int(token_embeddings.size(0))
        seq_len = int(token_embeddings.size(1))

        all_outputs: Dict[str, List[Any]] = {
            'proto_assignments': [],
            'proto_probs': [],
            'uncertainties': [],
            'span_preds': [],
            'gates': [],
            'h_augmented': [],
        }

        for b in range(batch_size):
            word_map = token_word_map[b] if token_word_map and len(token_word_map) > b else None

            batch_outputs = self.process_sequence(
                token_embeddings[b],
                token_types[b] if token_types and len(token_types) > b else [f"tok{i}" for i in range(seq_len)],
                device,
                word_map=word_map,
                train_mode=train_mode,
            )

            for k in all_outputs:
                all_outputs[k].append(batch_outputs[k])

        try:
            h_aug_list: List[torch.Tensor] = []
            max_seq_len = seq_len

            for b in range(batch_size):
                h_batch_list = all_outputs['h_augmented'][b]

                if len(h_batch_list) > 0 and isinstance(h_batch_list[0], torch.Tensor):
                    h_batch = torch.stack(h_batch_list, dim=0)

                    if h_batch.size(0) < max_seq_len:
                        pad = max_seq_len - h_batch.size(0)
                        h_batch = F.pad(h_batch, (0, 0, 0, pad), value=0)
                    elif h_batch.size(0) > max_seq_len:
                        h_batch = h_batch[:max_seq_len]
                else:
                    h_batch = token_embeddings[b].clone()

                h_aug_list.append(h_batch)

            all_outputs['h_augmented'] = torch.stack(h_aug_list, dim=0)
        except Exception:
            all_outputs['h_augmented'] = token_embeddings

        try:
            proto_assign_tensor = []
            for row in all_outputs['proto_assignments']:
                try:
                    stacked = torch.stack(
                        [x if isinstance(x, torch.Tensor) else torch.tensor(x) for x in row],
                        dim=0,
                    )
                    proto_assign_tensor.append(stacked)
                except Exception:
                    proto_assign_tensor.append(
                        torch.tensor(
                            [int(x) if not isinstance(x, torch.Tensor) else int(x.item()) for x in row],
                            dtype=torch.long,
                        )
                    )
            all_outputs['proto_assignments'] = proto_assign_tensor
        except Exception:
            pass

        return all_outputs

    def process_sequence(
        self,
        token_embeddings: torch.Tensor,
        token_types: List[Any],
        device: torch.device,
        word_map: Optional[Dict[int, Optional[str]]] = None,
        train_mode: bool = True,
    ) -> Dict[str, List[Any]]:
        seq_len = int(token_embeddings.size(0))

        outputs: Dict[str, List[Any]] = {
            'proto_assignments': [],
            'proto_probs': [],
            'uncertainties': [],
            'span_preds': [],
            'gates': [],
            'h_augmented': [],
        }

        for j in range(seq_len):
            raw_tok = token_types[j] if j < len(token_types) else f"tok{j}"
            if not isinstance(raw_tok, str):
                raw_tok = str(raw_tok) if raw_tok is not None else f"tok{j}"

            token_key = self.canonical_token_key(raw_tok, word_map, j)
            h_j = token_embeddings[j]

            if not token_key:
                outputs['proto_assignments'].append(torch.tensor(-1))
                outputs['proto_probs'].append([])
                outputs['uncertainties'].append(0.0)
                outputs['span_preds'].append(0.0)
                outputs['gates'].append(0.0)
                outputs['h_augmented'].append(h_j)
                continue

            if not self.should_track_token(token_key):
                outputs['proto_assignments'].append(torch.tensor(-1))
                outputs['proto_probs'].append([])
                outputs['uncertainties'].append(0.0)
                outputs['span_preds'].append(0.0)
                outputs['gates'].append(0.0)
                outputs['h_augmented'].append(h_j)
                continue

            with self.buffer_lock:
                if token_key not in self.buffers:
                    self.buffers[token_key] = deque(maxlen=self.buffer_size)
                    self.prototype_stores[token_key] = MemoryEfficientPrototypeStore(
                        self.embed_dim, self.max_protos
                    )

                try:
                    self.buffers[token_key].append(h_j.detach().clone().cpu())
                except Exception:
                    try:
                        self.buffers[token_key].append(h_j.cpu())
                    except Exception:
                        pass

            buffer_len = len(self.buffers[token_key])

            try:
                if self.enable_training_clustering and buffer_len >= max(self.n_min + 2, 10):
                    now = time.time()
                    last_t = self.last_cluster_time.get(token_key, 0.0)

                    if now - last_t >= self.cluster_cooldown_seconds:
                        self.last_cluster_time[token_key] = now

                        def bg_cluster(tok: str = token_key) -> None:
                            try:
                                with self.clustering_lock:
                                    self.cluster_buffer_to_prototypes_hierarchical(tok)
                            except Exception:
                                pass

                        th = threading.Thread(target=bg_cluster, daemon=True)
                        th.start()
                        with self.thread_lock:
                            self.active_threads.append(th)
            except Exception:
                pass

            with self.clustering_lock:
                store = self.prototype_stores.get(token_key, MemoryEfficientPrototypeStore(self.embed_dim, self.max_protos))

            centroids_snapshot: Optional[List[torch.Tensor]] = None
            with self.clustering_lock:
                try:
                    if hasattr(store, 'centroids') and len(store.centroids) > 0:
                        centroids_snapshot = []
                        for c in store.centroids:
                            try:
                                if isinstance(c, torch.Tensor):
                                    centroids_snapshot.append(c.clone().cpu())
                                else:
                                    centroids_snapshot.append(
                                        torch.from_numpy(
                                            np.asarray(c, dtype=np.float32)
                                        ).cpu()
                                    )
                            except Exception:
                                continue
                        if not centroids_snapshot:
                            centroids_snapshot = None
                except Exception:
                    centroids_snapshot = None

            assignment = -1
            prob_list: List[float] = []
            uncertainty = 0.0
            span_pred = 0.0
            gate_val = 0.0
            h_aug = h_j

            if centroids_snapshot and len(centroids_snapshot) >= 1:
                try:
                    try:
                        h_cpu = h_j.detach().cpu().numpy()
                    except Exception:
                        h_cpu = h_j.cpu().numpy()

                    try:
                        cents_np = np.stack([c.numpy() for c in centroids_snapshot], axis=0)
                    except Exception:
                        cents_np = np.stack([np.asarray(c, dtype=np.float32) for c in centroids_snapshot], axis=0)

                    dists_np = np.linalg.norm(cents_np - h_cpu[None, :], axis=1)

                    if dists_np.size > 0:
                        min_dist = float(dists_np.min())
                        min_idx = int(np.argmin(dists_np))

                        if len(centroids_snapshot) >= 2:
                            mean_dist = float(np.mean(dists_np))
                            std_dist = float(np.std(dists_np))
                            span_pred = float(np.clip(std_dist / (mean_dist + 1e-6), 0.0, 1.0))
                        else:
                            span_pred = float(np.clip((min_dist - store.mu) / (1e-3), 0.0, 1.0))

                        base_threshold = max(store.tau, 1e-3) if store.size() > 0 else 0.3
                        uncertainty_dist = float(np.clip(min_dist / (base_threshold * 2), 0.0, 1.0))

                        if len(centroids_snapshot) >= 2:
                            precisions = 1.0 / (dists_np**2 + 1e-6)
                            gate_weights = precisions / (np.sum(precisions) + 1e-6)
                            gate_val = float(np.max(gate_weights))
                        else:
                            gate_val = float(np.clip(1.0 - (min_dist - store.mu) / (1e-3), 0.0, 1.0))

                        if store.size() < self.max_protos and min_dist > store.get_adaptive_threshold(float(globals().get("DSCD_NEWSENSE_LAMBDA", DSCD_NEWSENSE_LAMBDA))):
                            store.add_prototype(h_j, time.time(), count=1)
                            assignment = store.size() - 1
                            centroids_snapshot.append(h_j.cpu())
                            cents_np = np.vstack([cents_np, h_cpu[None, :]])
                        else:
                            assignment = min_idx
                            try:
                                store.update_rolling_stats(min_dist)
                            except Exception:
                                pass

                        try:
                            dist_tensor = torch.from_numpy(dists_np).float().to(device)
                            probs_tensor = F.softmax(-dist_tensor, dim=0)
                            prob_list = probs_tensor.tolist()

                            entropy = -torch.sum(probs_tensor * torch.log(probs_tensor + 1e-10))
                            max_entropy = np.log(len(dists_np))
                            uncertainty_entropy = float(entropy.item() / max_entropy) if max_entropy > 0 else 0.0
                        except Exception:
                            exps = np.exp(-dists_np - np.max(-dists_np)) if dists_np.size > 0 else np.array([])
                            if exps.size > 0:
                                probs = exps / (exps.sum() + 1e-12)
                                prob_list = probs.tolist()
                                entropy_val = -np.sum(probs * np.log(probs + 1e-10))
                                max_entropy = np.log(len(dists_np))
                                uncertainty_entropy = float(entropy_val / max_entropy) if max_entropy > 0 else 0.0
                            else:
                                prob_list = []
                                uncertainty_entropy = 0.0

                        if len(centroids_snapshot) >= 2:
                            uncertainty = 0.4 * uncertainty_dist + 0.6 * uncertainty_entropy
                        else:
                            uncertainty = uncertainty_dist

                        if gate_val > 0.3 and 0 <= assignment < len(centroids_snapshot):
                            try:
                                centroid_t = centroids_snapshot[assignment]

                                if device != torch.device('cpu'):
                                    try:
                                        centroid_t = centroid_t.to(device)
                                    except Exception:
                                        pass

                                blend_weight = 0.3 if gate_val > 0.7 else 0.15
                                h_aug = h_j + blend_weight * (centroid_t - h_j)
                            except Exception:
                                h_aug = h_j

                except Exception as e:
                    if DEBUG_DISCOVERY:
                        print(f"[DSCD] Assignment error for {token_key}: {str(e)[:200]}")

            outputs['proto_assignments'].append(torch.tensor(assignment))
            outputs['proto_probs'].append(prob_list)
            outputs['uncertainties'].append(uncertainty)
            outputs['span_preds'].append(span_pred)
            outputs['gates'].append(gate_val)
            outputs['h_augmented'].append(h_aug)

        try:
            if not train_mode and len(self.prototype_stores) > 0 and VERBOSE_LOGGING:
                if self.last_periodic_check % PRINT_INTERVAL == 0:
                    self.print_clusters_summary()
                self.last_periodic_check += 1
        except Exception:
            pass

        return outputs

    def print_clusters_summary(self) -> None:
        try:
            items: List[Tuple[str, int, int, float, float, int]] = []

            with self.clustering_lock:
                for token, store in self.prototype_stores.items():
                    if is_punctuation_only(token):
                        continue

                    try:
                        proto_sample_count = sum(getattr(store, 'counts', []) or [])
                    except Exception:
                        proto_sample_count = 0

                    buffer_len = len(self.buffers.get(token, [])) if token in self.buffers else 0
                    total_count = proto_sample_count if proto_sample_count > 0 else buffer_len
                    protos = store.size()
                    mu = getattr(store, 'mu', 0.0)
                    tau = getattr(store, 'tau', 0.0)

                    items.append((token, total_count, protos, mu, tau, buffer_len))

            items.sort(key=lambda x: x[1], reverse=True)
            top_5 = items[:5]

            if VERBOSE_LOGGING:
                print("\n[CLUSTER] Top 5 clusters:")
                print("-" * 90)
                print(f"{'Rank':<6} {'Token':<14} {'Count':<12} {'Protos':<10} {'Mu':<14} {'Tau':<12}")
                print("-" * 90)
                for rank, (tok, cnt, prot, mu, tau, buf_len) in enumerate(top_5, 1):
                    tok_str = str(tok)[:14]
                    print(f"{rank:<6} {tok_str:<14} {cnt:<12} {prot:<10} {mu:<14.6f} {tau:<12.6f}")
                print("-" * 90)
        except Exception:
            pass

    def _simple_graph_cluster(self, embeddings: np.ndarray, threshold: float) -> np.ndarray:
        if embeddings is None or embeddings.shape[0] == 0:
            return np.array([], dtype=int)
        try:
            dif = embeddings[:, None, :] - embeddings[None, :, :]
            dists = np.linalg.norm(dif, axis=-1)
            adj = (dists <= threshold).astype(np.int32)
            n = adj.shape[0]
            parent = list(range(n))
            def find(x):
                while parent[x] != x:
                    parent[x] = parent[parent[x]]
                    x = parent[x]
                return x
            def union(a, b):
                ra, rb = find(a), find(b)
                if ra != rb:
                    parent[rb] = ra
            for i in range(n):
                for j in range(i+1, n):
                    if adj[i, j]:
                        union(i, j)
            comps = {}
            labels = np.zeros(n, dtype=int)
            for i in range(n):
                r = find(i)
                if r not in comps:
                    comps[r] = len(comps)
                labels[i] = comps[r]
            return labels
        except Exception:
            return np.zeros(embeddings.shape[0], dtype=int)

    def cluster_buffer_to_prototypes_hierarchical(self, token_type: str) -> bool:
        try:
            if is_punctuation_only(token_type):
                if DEBUG_DISCOVERY:
                    print(f"[DSCD-CLUSTER] Skipping punctuation token: {token_type}")
                return False

            if not self.should_track_token(token_type):
                if DEBUG_DISCOVERY:
                    print(f"[DSCD-CLUSTER] Skipping non-word token: {token_type}")
                return False

            with self.buffer_lock:
                if token_type not in self.buffers:
                    return False
                buf_snapshot = [e.clone() if isinstance(e, torch.Tensor) else e for e in self.buffers[token_type]]

            if len(buf_snapshot) < max(self.n_min + 2, 10):
                if DEBUG_DISCOVERY:
                    print(f"[DSCD-CLUSTER] {token_type}: buffer={len(buf_snapshot)} < min={max(self.n_min + 2, 10)}")
                return False

            emb_list: List[np.ndarray] = []
            for e in buf_snapshot:
                try:
                    if isinstance(e, torch.Tensor):
                        try:
                            emb_list.append(e.numpy())
                        except Exception:
                            emb_list.append(e.cpu().numpy())
                    else:
                        emb_list.append(np.asarray(e, dtype=np.float32))
                except Exception:
                    continue

            if len(emb_list) == 0:
                return False

            if len(emb_list) > self.max_clustering_points:
                idxs = np.random.choice(len(emb_list), size=self.max_clustering_points, replace=False)
                embeddings = np.stack([emb_list[i] for i in idxs], axis=0)
            else:
                embeddings = np.stack(emb_list, axis=0)

            if embeddings.shape[0] < 2:
                return False

            norms = np.linalg.norm(embeddings, axis=1)
            if np.all(norms < 1e-6):
                if DEBUG_DISCOVERY:
                    print(f"[DSCD-CLUSTER] {token_type}: all zero vectors, skipping")
                return False

            if DEBUG_DISCOVERY:
                print(
                    f"[DSCD-CLUSTER] {token_type}: buf={len(buf_snapshot)} "
                    f"sampled={embeddings.shape[0]} mean_norm={norms.mean():.4f}"
                )

            # ensure store exists
            with self.clustering_lock:
                store = self.prototype_stores.get(token_type)
                if store is None:
                    store = MemoryEfficientPrototypeStore(self.embed_dim, self.max_protos)
                    self.prototype_stores[token_type] = store

            protos_added = 0
            new_centroids: List[torch.Tensor] = []
            new_counts: List[int] = []
            new_times: List[float] = []

            # hierarchical via scipy
            if HAS_CLUSTERING and scipy_pdist is not None and linkage is not None and fcluster is not None:
                try:
                    condensed = scipy_pdist(embeddings, metric='euclidean')
                    if condensed.size > 0:
                        Z = linkage(condensed, method='average')
                        max_dist = float(np.max(condensed)) if condensed.size > 0 else 1.0
                        relative_threshold = float(self.dispersion_threshold)
                        absolute_threshold = relative_threshold * max_dist
                        clusters = fcluster(Z, t=absolute_threshold, criterion='distance') - 1

                        if clusters.size > 0:
                            max_c = int(clusters.max())
                            for c_id in range(max_c + 1):
                                mask = (clusters == c_id)
                                cluster_size = int(mask.sum())
                                if cluster_size >= self.n_min:
                                    centroid = embeddings[mask].mean(axis=0).astype(np.float32)
                                    centroid_tensor = torch.from_numpy(centroid).float()
                                    new_centroids.append(centroid_tensor)
                                    new_counts.append(cluster_size)
                                    new_times.append(time.time())
                                    protos_added += 1

                            if len(new_centroids) > self.max_protos:
                                sorted_indices = np.argsort(new_counts)[-self.max_protos:]
                                new_centroids = [new_centroids[i] for i in sorted_indices]
                                new_counts = [new_counts[i] for i in sorted_indices]
                                new_times = [new_times[i] for i in sorted_indices]
                                protos_added = len(new_centroids)

                            with self.clustering_lock:
                                store.centroids = new_centroids
                                store.counts = new_counts
                                store.creation_time = new_times
                                try:
                                    store.labels = torch.tensor(clusters)
                                except Exception:
                                    store.labels = None

                            if DEBUG_DISCOVERY and protos_added > 0:
                                print(f"[DSCD-CLUSTER] Hierarchical created {protos_added} prototypes for {token_type}")
                except Exception:
                    if DEBUG_DISCOVERY:
                        print(f"[DSCD-CLUSTER] Hierarchical clustering failed for {token_type}, falling back")

            # KMeans fallback
            if protos_added == 0 and HAS_KMEANS and KMeans is not None:
                try:
                    min_k = 1
                    max_k = min(self.max_protos, max(1, len(embeddings) // max(1, self.n_min)))
                    if max_k < min_k:
                        max_k = min_k

                    if len(embeddings) >= 20:
                        k_guess = min(max_k, max(2, int(np.sqrt(len(embeddings)) / 2)))
                    elif len(embeddings) >= 10:
                        k_guess = min(max_k, 2)
                    else:
                        k_guess = 1

                    k_guess = max(min_k, min(k_guess, len(embeddings)))

                    if k_guess >= 1 and len(embeddings) >= k_guess:
                        km = KMeans(n_clusters=k_guess, random_state=0, n_init=10).fit(embeddings)
                        labels = km.labels_

                        new_centroids = []
                        new_counts = []
                        new_times = []

                        for c in range(k_guess):
                            mask = (labels == c)
                            cluster_size = int(mask.sum())
                            if cluster_size >= self.n_min:
                                centroid = embeddings[mask].mean(axis=0).astype(np.float32)
                                centroid_tensor = torch.from_numpy(centroid).float()
                                new_centroids.append(centroid_tensor)
                                new_counts.append(cluster_size)
                                new_times.append(time.time())
                                protos_added += 1

                        if new_centroids:
                            with self.clustering_lock:
                                store.centroids = new_centroids
                                store.counts = new_counts
                                store.creation_time = new_times
                                try:
                                    store.labels = torch.tensor(labels)
                                except Exception:
                                    store.labels = None

                            if DEBUG_DISCOVERY and protos_added > 0:
                                print(f"[DSCD-CLUSTER] KMeans created {protos_added} prototypes for {token_type}")
                except Exception:
                    if DEBUG_DISCOVERY:
                        print(f"[DSCD-CLUSTER] KMeans failed for {token_type}, falling back")

            # Graph-cluster fallback
            if protos_added == 0:
                try:
                    try:
                        dif = embeddings[:, None, :] - embeddings[None, :, :]
                        dists = np.linalg.norm(dif, axis=-1)
                        max_d = float(np.max(dists))
                    except Exception:
                        max_d = float(np.max(np.linalg.norm(embeddings - embeddings.mean(axis=0), axis=1)))
                    threshold = float(self.dispersion_threshold) * (max_d if max_d > 0 else 1.0)
                    if threshold <= 0:
                        threshold = float(self.dispersion_threshold) + 1e-3
                    labels = self._simple_graph_cluster(embeddings, threshold)
                    unique_labels = np.unique(labels)
                    new_centroids = []
                    new_counts = []
                    new_times = []
                    for lbl in unique_labels:
                        mask = (labels == lbl)
                        cluster_size = int(mask.sum())
                        if cluster_size >= self.n_min:
                            centroid = embeddings[mask].mean(axis=0).astype(np.float32)
                            centroid_tensor = torch.from_numpy(centroid).float()
                            new_centroids.append(centroid_tensor)
                            new_counts.append(cluster_size)
                            new_times.append(time.time())
                            protos_added += 1

                    if new_centroids:
                        if len(new_centroids) > self.max_protos:
                            sorted_indices = np.argsort(new_counts)[-self.max_protos:]
                            new_centroids = [new_centroids[i] for i in sorted_indices]
                            new_counts = [new_counts[i] for i in sorted_indices]
                            new_times = [new_times[i] for i in sorted_indices]
                        with self.clustering_lock:
                            store.centroids = new_centroids
                            store.counts = new_counts
                            store.creation_time = new_times
                            try:
                                store.labels = torch.tensor(labels)
                            except Exception:
                                store.labels = None

                        if DEBUG_DISCOVERY and protos_added > 0:
                            print(f"[DSCD-CLUSTER] Graph-cluster fallback created {protos_added} prototypes for {token_type}")
                except Exception:
                    if DEBUG_DISCOVERY:
                        print(f"[DSCD-CLUSTER] Fallback clustering failed for {token_type}")

            # update cluster stats
            try:
                if store.centroids:
                    counts = store.counts if store.counts else [1] * len(store.centroids)
                    total_count = sum(counts)
                    mean_count = float(total_count) / max(1, len(counts))
                    self.cluster_stats[str(token_type)] = {
                        'num_prototypes': len(store.centroids),
                        'counts': [int(c) for c in counts],
                        'total_samples': int(total_count),
                        'mean_count': float(mean_count),
                        'mu': float(store.mu),
                        'tau': float(store.tau),
                    }
            except Exception:
                pass

            return store.size() > 0

        except Exception as e:
            if DEBUG_DISCOVERY:
                print(f"[DSCD-ERROR] Clustering error for {token_type}: {type(e).__name__} {str(e)[:200]}")
            return False

    def get_explanations(self, threshold_span: float = 0.3) -> List[Dict[str, Any]]:
        expl: List[Dict[str, Any]] = []
        with self.clustering_lock:
            for token_type, store in self.prototype_stores.items():
                if store.size() >= 2:
                    expl.append({'token': str(token_type), 'protos': store.size()})
        return expl

    def periodic_discovery_check(self, global_step: int, frequency: int) -> int:
        try:
            candidates: List[Tuple[str, float, int]] = []
            buffer_snapshot = {}
            already_clustered = set()

            with self.buffer_lock:
                for token in list(self.buffers.keys()):
                    buffer_snapshot[token] = len(self.buffers.get(token, []))

            with self.clustering_lock:
                for token in self.prototype_stores.keys():
                    if self.prototype_stores[token].size() >= 2:
                        already_clustered.add(token)

            for token, buffer_size in buffer_snapshot.items():
                if is_punctuation_only(token):
                    continue
                if token in already_clustered:
                    continue
                if buffer_size >= max(self.n_min + 2, 10):
                    try:
                        dispersion = self.get_dispersion(token)
                        if dispersion >= self.dispersion_threshold:
                            rank_score = dispersion * buffer_size
                            candidates.append((token, rank_score, buffer_size))
                    except:
                        continue

            if not candidates:
                return 0

            candidates.sort(key=lambda x: x[1], reverse=True)
            candidates_to_process = candidates[:min(MAX_TOKENS_PER_DISCOVERY, len(candidates))]

            return self.discover_homographs_for_tokens(
                [c[0] for c in candidates_to_process],
                self.n_min,
                self.dispersion_threshold,
                global_step,
            )

        except Exception as e:
            if DEBUG_DISCOVERY:
                print(f"[DSCD] periodic_discovery_check failed: {e}")
            return 0

    def get_prototype_summary(self) -> Dict[str, Any]:
        try:
            with self.clustering_lock:
                total_tokens = len(self.prototype_stores)
                total_prototypes = sum(s.size() for s in self.prototype_stores.values())
                homographs = sum(1 for s in self.prototype_stores.values() if s.size() >= 2)
                discovered = len(self.discovered_homographs)
            return {
                'total_tokens': total_tokens,
                'total_prototypes': total_prototypes,
                'num_homographs': homographs,
                'discovered_homographs': discovered,
            }
        except Exception:
            return {
                'total_tokens': 0,
                'total_prototypes': 0,
                'num_homographs': 0,
                'discovered_homographs': 0,
            }

    def get_discovered_homographs(self) -> Set[str]:
        with self.clustering_lock:
            return set(self.discovered_homographs)


print("=" * 80)
print("Cell 3: DSCD (Word-Level Homograph Disambiguation) - COMPLETE FIXED")
print("=" * 80)
print("CONFIGURATION:")
print(f"  ✅ Preferred clustering: scipy hierarchical (HAS_CLUSTERING={HAS_CLUSTERING})")
print(f"  ✅ KMeans fallback available: {HAS_KMEANS}")
print(f"  ✅ Graph-clustering fallback: enabled (numpy-based)")
print(f"  ✅ Parameters: max_protos={DSCD_MAX_PROTOS}, n_min={DSCD_N_MIN}, dispersion_threshold={DSCD_DISPERSION_THRESHOLD}")
print("=" * 80)

Cell 3: DSCD (Word-Level Homograph Disambiguation) - COMPLETE FIXED
CONFIGURATION:
  ✅ Preferred clustering: scipy hierarchical (HAS_CLUSTERING=True)
  ✅ KMeans fallback available: True
  ✅ Graph-clustering fallback: enabled (numpy-based)
  ✅ Parameters: max_protos=10, n_min=2, dispersion_threshold=0.15


In [7]:
# ==============================================================================
# CELL 4: ASBN MODULE - ADVERSARIAL SELECTIVE BATCH NORMALIZATION - FIXED
# ==============================================================================
import traceback
from typing import Any, List, Tuple, Optional, Dict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Defensive global extraction with defaults
try:
    _MAX_LENGTH = int(globals().get("MAX_LENGTH", 48))
except Exception:
    _MAX_LENGTH = 48

try:
    _ENABLE_ASBN_TRAINING = bool(globals().get("ENABLE_ASBN_TRAINING", True))
except Exception:
    _ENABLE_ASBN_TRAINING = True

try:
    _VERBOSE_LOGGING = bool(globals().get("VERBOSE_LOGGING", False))
except Exception:
    _VERBOSE_LOGGING = False

try:
    _DEBUG_DISCOVERY = bool(globals().get("DEBUG_DISCOVERY", False))
except Exception:
    _DEBUG_DISCOVERY = False

try:
    _DEBUG_TIMING = bool(globals().get("DEBUG_TIMING", False))
except Exception:
    _DEBUG_TIMING = False

try:
    _SOURCE_LANGUAGE = str(globals().get("SOURCE_LANGUAGE", "bn"))
except Exception:
    _SOURCE_LANGUAGE = "bn"

try:
    _GRL_ALPHA_START = float(globals().get("GRL_ALPHA_START", 0.1))
    _GRL_ALPHA_END = float(globals().get("GRL_ALPHA_END", 1.0))
    _GRL_ALPHA_SCHEDULE = str(globals().get("GRL_ALPHA_SCHEDULE", "linear"))
    try:
        _GRL_ALPHA_STEPS = int(globals().get("GRL_ALPHA_STEPS", 10000))
    except Exception:
        _GRL_ALPHA_STEPS = 10000
except Exception:
    _GRL_ALPHA_START = 0.1
    _GRL_ALPHA_END = 1.0
    _GRL_ALPHA_SCHEDULE = "linear"
    _GRL_ALPHA_STEPS = 10000

# Prepare safe fallbacks for DSCD/tokenizer helper functions to avoid crashes
_has_should_track_token_dscd = False
_should_track_token_dscd = lambda token_text: True  # conservative fallback: allow token tracking

try:
    func = globals().get("should_track_token", None)
    if callable(func):
        _should_track_token_dscd = func
        _has_should_track_token_dscd = True
    else:
        cls = globals().get("MemoryEfficientDSCDOnline", None)
        if cls is not None and hasattr(cls, "should_track_token"):
            try:
                probe = cls(embed_dim=int(globals().get("DSCD_EMBED_DIM", 512)))
                _should_track_token_dscd = lambda token_text: probe.should_track_token(token_text)
                _has_should_track_token_dscd = True
            except Exception:
                _has_should_track_token_dscd = False
except Exception:
    _has_should_track_token_dscd = False

_has_is_valid_token = False
_is_valid_token_fn = lambda token, *args, **kwargs: True
try:
    func = globals().get("is_valid_token", None)
    if callable(func):
        _is_valid_token_fn = func
        _has_is_valid_token = True
except Exception:
    _has_is_valid_token = False

_has_get_tokenizer_special_tokens = False
_get_tokenizer_special_tokens_fn = lambda tk: set()
try:
    func = globals().get("get_tokenizer_special_tokens", None)
    if callable(func):
        _get_tokenizer_special_tokens_fn = func
        _has_get_tokenizer_special_tokens = True
except Exception:
    _has_get_tokenizer_special_tokens = False

# Utility: safe clamp helper
def _safe_clamp_tensor(tensor: torch.Tensor, max_val: int) -> torch.Tensor:
    try:
        if tensor is None:
            return tensor
        return torch.clamp(tensor, 0, max_val)
    except Exception:
        return tensor

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = float(alpha)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        grad_input = -ctx.alpha * grad_output
        return grad_input, None

def gradient_reversal(x, alpha: float = 1.0):
    return GradientReversalFunction.apply(x, alpha)

class LightweightDiscriminator(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)

class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)

class MemoryEfficientASBNModule(nn.Module):
    def __init__(
        self,
        embed_dim: int,
        tokenizer=None,
        language: str = "bn",
        freq_threshold: float = 0.7,
        uncertainty_threshold: float = 0.3,
        gate_threshold: float = 0.5,
        warmup_steps: int = 50,
        encoder_grl_scale: float = 1.0,
    ):
        super().__init__()
        self.language = language
        self.tokenizer = tokenizer
        self.embed_dim = int(embed_dim)

        self.bn_source = nn.BatchNorm1d(self.embed_dim, track_running_stats=True)
        self.bn_target = nn.BatchNorm1d(self.embed_dim, track_running_stats=True)

        self.d_domain = DomainDiscriminator(self.embed_dim)
        self.d_freq = LightweightDiscriminator(self.embed_dim + 2)
        self.d_ctx = LightweightDiscriminator(self.embed_dim + 2)
        self.d_xl = LightweightDiscriminator(self.embed_dim)

        self.freq_threshold = float(freq_threshold)
        self.uncertainty_threshold = float(uncertainty_threshold)
        self.gate_threshold = float(gate_threshold)
        self.warmup_steps = int(warmup_steps)
        self.current_step = 0

        self.lambda_base = {"freq": 1.0, "ctx": 1.0, "xl": 1.0, "domain": 1.0}
        self.lambda_max = 2.0
        self.encoder_grl_scale = float(encoder_grl_scale)

        self.stats_reset_interval = 100
        self.stats = {
            "domain_loss": 0.0,
            "domain_accuracy": 0.0,
            "source_accuracy": 0.0,
            "target_accuracy": 0.0,
            "asbn_loss": 0.0,
            "num_updates": 0,
        }

        try:
            if tokenizer is not None:
                if _has_get_tokenizer_special_tokens and _get_tokenizer_special_tokens_fn is not None:
                    self.special_tokens = _get_tokenizer_special_tokens_fn(tokenizer)
                else:
                    self.special_tokens = set(getattr(tokenizer, "all_special_tokens", [])) or set()
            else:
                self.special_tokens = set()
        except Exception:
            self.special_tokens = set()

        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            print("=" * 80)
            print("[ASBN-INIT] Initialized MemoryEfficientASBNModule")
            print("=" * 80)
            print(f"  embed_dim: {self.embed_dim}")
            print(f"  warmup_steps: {self.warmup_steps}")
            print(f"  encoder_grl_scale: {self.encoder_grl_scale}")
            print(f"  GRL_ALPHA: {_GRL_ALPHA_START} → {_GRL_ALPHA_END} ({_GRL_ALPHA_SCHEDULE})")
            print(f"  GRL_STEPS: {_GRL_ALPHA_STEPS}")
            print(f"  freq_threshold: {self.freq_threshold}")
            print(f"  uncertainty_threshold: {self.uncertainty_threshold}")
            print(f"  gate_threshold: {self.gate_threshold}")
            print(f"  DSCD should_track available: {_has_should_track_token_dscd}")
            print("=" * 80)

    def get_grl_alpha(self, global_step: Optional[int] = None) -> float:
        if global_step is None:
            global_step = self.current_step
        step = max(0, int(global_step))

        schedule = str(_GRL_ALPHA_SCHEDULE).lower()
        if schedule == "linear":
            progress = min(1.0, float(step) / max(1.0, float(_GRL_ALPHA_STEPS)))
            alpha = _GRL_ALPHA_START + progress * (_GRL_ALPHA_END - _GRL_ALPHA_START)
        elif schedule == "exponential":
            progress = min(1.0, float(step) / max(1.0, float(_GRL_ALPHA_STEPS)))
            try:
                if _GRL_ALPHA_START > 0:
                    ratio = _GRL_ALPHA_END / _GRL_ALPHA_START
                    alpha = _GRL_ALPHA_START * (ratio ** progress)
                else:
                    alpha = _GRL_ALPHA_END * progress
            except Exception:
                alpha = _GRL_ALPHA_END
        else:
            alpha = _GRL_ALPHA_END

        alpha = float(np.clip(alpha, 0.0, 2.0))
        return alpha

    def get_asbn_stats(self) -> Dict[str, float]:
        return self.get_detailed_stats()

    def get_detailed_stats(self) -> Dict[str, float]:
        if self.stats["num_updates"] > 0:
            n = float(self.stats["num_updates"])
            return {
                "domain_loss": self.stats["domain_loss"] / n,
                "domain_accuracy": self.stats["domain_accuracy"] / n,
                "source_accuracy": self.stats["source_accuracy"] / n,
                "target_accuracy": self.stats["target_accuracy"] / n,
                "asbn_loss": self.stats["asbn_loss"] / n,
                "num_updates": self.stats["num_updates"],
            }
        return {
            "domain_loss": 0.0,
            "domain_accuracy": 0.0,
            "source_accuracy": 0.0,
            "target_accuracy": 0.0,
            "asbn_loss": 0.0,
            "num_updates": 0,
        }

    def reset_stats(self) -> None:
        self.stats = {
            "domain_loss": 0.0,
            "domain_accuracy": 0.0,
            "source_accuracy": 0.0,
            "target_accuracy": 0.0,
            "asbn_loss": 0.0,
            "num_updates": 0,
        }

    def critic_parameters(self):
        return (
            list(self.d_domain.parameters())
            + list(self.d_freq.parameters())
            + list(self.d_ctx.parameters())
            + list(self.d_xl.parameters())
        )

    def _ensure_discriminators_on_device(self, device: torch.device) -> None:
        try:
            for mod in (
                self.d_domain,
                self.d_freq,
                self.d_ctx,
                self.d_xl,
                self.bn_source,
                self.bn_target,
            ):
                try:
                    params = list(mod.parameters())
                    if params:
                        p = params[0]
                        if p.device != device:
                            mod.to(device)
                    else:
                        mod.to(device)
                except Exception:
                    try:
                        mod.to(device)
                    except Exception:
                        pass
        except Exception:
            if _VERBOSE_LOGGING:
                try:
                    print("[ASBN] Device migration warning")
                except Exception:
                    pass

    def _expand_domain_labels(self, domain_labels: Optional[torch.Tensor], batch_size: int) -> Optional[torch.Tensor]:
        if domain_labels is None:
            return None

        try:
            if not isinstance(domain_labels, torch.Tensor):
                domain_labels = torch.tensor(domain_labels, dtype=torch.long)

            if domain_labels.dim() == 0:
                domain_labels = domain_labels.unsqueeze(0)

            if domain_labels.size(0) == 1 and batch_size > 1:
                domain_labels = domain_labels.expand(batch_size).contiguous()
            elif domain_labels.size(0) != batch_size:
                if _VERBOSE_LOGGING:
                    print(f"[ASBN] Domain label mismatch: got {domain_labels.size(0)}, expected {batch_size}")
                if domain_labels.size(0) > 0:
                    domain_labels = domain_labels[0].unsqueeze(0).expand(batch_size).contiguous()
                else:
                    return None

            domain_labels = domain_labels.long()

            invalid_mask = (domain_labels < 0) | (domain_labels > 1)
            if invalid_mask.any():
                if _VERBOSE_LOGGING:
                    print(f"[ASBN] Invalid domain labels detected, clamping to [0, 1]")
                domain_labels = torch.clamp(domain_labels, 0, 1)

            return domain_labels

        except Exception:
            if _VERBOSE_LOGGING:
                print(f"[ASBN] Domain label expansion failed")
            return None

    def _parse_proto_probs_matrix(self, proto_probs: Any, batch_size: int, seq_len: int, device: torch.device) -> torch.Tensor:
        pmax = torch.full((batch_size, seq_len), 0.5, dtype=torch.float32, device=device)

        try:
            if proto_probs is None:
                return pmax

            if isinstance(proto_probs, torch.Tensor):
                if proto_probs.dim() == 3:
                    B, T, K = proto_probs.shape
                    p = proto_probs.detach().to(device)
                    b_max = min(batch_size, B)
                    t_max = min(seq_len, T)
                    pmax[:b_max, :t_max] = p[:b_max, :t_max].max(dim=2)[0]
                    return pmax

                if proto_probs.dim() == 2:
                    p = proto_probs.detach().to(device)
                    if batch_size >= 1:
                        t_max = min(seq_len, p.size(0))
                        if p.dim() == 2 and p.size(1) > 1:
                            pmax[0, :t_max] = p[:t_max].max(dim=1)[0]
                        else:
                            pmax[0, :t_max] = p[:t_max, 0] if p.dim() == 2 else p[:t_max]
                        return pmax

            if isinstance(proto_probs, (list, tuple)):
                if len(proto_probs) == batch_size:
                    for b in range(batch_size):
                        row = proto_probs[b]
                        if isinstance(row, torch.Tensor):
                            if row.dim() == 2:
                                t_max = min(seq_len, row.size(0))
                                pmax[b, :t_max] = row[:t_max].max(dim=1)[0].to(device)
                            elif row.dim() == 1:
                                t_max = min(seq_len, row.size(0))
                                pmax[b, :t_max] = row[:t_max].to(device)
                        elif isinstance(row, (list, tuple)):
                            for t in range(min(seq_len, len(row))):
                                try:
                                    val = row[t]
                                    if isinstance(val, torch.Tensor):
                                        pmax[b, t] = float(val.max().item()) if val.numel() > 1 else float(val.item())
                                    elif isinstance(val, (list, tuple, np.ndarray)):
                                        arr = np.asarray(val, dtype=np.float32)
                                        pmax[b, t] = float(np.max(arr)) if arr.size > 0 else 0.5
                                    elif isinstance(val, (int, float)):
                                        pmax[b, t] = float(val)
                                    else:
                                        pmax[b, t] = 0.5
                                except Exception:
                                    pmax[b, t] = 0.5
                else:
                    if batch_size == 1 and len(proto_probs) > 0:
                        row = proto_probs
                        for t in range(min(seq_len, len(row))):
                            try:
                                val = row[t]
                                if isinstance(val, torch.Tensor):
                                    pmax[0, t] = float(val.max().item()) if val.numel() > 1 else float(val.item())
                                elif isinstance(val, (list, tuple, np.ndarray)):
                                    arr = np.asarray(val, dtype=np.float32)
                                    pmax[0, t] = float(np.max(arr)) if arr.size > 0 else 0.5
                                elif isinstance(val, (int, float)):
                                    pmax[0, t] = float(val)
                                else:
                                    pmax[0, t] = 0.5
                            except Exception:
                                pmax[0, t] = 0.5

        except Exception:
            if _VERBOSE_LOGGING:
                print(f"[ASBN] parse_proto_probs exception")

        return pmax

    def _parse_scalar_matrix(self, mat: Any, batch_size: int, seq_len: int, device: torch.device,
                            default: float = 0.0) -> torch.Tensor:
        out = torch.full((batch_size, seq_len), float(default), dtype=torch.float32, device=device)

        try:
            if mat is None:
                return out

            if isinstance(mat, torch.Tensor):
                mat = mat.detach()
                if mat.dim() == 3:
                    B, T, _ = mat.shape
                    b_max = min(batch_size, B)
                    t_max = min(seq_len, T)
                    out[:b_max, :t_max] = mat[:b_max, :t_max, 0].to(device)
                elif mat.dim() == 2:
                    if mat.size(0) == batch_size:
                        t_max = min(seq_len, mat.size(1))
                        out[:, :t_max] = mat[:, :t_max].to(device)
                    elif batch_size == 1:
                        t_max = min(seq_len, mat.size(0))
                        out[0, :t_max] = mat[:t_max, 0].to(device) if mat.size(1) > 1 else mat[:t_max].to(device)
                elif mat.dim() == 1:
                    if batch_size == 1:
                        t_max = min(seq_len, mat.size(0))
                        out[0, :t_max] = mat[:t_max].to(device)

            elif isinstance(mat, (list, tuple)):
                if len(mat) == batch_size:
                    for b in range(batch_size):
                        row = mat[b]
                        if isinstance(row, torch.Tensor):
                            if row.dim() >= 1:
                                t_max = min(seq_len, row.size(0))
                                for t in range(t_max):
                                    out[b, t] = float(row[t].item())
                        elif isinstance(row, (list, tuple, np.ndarray)):
                            t_max = min(seq_len, len(row))
                            for t in range(t_max):
                                try:
                                    v = row[t]
                                    out[b, t] = float(v.item()) if isinstance(v, torch.Tensor) else float(v)
                                except Exception:
                                    out[b, t] = float(default)
                elif batch_size == 1 and len(mat) > 0:
                    row = mat
                    t_max = min(seq_len, len(row))
                    for t in range(t_max):
                        try:
                            v = row[t]
                            out[0, t] = float(v.item()) if isinstance(v, torch.Tensor) else float(v)
                        except Exception:
                            out[0, t] = float(default)

        except Exception:
            if _VERBOSE_LOGGING:
                try:
                    print("[ASBN] parse_scalar_matrix warning")
                except Exception:
                    pass

        return out

    def compute_lambda_scaled_tensor(self, pmax: torch.Tensor, uncertainty: torch.Tensor,
                                    gate: torch.Tensor, lambda_type: str) -> torch.Tensor:
        base = float(self.lambda_base.get(lambda_type, 1.0))
        lam = base * torch.ones_like(pmax, device=pmax.device)
        lam = torch.clamp(lam, min=0.1, max=float(self.lambda_max))
        lam = lam.contiguous()
        lam = torch.where(torch.isfinite(lam), lam, torch.ones_like(lam))
        return lam

    def forward(
        self,
        h: torch.Tensor,
        h_augmented: Optional[torch.Tensor] = None,
        proto_probs: Any = None,
        uncertainties: Any = None,
        gates: Any = None,
        token_word_map: Optional[List[Dict[int, str]]] = None,
        domain_labels: Optional[torch.Tensor] = None,
        global_step: Optional[int] = None,
    ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:

        if global_step is not None:
            self.current_step = int(global_step)

        if h_augmented is not None and isinstance(h_augmented, torch.Tensor) and h_augmented.dim() == 3:
            h_input = h_augmented
        else:
            h_input = h

        if not isinstance(h_input, torch.Tensor) or h_input.dim() != 3:
            dev = h_input.device if isinstance(h_input, torch.Tensor) else torch.device("cpu")
            zero = torch.tensor(0.0, device=dev)
            return h_input, {
                "encoder_loss": zero,
                "adversarial_loss": zero,
                "domain_loss": zero,
                "domain_accuracy": zero,
            }

        B, T, H = h_input.size()
        device = h_input.device

        domain_labels = self._expand_domain_labels(domain_labels, B)
        if domain_labels is not None:
            try:
                domain_labels = domain_labels.to(device)
            except Exception:
                pass

        h_normalized = h_input.clone()

        if domain_labels is not None and B * T >= 2:
            try:
                self._ensure_discriminators_on_device(device)
                h_flat = h_input.view(B * T, H)
                domain_expanded = domain_labels.unsqueeze(1).expand(B, T).reshape(-1)

                source_mask = domain_expanded == 0
                target_mask = domain_expanded == 1

                h_norm_flat = h_flat.clone()

                source_count = int(source_mask.sum().item())
                target_count = int(target_mask.sum().item())

                if source_count >= 2:
                    self.bn_source.train(self.training)
                    h_norm_flat[source_mask] = self.bn_source(h_flat[source_mask])
                elif source_count == 1:
                    self.bn_source.eval()
                    with torch.no_grad():
                        h_norm_flat[source_mask] = self.bn_source(h_flat[source_mask])

                if target_count >= 2:
                    self.bn_target.train(self.training)
                    h_norm_flat[target_mask] = self.bn_target(h_flat[target_mask])
                elif target_count == 1:
                    self.bn_target.eval()
                    with torch.no_grad():
                        h_norm_flat[target_mask] = self.bn_target(h_flat[target_mask])

                h_normalized = h_norm_flat.view(B, T, H)

                if _DEBUG_DISCOVERY and self.current_step % 500 == 0:
                    print(f"[ASBN-BN] step {self.current_step}: src={source_count}, tgt={target_count}")

            except Exception:
                h_normalized = h_input

        if self.current_step < self.warmup_steps:
            if _DEBUG_DISCOVERY and self.current_step % 50 == 0:
                print(f"[ASBN] Warmup: {self.current_step}/{self.warmup_steps}")
            zero = torch.tensor(0.0, device=device)
            return h_normalized, {
                "encoder_loss": zero,
                "adversarial_loss": zero,
                "domain_loss": zero,
                "domain_accuracy": zero,
            }

        if not self.training or not _ENABLE_ASBN_TRAINING:
            zero = torch.tensor(0.0, device=device)
            return h_normalized, {
                "encoder_loss": zero,
                "adversarial_loss": zero,
                "domain_loss": zero,
                "domain_accuracy": zero,
            }

        self._ensure_discriminators_on_device(device)
        self.d_domain.train()
        self.d_freq.train()
        self.d_ctx.train()
        self.d_xl.train()

        pmax_mat = self._parse_proto_probs_matrix(proto_probs, B, T, device)
        U_mat = self._parse_scalar_matrix(uncertainties, B, T, device, default=0.1)
        G_mat = self._parse_scalar_matrix(gates, B, T, device, default=0.0)

        sel_mask = torch.ones((B, T), dtype=torch.bool, device=device)
        batch_indices = torch.arange(B, device=device).unsqueeze(1).expand(B, T)

        # Token filtering: guarded, uses fallback that allows tokens if DSCD helper absent
        if token_word_map is not None and callable(_should_track_token_dscd):
            try:
                for b in range(min(B, len(token_word_map))):
                    wm = token_word_map[b]
                    if not wm or not isinstance(wm, dict):
                        continue
                    for t in range(T):
                        if t in wm:
                            try:
                                token_str = str(wm[t]) if wm[t] is not None else ""
                                if token_str:
                                    tracked = False
                                    try:
                                        tracked = bool(_should_track_token_dscd(token_str))
                                    except Exception:
                                        tracked = True
                                    if not tracked:
                                        sel_mask[b, t] = False
                            except Exception:
                                pass
            except Exception:
                if _VERBOSE_LOGGING:
                    print("[ASBN] Token filtering warning")

        sel_idx = sel_mask.view(-1).nonzero(as_tuple=False).squeeze(1)
        sel_idx = sel_idx if isinstance(sel_idx, torch.Tensor) else torch.tensor([], dtype=torch.long, device=device)
        sel_idx = sel_idx.to(device) if sel_idx.numel() > 0 else torch.tensor([], dtype=torch.long, device=device)

        batch_idx = batch_indices.view(-1)[sel_idx] if sel_idx.numel() > 0 else torch.tensor([], dtype=torch.long, device=device)

        if sel_idx.numel() == 0:
            if _DEBUG_DISCOVERY and self.current_step % 200 == 0:
                print("[ASBN] No valid tokens selected")
            zero = torch.tensor(0.0, device=device)
            return h_normalized, {
                "encoder_loss": zero,
                "adversarial_loss": zero,
                "domain_loss": zero,
                "domain_accuracy": zero,
            }

        h_flat = h_normalized.view(B * T, H)
        sel_emb = h_flat[sel_idx]

        pmax_flat = pmax_mat.view(-1)[sel_idx]
        U_flat = U_mat.view(-1)[sel_idx]
        G_flat = G_mat.view(-1)[sel_idx]

        seq_len_feature = float(T) / max(int(_MAX_LENGTH), 1)
        freq_feature = torch.stack([pmax_flat, U_flat], dim=1).to(device)
        ctx_feature = torch.stack([G_flat, torch.full_like(G_flat, seq_len_feature)], dim=1).to(device)
        xl_input = sel_emb

        grl_alpha = self.get_grl_alpha(global_step)

        freq_input = torch.cat([sel_emb, freq_feature], dim=1)
        ctx_input = torch.cat([sel_emb, ctx_feature], dim=1)

        xl_input_grl = gradient_reversal(xl_input, alpha=grl_alpha)
        freq_input_grl = gradient_reversal(freq_input, alpha=grl_alpha)
        ctx_input_grl = gradient_reversal(ctx_input, alpha=grl_alpha)

        freq_logits = self.d_freq(freq_input_grl)
        ctx_logits = self.d_ctx(ctx_input_grl)
        xl_logits = self.d_xl(xl_input_grl)

        freq_label = (pmax_flat > self.freq_threshold).long().to(device)
        ctx_label = (U_flat < self.uncertainty_threshold).long().to(device)
        xl_label = (G_flat > self.gate_threshold).long().to(device)

        loss_freq = F.cross_entropy(freq_logits, freq_label, reduction="none")
        loss_ctx = F.cross_entropy(ctx_logits, ctx_label, reduction="none")
        loss_xl = F.cross_entropy(xl_logits, xl_label, reduction="none")

        lam_freq = self.compute_lambda_scaled_tensor(pmax_flat, U_flat, G_flat, "freq")
        lam_ctx = self.compute_lambda_scaled_tensor(pmax_flat, U_flat, G_flat, "ctx")
        lam_xl = self.compute_lambda_scaled_tensor(pmax_flat, U_flat, G_flat, "xl")

        weighted = lam_freq * loss_freq + lam_ctx * loss_ctx + lam_xl * loss_xl
        mean_weighted = torch.mean(weighted) if weighted.numel() > 0 else torch.tensor(0.0, device=device)

        domain_loss = torch.tensor(0.0, device=device)
        domain_accuracy = torch.tensor(0.0, device=device)

        if domain_labels is not None and sel_idx.numel() > 0:
            try:
                domain_flat = domain_labels[batch_idx]

                domain_input = gradient_reversal(sel_emb, alpha=grl_alpha)
                domain_logits = self.d_domain(domain_input)

                domain_loss = F.cross_entropy(domain_logits, domain_flat)

                with torch.no_grad():
                    domain_preds = torch.argmax(domain_logits, dim=1)
                    domain_accuracy = (domain_preds == domain_flat).float().mean()

                    source_mask = domain_flat == 0
                    target_mask = domain_flat == 1

                    if source_mask.any():
                        source_acc = (domain_preds[source_mask] == domain_flat[source_mask]).float().mean()
                        self.stats["source_accuracy"] += float(source_acc.item())

                    if target_mask.any():
                        target_acc = (domain_preds[target_mask] == domain_flat[target_mask]).float().mean()
                        self.stats["target_accuracy"] += float(target_acc.item())

            except Exception:
                if _VERBOSE_LOGGING:
                    print(f"[ASBN] Domain loss failed")

        encoder_loss = self.encoder_grl_scale * (mean_weighted + domain_loss)

        try:
            with torch.no_grad():
                self.stats["domain_loss"] += float(domain_loss.item()) if isinstance(domain_loss, torch.Tensor) else float(domain_loss)
                self.stats["domain_accuracy"] += float(domain_accuracy.item()) if isinstance(domain_accuracy, torch.Tensor) else float(domain_accuracy)
                self.stats["asbn_loss"] += float(encoder_loss.item()) if isinstance(encoder_loss, torch.Tensor) else float(encoder_loss)
                self.stats["num_updates"] += 1

                if self.stats["num_updates"] >= self.stats_reset_interval:
                    if _DEBUG_DISCOVERY:
                        stats = self.get_detailed_stats()
                        print(f"\n[ASBN-STATS] After {stats['num_updates']} updates:")
                        print(f"  Domain loss: {stats['domain_loss']:.4f}")
                        print(f"  Domain acc: {stats['domain_accuracy']:.2%}")
                        print(f"  Source acc: {stats['source_accuracy']:.2%}")
                        print(f"  Target acc: {stats['target_accuracy']:.2%}")
                        print(f"  ASBN loss: {stats['asbn_loss']:.4f}")
                    self.reset_stats()
        except Exception:
            pass

        if _DEBUG_DISCOVERY and self.current_step % 500 == 0:
            try:
                print(f"\n[ASBN-STEP-{self.current_step}]")
                print(f"  GRL alpha: {grl_alpha:.3f}")
                print(f"  Selected tokens: {sel_idx.numel()}/{B*T}")
                print(f"  Encoder loss: {float(encoder_loss.item()) if isinstance(encoder_loss, torch.Tensor) else float(encoder_loss):.4f}")
                print(f"  Domain loss: {float(domain_loss.item()) if isinstance(domain_loss, torch.Tensor) else float(domain_loss):.4f}")
                print(f"  Domain acc: {float(domain_accuracy.item()) if isinstance(domain_accuracy, torch.Tensor) else float(domain_accuracy):.2%}")
            except Exception:
                pass

        return h_normalized, {
            "encoder_loss": encoder_loss,
            "adversarial_loss": mean_weighted,
            "domain_loss": domain_loss,
            "domain_accuracy": domain_accuracy,
        }

    def test_asbn(self, batch_size: int = 2, seq_len: int = 10) -> bool:
        print("\n" + "=" * 80)
        print("[ASBN-TEST] Testing ASBN Module")
        print("=" * 80)

        try:
            try:
                device = next(self.parameters()).device
            except StopIteration:
                device = torch.device("cpu")

            h = torch.randn(batch_size, seq_len, self.embed_dim, device=device)
            domain_labels = torch.randint(0, 2, (batch_size,), device=device)

            h_out, losses = self.forward(h, domain_labels=domain_labels)
            assert h_out.shape == h.shape, "Output shape mismatch"
            assert "domain_loss" in losses, "Missing domain_loss"
            print("  ✓ forward() with domain_labels")

            proto_probs = torch.rand(batch_size, seq_len, 3, device=device)
            uncertainties = torch.rand(batch_size, seq_len, device=device)
            gates = torch.rand(batch_size, seq_len, device=device)

            self.train()
            self.current_step = self.warmup_steps + 1

            h_out, losses = self.forward(
                h,
                proto_probs=proto_probs,
                uncertainties=uncertainties,
                gates=gates,
                domain_labels=domain_labels,
                global_step=self.current_step,
            )

            assert losses["encoder_loss"].item() >= 0.0, "Negative encoder loss"
            assert 0.0 <= losses["domain_accuracy"].item() <= 1.0, "Invalid domain accuracy"
            print("  ✓ forward() with full inputs")

            h_aug = torch.randn(batch_size, seq_len, self.embed_dim, device=device)
            h_out, losses = self.forward(
                h,
                h_augmented=h_aug,
                domain_labels=domain_labels,
                global_step=self.current_step,
            )
            assert h_out.shape == h_aug.shape, "h_augmented not used"
            print("  ✓ forward() with h_augmented")

            stats = self.get_detailed_stats()
            assert "domain_loss" in stats, "Missing stats"
            print("  ✓ Statistics tracking")

            print("\n✓ All ASBN tests passed")
            print("=" * 80 + "\n")
            return True

        except Exception as e:
            print(f"\n✗ ASBN test failed: {e}")
            traceback.print_exc()
            print("=" * 80 + "\n")
            return False

print("\n" + "=" * 80)
print("CELL 4: ASBN MODULE - FIXED AND ALIGNED")
print("=" * 80)
print("Configuration:")
print(f"  Warmup Steps: {int(globals().get('WARMUP_STEPS', 50))}")
print(f"  GRL Alpha: {_GRL_ALPHA_START:.3f} → {_GRL_ALPHA_END:.3f}")
print(f"  GRL Schedule: {_GRL_ALPHA_SCHEDULE}")
print(f"  GRL Steps: {_GRL_ALPHA_STEPS}")
print(f"  Encoder GRL Scale: {1.0}")
print(f"  Stats Reset: Every {int(50 * 2)} updates (default printed)")
print(f"  ASBN Training: {'ENABLED' if _ENABLE_ASBN_TRAINING else 'DISABLED'}")
print(f"  DSCD Integration: {'✓' if _has_should_track_token_dscd else '✗ (fallback used)'}")
print("=" * 80 + "\n")


CELL 4: ASBN MODULE - FIXED AND ALIGNED
Configuration:
  Warmup Steps: 4000
  GRL Alpha: 0.100 → 1.000
  GRL Schedule: linear
  GRL Steps: 500
  Encoder GRL Scale: 1.0
  Stats Reset: Every 100 updates (default printed)
  ASBN Training: ENABLED
  DSCD Integration: ✓



In [8]:
# ==============================================================================
# CELL 5: TRG MODULE - TRANSPARENT RATIONALE GENERATION - FIXED
# ==============================================================================
from typing import List, Dict, Tuple, Optional, Set, Any
from collections import deque
import traceback
import numpy as np
import torch
import torch.nn as nn
import threading
import time
import gc

_TRG_EVIDENCE_K = int(globals().get("TRG_EVIDENCE_K", 3))
_TRG_GEN_EMBED = int(globals().get("TRG_GEN_EMBED", 64))
_MAX_SILVER_BUFFER = int(globals().get("MAX_SILVER_BUFFER", 50))

_VERBOSE_LOGGING = bool(globals().get("VERBOSE_LOGGING", False))
_DEBUG_DISCOVERY = bool(globals().get("DEBUG_DISCOVERY", False))
_DEBUG_TIMING = bool(globals().get("DEBUG_TIMING", False))
_ENABLE_TRG_INFERENCE = bool(globals().get("ENABLE_TRG_INFERENCE", True))

_SOURCE_LANGUAGE = str(globals().get("SOURCE_LANGUAGE", "bn"))

_TRG_UNCERTAINTY_THRESHOLD = float(globals().get("TRG_UNCERTAINTY_THRESHOLD", globals().get("TAU_LOW", 0.15)))
_TRG_SPAN_THRESHOLD = float(globals().get("TRG_SPAN_THRESHOLD", globals().get("SPAN_THRESHOLD", 0.20)))
_TAU_HIGH = float(globals().get("TAU_HIGH", 0.85))
_TAU_LOW = float(globals().get("TAU_LOW", 0.15))
_TAU_ACCEPT = float(globals().get("TAU_ACCEPT", 0.80))
_TRG_TEMPERATURE = float(globals().get("TRG_TEMPERATURE", 1.0))
_MAX_EXPLANATIONS_PER_SENTENCE = int(globals().get("MAX_EXPLANATIONS_PER_SENTENCE", 10))

_BENGALI_PUNCT_SET = set(['।', '॥'])
_COMMON_PUNCT_SET = set(['.', ',', ';', ':', '!', '?', '"', "'", '-', '(', ')', '[', ']', '{', '}', '/', '\\'])
_TRG_PUNCT_SET = _BENGALI_PUNCT_SET | _COMMON_PUNCT_SET

_FUNCTION_WORDS = {
    'এবং', 'ও', 'কিন্তু', 'তবে', 'যদি', 'তাহলে', 'কারণ', 'যেমন',
    'যখন', 'তখন', 'যেহেতু', 'সেহেতু', 'অথবা', 'কিংবা', 'বা',
    'এই', 'সেই', 'ঐ', 'ওই', 'কোন', 'কোনো', 'যে', 'যা', 'যিনি',
    'একটি', 'একজন', 'কয়েকটি', 'অনেক', 'সব', 'সকল', 'কিছু', 'সবকিছু',
    'আমি', 'তুমি', 'সে', 'তিনি', 'আমরা', 'তোমরা', 'তারা', 'আপনি', 'আপনারা',
    'আমার', 'তোমার', 'তার', 'আমাদের', 'তোমাদের', 'তাদের', 'আপনার', 'আপনাদের',
    'কি', 'কী', 'কে', 'কেন', 'কখন', 'কোথায়', 'কীভাবে', 'কতটা',
    'না', 'নয়', 'নেই', 'নি', 'আছে', 'চিল', 'হবে', 'হয়',
    'থেকে', 'পর্যন্ত', 'জন্য', 'সঙ্গে', 'সাথে', 'দিয়ে', 'মধ্যে', 'উপর',
    'করা', 'করে', 'করেন', 'করছে', 'করবে', 'করলে', 'হওয়া', 'হয়ে', 'হয়েছে',
    'সেটি', 'তাই', 'এতে', 'সেইসাথে', 'এই যে', 'যাতে', 'যেন', 'তার ফলে'
}


def _is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").replace("</w>", "").strip()
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET or clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in _TRG_PUNCT_SET for c in clean)


def _fallback_is_valid_token(token: str, special_tokens: set, tokenizer=None, language: str = "bn") -> bool:
    if token is None:
        return False
    if not isinstance(token, str):
        try:
            token = str(token)
        except Exception:
            return False
    token = token.strip()
    if not token:
        return False
    if token in (special_tokens or set()):
        return False
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").replace("</w>", "").strip()
    if len(clean) < 2:
        return False
    if not any(c.isalpha() for c in clean):
        return False
    if _is_punctuation_only(token):
        return False
    if clean.isdigit():
        return False
    if language == "bn":
        bengali_chars = sum(1 for c in clean if '\u0980' <= c <= '\u09FF')
        if bengali_chars == 0:
            return False
        if len(clean) >= 2 and bengali_chars / len(clean) < 0.5:
            return False
    return True


def _is_word_start(raw_token: str, token_word_map: Optional[dict], idx: int) -> bool:
    if not isinstance(raw_token, str):
        return False
    try:
        if token_word_map is not None and isinstance(token_word_map, dict):
            if idx in token_word_map:
                w = token_word_map[idx]
                if isinstance(w, str) and w.strip():
                    return True
        if raw_token.startswith("▁") or raw_token.startswith("Ġ"):
            return True
        clean = raw_token.replace("▁", "").replace("Ġ", "").replace("##", "").replace("</w>", "").strip()
        if len(clean) < 2:
            return False
        if _is_punctuation_only(raw_token):
            return False
        if token_word_map is None and any(c.isalpha() for c in clean):
            return True
        return False
    except Exception:
        return False


_is_valid_token_fn = globals().get("is_valid_token", None)
_has_is_valid_token = callable(_is_valid_token_fn)
if not _has_is_valid_token:
    _is_valid_token_fn = _fallback_is_valid_token

_get_special_tokens_fn = globals().get("get_special_tokens", None)
if not callable(_get_special_tokens_fn):
    _get_special_tokens_fn = globals().get("get_tokenizer_special_tokens", None)
if not callable(_get_special_tokens_fn):
    _get_special_tokens_fn = lambda tk: set(getattr(tk, "all_special_tokens", [])) if tk is not None else set()


class ComprehensiveTRGExplanationTemplate:
    def __init__(self):
        self.explanation_templates = {
            "high_confidence": (
                "Chose '{sense}' with high confidence ({confidence:.1%}) based on: '{evidence}'.   "
                "Pattern matches learned data.   {alternatives_text}"
            ),
            "medium_confidence": (
                "Selected '{sense}' with moderate confidence ({confidence:.1%}). "
                "Evidence: '{evidence}'. Some uncertainty ({uncertainty:.1%}).   {alternatives_text}"
            ),
            "low_confidence": (
                "Uncertain; chose '{sense}' ({confidence:.1%}). "
                "Evidence: '{evidence}'.   {alternatives_text} Review recommended."
            ),
            "fallback": ("Token '{token}' analyzed.   Context: '{evidence}'."),
        }

    def generate_explanation(self, evidence: Dict) -> str:
        if not evidence or not isinstance(evidence, dict):
            return ""
        token = str(evidence.get("token", "unknown")).replace("▁", "").replace("Ġ", "")
        sense_info = evidence.get("chosen_sense", ("unknown", 0.5))
        if isinstance(sense_info, (tuple, list)) and len(sense_info) >= 2:
            sense_name, confidence = str(sense_info[0]), float(sense_info[1])
        else:
            sense_name, confidence = "unknown", 0.5
        uncertainty = float(evidence.get("uncertainty", 0.5))
        evidence_tokens = evidence.get("evidence_tokens", [])
        evidence_str = ", ".join([str(tok).replace("▁", "").replace("Ġ", "") for tok in evidence_tokens[:_TRG_EVIDENCE_K]]) or "limited context"
        alternatives = evidence.get("alternatives", [])
        alternatives_text = ""
        if isinstance(alternatives, list) and len(alternatives) > 0:
            alt_parts = []
            for alt in alternatives[:2]:
                if isinstance(alt, (tuple, list)) and len(alt) >= 2:
                    alt_name, alt_conf = str(alt[0]), float(alt[1])
                    alt_parts.append(f"'{alt_name}' ({alt_conf:.1%})")
            if alt_parts:
                alternatives_text = f"Alternatives: {', '.join(alt_parts)}."
        if confidence >= _TAU_ACCEPT:
            template_key = "high_confidence"
        elif confidence >= _TRG_UNCERTAINTY_THRESHOLD:
            template_key = "medium_confidence"
        else:
            template_key = "low_confidence"
        template = self.explanation_templates.get(template_key, self.explanation_templates["fallback"])
        try:
            return template.format(
                sense=sense_name,
                confidence=confidence,
                uncertainty=uncertainty,
                evidence=evidence_str,
                alternatives_text=alternatives_text,
                token=token,
            )
        except Exception:
            return f"Token '{token}' -> '{sense_name}' ({confidence:.1%})."


class MemoryEfficientTRGExtractor:
    def __init__(self, tokenizer=None, language: str = "bn", dscd_module=None):
        self.tokenizer = tokenizer
        self.language = language
        self.dscd_module = dscd_module
        self.span_clamp_warnings = 0
        self.last_warning_time = 0.0
        try:
            self.special_tokens = _get_special_tokens_fn(tokenizer) if callable(_get_special_tokens_fn) else set()
        except Exception:
            self.special_tokens = set()

    def extract_evidence_from_target(self, token_idx: int, span_start: int, span_end: int, tgt_preds: torch.Tensor) -> Optional[List[str]]:
        if not isinstance(token_idx, int) or token_idx < 0:
            return None
        if not isinstance(span_start, int) or not isinstance(span_end, int):
            return None
        if span_start < 0:
            return None
        if not isinstance(tgt_preds, (torch.Tensor, list)):
            return None
        seq_len = len(tgt_preds) if isinstance(tgt_preds, list) else int(tgt_preds.size(0))
        if span_end > seq_len:
            return None
        if span_start >= span_end:
            return None
        if token_idx < span_start or token_idx >= span_end:
            return None
        if token_idx >= seq_len:
            return None
        try:
            evidence_tokens: List[str] = []
            for i in range(span_start, span_end):
                if i == token_idx:
                    continue
                if isinstance(tgt_preds, list):
                    evidence_tokens.append(str(tgt_preds[i]))
                else:
                    try:
                        evidence_tokens.append(str(int(tgt_preds[i].item())))
                    except Exception:
                        evidence_tokens.append(f"token_{i}")
            return evidence_tokens if evidence_tokens else None
        except Exception:
            return None

    def extract_evidence_efficiently(self, token_idx: int, tokens: List[str], dscd_outputs: Dict, token_word_map: Optional[dict] = None, decoder_attention: Optional[torch.Tensor] = None) -> Dict:
        if not isinstance(tokens, list):
            return self._create_fallback_evidence(token_idx, [])
        if not isinstance(token_idx, int):
            return self._create_fallback_evidence(0, tokens)
        if token_idx < 0 or token_idx >= len(tokens):
            return self._create_fallback_evidence(max(0, min(token_idx, len(tokens) - 1)), tokens)
        raw_token = tokens[token_idx]
        try:
            is_valid = False
            try:
                is_valid = bool(_is_valid_token_fn(raw_token, self.special_tokens, self.tokenizer, language=self.language))
            except Exception:
                is_valid = _fallback_is_valid_token(raw_token, self.special_tokens, self.tokenizer, self.language)
        except Exception:
            is_valid = _fallback_is_valid_token(raw_token, self.special_tokens, self.tokenizer, self.language)
        if not is_valid:
            return self._create_fallback_evidence(token_idx, tokens)
        try:
            proto_probs = self._safe_extract_proto_probs(token_idx, dscd_outputs)
            uncertainty = self._safe_extract_uncertainty(token_idx, dscd_outputs)
            gate = self._safe_extract_gate(token_idx, dscd_outputs)
            span = self._safe_extract_span(token_idx, dscd_outputs)
            evidence_tokens: Optional[List[str]] = None
            if decoder_attention is not None and isinstance(decoder_attention, torch.Tensor):
                try:
                    vec = None
                    if decoder_attention.dim() == 4:
                        attn_avg = decoder_attention.mean(dim=(0, 1))
                        if attn_avg.dim() == 2 and token_idx < attn_avg.size(0):
                            vec = attn_avg[token_idx]
                    elif decoder_attention.dim() == 3:
                        attn_avg = decoder_attention.mean(dim=0)
                        if attn_avg.dim() == 2 and token_idx < attn_avg.size(0):
                            vec = attn_avg[token_idx]
                    elif decoder_attention.dim() == 2:
                        if token_idx < decoder_attention.size(0):
                            vec = decoder_attention[token_idx]
                    elif decoder_attention.dim() == 1:
                        vec = decoder_attention
                    if vec is not None and vec.numel() > 0:
                        k = min(5, int(max(1, vec.size(0))))
                        try:
                            top_k_indices = torch.topk(vec, k=k).indices.detach().cpu().numpy()
                        except Exception:
                            vals = vec.detach().cpu().numpy()
                            top_k_indices = np.argsort(vals)[-k:][::-1]
                        evidence_tokens = []
                        for i in top_k_indices:
                            if 0 <= int(i) < len(tokens) and int(i) != token_idx:
                                evidence_tokens.append(tokens[int(i)])
                except Exception:
                    evidence_tokens = None
            if evidence_tokens is None:
                evidence_tokens = self._extract_context_window(token_idx, tokens, token_word_map)
            seen: Dict[str, bool] = {}
            dedup_evidence: List[str] = []
            for t in (evidence_tokens or []):
                if t not in seen:
                    seen[t] = True
                    dedup_evidence.append(t)
            evidence_tokens = dedup_evidence[:_TRG_EVIDENCE_K]
            top_senses = self._compute_sense_alternatives_fast(proto_probs, temperature=_TRG_TEMPERATURE)
            chosen_sense = top_senses[0] if len(top_senses) > 0 else ("unknown", 0.5)
            alternatives = top_senses[1:3] if len(top_senses) > 1 else []
            if token_word_map and token_idx in token_word_map and isinstance(token_word_map[token_idx], str) and token_word_map[token_idx].strip():
                token_value = token_word_map[token_idx]
            else:
                token_value = raw_token
            return {
                "token": token_value,
                "token_idx": token_idx,
                "evidence_tokens": evidence_tokens,
                "chosen_sense": chosen_sense,
                "alternatives": alternatives,
                "uncertainty": float(uncertainty),
                "gate": float(gate),
                "span": float(span),
            }
        except Exception as e:
            if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                print(f"[TRG] Evidence error @ {token_idx}: {e}")
            return self._create_fallback_evidence(token_idx, tokens)

    def _extract_context_window(self, token_idx: int, tokens: List[str], token_word_map: Optional[dict]) -> List[str]:
        context_window = 2
        start_idx = max(0, token_idx - context_window)
        end_idx = min(len(tokens), token_idx + context_window + 1)
        evidence_tokens: List[str] = []
        for i in range(start_idx, end_idx):
            if i == token_idx or i >= len(tokens):
                continue
            rtok = tokens[i]
            clean_token = str(rtok).replace("▁", "").replace("Ġ", "").replace("</w>", "").strip()
            if not _is_word_start(rtok, token_word_map, i):
                if token_word_map is None and len(clean_token) >= 2 and any(c.isalpha() for c in clean_token):
                    pass
                else:
                    continue
            try:
                ok = False
                try:
                    ok = bool(_is_valid_token_fn(rtok, self.special_tokens, self.tokenizer, language=self.language))
                except Exception:
                    ok = _fallback_is_valid_token(rtok, self.special_tokens, self.tokenizer, self.language)
            except Exception:
                ok = _fallback_is_valid_token(rtok, self.special_tokens, self.tokenizer, self.language)
            if ok and len(clean_token) > 0:
                if token_word_map and isinstance(token_word_map.get(i, ""), str) and token_word_map[i].strip():
                    evidence_tokens.append(token_word_map[i].strip())
                else:
                    evidence_tokens.append(clean_token)
        return evidence_tokens

    def _safe_extract_proto_probs(self, token_idx: int, dscd_outputs: Dict) -> torch.Tensor:
        try:
            if not isinstance(dscd_outputs, dict):
                return torch.tensor([1.0], dtype=torch.float32)
            pp_all = dscd_outputs.get("proto_probs", None)
            if pp_all is None:
                return torch.tensor([1.0], dtype=torch.float32)
            # handle tensor (B,T,K) or list-of-lists or list of tensors
            if isinstance(pp_all, torch.Tensor):
                if pp_all.dim() == 3:
                    B, T, K = pp_all.shape
                    b_idx = 0
                    if token_idx < T:
                        return pp_all[b_idx, token_idx, :].detach().cpu().flatten()
                    else:
                        return pp_all[b_idx].detach().cpu().flatten()
                elif pp_all.dim() == 2:
                    if token_idx < pp_all.size(0):
                        return pp_all[token_idx].detach().cpu().flatten()
                    return pp_all.detach().cpu().flatten()
                else:
                    return pp_all.detach().cpu().flatten()
            if isinstance(pp_all, (list, tuple)):
                first = pp_all[0] if len(pp_all) > 0 else None
                if isinstance(first, torch.Tensor):
                    if first.dim() == 2 and token_idx < first.size(0):
                        return first[token_idx].detach().cpu().flatten()
                    return first.detach().cpu().flatten()
                if isinstance(first, (list, tuple, np.ndarray)):
                    try:
                        val = first[token_idx] if token_idx < len(first) else first[0]
                        return torch.as_tensor(val, dtype=torch.float32).flatten()
                    except Exception:
                        try:
                            val = first[0]
                            return torch.as_tensor(val, dtype=torch.float32).flatten()
                        except Exception:
                            pass
            # fallback
        except Exception:
            if _VERBOSE_LOGGING:
                print(f"[TRG] Proto_probs extraction failed for token {token_idx}, using default [1.0]")
        return torch.tensor([1.0], dtype=torch.float32)

    def _safe_extract_uncertainty(self, token_idx: int, dscd_outputs: Dict) -> float:
        try:
            if not isinstance(dscd_outputs, dict):
                return 0.5
            U_all = dscd_outputs.get("uncertainties", None)
            if U_all is None:
                return 0.5
            if isinstance(U_all, torch.Tensor):
                if U_all.dim() >= 2:
                    if token_idx < U_all.size(1):
                        return float(U_all[0, token_idx].item()) if U_all.dim() == 2 else float(U_all[token_idx].item())
                elif U_all.dim() == 1 and token_idx < U_all.size(0):
                    return float(U_all[token_idx].item())
            if isinstance(U_all, (list, tuple)):
                row = U_all[0] if len(U_all) > 0 else None
                if isinstance(row, torch.Tensor):
                    if row.dim() >= 1 and token_idx < row.size(0):
                        return float(row[token_idx].item())
                if isinstance(row, (list, tuple)) and token_idx < len(row):
                    val = row[token_idx]
                    return float(val.item()) if isinstance(val, torch.Tensor) else float(val)
        except Exception:
            pass
        return 0.5

    def _safe_extract_gate(self, token_idx: int, dscd_outputs: Dict) -> float:
        try:
            if not isinstance(dscd_outputs, dict):
                return 0.0
            G_all = dscd_outputs.get("gates", None)
            if G_all is None:
                return 0.0
            if isinstance(G_all, torch.Tensor):
                if G_all.dim() >= 2:
                    if token_idx < G_all.size(1):
                        return float(G_all[0, token_idx].item()) if G_all.dim() == 2 else float(G_all[token_idx].item())
                elif G_all.dim() == 1 and token_idx < G_all.size(0):
                    return float(G_all[token_idx].item())
            if isinstance(G_all, (list, tuple)):
                row = G_all[0] if len(G_all) > 0 else None
                if isinstance(row, torch.Tensor):
                    if row.dim() >= 1 and token_idx < row.size(0):
                        return float(row[token_idx].item())
                if isinstance(row, (list, tuple)) and token_idx < len(row):
                    val = row[token_idx]
                    return float(val.item()) if isinstance(val, torch.Tensor) else float(val)
        except Exception:
            pass
        return 0.0

    def _safe_extract_span(self, token_idx: int, dscd_outputs: Dict) -> float:
        try:
            if not isinstance(dscd_outputs, dict):
                return 0.0
            S_all = dscd_outputs.get("span_preds", None)
            if S_all is None:
                return 0.0
            if isinstance(S_all, torch.Tensor):
                if S_all.dim() >= 2:
                    if token_idx < S_all.size(1):
                        span_val = float(S_all[0, token_idx].item()) if S_all.dim() == 2 else float(S_all[token_idx].item())
                    elif S_all.dim() == 1 and token_idx < S_all.size(0):
                        span_val = float(S_all[token_idx].item())
                    else:
                        span_val = 0.0
                elif S_all.dim() == 1 and token_idx < S_all.size(0):
                    span_val = float(S_all[token_idx].item())
                else:
                    span_val = 0.0
            elif isinstance(S_all, (list, tuple)):
                row = S_all[0] if len(S_all) > 0 else None
                span_val = None
                if isinstance(row, torch.Tensor):
                    if row.dim() >= 1 and token_idx < row.size(0):
                        span_val = float(row[token_idx].item())
                elif isinstance(row, (list, tuple)) and token_idx < len(row):
                    val = row[token_idx]
                    span_val = float(val.item()) if isinstance(val, torch.Tensor) else float(val)
                if span_val is None:
                    span_val = 0.0
            else:
                span_val = 0.0
            if not np.isfinite(span_val):
                span_val = 0.0
            if span_val < 0.0 or span_val > 1.0:
                current_time = time.time()
                if self.span_clamp_warnings < 10 or (current_time - self.last_warning_time) > 60.0:
                    if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                        print(f"[TRG] Clamping span {span_val:.3f} -> [0.0, 1.0]")
                    self.span_clamp_warnings += 1
                    self.last_warning_time = current_time
                span_val = max(0.0, min(1.0, float(span_val)))
            return span_val
        except Exception:
            pass
        return 0.0

    def compute_span(self, sense_probs) -> float:
        try:
            if isinstance(sense_probs, dict):
                probs = list(sense_probs.values())
            else:
                probs = sense_probs
            if isinstance(probs, torch.Tensor):
                probs = probs.detach().cpu().numpy().flatten().tolist()
            if isinstance(probs, (np.ndarray, list)):
                probs = list(probs)
            if len(probs) < 2:
                return 0.0
            sorted_probs = sorted([float(p) for p in probs], reverse=True)
            span = float(sorted_probs[0]) - float(sorted_probs[1])
            return max(0.0, min(1.0, span))
        except Exception:
            return 0.0

    def _compute_sense_alternatives_fast(self, proto_probs: torch.Tensor, temperature: float = 1.0) -> List[Tuple[str, float]]:
        try:
            if not isinstance(proto_probs, torch.Tensor):
                proto_probs = torch.as_tensor(proto_probs, dtype=torch.float32)
            probs = proto_probs.flatten().float()
            if probs.numel() == 0:
                return [("unknown", 0.5)]
            probs = torch.clamp(probs, min=1e-10, max=1.0)
            if temperature != 1.0 and probs.numel() > 1:
                log_probs = torch.log(probs)
                scaled_log_probs = log_probs / float(temperature)
                probs = torch.softmax(scaled_log_probs, dim=0)
            if probs.numel() > 1:
                probs_sorted, indices = torch.sort(probs, descending=True)
                top_k = min(3, int(indices.numel()))
                return [(f"sense_{int(indices[i].item())}", float(probs_sorted[i].item())) for i in range(top_k)]
            else:
                return [("sense_0", float(probs[0].item()))]
        except Exception:
            return [("unknown", 0.5)]

    def _create_fallback_evidence(self, token_idx: int, tokens: List[str]) -> Dict:
        token = tokens[token_idx] if isinstance(tokens, list) and 0 <= token_idx < len(tokens) else "UNK"
        return {
            "token": token,
            "token_idx": token_idx,
            "evidence_tokens": [],
            "chosen_sense": ("unknown", 0.5),
            "alternatives": [],
            "uncertainty": 0.5,
            "gate": 0.0,
            "span": 0.0,
        }

    def get_homograph_tokens_from_dscd(self) -> Set[str]:
        homograph_tokens: Set[str] = set()
        try:
            if self.dscd_module is not None:
                if hasattr(self.dscd_module, "get_discovered_homographs"):
                    homograph_tokens = set(self.dscd_module.get_discovered_homographs())
                elif hasattr(self.dscd_module, "prototype_stores"):
                    for token, store in self.dscd_module.prototype_stores.items():
                        if hasattr(store, "size") and store.size() >= 2:
                            clean = str(token).replace("▁", "").replace("Ġ", "").replace("##", "").strip()
                            homograph_tokens.add(clean)
        except Exception:
            pass
        return homograph_tokens


class CompleteTRGWithExplanations(nn.Module):
    def __init__(self, embed_dim: Optional[int] = None, tokenizer=None, language: str = "bn", dscd_module=None):
        super().__init__()
        self.embed_dim = int(embed_dim) if embed_dim is not None else int(_TRG_GEN_EMBED)
        self.tokenizer = tokenizer
        self.language = language
        self.dscd_module = dscd_module
        if tokenizer is not None:
            try:
                self.special_tokens = _get_special_tokens_fn(tokenizer) if callable(_get_special_tokens_fn) else set(getattr(tokenizer, 'all_special_tokens', []))
            except Exception:
                self.special_tokens = set()
        else:
            self.special_tokens = set()
        self.template_system = ComprehensiveTRGExplanationTemplate()
        self.evidence_extractor = MemoryEfficientTRGExtractor(tokenizer, language=language, dscd_module=dscd_module)
        self.silver_buffer = deque(maxlen=int(_MAX_SILVER_BUFFER))
        self._silver_lock = threading.Lock()
        self.stats_reset_interval = 1000
        self.stats = {
            "explanations_generated": 0,
            "high_confidence_explanations": 0,
            "low_confidence_explanations": 0,
            "empty_evidence_count": 0,
            "total_evidence_tokens": 0,
            "tokens_filtered_word_start": 0,
            "tokens_filtered_validity": 0,
            "tokens_filtered_ambiguity": 0,
            "dscd_homographs_explained": 0,
        }
        self._stats_lock = threading.Lock()
        if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
            print("[TRG] Initialized:")
            print(f"  - Uncertainty: ADAPTIVE (base={_TRG_UNCERTAINTY_THRESHOLD:.2f})")
            print(f"  - Span: ADAPTIVE (base={_TRG_SPAN_THRESHOLD:.2f})")
            print(f"  - Temperature: {_TRG_TEMPERATURE:.2f}")
            print(f"  - TAU_HIGH: {_TAU_HIGH:.2f}")
            print(f"  - TAU_LOW: {_TAU_LOW:.2f}")
            print(f"  - TAU_ACCEPT: {_TAU_ACCEPT:.2f}")
            print(f"  - Evidence K: {_TRG_EVIDENCE_K}")
            print(f"  - Function availability: is_valid={_has_is_valid_token}, get_special={callable(_get_special_tokens_fn)}")

    def emergency_cleanup(self):
        try:
            self.clear_silver_buffer()
            self.reset_statistics()
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    def _update_stats(self, evidence: Dict, is_dscd_homograph: bool = False) -> None:
        with self._stats_lock:
            self.stats["explanations_generated"] += 1
            if is_dscd_homograph:
                self.stats["dscd_homographs_explained"] += 1
            if not evidence.get("evidence_tokens"):
                self.stats["empty_evidence_count"] += 1
            else:
                self.stats["total_evidence_tokens"] += len(evidence["evidence_tokens"])
            confidence = 0.5
            chosen = evidence.get("chosen_sense")
            if isinstance(chosen, (tuple, list)) and len(chosen) >= 2:
                try:
                    confidence = float(chosen[1])
                except Exception:
                    confidence = 0.5
            if confidence >= _TAU_ACCEPT:
                self.stats["high_confidence_explanations"] += 1
            elif confidence < _TRG_UNCERTAINTY_THRESHOLD:
                self.stats["low_confidence_explanations"] += 1
            if self.stats["explanations_generated"] >= self.stats_reset_interval:
                if _DEBUG_DISCOVERY:
                    current_stats = self.get_statistics()
                    print(f"\n[TRG-STATS] After {self.stats['explanations_generated']}:")
                    print(f"  High conf: {current_stats['high_confidence_rate']:.2%}")
                    print(f"  DSCD: {current_stats['dscd_homograph_rate']:.2%}")
                self.reset_statistics()

    def _add_to_silver_buffer(self, evidence: Dict, explanation: str, tokens: List[str]) -> None:
        try:
            conf = 0.5
            chosen = evidence.get("chosen_sense")
            if isinstance(chosen, (tuple, list)) and len(chosen) >= 2:
                conf = float(chosen[1])
            entry = {"token": str(evidence.get("token", "UNK"))[:20], "explanation": str(explanation)[:150], "confidence": conf}
            with self._silver_lock:
                self.silver_buffer.append(entry)
        except Exception:
            pass

    def generate_explanation_for_token(self, token_idx: int, tokens: List[str], dscd_outputs: Dict, token_word_map: Optional[dict] = None, decoder_attention: Optional[torch.Tensor] = None, is_dscd_homograph: bool = False) -> Tuple[str, Dict]:
        if self.training or not _ENABLE_TRG_INFERENCE:
            return "", {}
        if not isinstance(tokens, list) or not isinstance(token_idx, int):
            return "", {}
        if token_idx < 0 or token_idx >= len(tokens):
            return "", {}
        raw_token = tokens[token_idx]
        try:
            is_valid = False
            try:
                is_valid = bool(_is_valid_token_fn(raw_token, self.special_tokens, self.tokenizer, language=self.language))
            except Exception:
                is_valid = _fallback_is_valid_token(raw_token, self.special_tokens, self.tokenizer, self.language)
        except Exception:
            is_valid = _fallback_is_valid_token(raw_token, self.special_tokens, self.tokenizer, self.language)
        if not is_valid:
            return "", {}
        try:
            evidence = self.evidence_extractor.extract_evidence_efficiently(token_idx, tokens, dscd_outputs, token_word_map=token_word_map, decoder_attention=decoder_attention)
            explanation_text = self.template_system.generate_explanation(evidence)
            self._update_stats(evidence, is_dscd_homograph=is_dscd_homograph)
            self._add_to_silver_buffer(evidence, explanation_text, tokens)
            return explanation_text, evidence
        except Exception:
            return "", {}

    @staticmethod
    def _to_list_helper(x: Any) -> List[float]:
        if x is None:
            return []
        try:
            if isinstance(x, torch.Tensor):
                x = x.detach().cpu()
                if x.ndim == 0:
                    return [float(x.item())]
                if x.ndim == 1:
                    return [float(v.item()) for v in x]
                if x.ndim == 2:
                    if x.size(0) == 1:
                        return [float(v.item()) for v in x[0]]
                    else:
                        return [float(v.item()) for v in x.flatten()]
                if x.ndim >= 3:
                    return [float(v.item()) for v in x.flatten()]
            if isinstance(x, (list, tuple)):
                out: List[float] = []
                for v in x:
                    if isinstance(v, torch.Tensor):
                        v = v.detach().cpu()
                        if v.ndim == 0:
                            out.append(float(v.item()))
                        elif v.numel() > 0:
                            out.append(float(v.flatten()[0].item()))
                        else:
                            out.append(0.0)
                    elif isinstance(v, (int, float, np.number)):
                        out.append(float(v))
                    else:
                        try:
                            out.append(float(v))
                        except Exception:
                            out.append(0.0)
                return out
            if isinstance(x, (int, float, np.number)):
                return [float(x)]
            return [float(x)]
        except Exception:
            return []

    def compute_uncertainty_adaptive(self, proto_probs: Any, uncertainties: Any) -> Tuple[float, float]:
        try:
            U = self._to_list_helper(uncertainties)
            if not U or len(U) == 0:
                return float(_TRG_UNCERTAINTY_THRESHOLD), float(_TRG_UNCERTAINTY_THRESHOLD)
            U_arr = np.array(U, dtype=np.float32)
            U_arr = U_arr[np.isfinite(U_arr)]
            if len(U_arr) == 0:
                return float(_TRG_UNCERTAINTY_THRESHOLD), float(_TRG_UNCERTAINTY_THRESHOLD)
            median_u = float(np.median(U_arr))
            std_u = float(np.std(U_arr))
            adaptive_threshold = median_u + 0.5 * std_u
            adaptive_threshold = max(0.05, min(0.50, adaptive_threshold))
            return float(adaptive_threshold), float(median_u)
        except Exception:
            return float(_TRG_UNCERTAINTY_THRESHOLD), float(_TRG_UNCERTAINTY_THRESHOLD)

    def compute_span_adaptive(self, span_preds: Any) -> Tuple[float, float]:
        try:
            S = self._to_list_helper(span_preds)
            if not S or len(S) == 0:
                return float(_TRG_SPAN_THRESHOLD), float(_TRG_SPAN_THRESHOLD)
            S_arr = np.array(S, dtype=np.float32)
            S_arr = S_arr[np.isfinite(S_arr)]
            if len(S_arr) == 0:
                return float(_TRG_SPAN_THRESHOLD), float(_TRG_SPAN_THRESHOLD)
            median_s = float(np.median(S_arr))
            percentile_75 = float(np.percentile(S_arr, 75))
            adaptive_threshold = 0.5 * median_s + 0.5 * percentile_75
            adaptive_threshold = max(0.02, min(0.30, adaptive_threshold))
            return float(adaptive_threshold), float(median_s)
        except Exception:
            return float(_TRG_SPAN_THRESHOLD), float(_TRG_SPAN_THRESHOLD)

    def process_sentence_for_explanations(self, tokens: List[str], dscd_outputs: Dict, token_word_map: Optional[dict] = None, span_threshold: Optional[float] = None, uncertainty_threshold: Optional[float] = None, decoder_attention: Optional[torch.Tensor] = None, max_explanations: int = _MAX_EXPLANATIONS_PER_SENTENCE) -> List[Dict]:
        if self.training or not _ENABLE_TRG_INFERENCE:
            return []
        if span_threshold is None:
            span_threshold = float(_TRG_SPAN_THRESHOLD)
        if uncertainty_threshold is None:
            uncertainty_threshold = float(_TRG_UNCERTAINTY_THRESHOLD)
        explanations: List[Dict] = []
        try:
            if not tokens or not isinstance(tokens, list):
                return explanations
            if not isinstance(dscd_outputs, dict) or not dscd_outputs:
                return explanations
            U_all = dscd_outputs.get("uncertainties", [])
            S_all = dscd_outputs.get("span_preds", [])
            if not U_all or not U_all[0]:
                return explanations
            U = self._to_list_helper(U_all[0])
            S = self._to_list_helper(S_all[0]) if S_all and S_all[0] else [0.0] * len(tokens)
            seq_len = len(tokens)
            if len(U) < seq_len:
                U.extend([0.5] * (seq_len - len(U)))
            elif len(U) > seq_len:
                U = U[:seq_len]
            if len(S) < seq_len:
                S.extend([0.0] * (seq_len - len(S)))
            elif len(S) > seq_len:
                S = S[:seq_len]
            if not U:
                return explanations
            adaptive_u_threshold, median_u = self.compute_uncertainty_adaptive(dscd_outputs.get("proto_probs", None), U_all[0])
            adaptive_s_threshold, median_s = self.compute_span_adaptive(S_all[0] if S_all else None)
            strict_uncertainty = max(adaptive_u_threshold, uncertainty_threshold)
            strict_span = max(adaptive_s_threshold, span_threshold)
            if _DEBUG_DISCOVERY:
                print(f"[TRG-ADAPTIVE] U: median={median_u:.3f}, thresh={strict_uncertainty:.3f}")
                print(f"[TRG-ADAPTIVE] S: median={median_s:.3f}, thresh={strict_span:.3f}")
            dscd_homographs = self.evidence_extractor.get_homograph_tokens_from_dscd()
            candidates: List[Tuple[int, float, float, str, int, int]] = []
            local_stats = {"tokens_filtered_word_start": 0, "tokens_filtered_validity": 0, "tokens_filtered_ambiguity": 0}
            for idx in range(seq_len):
                tok = tokens[idx]
                clean_tok = tok.replace("▁", "").replace("Ġ", "").strip()
                if _is_punctuation_only(tok):
                    local_stats["tokens_filtered_validity"] += 1
                    continue
                if not _is_word_start(tok, token_word_map, idx):
                    local_stats["tokens_filtered_word_start"] += 1
                    continue
                try:
                    valid = bool(_is_valid_token_fn(tok, self.special_tokens, self.tokenizer, language=self.language))
                except Exception:
                    valid = _fallback_is_valid_token(tok, self.special_tokens, self.tokenizer, self.language)
                if not valid:
                    local_stats["tokens_filtered_validity"] += 1
                    continue
                if clean_tok in _FUNCTION_WORDS:
                    local_stats["tokens_filtered_validity"] += 1
                    continue
                if len(clean_tok) < 3 and not any('\u0980' <= c <= '\u09FF' for c in clean_tok):
                    local_stats["tokens_filtered_validity"] += 1
                    continue
                u = float(U[idx])
                s = float(S[idx])
                in_dscd = clean_tok in dscd_homographs
                if in_dscd:
                    priority = 1
                elif s >= strict_span and u >= strict_uncertainty:
                    priority = 2
                elif s >= strict_span:
                    priority = 3
                elif u >= strict_uncertainty:
                    priority = 4
                else:
                    local_stats["tokens_filtered_ambiguity"] += 1
                    continue
                candidates.append((idx, u, s, clean_tok, priority, idx))
            with self._stats_lock:
                self.stats["tokens_filtered_word_start"] += local_stats["tokens_filtered_word_start"]
                self.stats["tokens_filtered_validity"] += local_stats["tokens_filtered_validity"]
                self.stats["tokens_filtered_ambiguity"] += local_stats["tokens_filtered_ambiguity"]
            if not candidates:
                return explanations
            candidates.sort(key=lambda t: (t[4], -t[2], -t[1], t[5]))
            for (token_idx, u, s, clean_tok, priority, _) in candidates[: max_explanations]:
                try:
                    explanation_text, evidence = self.generate_explanation_for_token(token_idx, tokens, dscd_outputs, token_word_map=token_word_map, decoder_attention=decoder_attention, is_dscd_homograph=(priority == 1))
                    if explanation_text and evidence:
                        explanations.append({
                            "token_idx": token_idx,
                            "token": (token_word_map[token_idx] if token_word_map and token_idx in token_word_map else tokens[token_idx].replace("▁", "").replace("Ġ", "")),
                            "explanation": explanation_text,
                            "uncertainty": u,
                            "span": s,
                            "dscd_discovered": (priority == 1),
                            "priority": priority,
                        })
                except Exception:
                    continue
        except Exception:
            pass
        return explanations

    def get_statistics(self) -> Dict:
        with self._stats_lock:
            total = max(self.stats["explanations_generated"], 1)
            if self.stats["explanations_generated"] > 0:
                avg_evidence_tokens = (self.stats["total_evidence_tokens"] / total)
            else:
                avg_evidence_tokens = 0.0
            return {
                **self.stats.copy(),
                "high_confidence_rate": self.stats["high_confidence_explanations"] / total,
                "low_confidence_rate": self.stats["low_confidence_explanations"] / total,
                "empty_evidence_rate": self.stats["empty_evidence_count"] / total,
                "avg_evidence_tokens": avg_evidence_tokens,
                "silver_buffer_size": len(self.silver_buffer),
                "dscd_homograph_rate": self.stats["dscd_homographs_explained"] / total,
            }

    def reset_statistics(self) -> None:
        with self._stats_lock:
            self.stats = {
                "explanations_generated": 0,
                "high_confidence_explanations": 0,
                "low_confidence_explanations": 0,
                "empty_evidence_count": 0,
                "total_evidence_tokens": 0,
                "tokens_filtered_word_start": 0,
                "tokens_filtered_validity": 0,
                "tokens_filtered_ambiguity": 0,
                "dscd_homographs_explained": 0,
            }

    def clear_silver_buffer(self) -> None:
        with self._silver_lock:
            self.silver_buffer.clear()

    def test_trg(self, tokenizer=None) -> bool:
        print("\n" + "=" * 60)
        print("[TRG-TEST] Testing")
        print("=" * 60)
        try:
            tokens = ["▁আমি", "▁কল", "▁বন্ধ", "▁করেছি", "।"]
            dscd_outputs = {
                "proto_probs": [[torch.tensor([0.6, 0.4]) for _ in tokens]],
                "uncertainties": [[0.1, 0.5, 0.2, 0.1, 0.0]],
                "span_preds": [[0.05, 0.3, 0.1, 0.05, 0.0]],
                "gates": [[0.2, 0.8, 0.3, 0.2, 0.0]],
            }
            token_word_map = {0: "আমি", 1: "কল", 2: "বন্ধ", 3: "করেছি", 4: "।"}
            self.eval()
            explanations = self.process_sentence_for_explanations(tokens=tokens, dscd_outputs=dscd_outputs, token_word_map=token_word_map, max_explanations=3)
            print(f"  ✓ Generated {len(explanations)} explanations")
            if len(explanations) > 0:
                for i, expl in enumerate(explanations, 1):
                    print(f"    {i}. '{expl['token']}' (u={expl['uncertainty']:.2f})")
            stats = self.get_statistics()
            print(f"  ✓ Stats: {stats['explanations_generated']} total")
            self.reset_statistics()
            stats_after = self.get_statistics()
            assert stats_after["explanations_generated"] == 0
            print("  ✓ Reset OK")
            print("\n✓ All TRG tests passed")
            print("=" * 60 + "\n")
            return True
        except Exception as e:
            print(f"\n✗ Test failed: {e}")
            try:
                traceback.print_exc()
            except Exception:
                pass
            print("=" * 60 + "\n")
            return False


print("\n" + "=" * 80)
print("Cell 5: TRG Module - VERIFIED (FIXED)")
print("=" * 80)
print("Configuration:")
print(f"  - Uncertainty: ADAPTIVE (base={_TRG_UNCERTAINTY_THRESHOLD:.2f})")
print(f"  - Span: ADAPTIVE (base={_TRG_SPAN_THRESHOLD:.2f})")
print(f"  - Temperature: {_TRG_TEMPERATURE:.2f}")
print(f"  - TAU_HIGH: {_TAU_HIGH:.2f}")
print(f"  - TAU_LOW: {_TAU_LOW:.2f}")
print(f"  - TAU_ACCEPT: {_TAU_ACCEPT:.2f}")
print(f"  - Evidence K: {_TRG_EVIDENCE_K}")
print(f"  - Max Explanations: {_MAX_EXPLANATIONS_PER_SENTENCE}")
print("=" * 80 + "\n")


Cell 5: TRG Module - VERIFIED (FIXED)
Configuration:
  - Uncertainty: ADAPTIVE (base=0.15)
  - Span: ADAPTIVE (base=0.20)
  - Temperature: 1.00
  - TAU_HIGH: 0.85
  - TAU_LOW: 0.15
  - TAU_ACCEPT: 0.70
  - Evidence K: 3
  - Max Explanations: 10



In [9]:
# ===========================================================================================
# CELL 6: DUAL-PATH TATN MODEL - PATH 1 (WORD-LEVEL) + PATH 2 (SUBWORD-LEVEL) (INDICTRANS2) - FIXED
# ===========================================================================================
from typing import List, Dict, Optional, Any, Tuple, Union
import traceback
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput
import threading
import gc
import time
import re

_SOURCE_LANGUAGE = str(globals().get("SOURCE_LANGUAGE", "ben_Beng"))
_TARGET_LANGUAGE = str(globals().get("TARGET_LANGUAGE", "eng_Latn"))
_MODEL_NAME = str(globals().get("MODEL_NAME", "ai4bharat/indictrans2-indic-en-dist-200M"))

def _get_int_global(name: str, default: int) -> int:
    try:
        val = globals().get(name)
        if val is not None:
            return int(val)
    except Exception:
        pass
    return default

def _get_float_global(name: str, default: float) -> float:
    try:
        val = globals().get(name)
        if val is not None:
            return float(val)
    except Exception:
        pass
    return default

def _get_bool_global(name: str, default: bool) -> bool:
    try:
        val = globals().get(name)
        if val is not None:
            return bool(val)
    except Exception:
        pass
    return default

_DSCD_BUFFER_SIZE               = _get_int_global("DSCD_BUFFER_SIZE", 500)
_DSCD_MAX_PROTOS                = _get_int_global("DSCD_MAX_PROTOS", 10)
_DSCD_N_MIN                     = _get_int_global("DSCD_N_MIN", 2)
_DSCD_DISPERSION_THRESHOLD      = _get_float_global("DSCD_DISPERSION_THRESHOLD", 0.15)

_ENABLE_ASBN_TRAINING           = _get_bool_global("ENABLE_ASBN_TRAINING", True)
_ENABLE_ASBN_INFERENCE          = _get_bool_global("ENABLE_ASBN_INFERENCE", True)
_ENABLE_TRG_INFERENCE           = _get_bool_global("ENABLE_TRG_INFERENCE", True)
_MEMORY_CLEANUP_FREQUENCY       = _get_int_global("MEMORY_CLEANUP_FREQUENCY", 50)

_NUM_GPUS                       = _get_int_global("NUM_GPUS", torch.cuda.device_count() if torch.cuda.is_available() else 0)
_USE_GC                         = _get_bool_global("GRADIENT_CHECKPOINTING", True)
_DSCD_ENABLE_TRAINING_CLUSTERING = _get_bool_global("DSCD_ENABLE_TRAINING_CLUSTERING", True)

_LAMBDA_ASBN                    = _get_float_global("LAMBDA_ASBN", 0.0)
_LAMBDA_DSCD                    = _get_float_global("LAMBDA_DSCD", 0.15)

_VERBOSE_LOGGING                = _get_bool_global("VERBOSE_LOGGING", False)
_DEBUG_DISCOVERY                = bool(globals().get("DEBUG_DISCOVERY", False))
_DEBUG_TIMING                   = bool(globals().get("DEBUG_TIMING", False))

_PERIODIC_DISCOVERY_FREQUENCY   = _get_int_global("PERIODIC_DISCOVERY_FREQUENCY", 100)
_VALIDATION_CHECK_INTERVAL      = _get_int_global("VALIDATION_CHECK_INTERVAL", 200)

_SPAN_THRESHOLD                 = _get_float_global("SPAN_THRESHOLD", 0.20)
_UNCERTAINTY_THRESHOLD          = _get_float_global("UNCERTAINTY_THRESHOLD", 0.15)

_TRG_UNCERTAINTY_THRESHOLD      = _get_float_global("TRG_UNCERTAINTY_THRESHOLD", _get_float_global("TAU_LOW", 0.15))
_TAU_LOW                        = _get_float_global("TAU_LOW", 0.15)

_TRAIN_DOMAIN                   = _get_int_global("TRAIN_DOMAIN", 0)
_TEST_DOMAIN                    = _get_int_global("TEST_DOMAIN", 1)
_USE_DOMAIN_LABELS              = _get_bool_global("USE_DOMAIN_LABELS", True)

_LABEL_SMOOTHING_EPS            = float(globals().get("LABEL_SMOOTHING", 0.1))
_RDROP_ALPHA                    = float(globals().get("RDROP_ALPHA", 5.0))

_has_reconstruct_word_spans     = callable(globals().get("reconstruct_word_spans", None))
_has_postprocess_indictrans2    = callable(globals().get("postprocess_for_indictrans2", None))

_BENGALI_PUNCT_SET = set(['।', '॥'])
_COMMON_PUNCT_SET  = set(['.', ',', ';', ':', '!', '?', '"', "'", '-', '(', ')', '[', ']', '{', '}', '/', '\\'])
_PUNCT_SET = _BENGALI_PUNCT_SET | _COMMON_PUNCT_SET


def _is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = token.replace("▁", "").replace("Ġ", "").replace("##", "").replace("</w>", "").strip()
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET or clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in _PUNCT_SET for c in clean)


def _safe_get_last_hidden_state(enc_output):
    if enc_output is None:
        return None
    if hasattr(enc_output, "last_hidden_state"):
        return enc_output.last_hidden_state
    if isinstance(enc_output, (list, tuple)) and len(enc_output) > 0:
        return enc_output[0]
    return None


def build_token_word_map_sentencepiece(token_strings: List[str], fallback: bool = True) -> Dict[int, str]:
    word_map: Dict[int, str] = {}
    current_word = ""
    start_idx = None
    for i, token in enumerate(token_strings):
        if not token or token.startswith("<") or token.startswith("["):
            continue
        if token.startswith("▁"):
            if current_word and start_idx is not None:
                clean = current_word.replace("▁", "").strip()
                if clean and len(clean) >= 2 and not _is_punctuation_only(current_word):
                    word_map[start_idx] = clean
            current_word = token
            start_idx = i
        else:
            current_word += token
    if current_word and start_idx is not None:
        clean = current_word.replace("▁", "").strip()
        if clean and len(clean) >= 2 and not _is_punctuation_only(current_word):
            word_map[start_idx] = clean
    if fallback and not word_map:
        for i, tok in enumerate(token_strings):
            clean = tok.replace("▁", "").strip()
            if clean and len(clean) >= 2 and not _is_punctuation_only(tok):
                word_map[i] = clean
    return word_map


def normalize_dscd_outputs(
    raw: Dict[str, Any],
    batch_size: int,
    seq_len: int,
    device: torch.device,
    embed_dim: int,
    fallback_h: Optional[torch.Tensor] = None
) -> Dict[str, Any]:
    if fallback_h is None:
        fallback_h_augmented = torch.zeros(batch_size, seq_len, embed_dim, device=device, dtype=torch.float32)
    else:
        fallback_h_augmented = fallback_h.detach().clone()
    defaults = {
        "h_augmented": fallback_h_augmented,
        "proto_probs": [[torch.tensor([1.0], device=device, dtype=torch.float32) for _ in range(seq_len)] for _ in range(batch_size)],
        "uncertainties": [[torch.tensor(0.0, device=device, dtype=torch.float32) for _ in range(seq_len)] for _ in range(batch_size)],
        "gates": [[torch.tensor(0.0, device=device, dtype=torch.float32) for _ in range(seq_len)] for _ in range(batch_size)],
        "span_preds": [[torch.tensor(0.0, device=device, dtype=torch.float32) for _ in range(seq_len)] for _ in range(batch_size)],
        "proto_assignments": [torch.zeros(seq_len, dtype=torch.long, device=device) for _ in range(batch_size)],
    }
    if not isinstance(raw, dict):
        return defaults
    out = defaults.copy()
    try:
        if "h_augmented" in raw and raw["h_augmented"] is not None:
            h = raw["h_augmented"]
            if isinstance(h, torch.Tensor) and h.shape == (batch_size, seq_len, embed_dim):
                out["h_augmented"] = h.to(device)
            else:
                try:
                    out["h_augmented"] = h.to(device).reshape(batch_size, seq_len, embed_dim)
                except Exception:
                    out["h_augmented"] = fallback_h_augmented
    except Exception:
        out["h_augmented"] = fallback_h_augmented
    for list_key in ("proto_probs", "uncertainties", "gates", "span_preds"):
        if list_key in raw and raw[list_key] is not None:
            try:
                val = raw[list_key]
                if isinstance(val, list) and len(val) == batch_size:
                    safe_batch = []
                    for b_row in val:
                        if isinstance(b_row, list):
                            safe_row = []
                            for t_idx in range(seq_len):
                                try:
                                    if t_idx < len(b_row):
                                        v = b_row[t_idx]
                                        if isinstance(v, torch.Tensor):
                                            safe_row.append(v.detach().to(device))
                                        else:
                                            safe_row.append(torch.as_tensor(v, device=device, dtype=torch.float32))
                                    else:
                                        if list_key == "proto_probs":
                                            safe_row.append(torch.tensor([1.0], device=device, dtype=torch.float32))
                                        else:
                                            safe_row.append(torch.tensor(0.0, device=device, dtype=torch.float32))
                                except Exception:
                                    safe_row.append(torch.tensor(0.0, device=device, dtype=torch.float32))
                            safe_batch.append(safe_row)
                        else:
                            if list_key == "proto_probs":
                                safe_batch.append([torch.tensor([1.0], device=device, dtype=torch.float32) for _ in range(seq_len)])
                            else:
                                safe_batch.append([torch.tensor(0.0, device=device, dtype=torch.float32) for _ in range(seq_len)])
                    out[list_key] = safe_batch
            except Exception:
                pass
    try:
        if "proto_assignments" in raw and raw["proto_assignments"] is not None:
            pa = raw["proto_assignments"]
            if isinstance(pa, list) and len(pa) == batch_size:
                safe_pa = []
                for b_row in pa:
                    try:
                        if isinstance(b_row, torch.Tensor):
                            safe_pa.append(b_row.detach().to(device).long())
                        else:
                            safe_pa.append(torch.tensor(b_row, dtype=torch.long, device=device))
                    except Exception:
                        safe_pa.append(torch.zeros(seq_len, dtype=torch.long, device=device))
                out["proto_assignments"] = safe_pa
    except Exception:
        pass
    return out


def _norm_scalar_matrix(uncertainties, gates, gate_threshold=0.01):
    final_normalized = []
    batch_size = len(uncertainties)
    for b in range(batch_size):
        u_row = uncertainties[b]
        g_row = gates[b]
        seq_len = len(u_row)
        safe_row = []
        for t in range(seq_len):
            try:
                u_val = float(u_row[t]) if t < len(u_row) else 0.0
                g_val = float(g_row[t]) if t < len(g_row) else 0.0
                if g_val < gate_threshold:
                    norm_val = 0.0
                else:
                    norm_val = max(0.0, min(1.0, u_val))
                safe_row.append(norm_val)
            except Exception:
                safe_row.append(0.0)
        final_normalized.append(safe_row)
    return final_normalized


def _norm_proto_probs(proto_probs):
    return [[pp if isinstance(pp, torch.Tensor) else torch.tensor([1.0]) for pp in row] for row in proto_probs]


def _to_vec(x):
    if isinstance(x, torch.Tensor):
        return x.flatten().tolist()
    elif isinstance(x, (list, tuple)):
        return list(x)
    elif isinstance(x, (int, float)):
        return [float(x)]
    else:
        return [0.0]


def _extract_words_from_text(text: str) -> List[str]:
    if not text or not isinstance(text, str):
        return []
    text = text.strip()
    if not text:
        return []
    try:
        words = re.findall(r'[\u0980-\u09FF]+|[a-zA-Z]+|\d+', text)
        words = [w for w in words if w and len(w) > 0 and not _is_punctuation_only(w)]
        return words if words else []
    except Exception:
        return []


def get_cluster_count(model: torch.nn.Module) -> int:
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd and hasattr(dscd, "get_prototype_summary"):
            summary = dscd.get_prototype_summary()
            return summary.get("total_prototypes", 0)
    except Exception:
        pass
    return 0


def get_homograph_cluster_count(model: torch.nn.Module) -> int:
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd and hasattr(dscd, "get_prototype_summary"):
            summary = dscd.get_prototype_summary()
            return summary.get("num_homographs", 0)
    except Exception:
        pass
    return 0


class MemoryOptimizedTATNWithExplanations(nn.Module):
    def __init__(self, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.global_step = 0
        self._step_lock = threading.Lock()
        self.last_discovery_step = 0
        self.last_validation_step = 0

        self.device = globals().get("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        self.resize_model_embeddings = bool(globals().get("RESIZE_MODEL_EMBEDDINGS", True))

        try:
            self.base_model = AutoModelForSeq2SeqLM.from_pretrained(
                _MODEL_NAME,
                torch_dtype=torch.float32,
                use_cache=False,
                trust_remote_code=True
            )
            try:
                self.base_model.config.use_cache = False
            except Exception:
                pass
        except Exception as e:
            raise RuntimeError(f"Failed to load model {_MODEL_NAME}: {e}")

        try:
            self.base_model.to(self.device)
        except Exception:
            pass

        tokenizer_vocab_size = len(self.tokenizer) if hasattr(self.tokenizer, "__len__") else getattr(self.tokenizer, "vocab_size", 0)

        encoder_vocab_size = int(getattr(self.base_model.config, "encoder_vocab_size", 0) or 0)
        decoder_vocab_size = int(getattr(self.base_model.config, "decoder_vocab_size", 0) or 0)
        model_vocab_size   = int(getattr(self.base_model.config, "vocab_size", 0) or 0)

        print(f"[TATN-INIT] Vocabulary Analysis:")
        print(f"  Tokenizer vocab: {tokenizer_vocab_size}")
        print(f"  Config vocab_size: {model_vocab_size}")
        print(f"  Config encoder_vocab_size: {encoder_vocab_size}")
        print(f"  Config decoder_vocab_size: {decoder_vocab_size}")

        if encoder_vocab_size > 0:
            self.encoder_vocab_size = encoder_vocab_size
        elif decoder_vocab_size > 0:
            self.encoder_vocab_size = decoder_vocab_size
        else:
            self.encoder_vocab_size = model_vocab_size

        if decoder_vocab_size > 0:
            self.decoder_vocab_size = decoder_vocab_size
        elif encoder_vocab_size > 0:
            self.decoder_vocab_size = encoder_vocab_size
        else:
            self.decoder_vocab_size = model_vocab_size

        self.vocab_size = max(self.encoder_vocab_size, tokenizer_vocab_size, self.decoder_vocab_size)

        try:
            enc_emb = self.base_model.get_input_embeddings()
            dec_emb = self.base_model.get_output_embeddings()
            actual_enc_vocab = enc_emb.num_embeddings if enc_emb is not None else None
            actual_dec_vocab = dec_emb.num_embeddings if dec_emb is not None else None

            if actual_enc_vocab is not None:
                self.encoder_vocab_size = actual_enc_vocab
            if actual_dec_vocab is not None:
                self.decoder_vocab_size = actual_dec_vocab

            if tokenizer_vocab_size and tokenizer_vocab_size > max(self.encoder_vocab_size, self.decoder_vocab_size):
                if self.resize_model_embeddings:
                    try:
                        new_size = int(tokenizer_vocab_size)
                        print(f"[TATN-INIT] Resizing model embeddings to tokenizer size: {new_size}")
                        self.base_model.resize_token_embeddings(new_size)
                        enc_emb = self.base_model.get_input_embeddings()
                        dec_emb = self.base_model.get_output_embeddings()
                        if enc_emb is not None:
                            self.encoder_vocab_size = enc_emb.num_embeddings
                        if dec_emb is not None:
                            self.decoder_vocab_size = dec_emb.num_embeddings
                        self.vocab_size = max(self.encoder_vocab_size, self.decoder_vocab_size)
                    except Exception as e:
                        print(f"[TATN-INIT] resize_token_embeddings failed: {e}")
                else:
                    print(f"[TATN-INIT] Tokenizer has more tokens ({tokenizer_vocab_size}) than model vocab ({self.vocab_size}). Set RESIZE_MODEL_EMBEDDINGS=True to resize automatically.")

        except Exception as e:
            print(f"[TATN-INIT] Embedding inspection failed: {e}")

        print(f"[TATN-INIT] Final - Encoder: {self.encoder_vocab_size}, Decoder: {self.decoder_vocab_size}, Tokenizer: {tokenizer_vocab_size}")

        # Resolve language token IDs — validated against encoder/tokenizer vocab (not decoder)
        try:
            src_token_id = None
            tgt_token_id = None
            tk = self.tokenizer

            if hasattr(tk, "lang_code_to_id"):
                try:
                    src_token_id = tk.lang_code_to_id.get(_SOURCE_LANGUAGE, None)
                    tgt_token_id = tk.lang_code_to_id.get(_TARGET_LANGUAGE, None)
                except Exception:
                    src_token_id = None
                    tgt_token_id = None

            if src_token_id is None:
                try:
                    val = tk.convert_tokens_to_ids(_SOURCE_LANGUAGE)
                    if isinstance(val, list):
                        src_token_id = val[0] if val else None
                    else:
                        src_token_id = val
                except Exception:
                    try:
                        src_token_id = tk.convert_tokens_to_ids([_SOURCE_LANGUAGE])[0]
                    except Exception:
                        src_token_id = None

            if tgt_token_id is None:
                try:
                    val = tk.convert_tokens_to_ids(_TARGET_LANGUAGE)
                    if isinstance(val, list):
                        tgt_token_id = val[0] if val else None
                    else:
                        tgt_token_id = val
                except Exception:
                    try:
                        tgt_token_id = tk.convert_tokens_to_ids([_TARGET_LANGUAGE])[0]
                    except Exception:
                        tgt_token_id = None

            if (src_token_id is None or src_token_id == 0) and hasattr(tk, "get_vocab"):
                try:
                    vocab = tk.get_vocab()
                    if _SOURCE_LANGUAGE in vocab:
                        src_token_id = int(vocab[_SOURCE_LANGUAGE])
                    if _TARGET_LANGUAGE in vocab:
                        tgt_token_id = int(vocab[_TARGET_LANGUAGE])
                except Exception:
                    pass

            added_tokens = []
            try:
                if (src_token_id is None or src_token_id < 0) and hasattr(tk, "add_tokens"):
                    added = tk.add_tokens([_SOURCE_LANGUAGE])
                    if added and added > 0:
                        added_tokens.append(_SOURCE_LANGUAGE)
                if (tgt_token_id is None or tgt_token_id < 0) and hasattr(tk, "add_tokens"):
                    added = tk.add_tokens([_TARGET_LANGUAGE])
                    if added and added > 0:
                        added_tokens.append(_TARGET_LANGUAGE)
            except Exception:
                pass

            if added_tokens and self.resize_model_embeddings:
                try:
                    new_size = len(tk)
                    print(f"[TATN-INIT] Resizing embeddings after adding tokens: {new_size}")
                    self.base_model.resize_token_embeddings(new_size)
                    enc_emb = self.base_model.get_input_embeddings()
                    dec_emb = self.base_model.get_output_embeddings()
                    if enc_emb is not None:
                        self.encoder_vocab_size = enc_emb.num_embeddings
                    if dec_emb is not None:
                        self.decoder_vocab_size = dec_emb.num_embeddings
                    self.vocab_size = max(self.encoder_vocab_size, self.decoder_vocab_size)
                except Exception as e:
                    print(f"[TATN-INIT] resize_token_embeddings after adding tokens failed: {e}")

            try:
                if hasattr(tk, "convert_tokens_to_ids"):
                    try:
                        if src_token_id is None:
                            val = tk.convert_tokens_to_ids(_SOURCE_LANGUAGE)
                            src_token_id = val[0] if isinstance(val, list) and val else val
                        if tgt_token_id is None:
                            val = tk.convert_tokens_to_ids(_TARGET_LANGUAGE)
                            tgt_token_id = val[0] if isinstance(val, list) and val else val
                    except Exception:
                        pass
                if (src_token_id is None or src_token_id < 0) and hasattr(tk, "get_vocab"):
                    vocab = tk.get_vocab()
                    src_token_id = int(vocab.get(_SOURCE_LANGUAGE, -1))
                    tgt_token_id = int(vocab.get(_TARGET_LANGUAGE, -1))
            except Exception:
                pass

            if src_token_id is None or tgt_token_id is None or src_token_id < 0 or tgt_token_id < 0:
                msg = (
                    f"Language tokens not found for Source='{_SOURCE_LANGUAGE}' or Target='{_TARGET_LANGUAGE}'.\n"
                    f"Found: src={src_token_id}, tgt={tgt_token_id}.\n"
                    "Action: ensure tokenizer contains these language tokens (e.g., tokenizer.add_tokens([...]))\n"
                    "If you added tokens, set RESIZE_MODEL_EMBEDDINGS=True so model embeddings are resized."
                )
                raise RuntimeError(msg)

            try:
                src_token_id = int(src_token_id)
            except Exception:
                src_token_id = int(src_token_id) if src_token_id is not None else -1
            try:
                tgt_token_id = int(tgt_token_id)
            except Exception:
                tgt_token_id = int(tgt_token_id) if tgt_token_id is not None else -1

            # Validate against encoder_vocab_size (which covers full tokenizer vocab = 122706)
            # tgt_token_id for eng_Latn may be > decoder_vocab_size (32296) but valid in encoder/tokenizer vocab
            if src_token_id >= self.encoder_vocab_size or tgt_token_id >= self.encoder_vocab_size:
                if self.resize_model_embeddings:
                    try:
                        new_size = max(int(src_token_id) + 1, int(tgt_token_id) + 1, len(self.tokenizer))
                        print(f"[TATN-INIT] Resizing embeddings to accommodate language token ids: {new_size}")
                        self.base_model.resize_token_embeddings(new_size)
                        enc_emb = self.base_model.get_input_embeddings()
                        dec_emb = self.base_model.get_output_embeddings()
                        if enc_emb is not None:
                            self.encoder_vocab_size = enc_emb.num_embeddings
                        if dec_emb is not None:
                            self.decoder_vocab_size = dec_emb.num_embeddings
                        self.vocab_size = max(self.encoder_vocab_size, self.decoder_vocab_size)
                    except Exception as e:
                        raise RuntimeError(f"Language token ids exceed vocab and resize failed: {e}")
                else:
                    raise RuntimeError(
                        f"Language token ids out of bounds and RESIZE_MODEL_EMBEDDINGS is False.\n"
                        f"  src_id={src_token_id} >= encoder_vocab={self.encoder_vocab_size}\n"
                        f"  tgt_id={tgt_token_id} >= encoder_vocab={self.encoder_vocab_size}\n"
                    )

            # Always set decoder_start_token_id and forced_bos_token_id from tgt_token_id
            self.base_model.config.decoder_start_token_id = int(tgt_token_id)
            try:
                self.base_model.config.forced_bos_token_id = int(tgt_token_id)
            except Exception:
                pass

            self.src_token_id = int(src_token_id)
            self.tgt_token_id = int(tgt_token_id)

            print(f"[TATN-INIT] Language tokens resolved: {_SOURCE_LANGUAGE}={self.src_token_id}, {_TARGET_LANGUAGE}={self.tgt_token_id}")

        except Exception as e:
            raise RuntimeError(f"Language token setup failed: {e}")

        try:
            if _USE_GC:
                if hasattr(self.base_model, "gradient_checkpointing_enable"):
                    try:
                        self.base_model.gradient_checkpointing_enable()
                    except Exception:
                        pass
                else:
                    try:
                        self.base_model.config.gradient_checkpointing = True
                    except Exception:
                        pass
        except Exception:
            pass

        # Resolve actual embed_dim from embedding layer weights (not config.d_model which is wrong 1024)
        try:
            embedding_layer = self.base_model.get_input_embeddings()
            actual_embed_dim = (
                getattr(embedding_layer, "embedding_dim", None)
                or getattr(embedding_layer, "out_features", None)
                or (embedding_layer.weight.shape[-1] if hasattr(embedding_layer, "weight") else None)
                or int(getattr(self.base_model.config, "d_model", getattr(self.base_model.config, "hidden_size", 512)))
            )
        except Exception:
            actual_embed_dim = int(getattr(self.base_model.config, "d_model", getattr(self.base_model.config, "hidden_size", 512)))

        embed_dim = int(getattr(self.base_model.config, "d_model", getattr(self.base_model.config, "hidden_size", 512)))
        if actual_embed_dim is not None and actual_embed_dim != embed_dim:
            print(f"[TATN-INIT] WARNING: config d_model={embed_dim} differs from embedding_dim={actual_embed_dim}. Using embedding_dim={actual_embed_dim}.")
            embed_dim = int(actual_embed_dim)

        # Store embed_dim as instance attribute — used in all forward methods
        self.embed_dim = embed_dim

        dscd_cls = globals().get("MemoryEfficientDSCDOnline", None)
        if callable(dscd_cls):
            try:
                self.dscd = dscd_cls(
                    embed_dim=self.embed_dim,
                    tokenizer=tokenizer,
                    buffer_size=_DSCD_BUFFER_SIZE,
                    max_protos=_DSCD_MAX_PROTOS,
                    n_min=_DSCD_N_MIN,
                    language=_SOURCE_LANGUAGE,
                    dispersion_threshold=_DSCD_DISPERSION_THRESHOLD,
                    enable_training_clustering=_DSCD_ENABLE_TRAINING_CLUSTERING,
                    max_clustering_points=500,
                    max_candidates_per_step=1,
                )
                dscd_embed_dim = getattr(self.dscd, "embed_dim", None)
                if dscd_embed_dim is not None and dscd_embed_dim != self.embed_dim:
                    print(f"[TATN-INIT] DSCD embed_dim ({dscd_embed_dim}) != model embed_dim ({self.embed_dim}). Continuing but DSCD may behave unexpectedly.")
            except Exception as e:
                raise RuntimeError(f"Failed to instantiate MemoryEfficientDSCDOnline: {e}")
        else:
            raise RuntimeError("MemoryEfficientDSCDOnline not found in globals()")

        asbn_cls = globals().get("MemoryEfficientASBNModule", None)
        if callable(asbn_cls):
            try:
                self.asbn = asbn_cls(self.embed_dim, tokenizer, language=_SOURCE_LANGUAGE)
                asbn_embed_dim = getattr(self.asbn, "embed_dim", None)
                if asbn_embed_dim is not None and asbn_embed_dim != self.embed_dim:
                    print(f"[TATN-INIT] ASBN embed_dim ({asbn_embed_dim}) != model embed_dim ({self.embed_dim}). Continuing but ASBN may behave unexpectedly.")
            except Exception as e:
                print(f"[TATN-INIT] ASBN init failed: {e}, using stub")
                self.asbn = self._build_stub_asbn()
        else:
            self.asbn = self._build_stub_asbn()

        trg_cls = globals().get("CompleteTRGWithExplanations", None)
        if callable(trg_cls):
            try:
                self.trg = trg_cls(self.embed_dim, tokenizer, language=_SOURCE_LANGUAGE, dscd_module=self.dscd)
            except Exception as e:
                print(f"[TATN-INIT] TRG init failed: {e}, using stub")
                self.trg = self._build_stub_trg()
        else:
            self.trg = self._build_stub_trg()

        label_smoothing_cls = globals().get("LabelSmoothingLoss", None)
        if callable(label_smoothing_cls):
            try:
                self.label_smoothing_loss = label_smoothing_cls(
                    num_classes=self.decoder_vocab_size,
                    smoothing=_LABEL_SMOOTHING_EPS,
                    ignore_index=-100
                )
            except Exception:
                self.label_smoothing_loss = None
        else:
            self.label_smoothing_loss = None

        rdrop_cls = globals().get("RDropLoss", None)
        if callable(rdrop_cls):
            try:
                self.rdrop_loss = rdrop_cls(alpha=_RDROP_ALPHA)
            except Exception:
                self.rdrop_loss = None
        else:
            self.rdrop_loss = None

        try:
            embedding_layer = self.base_model.get_input_embeddings()
            if embedding_layer is None:
                print("[TATN-INIT] WARNING: Model has no input embeddings layer!")
            else:
                actual_dim_check = (
                    getattr(embedding_layer, "embedding_dim", None)
                    or getattr(embedding_layer, "out_features", None)
                    or (embedding_layer.weight.shape[-1] if hasattr(embedding_layer, "weight") else None)
                    or self.embed_dim
                )
                if actual_dim_check != self.embed_dim:
                    print(f"[TATN-INIT] WARNING: Using embed_dim={self.embed_dim}, embedding layer dim={actual_dim_check}")
                else:
                    print(f"[TATN-INIT] ✅ Embedding layer verified: dim={self.embed_dim}, vocab={self.encoder_vocab_size}")
        except Exception as e:
            print(f"[TATN-INIT] Embedding layer verification warning: {e}")

    def print_vocabulary_diagnostic(self, train_loader=None):
        print("\n" + "=" * 80)
        print("VOCABULARY DIAGNOSTIC")
        print("=" * 80)
        print("\n[CONFIG] Model configuration:")
        print(f"  config.vocab_size: {self.base_model.config.vocab_size}")
        print(f"  config.encoder_vocab_size: {getattr(self.base_model.config, 'encoder_vocab_size', 'NOT FOUND')}")
        print(f"  config.decoder_vocab_size: {getattr(self.base_model.config, 'decoder_vocab_size', 'NOT FOUND')}")
        print("\n[EMBEDDINGS] Actual embedding layer sizes:")
        try:
            enc_emb = self.base_model.get_input_embeddings()
            print(f"  Encoder embeddings: {enc_emb.num_embeddings if enc_emb else 'None'}")
        except Exception as e:
            print(f"  Encoder embeddings: ERROR - {e}")
        try:
            dec_emb = self.base_model.get_output_embeddings()
            print(f"  Decoder embeddings: {dec_emb.num_embeddings if dec_emb else 'None'}")
        except Exception as e:
            print(f"  Decoder embeddings: ERROR - {e}")
        print(f"\n[CELL6] Stored vocab sizes:")
        print(f"  model.vocab_size: {self.vocab_size}")
        print(f"  model.encoder_vocab_size: {getattr(self, 'encoder_vocab_size', 'NOT SET')}")
        print(f"  model.decoder_vocab_size: {getattr(self, 'decoder_vocab_size', 'NOT SET')}")
        print(f"  model.embed_dim: {getattr(self, 'embed_dim', 'NOT SET')}")
        print(f"  model.src_token_id: {getattr(self, 'src_token_id', 'NOT SET')}")
        print(f"  model.tgt_token_id: {getattr(self, 'tgt_token_id', 'NOT SET')}")
        print(f"\n[TOKENIZER] Tokenizer info:")
        try:
            print(f"  len(tokenizer): {len(self.tokenizer)}")
        except Exception:
            print("  len(tokenizer): ERROR")
        print(f"  tokenizer.vocab_size: {getattr(self.tokenizer, 'vocab_size', 'NOT FOUND')}")
        if train_loader is not None:
            print("\n[LABELS] Checking first batch labels...")
            try:
                first_batch = next(iter(train_loader))
                if "labels" in first_batch:
                    labels = first_batch["labels"]
                    valid_labels = labels[labels != -100]
                    print(f"  Labels shape: {labels.shape}")
                    print(f"  Valid labels (non -100): {len(valid_labels)}")
                    print(f"  Min label: {valid_labels.min().item() if len(valid_labels) > 0 else 'N/A'}")
                    print(f"  Max label: {valid_labels.max().item() if len(valid_labels) > 0 else 'N/A'}")
                    if len(valid_labels) > 0:
                        max_label = valid_labels.max().item()
                        if max_label >= self.decoder_vocab_size:
                            print(f"\n  ❌ PROBLEM FOUND!")
                            print(f"     Max label ({max_label}) >= decoder_vocab_size ({self.decoder_vocab_size})")
                            print(f"     Number of out-of-bounds labels: {torch.sum(valid_labels >= self.decoder_vocab_size).item()}")
                        else:
                            print(f"  ✅ Labels are within decoder vocab bounds")
            except Exception as e:
                print(f"  ERROR checking labels: {e}")
        print("=" * 80 + "\n")

    def _build_stub_asbn(self):
        class _StubASBN(nn.Module):
            def forward(self, h, **kwargs):
                dev = h.device if isinstance(h, torch.Tensor) else torch.device("cpu")
                zero = torch.tensor(0.0, device=dev, requires_grad=True)
                return h, {"encoder_loss": zero, "adversarial_loss": zero, "domain_loss": zero, "domain_accuracy": zero}
            def critic_parameters(self):
                return []
            def reset_stats(self):
                pass
            def get_detailed_stats(self):
                return {"domain_loss": 0.0, "domain_accuracy": 0.0, "source_accuracy": 0.0, "target_accuracy": 0.0, "asbn_loss": 0.0, "num_updates": 0}
            def get_asbn_stats(self):
                return self.get_detailed_stats()
        return _StubASBN()

    def _build_stub_trg(self):
        class _StubTRG:
            def process_sentence_for_explanations(self, *args, **kwargs):
                return []
            def get_statistics(self):
                return {}
            def reset_statistics(self):
                pass
        return _StubTRG()

    @staticmethod
    def _entropy_reg_from_proto_probs_static(proto_probs_list, gates_list=None, min_gate: float = 0.01) -> torch.Tensor:
        if not proto_probs_list or not isinstance(proto_probs_list, list):
            return torch.tensor(0.0)
        dev = None
        for row in proto_probs_list:
            if isinstance(row, list):
                for p in row:
                    if isinstance(p, torch.Tensor):
                        dev = p.device
                        break
            if dev is not None:
                break
        if dev is None:
            return torch.tensor(0.0)
        total = torch.tensor(0.0, device=dev)
        count = 0
        for b, row in enumerate(proto_probs_list):
            if not isinstance(row, list):
                continue
            gl = gates_list[b] if (gates_list and b < len(gates_list)) else None
            for j, probs in enumerate(row):
                if not isinstance(probs, torch.Tensor) or probs.numel() == 0:
                    continue
                if gl and j < len(gl):
                    try:
                        if float(gl[j]) < min_gate:
                            continue
                    except Exception:
                        pass
                try:
                    p = torch.clamp(probs.to(dev).float(), 1e-8, 1.0)
                    H = -torch.sum(p * torch.log(p))
                    if torch.isfinite(H):
                        total = total + H
                        count += 1
                except Exception:
                    continue
        if count == 0:
            return torch.tensor(0.0, device=dev)
        return total / count

    def _extract_word_embeddings(
        self,
        src_texts: List[str],
        device: torch.device,
        embed_dim: int
    ) -> Tuple[torch.Tensor, List[Dict[int, str]], List[List[str]]]:
        batch_size = len(src_texts)
        word_embeddings_batch = []
        token_word_map_batch  = []
        words_batch           = []
        max_words = 0
        try:
            embedding_layer = self.base_model.get_input_embeddings()
        except Exception:
            fallback_embs  = torch.zeros(batch_size, 1, embed_dim, device=device)
            fallback_maps  = [{0: "UNK"} for _ in range(batch_size)]
            fallback_words = [["UNK"] for _ in range(batch_size)]
            return fallback_embs, fallback_maps, fallback_words
        for text in src_texts:
            if not text or not isinstance(text, str):
                text = "UNK"
            text = text.strip()
            if not text:
                text = "UNK"
            words = _extract_words_from_text(text)
            if not words:
                words = ["UNK"]
            words_batch.append(words)
            word_embeddings = []
            word_map        = {}
            for idx, word in enumerate(words):
                try:
                    if not word:
                        word = "UNK"
                    try:
                        word_ids = self.tokenizer.encode(word, add_special_tokens=False)
                    except Exception:
                        try:
                            word_ids = self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(word))
                        except Exception:
                            word_ids = [self.tokenizer.unk_token_id if hasattr(self.tokenizer, 'unk_token_id') else 1]
                    if not isinstance(word_ids, (list, tuple)):
                        word_ids = [int(word_ids)]
                    word_ids = [int(wid) for wid in word_ids if 0 <= int(wid) < self.encoder_vocab_size]
                    if not word_ids:
                        unk_id = getattr(self.tokenizer, "unk_token_id", 1)
                        word_ids = [int(unk_id)]
                    word_ids_tensor = torch.tensor([word_ids], dtype=torch.long, device=device)
                    subword_embs    = embedding_layer(word_ids_tensor)
                    word_emb        = subword_embs.mean(dim=1).squeeze(0)
                    if torch.isnan(word_emb).any() or torch.isinf(word_emb).any():
                        word_emb = torch.zeros(embed_dim, device=device)
                    word_embeddings.append(word_emb)
                    word_map[idx] = word
                except Exception:
                    fallback_emb = torch.zeros(embed_dim, device=device)
                    word_embeddings.append(fallback_emb)
                    word_map[idx] = word if word else "UNK"
            if word_embeddings:
                try:
                    word_embs_tensor = torch.stack(word_embeddings, dim=0)
                    word_embeddings_batch.append(word_embs_tensor)
                    token_word_map_batch.append(word_map)
                    max_words = max(max_words, len(word_embeddings))
                except Exception:
                    fallback_emb = torch.zeros(1, embed_dim, device=device)
                    word_embeddings_batch.append(fallback_emb)
                    token_word_map_batch.append({0: "UNK"})
                    max_words = max(max_words, 1)
            else:
                fallback_emb = torch.zeros(1, embed_dim, device=device)
                word_embeddings_batch.append(fallback_emb)
                token_word_map_batch.append({0: "UNK"})
                max_words = max(max_words, 1)
        if max_words == 0:
            max_words = 1
        try:
            padded_word_embs = torch.zeros(batch_size, max_words, embed_dim, device=device)
            for i, word_embs in enumerate(word_embeddings_batch):
                try:
                    length = word_embs.size(0)
                    if length > max_words:
                        length = max_words
                    padded_word_embs[i, :length] = word_embs[:length]
                except Exception:
                    pass
        except Exception:
            padded_word_embs = torch.zeros(batch_size, 1, embed_dim, device=device)
        return padded_word_embs, token_word_map_batch, words_batch

    def _extract_domain_labels(
        self,
        batch_size: int,
        device: torch.device,
        src_texts: Optional[List[str]] = None
    ) -> Optional[torch.Tensor]:
        if not _USE_DOMAIN_LABELS:
            return None
        try:
            if self.training:
                return torch.full((batch_size,), _TRAIN_DOMAIN, dtype=torch.long, device=device)
            else:
                return torch.full((batch_size,), _TEST_DOMAIN, dtype=torch.long, device=device)
        except Exception:
            return None

    def forward_path1(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
        domain_labels: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        if input_ids is None or attention_mask is None:
            raise ValueError("input_ids and attention_mask cannot be None")
        batch_size = int(input_ids.size(0))
        device = input_ids.device
        embed_dim = self.embed_dim  # use stored value — never re-read from config

        if src_texts is None or not isinstance(src_texts, list) or len(src_texts) != batch_size:
            src_texts_extracted = []
            for b in range(batch_size):
                try:
                    ids_b = input_ids[b].detach().cpu().tolist()
                    text  = self.tokenizer.decode(ids_b, skip_special_tokens=True)
                    if not text or not text.strip():
                        text = "UNK"
                    src_texts_extracted.append(text.strip())
                except Exception:
                    src_texts_extracted.append("UNK")
            src_texts = src_texts_extracted
        for i in range(len(src_texts)):
            if not src_texts[i] or not isinstance(src_texts[i], str) or not src_texts[i].strip():
                src_texts[i] = "UNK"
        try:
            h, token_word_map, _words_batch = self._extract_word_embeddings(src_texts, device, embed_dim)
        except Exception:
            h = torch.zeros(batch_size, 1, embed_dim, device=device)
            token_word_map = [{0: "UNK"} for _ in range(batch_size)]
        max_words = int(h.size(1))
        h_detached = h.detach()
        try:
            raw_dscd = self.dscd.forward(
                h_detached,
                token_types=None,
                train_mode=self.training,
                input_ids=None,
                attention_mask=None,
                token_word_map=token_word_map
            )
        except Exception:
            raw_dscd = {
                "h_augmented": h_detached.clone(),
                "proto_probs": [[torch.tensor([1.0], dtype=torch.float32, device=device) for _ in range(max_words)] for _ in range(batch_size)],
                "uncertainties": [[torch.tensor(0.0, device=device) for _ in range(max_words)] for _ in range(batch_size)],
                "gates": [[torch.tensor(0.0, device=device) for _ in range(max_words)] for _ in range(batch_size)]
            }
        dscd = normalize_dscd_outputs(raw_dscd, batch_size, max_words, device, embed_dim, fallback_h=h_detached)
        h_aug = dscd.get("h_augmented", None)
        if h_aug is None:
            h_aug = torch.zeros(batch_size, max_words, embed_dim, device=device)
        if domain_labels is None:
            domain_labels = self._extract_domain_labels(batch_size=batch_size, device=device, src_texts=src_texts)
        asbn_loss = torch.zeros(1, device=device, requires_grad=True)
        if self.training and _ENABLE_ASBN_TRAINING and domain_labels is not None:
            try:
                _h_asbn, asbn_losses = self.asbn.forward(
                    h_aug,
                    proto_probs=dscd.get("proto_probs", None),
                    uncertainties=dscd.get("uncertainties", None),
                    gates=dscd.get("gates", None),
                    token_word_map=token_word_map,
                    domain_labels=domain_labels,
                    global_step=self.global_step
                )
                if isinstance(asbn_losses, dict):
                    encoder_loss = asbn_losses.get("encoder_loss", torch.zeros(1, device=device, requires_grad=True))
                    if isinstance(encoder_loss, torch.Tensor) and torch.isfinite(encoder_loss):
                        asbn_loss = encoder_loss if encoder_loss.requires_grad else torch.tensor(float(encoder_loss.item()), device=device, requires_grad=True)
            except Exception:
                asbn_loss = torch.zeros(1, device=device, requires_grad=True)
        dscd_reg = torch.zeros(1, device=device, requires_grad=True)
        try:
            dscd_reg_raw = self._entropy_reg_from_proto_probs_static(
                dscd.get("proto_probs", []),
                gates_list=dscd.get("gates", []),
                min_gate=0.01
            )
            if isinstance(dscd_reg_raw, torch.Tensor) and torch.isfinite(dscd_reg_raw):
                dscd_reg = dscd_reg_raw.to(device)
                if not dscd_reg.requires_grad:
                    dscd_reg = torch.tensor(float(dscd_reg.item()), device=device, requires_grad=True)
            else:
                dscd_reg = torch.tensor(0.01, device=device, requires_grad=True)
        except Exception:
            dscd_reg = torch.tensor(0.01, device=device, requires_grad=True)
        total_loss = _LAMBDA_ASBN * asbn_loss + _LAMBDA_DSCD * torch.clamp(dscd_reg, 0.0, 5.0)
        if not isinstance(total_loss, torch.Tensor):
            total_loss = torch.tensor(float(total_loss), device=device, requires_grad=True)
        if not torch.isfinite(total_loss) or total_loss.item() == 0.0:
            total_loss = torch.tensor(0.1, device=device, requires_grad=True)
        if not total_loss.requires_grad:
            total_loss = torch.tensor(float(total_loss.item()), device=device, requires_grad=True)
        self._last_path1_asbn_loss = float(asbn_loss.item()) if isinstance(asbn_loss, torch.Tensor) else 0.0
        self._last_path1_dscd_loss = float(dscd_reg.item()) if isinstance(dscd_reg, torch.Tensor) else 0.0
        return total_loss

    def forward_path2(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        labels: torch.Tensor,
        decoder_input_ids: Optional[torch.Tensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
        use_rdrop: bool = True
    ) -> torch.Tensor:
        if input_ids is None or attention_mask is None or labels is None:
            raise ValueError("input_ids, attention_mask, and labels cannot be None")
        batch_size = int(input_ids.size(0))
        device     = input_ids.device
        embed_dim  = self.embed_dim  # use stored value

        if torch.any(input_ids >= self.encoder_vocab_size) or torch.any(input_ids < 0):
            input_ids = torch.clamp(input_ids, 0, max(0, self.encoder_vocab_size - 1))
        valid_labels = labels[labels != -100]
        if valid_labels.numel() > 0 and valid_labels.max().item() >= self.decoder_vocab_size:
            if _DEBUG_DISCOVERY:
                print(f"[PATH2] Labels out of bounds, max={valid_labels.max().item()} vocab={self.decoder_vocab_size}")

        if decoder_input_ids is None or decoder_attention_mask is None:
            pad_id = getattr(self.tokenizer, "pad_token_id", 1)
            decoder_input_ids = labels.clone()
            decoder_input_ids = torch.where(
                decoder_input_ids == -100,
                torch.full_like(decoder_input_ids, pad_id),
                decoder_input_ids
            )
            # Use self.tgt_token_id as definitive BOS — never fall back to 0
            bos_column = torch.full(
                (batch_size, 1),
                self.tgt_token_id,
                dtype=torch.long,
                device=device
            )
            decoder_input_ids        = torch.cat([bos_column, decoder_input_ids[:, :-1]], dim=1)
            decoder_attention_mask   = (decoder_input_ids != pad_id).long()
            decoder_input_ids        = torch.clamp(decoder_input_ids, 0, max(0, self.decoder_vocab_size - 1))
        else:
            if torch.any(decoder_input_ids >= self.decoder_vocab_size) or torch.any(decoder_input_ids < 0):
                decoder_input_ids = torch.clamp(decoder_input_ids, 0, max(0, self.decoder_vocab_size - 1))

        # Run encoder explicitly to get hidden state for DSCD buffer accumulation during training
        try:
            model_core = getattr(self.base_model, "model", None)
            encoder    = getattr(model_core, "encoder", None) if model_core is not None else None
            if encoder is not None:
                with torch.no_grad():
                    enc_out = encoder(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        return_dict=True
                    )
                h_enc = _safe_get_last_hidden_state(enc_out)
                if h_enc is not None and isinstance(h_enc, torch.Tensor):
                    seq_len = h_enc.size(1)
                    # Build token_word_map if not provided
                    if token_word_map is None or not isinstance(token_word_map, list) or len(token_word_map) != batch_size:
                        token_word_map = []
                        for b in range(batch_size):
                            try:
                                ids_b   = input_ids[b].detach().cpu().tolist()
                                toks_b  = self.tokenizer.convert_ids_to_tokens(ids_b)
                                wm      = build_token_word_map_sentencepiece(toks_b, fallback=True)
                                token_word_map.append(wm if wm else {0: "UNK"})
                            except Exception:
                                token_word_map.append({0: "UNK"})
                    # Feed encoder hidden state into DSCD buffer in train_mode=True
                    try:
                        self.dscd.forward(
                            h_enc.detach(),
                            token_types=None,
                            train_mode=True,
                            input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_word_map=token_word_map
                        )
                    except Exception:
                        pass
        except Exception:
            pass

        seq_outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            labels=labels,
            use_cache=False,
            return_dict=True
        )
        translation_loss = getattr(seq_outputs, "loss", None)
        if translation_loss is None or not torch.isfinite(translation_loss):
            translation_loss = torch.tensor(10.0, device=device)
        else:
            translation_loss = torch.clamp(translation_loss, 0.0, 100.0)
        if self.label_smoothing_loss is not None:
            try:
                logits  = seq_outputs.logits
                ls_loss = self.label_smoothing_loss(logits, labels)
                if torch.isfinite(ls_loss):
                    translation_loss = ls_loss
            except Exception:
                pass
        rdrop_loss = torch.tensor(0.0, device=device)
        if use_rdrop and self.rdrop_loss is not None and self.training:
            try:
                seq_outputs2 = self.base_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    decoder_input_ids=decoder_input_ids,
                    decoder_attention_mask=decoder_attention_mask,
                    labels=labels,
                    use_cache=False,
                    return_dict=True
                )
                logits1    = seq_outputs.logits
                logits2    = seq_outputs2.logits
                rdrop_loss = self.rdrop_loss(logits1, logits2, labels)
                if not torch.isfinite(rdrop_loss):
                    rdrop_loss = torch.tensor(0.0, device=device)
            except Exception:
                rdrop_loss = torch.tensor(0.0, device=device)
        total_loss = translation_loss + rdrop_loss
        if not torch.isfinite(total_loss):
            total_loss = translation_loss
        return total_loss

    def _forward_path2_inference_return_dict(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
        return_dict: bool = True
    ) -> Union[Dict[str, Any], Tuple[torch.Tensor, Dict[str, Any], float]]:
        batch_size, seq_len = int(input_ids.size(0)), int(input_ids.size(1))
        device    = input_ids.device
        embed_dim = self.embed_dim  # use stored value

        if src_texts is None or not isinstance(src_texts, list) or len(src_texts) != batch_size:
            src_texts = []
            for b in range(batch_size):
                try:
                    ids_b = input_ids[b].detach().cpu().tolist()
                    text  = self.tokenizer.decode(ids_b, skip_special_tokens=True)
                    src_texts.append(text.strip() if text and text.strip() else "UNK")
                except Exception:
                    src_texts.append("UNK")

        if token_word_map is None or not isinstance(token_word_map, list) or len(token_word_map) != batch_size:
            token_word_map = []
            for b in range(batch_size):
                try:
                    ids_b  = input_ids[b].detach().cpu().tolist()
                    toks_b = self.tokenizer.convert_ids_to_tokens(ids_b)
                    wm     = build_token_word_map_sentencepiece(toks_b, fallback=True)
                    token_word_map.append(wm if wm else {0: "UNK"})
                except Exception:
                    token_word_map.append({0: "UNK"})

        enc_out = None
        try:
            model_core = getattr(self.base_model, "model", None)
            encoder    = getattr(model_core, "encoder", None) if model_core is not None else None
            if encoder is None:
                raise RuntimeError("base_model.model.encoder not found")
            enc_out = encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
        except Exception as e:
            if _DEBUG_DISCOVERY:
                print(f"[PATH2-INF] Encoder forward failed: {e}")
            h_enc = torch.zeros(batch_size, seq_len, embed_dim, device=device, dtype=torch.float32)
            enc_out = BaseModelOutput(last_hidden_state=h_enc)

        h_enc = _safe_get_last_hidden_state(enc_out)
        if h_enc is None or not isinstance(h_enc, torch.Tensor):
            h_enc = torch.zeros(batch_size, seq_len, embed_dim, device=device, dtype=torch.float32)
        if h_enc.size(-1) != embed_dim:
            try:
                h_enc = h_enc[..., :embed_dim]
            except Exception:
                h_enc = torch.zeros(batch_size, seq_len, embed_dim, device=device, dtype=torch.float32)

        try:
            # Only detach at inference — during training this path is never called for training steps
            raw_dscd = self.dscd.forward(
                h_enc.detach(),
                token_types=None,
                train_mode=self.training,
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_word_map=token_word_map
            )
        except Exception as e:
            if _DEBUG_DISCOVERY:
                print(f"[PATH2-INF] DSCD forward failed: {e}")
            raw_dscd = {
                "h_augmented": h_enc.detach().clone(),
                "proto_probs": [[torch.tensor([1.0], dtype=torch.float32, device=device) for _ in range(seq_len)] for _ in range(batch_size)],
                "uncertainties": [[torch.tensor(0.0, device=device) for _ in range(seq_len)] for _ in range(batch_size)],
                "gates": [[torch.tensor(0.0, device=device) for _ in range(seq_len)] for _ in range(batch_size)],
                "span_preds": [[torch.tensor(0.0, device=device) for _ in range(seq_len)] for _ in range(batch_size)],
                "proto_assignments": [torch.zeros(seq_len, dtype=torch.long, device=device) for _ in range(batch_size)]
            }

        dscd_out = normalize_dscd_outputs(raw_dscd, batch_size, seq_len, device, embed_dim, fallback_h=h_enc)
        h_aug    = dscd_out.get("h_augmented", None)
        if h_aug is None or not isinstance(h_aug, torch.Tensor):
            h_aug = h_enc

        out: Dict[str, Any] = {}
        out["sense_augmented_embeddings"] = h_aug
        out["encoder_last_hidden_state"]  = h_aug
        out["last_hidden_state"]          = h_aug
        out["dscd_outputs"] = {
            "h_augmented":      h_aug,
            "proto_probs":      dscd_out.get("proto_probs", []),
            "uncertainties":    dscd_out.get("uncertainties", []),
            "gates":            dscd_out.get("gates", []),
            "span_preds":       dscd_out.get("span_preds", []),
            "proto_assignments": dscd_out.get("proto_assignments", [])
        }
        try:
            out["dscd_summary"] = getattr(self.dscd, "get_prototype_summary", lambda: {})()
        except Exception:
            out["dscd_summary"] = {}

        if return_dict:
            if self.training:
                safe_out: Dict[str, Any] = {}
                for k, v in out.items():
                    if v is None:
                        continue
                    if isinstance(v, torch.Tensor):
                        safe_out[k] = v
                    elif isinstance(v, dict):
                        ok = True
                        for val in v.values():
                            if val is None:
                                continue
                            if isinstance(val, torch.Tensor):
                                continue
                            if isinstance(val, list):
                                all_t = True
                                for itm in val:
                                    if not (isinstance(itm, torch.Tensor) or itm is None):
                                        all_t = False
                                        break
                                if all_t:
                                    continue
                            ok = False
                            break
                        if ok:
                            safe_out[k] = v
                return safe_out
            return out
        else:
            return (h_aug, out["dscd_outputs"], 0.0)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        src_texts: Optional[List[str]] = None,
        token_word_map: Optional[List[dict]] = None,
        labels: Optional[torch.Tensor] = None,
        decoder_input_ids: Optional[torch.Tensor] = None,
        decoder_attention_mask: Optional[torch.Tensor] = None,
        use_dscd: bool = True,
        use_asbn: bool = False,
        return_dict: bool = True,
        domain_labels: Optional[torch.Tensor] = None,
        path: Optional[int] = None,
        use_rdrop: bool = False,
        **kwargs
    ):
        if path == 1:
            with self._step_lock:
                self.global_step += 1
            return self.forward_path1(
                input_ids=input_ids,
                attention_mask=attention_mask,
                src_texts=src_texts,
                token_word_map=token_word_map,
                domain_labels=domain_labels
            )
        if path == 2:
            with self._step_lock:
                self.global_step += 1
            if labels is not None and self.training:
                return self.forward_path2(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                    decoder_input_ids=decoder_input_ids,
                    decoder_attention_mask=decoder_attention_mask,
                    src_texts=src_texts,
                    token_word_map=token_word_map,
                    use_rdrop=use_rdrop
                )
            return self._forward_path2_inference_return_dict(
                input_ids=input_ids,
                attention_mask=attention_mask,
                src_texts=src_texts,
                token_word_map=token_word_map,
                return_dict=return_dict
            )
        with self._step_lock:
            self.global_step += 1
            current_step = self.global_step
        if input_ids is None or attention_mask is None:
            raise ValueError("input_ids and attention_mask cannot be None")
        if input_ids.dim() != 2 or attention_mask.dim() != 2:
            raise ValueError(f"Expected 2D tensors, got {input_ids.shape}, {attention_mask.shape}")
        batch_size = int(input_ids.size(0))
        if domain_labels is None:
            domain_labels = self._extract_domain_labels(
                batch_size=batch_size,
                device=input_ids.device,
                src_texts=src_texts
            )
        if (torch.cuda.is_available() and _MEMORY_CLEANUP_FREQUENCY > 0 and current_step % _MEMORY_CLEANUP_FREQUENCY == 0):
            for i in range(min(_NUM_GPUS, torch.cuda.device_count())):
                try:
                    with torch.cuda.device(i):
                        torch.cuda.empty_cache()
                except Exception:
                    pass
            if gc.isenabled():
                gc.collect()
        return self._forward_path2_inference_return_dict(
            input_ids=input_ids,
            attention_mask=attention_mask,
            src_texts=src_texts,
            token_word_map=token_word_map,
            return_dict=return_dict
        )


print("[Cell 6] MemoryOptimizedTATNWithExplanations (INDICTRANS2) loaded successfully - COMPLETE FIXED")


[Cell 6] MemoryOptimizedTATNWithExplanations (INDICTRANS2) loaded successfully - COMPLETE FIXED


In [10]:
# ===========================================================================================
# CELL 7: DUAL-PATH TRAINING LOOP - PATH 1 (DSCD) + PATH 2 (TRANSLATION) (INDICTRANS2) - FIXED
# ===========================================================================================
import os
import time
import math
import gc
import traceback
import sys
from datetime import datetime
from pathlib import Path
from collections import defaultdict, deque
from typing import Optional, Dict, Any, List

import numpy as np
import torch
from torch.cuda.amp import GradScaler, autocast as cuda_amp_autocast
from tqdm import tqdm
from contextlib import nullcontext
import threading

try:
    _VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except (NameError, TypeError):
    _VERBOSE_LOGGING = False

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except (NameError, TypeError):
    _DEBUG_DISCOVERY = False

DEBUG_PRINT_INTERVAL = 200
_cell7_dbg_counts = defaultdict(int)


def cell7_dbg(key: str, msg: str, limit: int = 10):
    if not (_VERBOSE_LOGGING or _DEBUG_DISCOVERY):
        return
    _cell7_dbg_counts[key] += 1
    if _cell7_dbg_counts[key] <= limit:
        print(f"[CELL7-DBG] {msg}")


try:
    _DEVICE = DEVICE
except (NameError, TypeError):
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    _EPOCHS = int(EPOCHS)
except (NameError, ValueError, TypeError):
    _EPOCHS = 1

try:
    _BATCH_SIZE = int(BATCH_SIZE)
except (NameError, ValueError, TypeError):
    _BATCH_SIZE = 8

try:
    _ACCUMULATION_STEPS = int(ACCUMULATION_STEPS)
except (NameError, ValueError, TypeError):
    _ACCUMULATION_STEPS = 1

try:
    _GRAD_CLIP_NORM = float(GRAD_CLIP_NORM)
except (NameError, ValueError, TypeError):
    _GRAD_CLIP_NORM = 1.0

try:
    _MEMORY_CLEANUP_FREQUENCY = int(MEMORY_CLEANUP_FREQUENCY)
except (NameError, ValueError, TypeError):
    _MEMORY_CLEANUP_FREQUENCY = 500

try:
    _USE_MULTI_GPU = bool(USE_MULTI_GPU)
    _NUM_GPUS = int(NUM_GPUS)
except (NameError, ValueError, TypeError):
    _NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
    _USE_MULTI_GPU = _NUM_GPUS > 1

try:
    _USE_AMP = bool(USE_AMP)
except (NameError, TypeError):
    _USE_AMP = True

try:
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
    _TARGET_LANGUAGE = str(TARGET_LANGUAGE)
except (NameError, TypeError):
    _SOURCE_LANGUAGE = "ben_Beng"
    _TARGET_LANGUAGE = "eng_Latn"

try:
    _MAX_LENGTH = int(MAX_LENGTH)
except (NameError, ValueError, TypeError):
    _MAX_LENGTH = 64

try:
    _VALIDATION_CHECK_INTERVAL = int(VALIDATION_CHECK_INTERVAL)
except (NameError, ValueError, TypeError):
    _VALIDATION_CHECK_INTERVAL = 200

try:
    _PERIODIC_DISCOVERY_FREQUENCY = int(PERIODIC_DISCOVERY_FREQUENCY)
except (NameError, ValueError, TypeError):
    _PERIODIC_DISCOVERY_FREQUENCY = 100

try:
    _TRAIN_DOMAIN = int(TRAIN_DOMAIN)
    _TEST_DOMAIN = int(TEST_DOMAIN)
except (NameError, ValueError, TypeError):
    _TRAIN_DOMAIN = 0
    _TEST_DOMAIN = 1

try:
    _LAMBDA_TRG = float(LAMBDA_TRG)
except (NameError, ValueError, TypeError):
    _LAMBDA_TRG = 0.15

try:
    _WARMUP_STEPS = int(WARMUP_STEPS)
except (NameError, ValueError, TypeError):
    _WARMUP_STEPS = 4000

try:
    _USE_LR_SCHEDULER = bool(USE_LR_SCHEDULER)
except (NameError, TypeError):
    _USE_LR_SCHEDULER = True

try:
    _USE_DUAL_PATH_TRAINING = bool(USE_DUAL_PATH_TRAINING)
except (NameError, TypeError):
    _USE_DUAL_PATH_TRAINING = True

try:
    _ENABLE_ASBN_TRAINING = bool(ENABLE_ASBN_TRAINING)
except (NameError, TypeError):
    _ENABLE_ASBN_TRAINING = True

try:
    _HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST_BN)
except (NameError, TypeError):
    _HOMOGRAPH_REFERENCE_LIST = {
        "কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা", "বার", "হার", "তারা",
        "পানি", "দল", "বাজার", "নাম", "কথা", "বই", "ঘর", "মন", "হাত"
    }
    _HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in _HOMOGRAPH_REFERENCE_LIST)

_BENGALI_PUNCT_SET = set(['।', '॥'])
_COMMON_PUNCT_SET  = set(['.', ',', ';', ':', '!', '?', '"', "'", '-', '(', ')', '[', ']', '{', '}', '/', '\\'])
_PUNCT_SET = _BENGALI_PUNCT_SET | _COMMON_PUNCT_SET


def _is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = (
        token.replace("▁", "")
        .replace("Ġ", "")
        .replace("##", "")
        .replace("</w>", "")
        .strip()
    )
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET:
        return True
    if clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in _PUNCT_SET for c in clean)


def clear_all_gpu_caches():
    gc.collect()
    if not torch.cuda.is_available():
        return
    try:
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
    except Exception:
        pass


def get_amp_ctx():
    if not _USE_AMP or not torch.cuda.is_available():
        return nullcontext()
    try:
        return cuda_amp_autocast()
    except Exception:
        return nullcontext()


def clamp_input_ids_to_vocab(
    tensor: Optional[torch.Tensor],
    vocab_size: int,
    name: str = "tensor"
) -> Optional[torch.Tensor]:
    if tensor is None or not isinstance(tensor, torch.Tensor):
        return tensor
    orig_device = tensor.device
    try:
        cpu_t = tensor.detach().cpu().long()
        if cpu_t.numel() == 0:
            return tensor
        mask_valid = cpu_t != -100
        if mask_valid.any():
            vals = cpu_t[mask_valid]
            vals = torch.clamp(vals, 0, max(0, int(vocab_size) - 1))
            cpu_t[mask_valid] = vals
        if orig_device.type == "cuda":
            return cpu_t.to(orig_device)
        return cpu_t
    except Exception:
        return tensor


_PROTOBUF_COMPAT_ERROR_SHOWN = globals().get("_PROTOBUF_COMPAT_ERROR_SHOWN", False)


def _get_dscd_homographs(model: torch.nn.Module) -> set:
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return set()
        if hasattr(dscd, "get_discovered_homographs"):
            try:
                discovered = dscd.get_discovered_homographs()
                return set(w for w in discovered if not _is_punctuation_only(w))
            except Exception:
                pass
        homographs = set()
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        if lock:
            with lock:
                stores_snapshot = dict(getattr(dscd, "prototype_stores", {}).items())
        else:
            stores_snapshot = dict(getattr(dscd, "prototype_stores", {}).items())
        for token, store in stores_snapshot.items():
            try:
                # Homograph criterion: must have >= 2 actual centroids (prototypes)
                n_protos = len(getattr(store, "centroids", []))
                if n_protos >= 2:
                    clean_token = (
                        str(token)
                        .replace("▁", "")
                        .replace("Ġ", "")
                        .replace("##", "")
                        .strip()
                        .lower()
                    )
                    if clean_token and not _is_punctuation_only(clean_token):
                        homographs.add(clean_token)
            except Exception:
                continue
        return homographs
    except Exception:
        return set()


def _get_cluster_count(model: torch.nn.Module) -> int:
    """Returns actual prototype (centroid) count, not token-type count."""
    try:
        core = model
        while hasattr(core, "module"):
            core = core.module
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return 0
        if hasattr(dscd, "get_prototype_summary"):
            try:
                summary = dscd.get_prototype_summary()
                return int(summary.get("total_prototypes", 0))
            except Exception:
                pass
        stores = getattr(dscd, "prototype_stores", None)
        if stores is None:
            return 0
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        if lock:
            with lock:
                return sum(len(getattr(s, "centroids", [])) for s in stores.values())
        else:
            return sum(len(getattr(s, "centroids", [])) for s in stores.values())
    except Exception:
        return 0


def _get_token_type_count(model: torch.nn.Module) -> int:
    """Returns number of tracked token types in prototype_stores."""
    try:
        core = model
        while hasattr(core, "module"):
            core = core.module
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return 0
        stores = getattr(dscd, "prototype_stores", None)
        if stores is None:
            return 0
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        if lock:
            with lock:
                return len(stores)
        else:
            return len(stores)
    except Exception:
        return 0


def _get_homograph_cluster_count(model: torch.nn.Module) -> int:
    """Returns number of token types with >= 2 centroids (true homographs)."""
    try:
        core = model
        while hasattr(core, "module"):
            core = core.module
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return 0
        if hasattr(dscd, "get_prototype_summary"):
            try:
                summary = dscd.get_prototype_summary()
                return int(summary.get("num_homographs", 0))
            except Exception:
                pass
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        if lock:
            with lock:
                stores_snapshot = dict(getattr(dscd, "prototype_stores", {}).items())
        else:
            stores_snapshot = dict(getattr(dscd, "prototype_stores", {}).items())
        homograph_count = 0
        for token, store in stores_snapshot.items():
            try:
                n_protos = len(getattr(store, "centroids", []))
                if n_protos >= 2:
                    clean_token = (
                        str(token)
                        .replace("▁", "")
                        .replace("Ġ", "")
                        .replace("##", "")
                        .strip()
                        .lower()
                    )
                    if clean_token and not _is_punctuation_only(clean_token):
                        homograph_count += 1
            except Exception:
                continue
        return homograph_count
    except Exception:
        return 0


def _get_dscd_safe(model: torch.nn.Module):
    try:
        core = model
        while hasattr(core, "module"):
            core = core.module
        return getattr(core, "dscd", None)
    except Exception:
        return None


def _print_top_clusters(model: torch.nn.Module, top_n: int = 5):
    dscd = _get_dscd_safe(model)
    if dscd is None:
        return
    try:
        dscd_homographs = _get_dscd_homographs(model)
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        if lock:
            with lock:
                stores_snapshot = list(getattr(dscd, "prototype_stores", {}).items())
        else:
            stores_snapshot = list(getattr(dscd, "prototype_stores", {}).items())
        items = []
        homograph_items = []
        for token, store in stores_snapshot:
            try:
                total_count = sum(getattr(store, "counts", []) or [])
                n_protos    = len(getattr(store, "centroids", []))
                buf_len     = len(dscd.buffers.get(token, [])) if hasattr(dscd, "buffers") else 0
                clean_token = str(token).replace("▁", "").replace("Ġ", "").strip().lower()
                if _is_punctuation_only(clean_token):
                    continue
                is_homograph = clean_token in dscd_homographs
                item = (token, total_count, n_protos, buf_len, is_homograph)
                items.append(item)
                if is_homograph:
                    homograph_items.append(item)
            except Exception:
                continue
        items.sort(key=lambda x: x[1], reverse=True)
        print(f"\n[CLUSTER] Top {min(top_n, len(items))} clusters:")
        print("-" * 90)
        print(f"{'Rank':<6}{'Token':<15}{'Count':<12}{'Protos':<10}{'Mu':<15}{'Tau':<12}")
        print("-" * 90)
        for rank, (tok, cnt, n_protos, buf_len, is_homo) in enumerate(items[:top_n], 1):
            tok_str     = str(tok)
            tok_display = tok_str[:12] if len(tok_str) > 12 else tok_str
            core_ref    = _get_dscd_safe(model)
            store_ref   = getattr(core_ref, "prototype_stores", {}).get(tok, None) if core_ref else None
            mu_val      = float(getattr(store_ref, "mu", 0.0)) if store_ref is not None else 0.0
            tau_val     = float(getattr(store_ref, "tau", 0.0)) if store_ref is not None else 0.0
            print(
                f"{rank:<6}{tok_display:<15}{cnt:<12}{n_protos:<10}"
                f"{mu_val:<15.6f}{tau_val:<12.6f}"
            )
        print("-" * 90)
        if (_DEBUG_DISCOVERY or _VERBOSE_LOGGING) and homograph_items:
            print(f"[CLUSTER-DBG] DSCD-discovered homographs: {len(homograph_items)}")
            for tok, cnt, n_protos, buf_len, _ in homograph_items[:5]:
                print(f"  HOMO {str(tok)[:20]:20s} samples={cnt:4d} protos={n_protos}")
    except Exception:
        pass


def _check_discovery_status(model: torch.nn.Module, global_step: int):
    try:
        core = model
        while hasattr(core, "module"):
            core = core.module
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return
        if hasattr(dscd, "get_prototype_summary"):
            try:
                summary = dscd.get_prototype_summary()
                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    print(f"[DISCOVERY-STATUS] Step {global_step}:")
                    print(f"  - Total tokens:     {summary.get('total_tokens', 0)}")
                    print(f"  - Homographs:       {summary.get('num_homographs', 0)}")
                    print(f"  - Total prototypes: {summary.get('total_prototypes', 0)}")
                    print(f"  - Quality score:    {summary.get('quality_score', 0.0):.1%}")
            except Exception:
                pass
        if hasattr(dscd, "discovered_log") and dscd.discovered_log:
            total_discovered = len(dscd.discovered_log)
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[DISCOVERY-STATUS] Discovery events: {total_discovered}")
                recent = dscd.discovered_log[-3:] if len(dscd.discovered_log) >= 3 else dscd.discovered_log
                for entry in recent:
                    discovered = entry.get("discovered", 0)
                    candidates = entry.get("candidates", 0)
                    print(f"  - {discovered}/{candidates} homographs discovered")
        else:
            if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                print(f"[DISCOVERY-STATUS] No discoveries yet at step {global_step}")
    except Exception:
        pass


def _print_gpu_mem(prefix: str = ""):
    if not torch.cuda.is_available():
        return
    try:
        lines = [f"{prefix} GPU mem (GB):"]
        for i in range(torch.cuda.device_count()):
            try:
                alloc = torch.cuda.memory_allocated(i) / (1024 ** 3)
                resv  = torch.cuda.memory_reserved(i)  / (1024 ** 3)
                lines.append(f"  GPU {i}: alloc={alloc:.2f} resv={resv:.2f}")
            except Exception:
                lines.append(f"  GPU {i}: mem query failed")
        print("\n".join(lines))
    except Exception:
        pass


def _is_valid_translation(translation: str) -> bool:
    if not translation or not isinstance(translation, str):
        return False
    t = translation.strip()
    if not t or len(t) < 3:
        return False
    if t in ("Error occurred", "Translation generation failed", "ERROR DURING TRANSLATION", "..", "...", "."):
        return False
    words = [w for w in t.split() if w.strip()]
    return len(words) >= 1


@torch.inference_mode()
def comprehensive_epoch_validation(
    model: torch.nn.Module,
    tokenizer,
    epoch: int,
    global_step: int,
    source_lang: str,
    target_lang: str,
    max_length: int,
    device: torch.device
) -> Dict[str, Any]:
    global _PROTOBUF_COMPAT_ERROR_SHOWN
    print("\n" + "=" * 80)
    print(f"EPOCH {epoch} COMPREHENSIVE VALIDATION (Step {global_step})")
    print("=" * 80)

    core_model = model.module if hasattr(model, "module") else model
    was_training = core_model.training
    if not isinstance(device, torch.device):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dscd_homographs = _get_dscd_homographs(model)
    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
        print(f"[VALIDATION] DSCD discovered homographs: {len(dscd_homographs)}")
        if dscd_homographs:
            print(f"[VALIDATION] Sample: {list(dscd_homographs)[:10]}")

    validation_results: Dict[str, Any] = {
        "epoch":                          epoch,
        "step":                           global_step,
        "translations_success":           0,
        "translations_failed":            0,
        "explanations_generated":         0,
        "dscd_homographs_explained":      0,
        "reference_homographs_explained": 0,
        "avg_explanation_confidence":     0.0,
        "dscd_quality_score":             0.0,
        "dscd_multi_sense_tokens":        0,
        "dscd_total_prototypes":          0,
        "asbn_domain_loss":               0.0,
        "asbn_domain_accuracy":           0.0,
        "asbn_source_accuracy":           0.0,
        "asbn_target_accuracy":           0.0,
        "trg_total_explanations":         0,
        "validation_completed":           False,
    }

    try:
        core_model.eval()
        val_sentences = [
            ("আমি কল বন্ধ করেছি।",          "I turned off the tap",                  "কল=tap/call"),
            ("কাল আমি বই কিনব।",             "Tomorrow I will buy a book",            "কাল=tomorrow/yesterday"),
            ("পাতা ঝরে পড়েছে।",             "The leaf has fallen",                   "পাতা=leaf/page"),
            ("তিনি ব্যাংক গেছেন।",           "He went to the bank",                   "ব্যাংক=bank/embankment"),
            ("আমি ভালো আছি।",               "I am fine",                             "No ambiguity"),
            ("সে খুব মিষ্টি কথা বলে।",       "She speaks sweetly",                    "No ambiguity"),
            ("এটা আমার বই।",                "This is my book",                       "No ambiguity"),
            ("আজ আবহাওয়া ভালো।",            "Weather is good today",                 "No ambiguity"),
            ("ফল খুব সুস্বাদু।",             "The fruit is delicious",                "ফল=fruit/result"),
            ("মাথা ব্যথা করছে।",             "Head is aching",                        "মাথা=head/top"),
        ]

        print(f"\n[VALIDATION] Testing {len(val_sentences)} samples:")
        print("-" * 80)

        confidences: List[float] = []
        dscd_homograph_words_detected: set = set()
        reference_homograph_words_detected: set = set()

        try:
            try:
                if hasattr(tokenizer, "src_lang") and hasattr(tokenizer, "tgt_lang"):
                    tokenizer.src_lang = source_lang
                    tokenizer.tgt_lang = target_lang
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print(f"[VALIDATION] Tokenizer languages: src={source_lang}, tgt={target_lang}")
                elif hasattr(tokenizer, "set_src_lang_special_tokens") and hasattr(tokenizer, "set_tgt_lang_special_tokens"):
                    tokenizer.set_src_lang_special_tokens(source_lang)
                    tokenizer.set_tgt_lang_special_tokens(target_lang)
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print(f"[VALIDATION] IndicTrans2 tokenizer configured: src={source_lang}, tgt={target_lang}")
                else:
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        print(f"[VALIDATION] Tokenizer language setting not supported")
            except Exception:
                pass

            for idx, (src, expected, note) in enumerate(val_sentences, 1):
                try:
                    translation       = ""
                    explanation_status = ""
                    error_detail      = ""

                    if "translate_with_explanations" in globals():
                        try:
                            res = globals()["translate_with_explanations"](
                                model,
                                tokenizer,
                                src,
                                source_lang=source_lang,
                                target_lang=target_lang,
                                device=device,
                                max_length=max_length,
                            )
                            translation = str(res.get("translation", "") or "")
                            exps        = list(res.get("explanations", []) or [])
                            error_info  = res.get("error", "")

                            if _is_valid_translation(translation):
                                validation_results["translations_success"] += 1
                            else:
                                validation_results["translations_failed"] += 1
                                error_detail = f" ({error_info})" if error_info else " (empty or garbage result)"

                            validation_results["explanations_generated"] += len(exps)

                            if exps:
                                explanation_status = f"{len(exps)} expl"
                                for exp in exps:
                                    try:
                                        conf = float(exp.get("confidence", 0.5))
                                        confidences.append(conf)
                                        word       = str(exp.get("ambiguous_word", "")).strip()
                                        clean_word = word.replace("▁", "").replace("Ġ", "").lower()
                                        if clean_word and not _is_punctuation_only(clean_word):
                                            if clean_word in dscd_homographs:
                                                validation_results["dscd_homographs_explained"] += 1
                                                dscd_homograph_words_detected.add(clean_word)
                                            if clean_word in _HOMOGRAPH_REFERENCE_LIST:
                                                validation_results["reference_homographs_explained"] += 1
                                                reference_homograph_words_detected.add(clean_word)
                                    except Exception:
                                        pass
                            else:
                                explanation_status = "no expl"

                        except Exception as e:
                            explanation_status = f"error: {type(e).__name__}"
                            error_detail       = f" ({str(e)[:50]})"
                            translation        = ""
                            validation_results["translations_failed"] += 1
                    else:
                        explanation_status = "unavailable"
                        error_detail       = " (function not found)"
                        validation_results["translations_failed"] += 1

                    if _is_valid_translation(translation):
                        t_display = translation.strip()
                        if len(t_display) > 80:
                            t_display = t_display[:80] + "..."
                        print(f"  {idx:2d}. {explanation_status:15s} {note[:30]:30s} -> {t_display}")
                    else:
                        t_raw = f"'{translation}'" if translation else "[EMPTY]"
                        print(f"  {idx:2d}. Translation failed: {note[:30]:30s} got={t_raw}{error_detail}")
                        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                            print(f"       [DEBUG] Bengali input: {src[:50]}")

                except Exception as e:
                    validation_results["translations_failed"] += 1
                    print(f"  {idx:2d}. ERROR: {note[:30]:30s} -> {type(e).__name__}")
                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                        try:
                            traceback.print_exc()
                        except Exception:
                            pass

        finally:
            if torch.cuda.is_available():
                try:
                    torch.cuda.synchronize()
                except Exception:
                    pass
            clear_all_gpu_caches()

        print("\n" + "-" * 80)
        print("[VALIDATION] DSCD Prototype Quality Check:")
        try:
            dscd = getattr(core_model, "dscd", None)
            if dscd is not None and hasattr(dscd, "validate_prototypes"):
                lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
                if lock:
                    with lock:
                        quality_results = dscd.validate_prototypes(cluster_missing=False)
                else:
                    quality_results = dscd.validate_prototypes(cluster_missing=False)
                validation_results["dscd_quality_score"]      = quality_results.get("quality_score", 0.0)
                validation_results["dscd_multi_sense_tokens"] = quality_results.get("multi_sense_tokens", 0)
                validation_results["dscd_total_prototypes"]   = quality_results.get("total_prototypes", 0)
                print(f"  - Quality Score: {validation_results['dscd_quality_score']:.1%}")
                print(f"  - Multi-sense tokens: {validation_results['dscd_multi_sense_tokens']}")
                print(f"  - Total prototypes: {validation_results['dscd_total_prototypes']}")
            else:
                print("  - Validation not available")
        except Exception:
            print("  - Validation failed")

        print("\n" + "-" * 80)
        print("[VALIDATION] ASBN Training Statistics:")
        try:
            asbn = getattr(core_model, "asbn", None)
            if asbn is not None and hasattr(asbn, "get_detailed_stats"):
                asbn_stats = asbn.get_detailed_stats()
                validation_results["asbn_domain_loss"]       = asbn_stats.get("domain_loss", 0.0)
                validation_results["asbn_domain_accuracy"]   = asbn_stats.get("domain_accuracy", 0.0)
                validation_results["asbn_source_accuracy"]   = asbn_stats.get("source_accuracy", 0.0)
                validation_results["asbn_target_accuracy"]   = asbn_stats.get("target_accuracy", 0.0)
                print(f"  - Domain Loss:     {validation_results['asbn_domain_loss']:.4f}")
                print(f"  - Domain Accuracy: {validation_results['asbn_domain_accuracy']:.2%}")
            else:
                print("  - ASBN statistics not available")
        except Exception:
            print("  - ASBN stats retrieval failed")

        print("\n" + "-" * 80)
        print("[VALIDATION] TRG Explanation Statistics:")
        try:
            trg = getattr(core_model, "trg", None)
            if trg is not None and hasattr(trg, "get_statistics"):
                trg_stats = trg.get_statistics()
                validation_results["trg_total_explanations"] = trg_stats.get("explanations_generated", 0)
                print(f"  - Total explanations: {validation_results['trg_total_explanations']}")
                print(f"  - High confidence rate: {trg_stats.get('high_confidence_rate', 0):.1%}")
                print(f"  - DSCD homograph rate: {trg_stats.get('dscd_homograph_rate', 0):.1%}")
            else:
                print("  - TRG statistics not available")
        except Exception:
            print("  - TRG stats retrieval failed")

        if confidences:
            validation_results["avg_explanation_confidence"] = sum(confidences) / len(confidences)

        print("-" * 80)
        print("\n[VALIDATION] Summary:")
        print(f"  - Translations: {validation_results['translations_success']}/{len(val_sentences)} successful")
        print(f"  - Explanations generated: {validation_results['explanations_generated']}")
        print(f"  - Avg explanation confidence: {validation_results['avg_explanation_confidence']:.3f}")
        print(f"  - DSCD homographs explained: {validation_results['dscd_homographs_explained']}")
        print(f"  - Reference homographs explained: {validation_results['reference_homographs_explained']}")
        if dscd_homograph_words_detected:
            print(f"  - DSCD homographs detected: {', '.join(sorted(dscd_homograph_words_detected))}")
        print(f"  - DSCD Quality Score: {validation_results['dscd_quality_score']:.1%}")

        try:
            use_dual_path = _USE_DUAL_PATH_TRAINING
        except Exception:
            use_dual_path = True

        if use_dual_path and "training_stats" in globals():
            try:
                stats = globals()["training_stats"]
                print("\n" + "-" * 80)
                print("[VALIDATION] Path 1 Diagnostics (DSCD + ASBN):")
                if stats.get("path1_forward_losses"):
                    avg_fwd = np.mean(stats["path1_forward_losses"][-100:])
                    print(f"  - Forward Loss (recent avg): {avg_fwd:.4f}")
                if stats.get("path1_backward_losses"):
                    avg_bwd = np.mean(stats["path1_backward_losses"][-100:])
                    print(f"  - Backward Loss (recent avg): {avg_bwd:.4f}")
                print(f"  - DSCD Clusters: {validation_results['dscd_total_prototypes']}")
                print(f"  - Multi-sense Tokens: {validation_results['dscd_multi_sense_tokens']}")
                print(f"  - DSCD Quality: {validation_results['dscd_quality_score']:.1%}")
                if dscd_homograph_words_detected:
                    print(f"  - Homographs (recent): {', '.join(list(sorted(dscd_homograph_words_detected))[:5])}")
                print(f"  - ASBN Domain Loss: {validation_results['asbn_domain_loss']:.4f}")
                print(f"  - ASBN Domain Accuracy: {validation_results['asbn_domain_accuracy']:.2%}")
                print("\n" + "-" * 80)
                print("[VALIDATION] Path 2 Diagnostics (Translation):")
                if stats.get("path2_forward_losses"):
                    avg_fwd = np.mean(stats["path2_forward_losses"][-100:])
                    print(f"  - Forward Loss (recent avg): {avg_fwd:.4f}")
                if stats.get("path2_backward_losses"):
                    avg_bwd = np.mean(stats["path2_backward_losses"][-100:])
                    print(f"  - Backward Loss (recent avg): {avg_bwd:.4f}")
                if stats.get("path2_losses"):
                    avg_total = np.mean(stats["path2_losses"][-100:])
                    print(f"  - Translation Loss (recent avg): {avg_total:.4f}")
            except Exception:
                pass

        warnings: List[str] = []
        if validation_results["translations_failed"] > len(val_sentences) // 2:
            warnings.append("High translation failure rate")
        if validation_results["explanations_generated"] == 0:
            warnings.append("No explanations generated")
        if validation_results["dscd_quality_score"] < 0.3:
            warnings.append("Low DSCD quality score")
        if validation_results["dscd_multi_sense_tokens"] < 10:
            warnings.append("Very few multi-sense tokens")
        if warnings:
            print("\n[VALIDATION] Health Warnings:")
            for w in warnings:
                print(f"  - {w}")
        else:
            print("\n[VALIDATION] All systems healthy")

        validation_results["validation_completed"] = True

    except Exception as e:
        print(f"\n[VALIDATION] Critical error: {type(e).__name__}: {str(e)[:200]}")
        if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
            try:
                traceback.print_exc()
            except Exception:
                pass
        validation_results["validation_completed"] = False
    finally:
        if was_training:
            core_model.train()
        clear_all_gpu_caches()

    print("=" * 80 + "\n")
    return validation_results


def train_memory_efficient_tatn(
    model: torch.nn.Module,
    tokenizer,
    train_loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    phi_optimizer: Optional[torch.optim.Optimizer] = None,
    epochs: Optional[int] = None,
    accumulation_steps: Optional[int] = None,
    validate_every: Optional[int] = None,
    enable_validation: bool = True,
    use_dual_path: bool = None,
) -> torch.nn.Module:
    if epochs is None:
        epochs = _EPOCHS
    if accumulation_steps is None:
        accumulation_steps = _ACCUMULATION_STEPS
    if validate_every is None:
        validate_every = _VALIDATION_CHECK_INTERVAL
    if use_dual_path is None:
        use_dual_path = _USE_DUAL_PATH_TRAINING

    core_model         = model.module if hasattr(model, "module") else model
    encoder_vocab_size = getattr(core_model, "encoder_vocab_size", 122706)
    decoder_vocab_size = getattr(core_model, "decoder_vocab_size", 32296)

    print(f"[TRAIN] Starting training: epochs={epochs}, batch={_BATCH_SIZE}, accum_steps={accumulation_steps}")
    print(f"[TRAIN] Vocabulary: encoder={encoder_vocab_size}, decoder={decoder_vocab_size}")
    print(f"[TRAIN] Validation: {'enabled' if enable_validation and validate_every > 0 else 'disabled'}")
    print(f"[TRAIN] DP enabled: {_USE_MULTI_GPU}, GPUs: {_NUM_GPUS}, Device: {_DEVICE}")
    print(f"[TRAIN] Discovery frequency: {_PERIODIC_DISCOVERY_FREQUENCY} steps")
    print(f"[TRAIN] Dual-path training: {'ENABLED' if use_dual_path else 'DISABLED (default path=2)'}")
    print(f"[TRAIN] ASBN Training: {'ENABLED' if _ENABLE_ASBN_TRAINING else 'DISABLED'}")
    print(f"[TRAIN] ASBN Optimizer: {'Active' if phi_optimizer is not None else 'Not provided'}")
    print("[TRAIN] Checkpoint: Will save to /kaggle/working/tatn_final.pt after all epochs\n")

    if "translate_with_explanations" not in globals():
        print("[TRAIN] ⚠️  WARNING: translate_with_explanations not found in globals!")
        print("[TRAIN] Validation will fail. Please ensure Cell 8 is executed before training.")

    model.train()
    clear_all_gpu_caches()

    scaler    = GradScaler(enabled=(_USE_AMP and torch.cuda.is_available()))
    scheduler = None

    if _USE_LR_SCHEDULER:
        try:
            from transformers import get_cosine_schedule_with_warmup
            total_steps  = len(train_loader) * epochs
            warmup_steps = _WARMUP_STEPS
            scheduler    = get_cosine_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )
            print(f"[TRAIN] ✅ Learning rate scheduler created:")
            print(f"[TRAIN]    - Type: Cosine with warmup")
            print(f"[TRAIN]    - Total steps: {total_steps}")
            print(f"[TRAIN]    - Warmup steps: {warmup_steps}")
            print(f"[TRAIN]    - Initial LR: {optimizer.param_groups[0]['lr']:.2e}")
        except Exception:
            scheduler = None
    else:
        print("[TRAIN] Learning rate scheduler disabled (USE_LR_SCHEDULER=False)\n")

    global_step       = 0
    accumulated_steps = 0
    pending_validation = False

    training_stats: Dict[str, Any] = {
        "total_loss":                    [],
        "epoch_losses":                  [],
        "backward_losses":               [],
        "batches_processed":             0,
        "optimizer_updates":             0,
        "skipped_batches":               0,
        "oom_errors":                    0,
        "runtime_errors":                0,
        "exceptions":                    0,
        "epoch_validations":             [],
        "dscd_quality_history":          [],
        "multi_sense_ratio_history":     [],
        "asbn_domain_accuracy_history":  [],
        "asbn_domain_loss_history":      [],
        "trg_explanation_history":       [],
        "discovery_runs":                0,
        "discovery_homographs_found":    0,
        "learning_rates":                [],
        "path1_batches":                 0,
        "path2_batches":                 0,
        "path1_losses":                  [],
        "path2_losses":                  [],
        "path1_forward_losses":          [],
        "path1_backward_losses":         [],
        "path2_forward_losses":          [],
        "path2_backward_losses":         [],
        "vocab_clamp_warnings":          0,
    }
    globals()["training_stats"] = training_stats

    last_forward_loss     = 0.0
    last_backward_loss    = 0.0
    cached_cluster_count  = 0
    cached_homograph_count = 0

    for epoch in range(1, epochs + 1):
        epoch_start  = time.time()
        epoch_losses: List[float] = []
        skip_reasons = defaultdict(int)

        print(f"\n{'=' * 80}")
        print(f"EPOCH {epoch}/{epochs} STARTED")
        print(f"{'=' * 80}\n")

        try:
            core = model.module if hasattr(model, "module") else model
            trg  = getattr(core, "trg", None)
            if trg and hasattr(trg, "reset_statistics"):
                try:
                    trg.reset_statistics()
                    print(f"[TRAIN] TRG statistics reset for epoch {epoch}")
                except Exception:
                    pass
        except Exception:
            pass

        try:
            core = model.module if hasattr(model, "module") else model
            asbn = getattr(core, "asbn", None)
            if asbn and hasattr(asbn, "reset_stats"):
                try:
                    asbn.reset_stats()
                    print(f"[TRAIN] ASBN statistics reset for epoch {epoch}")
                except Exception:
                    pass
        except Exception:
            pass

        try:
            optimizer.zero_grad(set_to_none=True)
        except Exception:
            pass

        progress  = None
        batch_idx = 0
        try:
            progress = tqdm(
                total=len(train_loader),
                desc=f"Epoch {epoch}/{epochs}",
                ncols=120,
                leave=False,
                position=0,
                file=sys.stdout
            )
        except Exception:
            progress = None

        try:
            for batch in train_loader:
                batch_idx  += 1
                global_step += 1
                training_stats["batches_processed"] += 1

                if (_DEBUG_DISCOVERY or _VERBOSE_LOGGING) and global_step % DEBUG_PRINT_INTERVAL == 0:
                    print(f"\n[TRAIN-DEBUG] Epoch {epoch} Batch {batch_idx} GlobalStep {global_step}")
                    _check_discovery_status(model, global_step)

                if _PERIODIC_DISCOVERY_FREQUENCY and _PERIODIC_DISCOVERY_FREQUENCY > 0:
                    if global_step % _PERIODIC_DISCOVERY_FREQUENCY == 0:
                        try:
                            core = model.module if hasattr(model, "module") else model
                            dscd = getattr(core, "dscd", None)
                            if dscd is not None and hasattr(dscd, "periodic_discovery_check"):
                                print(f"\n[DISCOVERY] Running periodic check at step {global_step}...")
                                try:
                                    num_discovered = dscd.periodic_discovery_check(global_step, _PERIODIC_DISCOVERY_FREQUENCY)
                                except TypeError:
                                    try:
                                        num_discovered = dscd.periodic_discovery_check(
                                            global_step=global_step,
                                            frequency=_PERIODIC_DISCOVERY_FREQUENCY
                                        )
                                    except Exception:
                                        try:
                                            num_discovered = dscd.periodic_discovery_check()
                                        except Exception:
                                            num_discovered = 0
                                training_stats["discovery_runs"] += 1
                                training_stats["discovery_homographs_found"] += int(num_discovered or 0)
                                if num_discovered and num_discovered > 0:
                                    print(f"[DISCOVERY] Found {num_discovered} new homographs!")
                                else:
                                    print(f"[DISCOVERY] No new homographs found this check")
                                cached_cluster_count   = _get_cluster_count(model)
                                cached_homograph_count = _get_homograph_cluster_count(model)
                        except Exception:
                            pass

                if enable_validation and validate_every and validate_every > 0 and (global_step % validate_every == 0):
                    if accumulated_steps == 0:
                        try:
                            optimizer.zero_grad(set_to_none=True)
                        except Exception:
                            pass
                        val_result = comprehensive_epoch_validation(
                            model, tokenizer, epoch, global_step,
                            _SOURCE_LANGUAGE, _TARGET_LANGUAGE, _MAX_LENGTH, _DEVICE
                        )
                        if val_result:
                            training_stats["epoch_validations"].append(val_result)
                        cached_cluster_count   = _get_cluster_count(model)
                        cached_homograph_count = _get_homograph_cluster_count(model)
                    else:
                        pending_validation = True

                if batch is None:
                    training_stats["skipped_batches"] += 1
                    skip_reasons["batch_none"] += 1
                    if progress is not None:
                        progress.update(1)
                    continue

                max_attempts = 2
                attempt      = 0

                try:
                    input_ids      = batch.get("input_ids", None)
                    attention_mask = batch.get("attention_mask", None)
                    labels         = batch.get("labels", None)
                except Exception:
                    training_stats["skipped_batches"] += 1
                    skip_reasons["bad_batch_structure"] += 1
                    if progress is not None:
                        progress.update(1)
                    continue

                # Check both key variants for src_text
                src_texts_batch = batch.get("src_texts", None) or batch.get("src_text", None)

                domain_labels = batch.get("domain_labels", None)
                if domain_labels is not None:
                    if not isinstance(domain_labels, torch.Tensor):
                        domain_labels = None
                    elif domain_labels.dim() == 0:
                        domain_labels = domain_labels.unsqueeze(0)
                if domain_labels is None:
                    try:
                        domain_labels = torch.full(
                            (int(input_ids.size(0)),),
                            _TRAIN_DOMAIN,
                            dtype=torch.long,
                            device=torch.device("cpu")
                        )
                    except Exception:
                        domain_labels = torch.full(
                            (_BATCH_SIZE,),
                            _TRAIN_DOMAIN,
                            dtype=torch.long,
                            device=torch.device("cpu")
                        )

                try:
                    batch_size = int(input_ids.size(0))
                except Exception:
                    batch_size = _BATCH_SIZE

                if _USE_MULTI_GPU and _NUM_GPUS > 1:
                    keep = (batch_size // _NUM_GPUS) * _NUM_GPUS
                    if keep == 0:
                        training_stats["skipped_batches"] += 1
                        skip_reasons["dp_keep_zero"] += 1
                        if progress is not None:
                            progress.update(1)
                        continue
                    if keep != batch_size:
                        input_ids      = input_ids[:keep]
                        attention_mask = attention_mask[:keep]
                        labels         = labels[:keep]
                        domain_labels  = domain_labels[:keep]
                        if src_texts_batch is not None and isinstance(src_texts_batch, list):
                            src_texts_batch = src_texts_batch[:keep]
                        batch_size = keep

                try:
                    if use_dual_path:
                        selected_path = 1 if batch_idx % 2 == 1 else 2
                    else:
                        selected_path = 2
                except Exception:
                    selected_path = 2

                try:
                    if selected_path == 1 and _ENABLE_ASBN_TRAINING:
                        unique_domains = 1
                        try:
                            if isinstance(domain_labels, torch.Tensor):
                                unique_domains = int(torch.unique(domain_labels.cpu()).numel())
                            else:
                                unique_domains = len(set(domain_labels))
                        except Exception:
                            unique_domains = 1
                        if batch_size < 2 or unique_domains < 2:
                            selected_path = 2
                except Exception:
                    pass

                try:
                    valid_label_count = int((labels != -100).sum().item()) if isinstance(labels, torch.Tensor) else 0
                except Exception:
                    valid_label_count = 0

                if selected_path == 2 and valid_label_count == 0:
                    training_stats["skipped_batches"] += 1
                    skip_reasons["no_valid_labels"] += 1
                    if progress is not None:
                        progress.update(1)
                    continue

                try:
                    encoder_vocab_size_gl = int(globals().get("ENCODER_VOCAB_SIZE", encoder_vocab_size))
                except Exception:
                    encoder_vocab_size_gl = int(encoder_vocab_size)

                try:
                    decoder_vocab_size_gl = int(globals().get("DECODER_VOCAB_SIZE", decoder_vocab_size))
                except Exception:
                    decoder_vocab_size_gl = int(decoder_vocab_size)

                try:
                    input_ids = clamp_input_ids_to_vocab(input_ids, encoder_vocab_size_gl, "input_ids")
                except Exception:
                    pass
                try:
                    labels = clamp_input_ids_to_vocab(labels, decoder_vocab_size_gl, "labels")
                except Exception:
                    pass
                if "decoder_input_ids" in batch:
                    try:
                        batch["decoder_input_ids"] = clamp_input_ids_to_vocab(
                            batch["decoder_input_ids"], decoder_vocab_size_gl, "decoder_input_ids"
                        )
                    except Exception:
                        pass

                try:
                    input_ids      = input_ids.to(_DEVICE, non_blocking=True)
                    attention_mask = attention_mask.to(_DEVICE, non_blocking=True)
                    labels         = labels.to(_DEVICE, non_blocking=True)
                    domain_labels  = domain_labels.to(_DEVICE, non_blocking=True)
                    if "decoder_input_ids" in batch:
                        batch["decoder_input_ids"] = batch["decoder_input_ids"].to(_DEVICE, non_blocking=True)
                except Exception:
                    pass

                while attempt <= max_attempts:
                    try:
                        if selected_path == 1:
                            training_stats["path1_batches"] += 1
                            forward_kwargs = {
                                "input_ids":       input_ids,
                                "attention_mask":  attention_mask,
                                "src_texts":       src_texts_batch,
                                "token_word_map":  batch.get("token_word_map", None),
                                "domain_labels":   domain_labels,
                            }
                            amp_ctx = get_amp_ctx()
                            with amp_ctx:
                                try:
                                    core = model.module if hasattr(model, "module") else model
                                    if hasattr(core, "forward_path1"):
                                        forward_out = core.forward_path1(**forward_kwargs)
                                    else:
                                        forward_kwargs["labels"] = None
                                        forward_out = model(**{**forward_kwargs, "use_cache": False, "path": 1})
                                except Exception:
                                    forward_kwargs["labels"] = None
                                    forward_out = model(**{**forward_kwargs, "use_cache": False, "path": 1})
                        else:
                            training_stats["path2_batches"] += 1
                            forward_kwargs = {
                                "input_ids":       input_ids,
                                "attention_mask":  attention_mask,
                                "labels":          labels,
                                "src_texts":       src_texts_batch,
                                "token_word_map":  batch.get("token_word_map", None),
                            }
                            amp_ctx = get_amp_ctx()
                            with amp_ctx:
                                try:
                                    core = model.module if hasattr(model, "module") else model
                                    if hasattr(core, "forward_path2"):
                                        forward_out = core.forward_path2(**forward_kwargs, use_rdrop=True)
                                    else:
                                        forward_out = model(**{**forward_kwargs, "use_cache": False, "path": 2})
                                except Exception:
                                    forward_out = model(**{**forward_kwargs, "use_cache": False, "path": 2})

                        if isinstance(forward_out, torch.Tensor):
                            loss_tensor = forward_out
                        elif isinstance(forward_out, dict) and "loss" in forward_out:
                            loss_tensor = forward_out["loss"]
                        elif isinstance(forward_out, dict) and "asbn_loss" in forward_out:
                            loss_tensor = forward_out["asbn_loss"]
                        elif isinstance(forward_out, (list, tuple)) and len(forward_out) > 0 and isinstance(forward_out[0], torch.Tensor):
                            loss_tensor = forward_out[0]
                        else:
                            raise RuntimeError("Forward did not return a recognizable loss tensor")

                        if not isinstance(loss_tensor, torch.Tensor):
                            loss_tensor = torch.tensor(float(loss_tensor), device=_DEVICE)
                        else:
                            loss_tensor = loss_tensor.to(_DEVICE)

                        if loss_tensor.numel() > 1:
                            loss_val    = float(loss_tensor.mean().item())
                            loss_tensor = loss_tensor.mean()
                        else:
                            loss_val = float(loss_tensor.item())

                        last_forward_loss = loss_val
                        epoch_losses.append(loss_val)
                        training_stats["total_loss"].append(loss_val)

                        if selected_path == 1:
                            training_stats["path1_losses"].append(loss_val)
                            training_stats["path1_forward_losses"].append(loss_val)
                        else:
                            training_stats["path2_losses"].append(loss_val)
                            training_stats["path2_forward_losses"].append(loss_val)

                        loss_scaled        = loss_tensor / max(1, accumulation_steps)
                        last_backward_loss = float(loss_scaled.item())
                        training_stats["backward_losses"].append(last_backward_loss)

                        if selected_path == 1:
                            training_stats["path1_backward_losses"].append(last_backward_loss)
                        else:
                            training_stats["path2_backward_losses"].append(last_backward_loss)

                        try:
                            if scaler.is_enabled():
                                scaler.scale(loss_scaled).backward()
                            else:
                                loss_scaled.backward()
                            if torch.cuda.is_available():
                                torch.cuda.empty_cache()
                        except RuntimeError as e:
                            raise

                        accumulated_steps += 1

                        if accumulated_steps >= accumulation_steps:
                            try:
                                if scaler.is_enabled():
                                    scaler.unscale_(optimizer)
                                    torch.nn.utils.clip_grad_norm_(model.parameters(), _GRAD_CLIP_NORM)
                                    scaler.step(optimizer)
                                    scaler.update()
                                else:
                                    torch.nn.utils.clip_grad_norm_(model.parameters(), _GRAD_CLIP_NORM)
                                    optimizer.step()
                                if scheduler is not None:
                                    try:
                                        scheduler.step()
                                    except Exception:
                                        pass
                                    current_lr = optimizer.param_groups[0]["lr"]
                                    training_stats["learning_rates"].append(current_lr)
                                optimizer.zero_grad(set_to_none=True)
                                training_stats["optimizer_updates"] += 1
                                if torch.cuda.is_available():
                                    torch.cuda.empty_cache()
                            except RuntimeError as e:
                                raise
                            finally:
                                accumulated_steps = 0
                                if pending_validation:
                                    try:
                                        optimizer.zero_grad(set_to_none=True)
                                    except Exception:
                                        pass
                                    val_result = comprehensive_epoch_validation(
                                        model, tokenizer, epoch, global_step,
                                        _SOURCE_LANGUAGE, _TARGET_LANGUAGE, _MAX_LENGTH, _DEVICE
                                    )
                                    if val_result:
                                        training_stats["epoch_validations"].append(val_result)
                                    pending_validation     = False
                                    cached_cluster_count   = _get_cluster_count(model)
                                    cached_homograph_count = _get_homograph_cluster_count(model)

                        if global_step % DEBUG_PRINT_INTERVAL == 0:
                            _print_gpu_mem("[TRAIN-DEBUG]")
                            cached_cluster_count   = _get_cluster_count(model)
                            cached_homograph_count = _get_homograph_cluster_count(model)
                            path_str = f"p{selected_path}" if use_dual_path else "p2"
                            print(
                                f"[TRAIN-DEBUG] step={global_step} {path_str} "
                                f"loss={last_forward_loss:.4f} "
                                f"protos={cached_cluster_count} "
                                f"homographs={cached_homograph_count}"
                            )
                            _print_top_clusters(model, top_n=5)

                        if global_step % _MEMORY_CLEANUP_FREQUENCY == 0:
                            clear_all_gpu_caches()
                        break

                    except RuntimeError as e:
                        if "out of memory" in str(e).lower():
                            training_stats["oom_errors"] += 1
                            attempt += 1
                            print(f"\n[OOM] Step {global_step} attempt={attempt} - reducing batch and retrying")
                            try:
                                optimizer.zero_grad(set_to_none=True)
                            except Exception:
                                pass
                            for p in model.parameters():
                                if p.grad is not None:
                                    p.grad = None
                            clear_all_gpu_caches()
                            if attempt <= max_attempts:
                                try:
                                    keep           = max(1, int(input_ids.size(0) // 2))
                                    input_ids      = input_ids[:keep]
                                    attention_mask = attention_mask[:keep]
                                    labels         = labels[:keep]
                                    domain_labels  = domain_labels[:keep]
                                    if "decoder_input_ids" in batch:
                                        batch["decoder_input_ids"] = batch["decoder_input_ids"][:keep]
                                    if src_texts_batch is not None and isinstance(src_texts_batch, list):
                                        src_texts_batch = src_texts_batch[:keep]
                                    batch_size = keep
                                    if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                                        print(f"[OOM-RETRY] Reduced batch to {keep}, retrying")
                                    continue
                                except Exception:
                                    training_stats["skipped_batches"] += 1
                                    skip_reasons["oom_could_not_reduce"] += 1
                                    break
                            else:
                                training_stats["skipped_batches"] += 1
                                skip_reasons["oom_max_attempts"] += 1
                                break
                        else:
                            training_stats["runtime_errors"] += 1
                            skip_reasons["runtime"] += 1
                            print(f"\nRUNTIME ERROR during forward/backward: {type(e).__name__}: {e}")
                            traceback.print_exc()
                            try:
                                optimizer.zero_grad(set_to_none=True)
                            except Exception:
                                pass
                            accumulated_steps = 0
                            break

                    except Exception:
                        training_stats["exceptions"] += 1
                        skip_reasons["exceptions"] += 1
                        try:
                            traceback.print_exc()
                        except Exception:
                            pass
                        try:
                            optimizer.zero_grad(set_to_none=True)
                        except Exception:
                            pass
                        accumulated_steps = 0
                        break

                processed_batches  = training_stats["batches_processed"] - training_stats["skipped_batches"]
                expected_updates   = max(1, math.floor(processed_batches / max(1, accumulation_steps)))
                success_rate       = 100.0 * training_stats["optimizer_updates"] / expected_updates if expected_updates > 0 else 0.0
                next_disc          = 0
                try:
                    if _PERIODIC_DISCOVERY_FREQUENCY and _PERIODIC_DISCOVERY_FREQUENCY > 0:
                        next_disc = _PERIODIC_DISCOVERY_FREQUENCY - (global_step % _PERIODIC_DISCOVERY_FREQUENCY)
                except Exception:
                    next_disc = 0

                postfix = {
                    "fwd":  f"{last_forward_loss:.2f}",
                    "bwd":  f"{last_backward_loss:.2f}",
                    "rate": f"{success_rate:.1f}%",
                    "disc": next_disc,
                    # clus now shows actual prototype count, not token-type count
                    "clus": cached_cluster_count,
                    "homo": cached_homograph_count,
                }
                if use_dual_path:
                    postfix["path"] = f"{selected_path}"
                if scheduler is not None and training_stats["learning_rates"]:
                    postfix["lr"] = f"{training_stats['learning_rates'][-1]:.1e}"

                if progress is not None:
                    try:
                        progress.set_postfix(postfix, refresh=False)
                    except Exception:
                        pass
                    progress.update(1)

        finally:
            if progress is not None:
                try:
                    progress.close()
                except Exception:
                    pass

        if accumulated_steps > 0:
            try:
                if scaler.is_enabled():
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), _GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), _GRAD_CLIP_NORM)
                    optimizer.step()
                if scheduler is not None:
                    try:
                        scheduler.step()
                    except Exception:
                        pass
                optimizer.zero_grad(set_to_none=True)
                training_stats["optimizer_updates"] += 1
            except Exception:
                pass
            finally:
                accumulated_steps = 0

        epoch_duration_min = (time.time() - epoch_start) / 60.0
        processed_batches  = training_stats["batches_processed"] - training_stats["skipped_batches"]
        expected_updates   = max(1, math.floor(processed_batches / max(1, accumulation_steps)))
        success_rate       = 100.0 * training_stats["optimizer_updates"] / expected_updates if expected_updates > 0 else 0.0
        cached_cluster_count   = _get_cluster_count(model)
        cached_homograph_count = _get_homograph_cluster_count(model)
        avg_epoch_loss         = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        training_stats["epoch_losses"].append(avg_epoch_loss)

        print("\n" + "=" * 80)
        print(f"EPOCH {epoch}/{epochs} SUMMARY")
        print("=" * 80)
        print(f"  Duration (min):    {epoch_duration_min:.2f}")
        print(f"  Optimizer updates: {training_stats['optimizer_updates']}")
        print(f"  Batches:           processed={processed_batches}, skipped={training_stats['skipped_batches']}")
        print(f"  Success rate:      {success_rate:.1f}%")
        print(f"  Total prototypes:  {cached_cluster_count}")
        print(f"  Homograph clusters:{cached_homograph_count}")
        print(f"  Avg epoch loss:    {avg_epoch_loss:.6f}")
        print(f"  Discovery runs:    {training_stats['discovery_runs']}")
        print(f"  Homographs found:  {training_stats['discovery_homographs_found']}")
        if use_dual_path:
            print(f"  Path 1 batches:    {training_stats['path1_batches']}")
            print(f"  Path 2 batches:    {training_stats['path2_batches']}")
            if training_stats["path1_losses"]:
                print(f"  Path 1 avg loss:   {np.mean(training_stats['path1_losses']):.4f}")
            if training_stats["path2_losses"]:
                print(f"  Path 2 avg loss:   {np.mean(training_stats['path2_losses']):.4f}")
        if scheduler is not None and training_stats["learning_rates"]:
            print(f"  Current LR:        {training_stats['learning_rates'][-1]:.2e}")
        if skip_reasons:
            print("  Skip reasons:")
            for k, v in sorted(skip_reasons.items(), key=lambda x: -x[1]):
                print(f"    - {k}: {v}")
        print("=" * 80)

        if enable_validation:
            try:
                optimizer.zero_grad(set_to_none=True)
            except Exception:
                pass
            try:
                validation_results = comprehensive_epoch_validation(
                    model=model,
                    tokenizer=tokenizer,
                    epoch=epoch,
                    global_step=global_step,
                    source_lang=_SOURCE_LANGUAGE,
                    target_lang=_TARGET_LANGUAGE,
                    max_length=_MAX_LENGTH,
                    device=_DEVICE
                )
                if validation_results and validation_results.get("validation_completed", False):
                    training_stats["epoch_validations"].append(validation_results)
                    training_stats["dscd_quality_history"].append(validation_results.get("dscd_quality_score", 0.0))
                    training_stats["asbn_domain_accuracy_history"].append(validation_results.get("asbn_domain_accuracy", 0.0))
                    training_stats["asbn_domain_loss_history"].append(validation_results.get("asbn_domain_loss", 0.0))
                    training_stats["trg_explanation_history"].append(validation_results.get("trg_total_explanations", 0))
                    try:
                        dscd = getattr(model.module if hasattr(model, "module") else model, "dscd", None)
                        if dscd is not None:
                            lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
                            if lock:
                                with lock:
                                    total_tokens = len(getattr(dscd, "prototype_stores", {}))
                            else:
                                total_tokens = len(getattr(dscd, "prototype_stores", {}))
                            multi_sense = validation_results.get("dscd_multi_sense_tokens", 0)
                            ratio       = multi_sense / total_tokens if total_tokens > 0 else 0.0
                            training_stats["multi_sense_ratio_history"].append(ratio)
                        else:
                            training_stats["multi_sense_ratio_history"].append(0.0)
                    except Exception:
                        training_stats["multi_sense_ratio_history"].append(0.0)
                else:
                    print("[TRAIN] Validation incomplete")
            except Exception as e:
                if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
                    try:
                        traceback.print_exc()
                    except Exception:
                        pass

    try:
        checkpoint_path = Path("/kaggle/working/tatn_final.pt")
        core_model      = model.module if hasattr(model, "module") else model
        dscd_state      = {}
        try:
            if hasattr(core_model, "dscd"):
                try:
                    dscd_state = core_model.dscd.state_dict()
                except Exception:
                    dscd_state = {}
        except Exception:
            dscd_state = {}

        checkpoint_data = {
            "epochs_trained":       epochs,
            "global_steps":         global_step,
            "final_train_loss":     training_stats["epoch_losses"][-1] if training_stats["epoch_losses"] else 0.0,
            "model_state_dict":     core_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict":    scaler.state_dict() if scaler is not None else None,
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "training_stats":       training_stats,
            "dscd_state":           dscd_state,
            "config": {
                "SPAN_THRESHOLD":               globals().get("SPAN_THRESHOLD", 0.20),
                "TAU_LOW":                      globals().get("TAU_LOW", 0.15),
                "LAMBDA_ASBN":                  globals().get("LAMBDA_ASBN", 0.3),
                "LAMBDA_DSCD":                  globals().get("LAMBDA_DSCD", 0.15),
                "LAMBDA_TRG":                   globals().get("LAMBDA_TRG", 0.15),
                "LAMBDA_TOKEN":                 globals().get("LAMBDA_TOKEN", 0.3),
                "LAMBDA_CONFIDENCE":            globals().get("LAMBDA_CONFIDENCE", 0.2),
                "LAMBDA_LENGTH":                globals().get("LAMBDA_LENGTH", 0.1),
                "TRG_TEMPERATURE":              globals().get("TRG_TEMPERATURE", 1.0),
                "PERIODIC_DISCOVERY_FREQUENCY": _PERIODIC_DISCOVERY_FREQUENCY,
                "NUM_EPOCHS":                   epochs,
                "BATCH_SIZE":                   _BATCH_SIZE,
                "LEARNING_RATE":                optimizer.param_groups[0]["lr"] if optimizer and optimizer.param_groups else 0.0,
                "WARMUP_STEPS":                 _WARMUP_STEPS,
                "USE_LR_SCHEDULER":             _USE_LR_SCHEDULER,
                "USE_DUAL_PATH_TRAINING":       use_dual_path,
            },
        }
        torch.save(checkpoint_data, checkpoint_path)
        file_size_mb = checkpoint_path.stat().st_size / (1024 ** 2)
        print("\nFINAL CHECKPOINT SAVED")
        print(f"   Path: {checkpoint_path}")
        print(f"   Size: {file_size_mb:.2f} MB")
        print(f"   Epochs trained: {epochs}")
        print(f"   Global steps: {global_step}")
        print(f"   Final train loss: {training_stats['epoch_losses'][-1] if training_stats['epoch_losses'] else 0.0:.4f}")
        print(f"   LR scheduler used: {'Yes' if scheduler is not None else 'No'}")
        print(f"   Dual-path mode: {'Yes' if use_dual_path else 'No (default path=2)'}")
        print(f"{'=' * 80}\n")
    except Exception as e:
        print(f"FINAL CHECKPOINT SAVE FAILED: {type(e).__name__}")
        if _DEBUG_DISCOVERY or _VERBOSE_LOGGING:
            try:
                traceback.print_exc()
            except Exception:
                pass

    print("\n" + "=" * 80)
    print("TRAINING COMPLETED - FINAL SUMMARY")
    print("=" * 80)
    processed_batches  = training_stats["batches_processed"] - training_stats["skipped_batches"]
    expected_updates   = max(1, math.floor(processed_batches / max(1, accumulation_steps)))
    success_rate       = 100.0 * training_stats["optimizer_updates"] / expected_updates if expected_updates > 0 else 0.0
    print(f"[TRAIN] Success Rate:           {success_rate:.1f}%")
    print(f"[TRAIN] Total Steps:            {global_step}")
    print(f"[TRAIN] Total Prototypes:       {cached_cluster_count}")
    print(f"[TRAIN] Homograph Clusters:     {cached_homograph_count}")
    print(f"[TRAIN] Discovery Runs:         {training_stats['discovery_runs']}")
    print(f"[TRAIN] Total Homographs Found: {training_stats['discovery_homographs_found']}")
    print(f"[TRAIN] LR Scheduler:           {'Enabled' if scheduler is not None else 'Disabled'}")
    print(f"[TRAIN] Dual-Path Mode:         {'Enabled' if use_dual_path else 'Disabled'}")
    if use_dual_path:
        print(f"[TRAIN] Path 1 Total Batches:   {training_stats['path1_batches']}")
        print(f"[TRAIN] Path 2 Total Batches:   {training_stats['path2_batches']}")
    if training_stats["dscd_quality_history"]:
        print("\n[TRAIN] DSCD Quality Score Trend:")
        for i, score in enumerate(training_stats["dscd_quality_history"], 1):
            print(f"  Epoch {i}: {score:.1%}")
    print("=" * 80)
    return model


print("\n" + "=" * 80)
print("Cell 7: DUAL-PATH Training Loop - PATH 1 (DSCD) + PATH 2 (TRANSLATION) (IndicTrans2) - FIXED")
print("=" * 80)
print("Configuration:")
print(f"  Source language:        {_SOURCE_LANGUAGE}")
print(f"  Target language:        {_TARGET_LANGUAGE}")
print(f"  Discovery frequency:    {_PERIODIC_DISCOVERY_FREQUENCY} steps")
print(f"  Validation interval:    {_VALIDATION_CHECK_INTERVAL} steps")
print(f"  Train domain:           {_TRAIN_DOMAIN}")
print(f"  Test domain:            {_TEST_DOMAIN}")
print(f"  Gradient clip:          {_GRAD_CLIP_NORM}")
print(f"  Memory cleanup:         Every {_MEMORY_CLEANUP_FREQUENCY} steps")
print(f"  Lambda TRG:             {_LAMBDA_TRG}")
print(f"  Warmup steps:           {_WARMUP_STEPS}")
print(f"  LR scheduler:           {'Enabled' if _USE_LR_SCHEDULER else 'Disabled'}")
print(f"  Dual-path training:     {'Enabled' if _USE_DUAL_PATH_TRAINING else 'Disabled (default)'}")
print("=" * 80 + "\n")



Cell 7: DUAL-PATH Training Loop - PATH 1 (DSCD) + PATH 2 (TRANSLATION) (IndicTrans2) - FIXED
Configuration:
  Source language:        ben_Beng
  Target language:        eng_Latn
  Discovery frequency:    100 steps
  Validation interval:    200 steps
  Train domain:           0
  Test domain:            1
  Gradient clip:          1.0
  Memory cleanup:         Every 50 steps
  Lambda TRG:             0.15
  Warmup steps:           4000
  LR scheduler:           Enabled
  Dual-path training:     Enabled



In [11]:
# ===========================================================================================
# CELL 8: INFERENCE & EVALUATION PIPELINE (DUAL-PATH COMPATIBLE) (INDICTRANS2) - FIXED
# ===========================================================================================
import os
import time
import math
import torch
import traceback
from typing import List, Dict, Any, Tuple, Optional
from collections import defaultdict
from transformers.modeling_outputs import BaseModelOutput
import threading
import gc

try:
    SOURCE_LANG = str(SOURCE_LANGUAGE)
    TARGET_LANG = str(TARGET_LANGUAGE)
except Exception:
    SOURCE_LANG = "ben_Beng"
    TARGET_LANG = "eng_Latn"

try:
    MAXLEN = int(MAX_LENGTH)
except Exception:
    MAXLEN = 64

try:
    DEVICE = DEVICE
except Exception:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except Exception:
    VERBOSE_LOGGING = False

try:
    DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except Exception:
    DEBUG_DISCOVERY = False

try:
    DEBUG_TIMING = bool(DEBUG_TIMING)
except Exception:
    DEBUG_TIMING = False

try:
    USE_MULTI_GPU = bool(USE_MULTI_GPU)
except Exception:
    USE_MULTI_GPU = torch.cuda.is_available() and (torch.cuda.device_count() > 1)

try:
    SPAN_THRESHOLD = float(SPAN_THRESHOLD)
except Exception:
    SPAN_THRESHOLD = 0.20

try:
    UNCERTAINTY_THRESHOLD = float(UNCERTAINTY_THRESHOLD)
except Exception:
    UNCERTAINTY_THRESHOLD = 0.15

try:
    HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST_BN)
except Exception:
    HOMOGRAPH_REFERENCE_LIST = {
        "কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা", "বার", "হার", "তারা",
        "পানি", "দল", "বাজার", "নাম", "কথা", "বই", "ঘর", "মন", "হাত"
    }
    HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST)

try:
    TEST_DOMAIN = int(TEST_DOMAIN)
except Exception:
    TEST_DOMAIN = 1

BENGALI_PUNCT_SET = set(['।', '॥'])
COMMON_PUNCT_SET = set(['.', ',', ';', ':', '!', '?', '"', "'", '-', '(', ')', '[', ']', '{', '}', '/', '\\'])
PUNCT_SET = BENGALI_PUNCT_SET | COMMON_PUNCT_SET


def is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = (
        token.replace("▁", "")
        .replace("Ġ", "")
        .replace("##", "")
        .replace("</w>", "")
        .strip()
    )
    if not clean:
        return False
    if clean in BENGALI_PUNCT_SET or clean in COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in PUNCT_SET for c in clean)


def clean_token(token: str) -> str:
    if not isinstance(token, str):
        return ""
    cleaned = token.replace("▁", "").replace("Ġ", "").replace("##", "").strip()
    for punct in [".", ",", "!", "?", ";", ":", "-"]:
        cleaned = cleaned.replace(punct, "")
    return cleaned.lower()


def get_dscd_homographs(model: torch.nn.Module) -> set:
    try:
        core = model.module if hasattr(model, 'module') else model
        dscd = getattr(core, 'dscd', None)
        if dscd is None:
            return set()
        if hasattr(dscd, 'get_discovered_homographs'):
            try:
                discovered = dscd.get_discovered_homographs()
                return set(w for w in discovered if not is_punctuation_only(w))
            except Exception:
                pass
        homographs = set()
        lock = getattr(dscd, 'buffer_lock', None) or getattr(dscd, 'clustering_lock', None)
        if lock:
            with lock:
                items = list(dscd.prototype_stores.items())
        else:
            items = list(getattr(dscd, 'prototype_stores', {}).items())
        for token, store in items:
            try:
                if hasattr(store, 'size') and store.size() >= 2:
                    clean_tok = clean_token(str(token))
                    if clean_tok and not is_punctuation_only(str(token)):
                        homographs.add(clean_tok)
            except Exception:
                continue
        return homographs
    except Exception:
        return set()


class InferenceStatistics:
    def __init__(self):
        self._lock = threading.Lock()
        self.reset()

    def reset(self):
        with self._lock:
            self.total_inferences = 0
            self.successful_translations = 0
            self.failed_translations = 0
            self.total_explanations = 0
            self.high_confidence_explanations = 0
            self.low_confidence_explanations = 0
            self.total_confidence = 0.0
            self.dscd_homographs_explained = set()
            self.reference_homographs_explained = set()
            self.avg_span = 0.0
            self.avg_uncertainty = 0.0
            self.dscd_empty_warnings = 0
            self.token_counts = defaultdict(int)
            self.token_confidences = defaultdict(list)

    def record_inference(self, result: Dict[str, Any], dscd_homographs: Optional[set] = None):
        with self._lock:
            self.total_inferences += 1
            if result.get('translation') and result['translation'] != "ERROR DURING TRANSLATION":
                self.successful_translations += 1
            else:
                self.failed_translations += 1
            explanations = result.get('explanations', [])
            self.total_explanations += len(explanations)
            for exp in explanations:
                try:
                    conf = exp.get('confidence', 0.5)
                    self.total_confidence += float(conf)
                    if conf >= 0.65:
                        self.high_confidence_explanations += 1
                    elif conf < 0.4:
                        self.low_confidence_explanations += 1
                    word = str(exp.get('ambiguous_word', '')).strip()
                    if is_punctuation_only(word):
                        continue
                    clean_word = clean_token(word)
                    if not clean_word:
                        continue
                    self.token_counts[clean_word] += 1
                    self.token_confidences[clean_word].append(float(conf))
                    if dscd_homographs and clean_word in dscd_homographs:
                        self.dscd_homographs_explained.add(clean_word)
                    if clean_word in HOMOGRAPH_REFERENCE_LIST:
                        self.reference_homographs_explained.add(clean_word)
                    self.avg_span += float(exp.get('span', 0.0))
                    self.avg_uncertainty += float(exp.get('uncertainty', 0.0))
                except Exception:
                    pass

    def get_summary(self) -> Dict[str, Any]:
        with self._lock:
            total_exp = max(self.total_explanations, 1)
            unique_tokens = len(self.token_counts)
            diversity_ratio = unique_tokens / total_exp if total_exp > 0 else 0.0
            return {
                'total_inferences': self.total_inferences,
                'successful_translations': self.successful_translations,
                'failed_translations': self.failed_translations,
                'success_rate': self.successful_translations / max(self.total_inferences, 1),
                'total_explanations': self.total_explanations,
                'explanations_per_inference': self.total_explanations / max(self.total_inferences, 1),
                'high_confidence_rate': self.high_confidence_explanations / total_exp,
                'low_confidence_rate': self.low_confidence_explanations / total_exp,
                'avg_confidence': self.total_confidence / total_exp,
                'avg_span': self.avg_span / total_exp,
                'avg_uncertainty': self.avg_uncertainty / total_exp,
                'dscd_homographs_explained': list(self.dscd_homographs_explained),
                'reference_homographs_explained': list(self.reference_homographs_explained),
                'dscd_empty_warnings': self.dscd_empty_warnings,
                'unique_tokens_explained': unique_tokens,
                'diversity_ratio': diversity_ratio,
            }

    def print_summary(self):
        summary = self.get_summary()
        print("\n" + "=" * 80)
        print("INFERENCE STATISTICS SUMMARY")
        print("=" * 80)
        print(f"Total inferences: {summary['total_inferences']}")
        print(f"Success rate: {summary['success_rate']:.1%}")
        print(f"Total explanations: {summary['total_explanations']}")
        print(f"Explanations per inference: {summary['explanations_per_inference']:.2f}")
        print(f"Unique tokens explained: {summary['unique_tokens_explained']}")
        print(f"Diversity ratio: {summary['diversity_ratio']:.2%}")
        print(f"Avg confidence: {summary['avg_confidence']:.3f}")
        print(f"High confidence rate: {summary['high_confidence_rate']:.1%}")
        print(f"Avg span: {summary['avg_span']:.3f}")
        print(f"Avg uncertainty: {summary['avg_uncertainty']:.3f}")
        if summary['dscd_homographs_explained']:
            print(f"\nDSCD homographs explained ({len(summary['dscd_homographs_explained'])})")
            print(f"  {', '.join(summary['dscd_homographs_explained'])}")
        if summary['reference_homographs_explained']:
            print(f"\nReference homographs explained ({len(summary['reference_homographs_explained'])})")
            print(f"  {', '.join(summary['reference_homographs_explained'])}")
        if summary['dscd_empty_warnings'] > 0:
            print(f"\nDSCD empty warnings: {summary['dscd_empty_warnings']}")
        print("=" * 80 + "\n")


INFERENCE_STATS = InferenceStatistics()


def to_device_batch(enc: Any, device: torch.device):
    try:
        if hasattr(enc, "to") and callable(getattr(enc, "to")):
            return enc.to(device)
    except Exception:
        pass
    if isinstance(enc, dict):
        out = {}
        try:
            for k, v in enc.items():
                try:
                    if isinstance(v, torch.Tensor):
                        out[k] = v.to(device)
                    elif isinstance(v, dict):
                        out[k] = to_device_batch(v, device)
                    elif isinstance(v, (list, tuple)):
                        out[k] = [
                            t.to(device) if isinstance(t, torch.Tensor) else t
                            for t in v
                        ]
                    else:
                        out[k] = v
                except Exception:
                    out[k] = v
            return out
        except Exception:
            return enc
    return enc


def extract_dscd_outputs(raw_out: Any) -> Dict[str, Any]:
    if raw_out is None:
        return {}
    if isinstance(raw_out, dict):
        if "dscd_outputs" in raw_out and isinstance(raw_out["dscd_outputs"], dict):
            return raw_out["dscd_outputs"]
        if "dscd" in raw_out and isinstance(raw_out["dscd"], dict):
            return raw_out["dscd"]
        if "proto_probs" in raw_out or "uncertainties" in raw_out:
            return raw_out
        for key in ("dscd_outputs", "dscd", "dscd_out"):
            if key in raw_out and isinstance(raw_out[key], dict):
                return raw_out[key]
        return raw_out
    if isinstance(raw_out, (list, tuple)):
        for item in raw_out:
            if isinstance(item, dict):
                return extract_dscd_outputs(item)
    return {}


def is_subword_token(token: str) -> bool:
    if not token or len(token.strip()) == 0:
        return True
    token = token.strip()
    if is_punctuation_only(token):
        return True
    if (
        token.startswith("##")
        or token.startswith("▁▁")
        or token.startswith("@@")
        or token.startswith("▁")
    ):
        return True
    if len(token) < 2:
        return True
    if (len(token) == 1 and token in PUNCT_SET) or token.isdigit():
        return True
    return False


def should_filter_explanation(expl: Dict[str, Any], span_th: float, u_th: float) -> bool:
    try:
        token = expl.get('ambiguous_word', expl.get('token', ''))
        if is_punctuation_only(str(token)):
            return True
        span = float(expl.get('span', 0.0))
        uncertainty = float(expl.get('uncertainty', 0.0))
        if is_subword_token(str(token)):
            return True
        if span < span_th and uncertainty < u_th:
            return True
        return False
    except Exception:
        return True


def has_bengali_chars(text: str) -> bool:
    if not text or not isinstance(text, str):
        return False
    return any('\u0980' <= c <= '\u09FF' for c in text)


def get_target_lang_token_id(tokenizer, model_core, target_lang: str, vocab_size: int) -> int:
    try:
        # Priority 1: use tgt_token_id already resolved in Cell 6 TATN __init__
        try:
            tgt_id = getattr(model_core, 'tgt_token_id', None)
            if tgt_id is not None and 0 <= int(tgt_id) < vocab_size:
                return int(tgt_id)
        except Exception:
            pass

        try:
            if hasattr(tokenizer, 'get_lang_id'):
                lang_id = tokenizer.get_lang_id(target_lang)
                if lang_id is not None and 0 <= int(lang_id) < vocab_size:
                    return int(lang_id)
        except Exception:
            pass

        try:
            mapping = getattr(tokenizer, 'lang_code_to_id', None)
            if isinstance(mapping, dict):
                lid = mapping.get(target_lang, None)
                if lid is not None and 0 <= int(lid) < vocab_size:
                    return int(lid)
        except Exception:
            pass

        try:
            conv = getattr(tokenizer, 'convert_tokens_to_ids', None)
            if callable(conv):
                lid = conv(target_lang)
                if isinstance(lid, list) and len(lid) > 0:
                    lid = int(lid[0])
                if isinstance(lid, int) and 0 <= lid < vocab_size:
                    return int(lid)
        except Exception:
            pass

        try:
            cfg = getattr(model_core, 'config', None)
            if cfg is None:
                core_base = getattr(model_core, 'base_model', None)
                cfg = getattr(core_base, 'config', None) if core_base is not None else None
            if cfg is not None:
                cand = getattr(cfg, 'forced_bos_token_id', None)
                if cand is not None and 0 <= int(cand) < vocab_size:
                    return int(cand)
                cand2 = getattr(cfg, 'decoder_start_token_id', None)
                if cand2 is not None and 0 <= int(cand2) < vocab_size:
                    return int(cand2)
        except Exception:
            pass

        return int(max(0, vocab_size - 1))
    except Exception:
        return int(max(0, vocab_size - 1))


def _get_indic_processor():
    ip = globals().get('indic_processor_instance', None)
    if ip is not None:
        return ip
    try:
        _has = globals().get('_HAS_INDIC_PROCESSOR', False)
        IndicProc = globals().get('IndicProcessor', None)
        if _has and IndicProc is not None:
            return IndicProc(inference=True)
    except Exception:
        pass
    return None


def _preprocess_text(input_sentence: str, source_lang: str, target_lang: str) -> str:
    if 'preprocess_for_indictrans2' in globals():
        try:
            indic_processor = _get_indic_processor()
            if indic_processor:
                preprocessed_batch = globals()['preprocess_for_indictrans2'](
                    [input_sentence],
                    src_lang=source_lang,
                    tgt_lang=target_lang,
                    indic_processor=indic_processor
                )
                if preprocessed_batch and len(preprocessed_batch) > 0:
                    return preprocessed_batch[0]
        except Exception:
            pass
    return f"{source_lang} {target_lang} {input_sentence}"


def _preprocess_batch_texts(batch: List[str], source_lang: str, target_lang: str) -> List[str]:
    if 'preprocess_for_indictrans2' in globals():
        try:
            indic_processor = _get_indic_processor()
            if indic_processor:
                preprocessed = globals()['preprocess_for_indictrans2'](
                    batch,
                    src_lang=source_lang,
                    tgt_lang=target_lang,
                    indic_processor=indic_processor
                )
                if preprocessed and len(preprocessed) == len(batch):
                    return preprocessed
        except Exception:
            pass
    return [f"{source_lang} {target_lang} {text}" for text in batch]


@torch.inference_mode()
def translate_with_explanations(
    model,
    tokenizer,
    input_sentence: str,
    source_lang: str = "ben_Beng",
    target_lang: str = "eng_Latn",
    device: Optional[torch.device] = None,
    max_length: Optional[int] = None,
    span_threshold: Optional[float] = None,
    uncertainty_threshold: Optional[float] = None,
    track_stats: bool = True,
) -> Dict[str, Any]:
    device = DEVICE if device is None else device
    max_len = MAXLEN if max_length is None else int(max_length)
    span_th = SPAN_THRESHOLD if span_threshold is None else float(span_threshold)
    u_th = UNCERTAINTY_THRESHOLD if uncertainty_threshold is None else float(uncertainty_threshold)

    if not input_sentence or not input_sentence.strip():
        return {
            "input_sentence": input_sentence,
            "translation": "",
            "ambiguous_words_detected": 0,
            "explanations": [],
            "quality_metrics": {},
            "dscd_validated": False,
            "error": "Empty input"
        }

    if not has_bengali_chars(input_sentence):
        if DEBUG_DISCOVERY or VERBOSE_LOGGING:
            print(f"[INF] WARNING: Input does not contain Bengali characters: {input_sentence[:50]}")

    try:
        if hasattr(tokenizer, 'src_lang') and hasattr(tokenizer, 'tgt_lang'):
            try:
                tokenizer.src_lang = source_lang
                tokenizer.tgt_lang = target_lang
            except Exception:
                pass
        elif hasattr(tokenizer, 'set_src_lang_special_tokens') and hasattr(tokenizer, 'set_tgt_lang_special_tokens'):
            try:
                tokenizer.set_src_lang_special_tokens(source_lang)
                tokenizer.set_tgt_lang_special_tokens(target_lang)
            except Exception:
                pass
    except Exception:
        pass

    if DEBUG_DISCOVERY or VERBOSE_LOGGING:
        print(f"\n[INF] Starting inference:")
        print(f"[INF]   Input: {input_sentence[:60]}")
        print(f"[INF]   Languages: {source_lang} -> {target_lang}")
        print(f"[INF]   Thresholds: span={span_th:.2f}, uncertainty={u_th:.2f}")

    dscd_homographs = get_dscd_homographs(model)

    try:
        text_to_tokenize = _preprocess_text(input_sentence, source_lang, target_lang)

        enc = tokenizer(
            text_to_tokenize,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len,
        )
        enc = to_device_batch(enc, device)

        model.eval()
        # Always unwrap DataParallel — generate() must run on the core model
        core = model.module if (USE_MULTI_GPU and hasattr(model, "module")) else model
        src_texts = [input_sentence]
        dscd_validated = False

        try:
            dscd = getattr(core, 'dscd', None)
            if dscd:
                lock = getattr(dscd, 'buffer_lock', None) or getattr(dscd, 'clustering_lock', None)
                num_stores = 0
                multi_sense = 0
                if lock:
                    with lock:
                        num_stores = len(getattr(dscd, 'prototype_stores', {}))
                        multi_sense = sum(1 for store in dscd.prototype_stores.values() if hasattr(store, 'centroids') and len(store.centroids) >= 2)
                else:
                    num_stores = len(getattr(dscd, 'prototype_stores', {}))
                    multi_sense = sum(1 for store in dscd.prototype_stores.values() if hasattr(store, 'centroids') and len(store.centroids) >= 2)
                if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                    print(f"[INF] DSCD state: {num_stores} tokens, {multi_sense} multi-sense, {len(dscd_homographs)} discovered")
                if num_stores == 0:
                    if track_stats:
                        INFERENCE_STATS.dscd_empty_warnings += 1
                else:
                    dscd_validated = True
        except Exception:
            pass

        with torch.inference_mode():
            # Step 1: DSCD forward pass for explanation outputs
            dscd_out_dict: Dict[str, Any] = {}
            try:
                fwd_outputs = core(
                    input_ids=enc.get("input_ids"),
                    attention_mask=enc.get("attention_mask"),
                    src_texts=src_texts,
                    token_word_map=None,
                    labels=None,
                    return_dict=True,
                    path=2
                )
                dscd_out_dict = extract_dscd_outputs(fwd_outputs)
                if DEBUG_DISCOVERY:
                    print(f"[INF] DSCD outputs extracted: {list(dscd_out_dict.keys()) if isinstance(dscd_out_dict, dict) else 'NOT_DICT'}")
            except Exception as e:
                if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                    print(f"[INF] DSCD forward error: {e}")
                dscd_out_dict = {}

            # Step 2: Get DSCD-augmented encoder hidden state
            # Primary path: run encoder directly on base_model using DSCD-augmented embeddings
            h_sense = None
            try:
                fwd_out = core(
                    input_ids=enc.get("input_ids"),
                    attention_mask=enc.get("attention_mask"),
                    src_texts=src_texts,
                    token_word_map=None,
                    labels=None,
                    return_dict=True,
                    path=2
                )
                if isinstance(fwd_out, dict):
                    # Prefer sense_augmented_embeddings (DSCD output) over raw encoder output
                    h_sense = (
                        fwd_out.get('sense_augmented_embeddings')
                        or fwd_out.get('encoder_last_hidden_state')
                        or None
                    )
                else:
                    h_sense = getattr(fwd_out, 'encoder_last_hidden_state', None)
            except Exception as e:
                if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                    print(f"[INF] DSCD forward (step 2) error: {e}")
                h_sense = None

            # Fallback: run base_model encoder directly if TATN forward didn't return encoder hidden state
            if h_sense is None:
                try:
                    base_model = getattr(core, 'base_model', None)
                    if base_model is not None:
                        encoder = getattr(getattr(base_model, 'model', base_model), 'encoder', None)
                        if encoder is None:
                            encoder = getattr(base_model, 'encoder', None)
                        if encoder is not None:
                            enc_out = encoder(
                                input_ids=enc.get("input_ids"),
                                attention_mask=enc.get("attention_mask"),
                                return_dict=True
                            )
                            h_sense = getattr(enc_out, 'last_hidden_state', None)
                            if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                                print(f"[INF] Used direct encoder fallback, shape: {h_sense.shape if h_sense is not None else 'None'}")
                except Exception as enc_err:
                    if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                        print(f"[INF] Direct encoder fallback failed: {enc_err}")
                    h_sense = None

            if h_sense is None:
                return {
                    "input_sentence": input_sentence,
                    "translation": "",
                    "ambiguous_words_detected": 0,
                    "explanations": [],
                    "quality_metrics": {},
                    "dscd_validated": dscd_validated,
                    "error": "No encoder outputs found in forward pass"
                }

            if not isinstance(h_sense, torch.Tensor):
                try:
                    h_sense = torch.as_tensor(h_sense, dtype=torch.float32, device=device)
                except Exception:
                    h_sense = torch.tensor(h_sense, dtype=torch.float32, device=device)
            else:
                h_sense = h_sense.to(device)
                if h_sense.dtype != torch.float32:
                    try:
                        h_sense = h_sense.float()
                    except Exception:
                        pass

            encoder_outputs_wrapped = BaseModelOutput(
                last_hidden_state=h_sense,
                hidden_states=None,
                attentions=None
            )

            # Step 3: resolve generator — MUST use base_model.generate(), not core.generate()
            # core is the TATN wrapper whose forward() is not a seq2seq generate path
            base_model = getattr(core, 'base_model', None)
            generator = None
            gen_owner = None

            if base_model is not None and hasattr(base_model, 'generate') and callable(base_model.generate):
                generator = base_model
                gen_owner = base_model
            elif hasattr(core, 'generate') and callable(core.generate):
                # Only use core.generate as last resort (may not work with TATN wrapper)
                generator = core
                gen_owner = core
            else:
                if hasattr(model, 'generate') and callable(model.generate):
                    generator = model
                    gen_owner = model

            if generator is None:
                return {
                    "input_sentence": input_sentence,
                    "translation": "",
                    "ambiguous_words_detected": 0,
                    "explanations": [],
                    "quality_metrics": {},
                    "dscd_validated": dscd_validated,
                    "error": "No generate() method available on model"
                }

            # Resolve decoder vocab size
            decoder_vocab_size = None
            try:
                decoder_vocab_size = getattr(core, 'decoder_vocab_size', None)
                if decoder_vocab_size is None and hasattr(core, 'config'):
                    decoder_vocab_size = getattr(core.config, 'decoder_vocab_size', None)
                if decoder_vocab_size is None:
                    base_cfg = getattr(getattr(core, 'base_model', None), 'config', None)
                    if base_cfg is not None:
                        decoder_vocab_size = getattr(base_cfg, 'decoder_vocab_size', None) or getattr(base_cfg, 'vocab_size', None)
            except Exception:
                decoder_vocab_size = None
            if decoder_vocab_size is None:
                decoder_vocab_size = 32296  # IndicTrans2 200M decoder vocab

            forced_id = get_target_lang_token_id(tokenizer, core, target_lang, int(decoder_vocab_size))
            if forced_id is None:
                forced_id = int(decoder_vocab_size - 1)
            if forced_id >= int(decoder_vocab_size) or forced_id < 0:
                if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                    print(f"[INF] forced_id ({forced_id}) out of range for decoder_vocab_size ({decoder_vocab_size}) - clamping")
                forced_id = max(0, int(decoder_vocab_size - 1))
            if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                print(f"[INF] Target language: {target_lang} -> Token ID: {forced_id}")
                print(f"[INF] Decoder vocab size: {decoder_vocab_size}, Token ID valid: {forced_id < decoder_vocab_size}")

            old_forced_bos = None
            cfg = None
            try:
                cfg = getattr(gen_owner, 'config', None)
                if cfg is None:
                    cfg = getattr(getattr(gen_owner, 'base_model', None), 'config', None)
                if cfg is not None:
                    old_forced_bos = getattr(cfg, 'forced_bos_token_id', None)
                    cfg.forced_bos_token_id = int(forced_id)
            except Exception:
                old_forced_bos = None

            generated = None
            last_exc = None
            gen_attempts = [
                {"num_beams": 5, "max_length": min(max_len, 128)},
                {"num_beams": 2, "max_length": min(max_len, 64)},
                {"num_beams": 1, "max_length": min(max_len, 48)},
            ]

            try:
                for gen_cfg in gen_attempts:
                    try:
                        generated = generator.generate(
                            encoder_outputs=encoder_outputs_wrapped,
                            attention_mask=enc.get("attention_mask"),
                            max_length=gen_cfg["max_length"],
                            num_beams=gen_cfg["num_beams"],
                            early_stopping=True,
                            forced_bos_token_id=int(forced_id),
                        )
                        break
                    except RuntimeError as oom_err:
                        last_exc = oom_err
                        msg = str(oom_err).lower()
                        if "out of memory" in msg:
                            if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                                print(f"[INF] OOM during generation (beams={gen_cfg['num_beams']}, len={gen_cfg['max_length']}), retrying")
                            try:
                                torch.cuda.empty_cache()
                            except Exception:
                                pass
                            continue
                        else:
                            raise
                if generated is None and last_exc is not None:
                    raise last_exc
            finally:
                try:
                    if old_forced_bos is not None and cfg is not None:
                        cfg.forced_bos_token_id = old_forced_bos
                except Exception:
                    pass

            raw_translation = ""
            try:
                if generated is not None and generated.numel() > 0:
                    gen_ids = generated[0].detach().cpu().tolist()
                    raw_translation = tokenizer.decode(gen_ids, skip_special_tokens=True)
                else:
                    raw_translation = ""
            except Exception:
                try:
                    raw_translation = tokenizer.decode(generated[0], skip_special_tokens=True) if generated is not None and len(generated) > 0 else ""
                except Exception:
                    raw_translation = ""

            translation = raw_translation
            if 'postprocess_for_indictrans2' in globals():
                try:
                    translation = globals()['postprocess_for_indictrans2'](raw_translation, target_lang=target_lang)
                except Exception:
                    translation = raw_translation

            if DEBUG_DISCOVERY:
                print(f"[INF] Translation: {translation[:60] if translation else 'EMPTY'}")

            if not translation or not translation.strip():
                error_msg = "Empty generation result"
                if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                    print(f"[INF] ERROR: {error_msg}")
                return {
                    "input_sentence": input_sentence,
                    "translation": "",
                    "ambiguous_words_detected": 0,
                    "explanations": [],
                    "quality_metrics": {},
                    "dscd_validated": dscd_validated,
                    "error": error_msg
                }

            # Step 4: TRG explanations
            out_explanations: List[Dict[str, Any]] = []
            try:
                trg = getattr(core, 'trg', None)
                if trg and hasattr(trg, 'process_sentence_for_explanations'):
                    try:
                        tokens_list = tokenizer.convert_ids_to_tokens(enc['input_ids'][0].detach().cpu().tolist())
                        if DEBUG_DISCOVERY:
                            print(f"[INF] Calling TRG with {len(tokens_list)} tokens")
                        trg_explanations = trg.process_sentence_for_explanations(
                            tokens=tokens_list,
                            dscd_outputs=dscd_out_dict,
                            token_word_map=None,
                            uncertainty_threshold=u_th,
                            decoder_attention=None
                        )
                        if DEBUG_DISCOVERY:
                            print(f"[INF] TRG returned {len(trg_explanations) if isinstance(trg_explanations, list) else 0} explanations")
                        if isinstance(trg_explanations, list):
                            for exp in trg_explanations:
                                try:
                                    raw_word = exp.get('token', '')
                                    if is_punctuation_only(str(raw_word)):
                                        continue
                                    clean_word = clean_token(str(raw_word)) if raw_word else ''
                                    if not clean_word:
                                        continue
                                    if should_filter_explanation(exp, span_th, u_th):
                                        continue
                                    s = float(exp.get('span', 0.0))
                                    u = float(exp.get('uncertainty', 0.0))
                                    confidence = max(s, u)
                                    expl_text = exp.get('explanation', '')
                                    if not expl_text:
                                        continue
                                    out_explanations.append({
                                        "ambiguous_word": clean_word,
                                        "position": exp.get("token_idx", "N/A"),
                                        "explanation": expl_text,
                                        "uncertainty": u,
                                        "span": s,
                                        "confidence": confidence,
                                        "is_real_amb": bool((s > span_th) or (u > u_th)),
                                    })
                                except Exception:
                                    continue
                    except Exception:
                        pass
                else:
                    if DEBUG_DISCOVERY:
                        print("[INF] TRG not available or missing process_sentence_for_explanations()")
            except Exception:
                pass

            real_amb_count = sum(1 for e in out_explanations if e.get('is_real_amb', False))
            quality_metrics = {
                'total_raw_explanations': len(out_explanations),
                'filtered_explanations': 0,
                'high_confidence_count': sum(1 for e in out_explanations if e.get('confidence', 0) >= 0.65),
                'low_confidence_count': sum(1 for e in out_explanations if e.get('confidence', 0) < 0.4),
                'avg_confidence': sum(e.get('confidence', 0) for e in out_explanations) / max(len(out_explanations), 1),
                'avg_span': sum(e.get('span', 0) for e in out_explanations) / max(len(out_explanations), 1),
                'avg_uncertainty': sum(e.get('uncertainty', 0) for e in out_explanations) / max(len(out_explanations), 1),
            }

            if DEBUG_DISCOVERY:
                print(f"[INF] Final: {len(out_explanations)} explanations (real ambiguous: {real_amb_count})")

            result = {
                "input_sentence": input_sentence,
                "translation": translation,
                "ambiguous_words_detected": int(real_amb_count),
                "explanations": out_explanations,
                "quality_metrics": quality_metrics,
                "dscd_validated": dscd_validated,
            }

            if track_stats:
                INFERENCE_STATS.record_inference(result, dscd_homographs=dscd_homographs)

            return result

    except Exception as e:
        if DEBUG_DISCOVERY or VERBOSE_LOGGING:
            print(f"[INF] ERROR: {type(e).__name__}: {str(e)[:200]}")
            try:
                traceback.print_exc()
            except Exception:
                pass
        error_result = {
            "input_sentence": input_sentence,
            "translation": "ERROR DURING TRANSLATION",
            "ambiguous_words_detected": 0,
            "explanations": [],
            "quality_metrics": {},
            "dscd_validated": False,
            "error": f"{type(e).__name__}: {str(e)[:150]}",
        }
        if track_stats:
            INFERENCE_STATS.record_inference(error_result, dscd_homographs=dscd_homographs)
        return error_result
    finally:
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        try:
            if gc.isenabled():
                gc.collect()
        except Exception:
            pass


def demonstrate_system(model, tokenizer, sentences: Optional[List[str]] = None):
    if sentences is None:
        sentences = [
            "আমি কল বন্ধ করেছি।",
            "কাল আমি বই কিনব।",
            "পাতা ঝরে পড়েছে।",
            "তিনি ব্যাংক গেছেন।",
            "আজ ভাল আবহাওয়া।",
        ]
    print("=" * 80)
    print("TATN DEMO: Translation + Explanations")
    print("=" * 80)
    INFERENCE_STATS.reset()
    for s in sentences:
        print(f"\nInput: {s}")
        res = translate_with_explanations(model, tokenizer, s, source_lang=SOURCE_LANG, target_lang=TARGET_LANG)
        print("Translation:", res.get("translation", ""))
        print("Ambiguous words detected:", res.get("ambiguous_words_detected", 0))
        quality = res.get("quality_metrics", {})
        if quality:
            print(
                f"Quality: conf={quality.get('avg_confidence', 0):.3f}, "
                f"high={quality.get('high_confidence_count', 0)}, "
                f"low={quality.get('low_confidence_count', 0)}"
            )
        if res.get("explanations"):
            for idx, ex in enumerate(res["explanations"], 1):
                print(
                    f"  {idx}. '{ex['ambiguous_word']}' "
                    f"pos={ex['position']} conf={ex.get('confidence', 0):.3f}"
                )
                print("     ", ex.get("explanation", "")[:200])
        else:
            print("  No explanations")
    print("=" * 80)
    INFERENCE_STATS.print_summary()


def dscd_discovery_warmup(
    model,
    tokenizer,
    num_sents: int = 8000,
    batch_size: int = 64,
    max_len: Optional[int] = None,
):
    if max_len is None:
        max_len = MAXLEN
    core = model.module if (USE_MULTI_GPU and hasattr(model, "module")) else model
    try:
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            print("[WARMUP] Model has no dscd component")
            return
        print("\n" + "=" * 80)
        print("[WARMUP] Starting DSCD discovery warmup")
        print("=" * 80)
        orig_enable = getattr(dscd, "enable_training_clustering", False)
        orig_n_min = getattr(dscd, "n_min", None)
        orig_buffer = getattr(dscd, "buffer_size", None)
        try:
            if hasattr(dscd, "enable_training_clustering"):
                dscd.enable_training_clustering = True
            if hasattr(dscd, "n_min"):
                dscd.n_min = max(3, int(getattr(dscd, "n_min", 5)))
            if hasattr(dscd, "buffer_size"):
                dscd.buffer_size = max(200, int(getattr(dscd, "buffer_size", 300)))
        except Exception:
            pass
        texts: List[str] = []
        try:
            if "load_and_preprocess_optimized" in globals():
                pairs = globals()['load_and_preprocess_optimized'](num_sents)
                texts = [bn for (bn, _) in pairs][:num_sents]
            else:
                base = [
                    "আমি কল বন্ধ করেছি।",
                    "কাল আমি বই কিনব।",
                    "পাতা ঝরে পড়েছে।",
                    "তিনি ব্যাংক গেছেন।",
                ]
                while len(texts) < num_sents:
                    texts.extend(base)
                texts = texts[:num_sents]
        except Exception:
            texts = ["আমি কল বন্ধ করেছি।"] * num_sents
        processed = 0
        core.eval()
        print(f"\n[WARMUP] Processing {len(texts)} sentences (batch={batch_size})...")
        start_time = time.time()
        last_print = start_time
        with torch.inference_mode():
            for i in range(0, len(texts), batch_size):
                batch = texts[i: i + batch_size]
                try:
                    batch_to_tokenize = _preprocess_batch_texts(batch, SOURCE_LANG, TARGET_LANG)
                    enc = tokenizer(
                        batch_to_tokenize,
                        return_tensors="pt",
                        padding=True,
                        truncation=True,
                        max_length=max_len
                    )
                    enc = to_device_batch(enc, DEVICE)
                    core(
                        input_ids=enc.get("input_ids"),
                        attention_mask=enc.get("attention_mask"),
                        src_texts=batch,
                        token_word_map=None,
                        labels=None,
                        return_dict=True,
                        path=2
                    )
                    processed += len(batch)
                    current_time = time.time()
                    if (i // batch_size) % 10 == 0 or (current_time - last_print) > 5:
                        elapsed = current_time - start_time
                        rate = processed / elapsed if elapsed > 0 else 0
                        eta = (len(texts) - processed) / rate if rate > 0 else 0
                        print(f"[WARMUP] {processed}/{len(texts)} ({processed/len(texts)*100:.1f}%) | {rate:.1f} sent/s | ETA {eta:.0f}s")
                        last_print = current_time
                    del enc
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        print(f"[WARMUP] OOM at batch {i//batch_size}, cleaning up...")
                        try:
                            torch.cuda.empty_cache()
                        except Exception:
                            pass
                        gc.collect()
                        continue
                    else:
                        print(f"[WARMUP] Batch {i//batch_size} failed: {str(e)[:100]}")
                        continue
                except Exception as e:
                    print(f"[WARMUP] Batch {i//batch_size} failed: {str(e)[:100]}")
                    continue
        total_time = time.time() - start_time
        print(f"\n[WARMUP] Completed in {total_time:.1f}s ({processed/total_time:.1f} sent/s)")
        print("-" * 80)
        try:
            lock = None
            if hasattr(dscd, 'buffer_lock'):
                lock = dscd.buffer_lock
            elif hasattr(dscd, 'clustering_lock'):
                lock = dscd.clustering_lock
            if lock:
                with lock:
                    stores = dict(dscd.prototype_stores)
            else:
                stores = dict(dscd.prototype_stores)
            num_types = len(stores)
            total_protos = (sum(store.size() for store in stores.values()) if stores else 0)
            multi = (sum(1 for store in stores.values() if store.size() >= 2) if stores else 0)
            print("[WARMUP] Summary:")
            print(f"  - Token types: {num_types}")
            print(f"  - Total prototypes: {total_protos}")
            print(f"  - Multi-sense tokens: {multi}")
            if num_types > 0:
                print(f"  - Multi-sense ratio: {multi/num_types:.1%}")
            dscd_homographs = get_dscd_homographs(model)
            print(f"\n[WARMUP] Discovered Homographs: {len(dscd_homographs)}")
            if dscd_homographs:
                print(f"  Sample: {list(dscd_homographs)[:10]}")
            reference_found = dscd_homographs.intersection(HOMOGRAPH_REFERENCE_LIST)
            print(f"\n[WARMUP] Reference List Comparison:")
            print(f"  - Reference list: {len(HOMOGRAPH_REFERENCE_LIST)} words")
            print(f"  - Found in DSCD: {len(reference_found)}")
            print(f"  - Coverage: {len(reference_found)/len(HOMOGRAPH_REFERENCE_LIST):.1%}")
            if num_types == 0:
                print("\n[WARMUP] CRITICAL: NO PROTOTYPES CREATED")
            elif len(reference_found) < len(HOMOGRAPH_REFERENCE_LIST) // 2:
                print("\n[WARMUP] WARNING: < 50% reference coverage")
            else:
                print("\n[WARMUP] SUCCESS")
        except Exception:
            print("[WARMUP] Validation failed during summary")
    finally:
        try:
            if dscd is not None:
                if hasattr(dscd, "enable_training_clustering"):
                    dscd.enable_training_clustering = orig_enable
                if hasattr(dscd, "n_min") and orig_n_min is not None:
                    dscd.n_min = orig_n_min
                if hasattr(dscd, "buffer_size") and orig_buffer is not None:
                    dscd.buffer_size = orig_buffer
        except Exception:
            pass
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        try:
            if gc.isenabled():
                gc.collect()
        except Exception:
            pass
        print("=" * 80 + "\n")


def final_evaluation_with_bleu(
    model,
    tokenizer,
    test_data: List[Tuple[str, str]],
    device: Optional[torch.device] = None,
    max_length: Optional[int] = None,
    batch_size: int = 16,
) -> Dict[str, Any]:
    device = DEVICE if device is None else device
    max_len = MAXLEN if max_length is None else int(max_length)
    print("\n" + "=" * 80)
    print("FINAL EVALUATION WITH BLEU/CHRF++")
    print("=" * 80)
    print(f"Test samples: {len(test_data)}")
    print(f"Batch size: {batch_size}")
    print(f"Max length: {max_len}")
    print("=" * 80 + "\n")
    INFERENCE_STATS.reset()
    predictions = []
    references = []
    translations_with_explanations = []
    model.eval()
    try:
        from sacrebleu.metrics import BLEU, CHRF
        bleu_metric = BLEU()
        chrf_metric = CHRF()
        metrics_available = True
    except Exception:
        print("[EVAL] WARNING: sacrebleu not available, BLEU/CHRF scores will not be computed")
        metrics_available = False
    start_time = time.time()
    with torch.inference_mode():
        for i in range(0, len(test_data), batch_size):
            batch = test_data[i:i+batch_size]
            for src, ref in batch:
                try:
                    result = translate_with_explanations(
                        model, tokenizer, src,
                        source_lang=SOURCE_LANG,
                        target_lang=TARGET_LANG,
                        device=device,
                        max_length=max_len,
                        track_stats=True
                    )
                    translation = result.get('translation', '')
                    predictions.append(translation)
                    references.append(ref)
                    translations_with_explanations.append({
                        'source': src,
                        'reference': ref,
                        'translation': translation,
                        'explanations': result.get('explanations', []),
                        'ambiguous_words': result.get('ambiguous_words_detected', 0)
                    })
                except Exception as e:
                    if DEBUG_DISCOVERY or VERBOSE_LOGGING:
                        print(f"[EVAL] Translation failed for: {src[:50]} - {type(e).__name__}")
                    predictions.append("")
                    references.append(ref)
                    translations_with_explanations.append({
                        'source': src,
                        'reference': ref,
                        'translation': "ERROR",
                        'explanations': [],
                        'ambiguous_words': 0
                    })
            if (i // batch_size) % 10 == 0:
                elapsed = time.time() - start_time
                processed = min(i + batch_size, len(test_data))
                rate = processed / elapsed if elapsed > 0 else 0
                eta = (len(test_data) - processed) / rate if rate > 0 else 0
                print(f"[EVAL] {processed}/{len(test_data)} ({processed/len(test_data)*100:.1f}%) | {rate:.1f} sent/s | ETA {eta:.0f}s")
    total_time = time.time() - start_time
    print(f"\n[EVAL] Translation completed in {total_time:.1f}s ({len(test_data)/total_time:.1f} sent/s)")
    results = {
        'total_samples': len(test_data),
        'successful_translations': sum(1 for p in predictions if p and p != "ERROR"),
        'failed_translations': sum(1 for p in predictions if not p or p == "ERROR"),
        'total_time': total_time,
        'throughput': len(test_data) / total_time,
        'predictions': predictions,
        'references': references,
        'translations_with_explanations': translations_with_explanations,
    }
    if metrics_available and predictions and references:
        try:
            valid_preds = []
            valid_refs = []
            for p, r in zip(predictions, references):
                if p and p != "ERROR" and r:
                    valid_preds.append(p)
                    valid_refs.append(r)
            if valid_preds:
                bleu_score = bleu_metric.corpus_score(valid_preds, [valid_refs])
                chrf_score = chrf_metric.corpus_score(valid_preds, [valid_refs])
                results['bleu'] = float(bleu_score.score)
                results['chrf'] = float(chrf_score.score)
                print("\n" + "=" * 80)
                print("METRIC SCORES")
                print("=" * 80)
                print(f"BLEU:    {results['bleu']:.2f}")
                print(f"CHRF++:  {results['chrf']:.2f}")
                print(f"Valid samples: {len(valid_preds)}/{len(predictions)}")
                print("=" * 80)
            else:
                print("[EVAL] WARNING: No valid translations for BLEU/CHRF computation")
                results['bleu'] = 0.0
                results['chrf'] = 0.0
        except Exception as e:
            print(f"[EVAL] Metric computation failed: {type(e).__name__}: {str(e)[:100]}")
            results['bleu'] = 0.0
            results['chrf'] = 0.0
    else:
        results['bleu'] = 0.0
        results['chrf'] = 0.0
    print("\n" + "=" * 80)
    print("EVALUATION SUMMARY")
    print("=" * 80)
    print(f"Total samples: {results['total_samples']}")
    print(f"Successful: {results['successful_translations']}")
    print(f"Failed: {results['failed_translations']}")
    print(f"Success rate: {results['successful_translations']/results['total_samples']:.1%}")
    print(f"Throughput: {results['throughput']:.1f} sent/s")
    print("=" * 80 + "\n")
    INFERENCE_STATS.print_summary()
    return results


def load_checkpoint_for_resume(
    model: torch.nn.Module, optimizer, checkpoint_path: str
) -> Tuple[bool, int, int, float]:
    if not os.path.exists(checkpoint_path):
        print(f"[CHECKPOINT] Not found: {checkpoint_path}")
        return False, 0, 0, 0.0
    try:
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    except Exception as e:
        print(f"[CHECKPOINT] Load failed: {e}")
        return False, 0, 0, 0.0
    core = model.module if (USE_MULTI_GPU and hasattr(model, "module")) else model
    state = ckpt.get("model_state_dict", ckpt)
    try:
        core.load_state_dict(state, strict=False)
    except Exception as e:
        print(f"[CHECKPOINT] model.load_state_dict failed: {e}")
        try:
            if isinstance(state, dict):
                new_state = {}
                for k, v in state.items():
                    new_key = k.replace("module.", "") if k.startswith("module.") else k
                    new_state[new_key] = v
                core.load_state_dict(new_state, strict=False)
        except Exception:
            pass
    try:
        if optimizer is not None and "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    except Exception as e:
        print(f"[CHECKPOINT] optimizer.load_state_dict failed: {e}")
    try:
        if "dscd_state" in ckpt and ckpt["dscd_state"]:
            dscd_state = ckpt["dscd_state"]
            print("[CHECKPOINT] Restoring DSCD...")
            dscd = core.dscd if hasattr(core, 'dscd') else None
            if dscd and hasattr(dscd, 'load_state_dict'):
                lock = getattr(dscd, 'buffer_lock', None) or getattr(dscd, 'clustering_lock', None)
                if lock:
                    with lock:
                        dscd.load_state_dict(dscd_state)
                        num_tokens = len(dscd.prototype_stores)
                        total_protos = sum(store.size() for store in dscd.prototype_stores.values())
                        multi_sense = sum(1 for store in dscd.prototype_stores.values() if store.size() >= 2)
                else:
                    dscd.load_state_dict(dscd_state)
                    num_tokens = len(dscd.prototype_stores)
                    total_protos = sum(store.size() for store in dscd.prototype_stores.values())
                    multi_sense = sum(1 for store in dscd.prototype_stores.values() if store.size() >= 2)
                print("[CHECKPOINT] DSCD restored:")
                print(f"  - Tokens: {num_tokens}")
                print(f"  - Prototypes: {total_protos}")
                print(f"  - Multi-sense: {multi_sense}")
                if num_tokens == 0:
                    print("[CHECKPOINT] WARNING: DSCD state empty - consider running warmup")
            else:
                print("[CHECKPOINT] Model has no dscd.load_state_dict")
        else:
            print("[CHECKPOINT] No DSCD state in checkpoint")
    except Exception as e:
        print(f"[CHECKPOINT] DSCD restore failed: {e}")
    epoch = int(ckpt.get("epochs_trained", ckpt.get("epoch", 0)))
    step = int(ckpt.get("global_steps", ckpt.get("global_step", ckpt.get("step", 0))))
    avg_loss = float(ckpt.get("final_train_loss", ckpt.get("avg_epoch_loss", ckpt.get("avg_loss", 0.0))))
    print(f"[CHECKPOINT] Loaded: epoch={epoch} step={step} loss={avg_loss:.6f}")
    return True, epoch, step, avg_loss


print("\n" + "=" * 80)
print("Cell 8: Inference & Evaluation Pipeline (DUAL-PATH COMPATIBLE) (IndicTrans2) - FIXED")
print("=" * 80)
print("Configuration:")
print(f"  - Source language: {SOURCE_LANG}")
print(f"  - Target language: {TARGET_LANG}")
print(f"  - Span threshold: {SPAN_THRESHOLD}")
print(f"  - Uncertainty threshold: {UNCERTAINTY_THRESHOLD}")
print(f"  - Max length: {MAXLEN}")
print(f"  - Device: {DEVICE}")
print("\n✅ FIXES APPLIED:")
print("  ✅ generator = core.base_model (not core) — bypasses TATN wrapper for generate()")
print("  ✅ h_sense prefers sense_augmented_embeddings over encoder_last_hidden_state")
print("  ✅ Direct base_model encoder fallback when TATN forward returns no encoder output")
print("  ✅ decoder_vocab_size defaults to 32296 (IndicTrans2 200M actual decoder vocab)")
print("  ✅ get_target_lang_token_id reads core.tgt_token_id first (Cell 6 resolved value)")
print("  ✅ Removed span_th/u_th clamping to 0.10 — now respects Cell 0 values (0.20/0.15)")
print("  ✅ _preprocess_text/_preprocess_batch_texts helpers — clean IndicProcessor handling")
print("  ✅ gen_attempts starts at num_beams=5 (was 2) for better translation quality")
print("=" * 80 + "\n")



Cell 8: Inference & Evaluation Pipeline (DUAL-PATH COMPATIBLE) (IndicTrans2) - FIXED
Configuration:
  - Source language: ben_Beng
  - Target language: eng_Latn
  - Span threshold: 0.2
  - Uncertainty threshold: 0.15
  - Max length: 128
  - Device: cuda

✅ FIXES APPLIED:
  ✅ generator = core.base_model (not core) — bypasses TATN wrapper for generate()
  ✅ h_sense prefers sense_augmented_embeddings over encoder_last_hidden_state
  ✅ Direct base_model encoder fallback when TATN forward returns no encoder output
  ✅ decoder_vocab_size defaults to 32296 (IndicTrans2 200M actual decoder vocab)
  ✅ get_target_lang_token_id reads core.tgt_token_id first (Cell 6 resolved value)
  ✅ Removed span_th/u_th clamping to 0.10 — now respects Cell 0 values (0.20/0.15)
  ✅ _preprocess_text/_preprocess_batch_texts helpers — clean IndicProcessor handling
  ✅ gen_attempts starts at num_beams=5 (was 2) for better translation quality



In [12]:
# ===========================================================================================
# CELL 9: COMPREHENSIVE TESTING & EVALUATION (DUAL-PATH COMPATIBLE) (INDICTRANS2) - FIXED
# ===========================================================================================
from typing import Dict, List, Tuple, Optional, Any
import torch
import traceback
import time
import functools
from collections import defaultdict

try:
    _USE_MULTI_GPU = bool(USE_MULTI_GPU)
except (NameError, TypeError):
    _USE_MULTI_GPU = torch.cuda.is_available() and torch.cuda.device_count() > 1

try:
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
except (NameError, TypeError):
    _SOURCE_LANGUAGE = "ben_Beng"

try:
    _TARGET_LANGUAGE = str(TARGET_LANGUAGE)
except (NameError, TypeError):
    _TARGET_LANGUAGE = "eng_Latn"

try:
    _VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
except (NameError, TypeError):
    _VERBOSE_LOGGING = False

try:
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except (NameError, TypeError):
    _DEBUG_DISCOVERY = False

try:
    _DEBUG_TIMING = bool(DEBUG_TIMING)
except (NameError, TypeError):
    _DEBUG_TIMING = False

try:
    _SPAN_THRESHOLD = float(SPAN_THRESHOLD)
except (NameError, ValueError, TypeError):
    _SPAN_THRESHOLD = 0.20

try:
    _UNCERTAINTY_THRESHOLD = float(UNCERTAINTY_THRESHOLD)
except (NameError, ValueError, TypeError):
    _UNCERTAINTY_THRESHOLD = 0.15

try:
    _MAX_LENGTH = int(MAX_LENGTH)
except (NameError, ValueError, TypeError):
    _MAX_LENGTH = 64

try:
    _DEVICE = DEVICE
except (NameError, TypeError):
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    _HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in HOMOGRAPH_REFERENCE_LIST_BN)
except (NameError, TypeError):
    _HOMOGRAPH_REFERENCE_LIST = {
        "কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা", "বার", "হার", "তারা",
        "পানি", "দল", "বাজার", "নাম", "কথা", "বই", "ঘর", "মন", "হাত"
    }
    _HOMOGRAPH_REFERENCE_LIST = set(str(w).lower() for w in _HOMOGRAPH_REFERENCE_LIST)

_BENGALI_PUNCT_SET = set(['।', '॥'])
_COMMON_PUNCT_SET  = set(['.', ',', ';', ':', '!', '?', '"', "'", '-', '(', ')', '[', ']', '{', '}', '/', '\\'])
_PUNCT_SET = _BENGALI_PUNCT_SET | _COMMON_PUNCT_SET

_MIN_TRANSLATION_LEN_CHARS = 5
_MIN_TRANSLATION_LEN_WORDS = 1


def _is_punctuation_only(token: str) -> bool:
    if not token or not isinstance(token, str):
        return False
    clean = (
        token.replace("▁", "")
        .replace("Ġ", "")
        .replace("##", "")
        .replace("</w>", "")
        .strip()
    )
    if not clean:
        return False
    if clean in _BENGALI_PUNCT_SET:
        return True
    if clean in _COMMON_PUNCT_SET:
        return True
    if len(clean) == 1 and not clean.isalnum():
        return True
    return all(c in _PUNCT_SET for c in clean)


def _is_valid_translation(translation: str) -> bool:
    if not translation or not isinstance(translation, str):
        return False
    t = translation.strip()
    if not t:
        return False
    if t in (
        "Error occurred",
        "Translation generation failed",
        "ERROR DURING TRANSLATION",
        "..",
        "...",
        ".",
    ):
        return False
    if len(t) < _MIN_TRANSLATION_LEN_CHARS:
        return False
    words = [w for w in t.split() if w.strip()]
    if len(words) < _MIN_TRANSLATION_LEN_WORDS:
        return False
    return True


def _get_cluster_count(model: torch.nn.Module) -> int:
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return 0
        if hasattr(dscd, "get_prototype_summary"):
            try:
                summary = dscd.get_prototype_summary()
                return int(summary.get("total_prototypes", 0))
            except Exception:
                pass
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        if lock:
            with lock:
                stores = getattr(dscd, "prototype_stores", {}) or {}
                return sum(
                    len(getattr(store, "centroids", []))
                    for store in stores.values()
                )
        else:
            stores = getattr(dscd, "prototype_stores", {}) or {}
            return sum(
                len(getattr(store, "centroids", []))
                for store in stores.values()
            )
    except Exception:
        return 0


def _get_dscd_homographs(model: torch.nn.Module) -> set:
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return set()
        if hasattr(dscd, "get_discovered_homographs"):
            try:
                discovered = dscd.get_discovered_homographs()
                return set(w for w in discovered if not _is_punctuation_only(w))
            except Exception:
                pass
        homographs = set()
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        if lock:
            with lock:
                prototype_stores = getattr(dscd, "prototype_stores", {}) or {}
                items = list(prototype_stores.items())
        else:
            prototype_stores = getattr(dscd, "prototype_stores", {}) or {}
            items = list(prototype_stores.items())
        for token, store in items:
            try:
                n_protos = len(getattr(store, "centroids", []))
                if n_protos >= 2:
                    if _is_punctuation_only(str(token)):
                        continue
                    clean_token = (
                        str(token)
                        .replace("▁", "")
                        .replace("Ġ", "")
                        .replace("##", "")
                        .strip()
                        .lower()
                    )
                    if clean_token:
                        homographs.add(clean_token)
            except Exception:
                continue
        return homographs
    except Exception:
        return set()


def _print_top_clusters(model: torch.nn.Module, top_n: int = 5):
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        if lock:
            with lock:
                prototype_stores = dict(getattr(dscd, "prototype_stores", {}) or {})
        else:
            prototype_stores = dict(getattr(dscd, "prototype_stores", {}) or {})
        if not prototype_stores:
            print("[CLUSTER] No clusters found yet")
            return
        cluster_info = []
        for token, store in prototype_stores.items():
            try:
                total_count = sum(getattr(store, "counts", []))
            except Exception:
                total_count = 0
            try:
                n_protos = len(getattr(store, "centroids", []))
            except Exception:
                n_protos = 0
            cluster_info.append({
                "token":  token,
                "count":  total_count,
                "protos": n_protos,
                "mu":     getattr(store, "mu", 0.0),
                "tau":    getattr(store, "tau", 0.0),
            })
        cluster_info.sort(key=lambda x: x["count"], reverse=True)
        print(f"\n[CLUSTER] Top {min(top_n, len(cluster_info))} clusters:")
        print("-" * 90)
        print(f"{'Rank':<6}{'Token':<15}{'Count':<12}{'Protos':<10}{'Mu':<15}{'Tau':<12}")
        print("-" * 90)
        for rank, info in enumerate(cluster_info[:top_n], 1):
            token_str = str(info["token"])
            token_display = token_str[:12] if len(token_str) > 12 else token_str
            print(
                f"{rank:<6}{token_display:<15}{info['count']:<12}{info['protos']:<10}"
                f"{info['mu']:<15.6f}{info['tau']:<12.6f}"
            )
        print("-" * 90)
    except Exception as e:
        if _DEBUG_DISCOVERY:
            print(f"[CLUSTER] Error: {str(e)[:100]}")


def _timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        if _DEBUG_TIMING:
            start  = time.time()
            result = func(*args, **kwargs)
            elapsed = time.time() - start
            print(f"[TIMING] {func.__name__}: {elapsed:.2f}s")
            return result
        else:
            return func(*args, **kwargs)
    return wrapper


@_timed
def comprehensive_post_training_testing(
    model: torch.nn.Module,
    tokenizer,
    run_warmup: bool = True,
    compare_baseline: bool = False,
    baseline_metrics: Optional[Dict[str, Any]] = None
) -> Dict[str, Any]:
    print("\n" + "=" * 80)
    print("COMPREHENSIVE POST-TRAINING EVALUATION")
    print("=" * 80)

    if "translate_with_explanations" not in globals() or not callable(globals().get("translate_with_explanations")):
        print("[EVAL] ERROR: translate_with_explanations not found or not callable!")
        print("[EVAL] Cell 8 must be executed before running evaluation.")
        return {
            "error": "translate_with_explanations not found",
            "total_tests": 0,
            "successful_translations": 0,
        }

    test_sentences: List[Tuple[str, str, str, List[str]]] = [
        ("আমি কল বন্ধ করেছি।",
         "I turned off the tap",
         "কল = tap/call",
         ["কল"]),
        ("কাল আমি বই কিনব।",
         "Tomorrow I will buy a book",
         "কাল = tomorrow/yesterday",
         ["কাল"]),
        ("পাতা ঝরে পড়েছে।",
         "The leaf has fallen",
         "পাতা = leaf/page",
         ["পাতা"]),
        ("তিনি ব্যাংক গেছেন।",
         "He went to the bank",
         "ব্যাংক = bank/embankment",
         ["ব্যাংক"]),
        ("ফল খুব সুস্বাদু।",
         "The fruit is delicious",
         "ফল = fruit/result",
         ["ফল"]),
        ("মাথা ব্যথা করছে।",
         "Head is aching",
         "মাথা = head/top",
         ["মাথা"]),
        ("কল থেকে কল এসেছে।",
         "A call came from the tap",
         "Multiple কল",
         ["কল"]),
        ("কালকে কাল মেঘ দেখা গেছে।",
         "Yesterday black clouds were seen",
         "Multiple কাল",
         ["কাল"]),
        ("আজ ভাল আবহাওয়া।",
         "Weather is good today",
         "Simple",
         []),
        ("আমি ভালো আছি।",
         "I am fine",
         "Simple",
         []),
        ("সে খুব মিষ্টি কথা বলে।",
         "She speaks sweetly",
         "Simple",
         []),
        ("এটা আমার বই।",
         "This is my book",
         "Simple",
         []),
        ("তিনি ব্যাংকে কাজ করেন এবং ব্যাংকের কাছে বসে থাকেন।",
         "He works at the bank and sits by the embankment",
         "Long with multiple",
         ["ব্যাংক"]),
    ]

    # Always pass the original model — Cell 8 handles DataParallel unwrapping internally
    core_model = model.module if (_USE_MULTI_GPU and hasattr(model, "module")) else model
    core_model.eval()

    quality_metrics: Dict[str, Any] = {
        "total_confidence":       0.0,
        "confidence_samples":     0,
        "high_confidence_count":  0,
        "medium_confidence_count": 0,
        "low_confidence_count":   0,
        "confidences":            [],
        "spans":                  [],
        "uncertainties":          [],
    }

    homograph_tracking: Dict[str, Any] = {
        "test_expected_homographs":  set(),
        "dscd_discovered_homographs": set(),
        "explained_homographs":      set(),
        "homograph_explanations":    defaultdict(list),
    }

    error_tracking: Dict[str, Any] = {
        "translation_failures": 0,
        "dscd_failures":        0,
        "trg_failures":         0,
        "timeout_errors":       0,
        "oom_errors":           0,
        "other_errors":         0,
        "error_details":        [],
        "per_test_status":      [],
    }

    timing_metrics: Dict[str, Any] = {
        "total_time":       0.0,
        "per_test_times":   [],
        "avg_test_time":    0.0,
    }

    discovery_validated = False
    try:
        dscd = getattr(core_model, "dscd", None)
        if dscd is not None and hasattr(dscd, "discovered_log"):
            lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
            if lock:
                with lock:
                    discovered_log = getattr(dscd, "discovered_log", [])
                    discovery_validated = bool(discovered_log)
            else:
                discovered_log = getattr(dscd, "discovered_log", [])
                discovery_validated = bool(discovered_log)
    except Exception:
        discovery_validated = False

    asbn_stats: Dict[str, Any] = {}
    try:
        asbn = getattr(core_model, "asbn", None)
        if asbn is not None:
            if hasattr(asbn, "get_detailed_stats"):
                asbn_stats = asbn.get_detailed_stats()
            elif hasattr(asbn, "get_asbn_stats"):
                asbn_stats = asbn.get_asbn_stats()
            if asbn_stats and _DEBUG_DISCOVERY:
                print(f"[EVAL] ASBN: domain_acc={asbn_stats.get('domain_accuracy', 0):.2%}")
    except Exception:
        asbn_stats = {}

    trg_stats: Dict[str, Any] = {}
    try:
        trg = getattr(core_model, "trg", None)
        if trg is not None and hasattr(trg, "get_statistics"):
            trg_stats = trg.get_statistics()
            if _DEBUG_DISCOVERY:
                print(f"[EVAL] TRG: {trg_stats.get('explanations_generated', 0)} total")
    except Exception:
        trg_stats = {}

    homograph_tracking["dscd_discovered_homographs"] = _get_dscd_homographs(core_model)
    print(f"[EVAL] DSCD discovered: {len(homograph_tracking['dscd_discovered_homographs'])}")
    if homograph_tracking["dscd_discovered_homographs"] and _DEBUG_DISCOVERY:
        print(f"[EVAL] Sample: {list(homograph_tracking['dscd_discovered_homographs'])[:10]}")

    if run_warmup:
        try:
            dscd = getattr(core_model, "dscd", None)
            if dscd is not None:
                lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
                if lock:
                    with lock:
                        stores      = getattr(dscd, "prototype_stores", None)
                        store_count = len(stores) if stores else 0
                else:
                    stores      = getattr(dscd, "prototype_stores", None)
                    store_count = len(stores) if stores else 0
                if store_count == 0 and "dscd_discovery_warmup" in globals() and callable(globals().get("dscd_discovery_warmup")):
                    try:
                        print("[EVAL] DSCD stores empty, running warmup (num_sents=4000)...")
                        globals()["dscd_discovery_warmup"](model, tokenizer, num_sents=4000, batch_size=64)
                        homograph_tracking["dscd_discovered_homographs"] = _get_dscd_homographs(core_model)
                    except Exception as e:
                        print(f"[EVAL] Warmup failed: {e}")
        except Exception:
            pass

    total_tests            = len(test_sentences)
    successful_translations = 0
    total_explanations     = 0
    total_high_span        = 0
    total_real_ambiguous   = 0

    print(f"\n[EVAL] Running {total_tests} tests...")
    print("-" * 80)

    # Initialize eval_start before try so finally block always has it
    eval_start = time.time()

    try:
        try:
            if hasattr(tokenizer, "src_lang") and hasattr(tokenizer, "tgt_lang"):
                tokenizer.src_lang = _SOURCE_LANGUAGE
                tokenizer.tgt_lang = _TARGET_LANGUAGE
            elif hasattr(tokenizer, "set_src_lang_special_tokens") and hasattr(tokenizer, "set_tgt_lang_special_tokens"):
                tokenizer.set_src_lang_special_tokens(_SOURCE_LANGUAGE)
                tokenizer.set_tgt_lang_special_tokens(_TARGET_LANGUAGE)
        except Exception:
            pass

        def _is_real_amb(expl: Dict[str, Any]) -> bool:
            try:
                s = float(expl.get("span", 0.0))
                u = float(expl.get("uncertainty", 0.0))
                return (s > _SPAN_THRESHOLD) or (u > _UNCERTAINTY_THRESHOLD)
            except Exception:
                return False

        def _compute_similarity(pred: str, expected: str) -> float:
            try:
                pred_words = set(pred.lower().split())
                exp_words  = set(expected.lower().split())
                if not exp_words:
                    return 0.0
                overlap = len(pred_words & exp_words)
                return overlap / len(exp_words)
            except Exception:
                return 0.0

        for _, _, _, expected_homos in test_sentences:
            homograph_tracking["test_expected_homographs"].update(
                [h.lower() for h in expected_homos]
            )

        for idx, (src_text, expected_translation, desc, expected_homos) in enumerate(test_sentences, 1):
            test_start = time.time()

            print(f"\nTest {idx}/{total_tests}: {desc}")
            print("=" * 60)

            test_status: Dict[str, Any] = {
                "test_id":            idx,
                "success":            False,
                "translation_ok":     False,
                "explanations_count": 0,
                "error":              None,
            }

            try:
                # Pass original model — Cell 8 handles DataParallel unwrapping
                result = globals()["translate_with_explanations"](
                    model,
                    tokenizer,
                    src_text,
                    source_lang=_SOURCE_LANGUAGE,
                    target_lang=_TARGET_LANGUAGE,
                    device=_DEVICE,
                    max_length=_MAX_LENGTH,
                    span_threshold=_SPAN_THRESHOLD,
                    uncertainty_threshold=_UNCERTAINTY_THRESHOLD,
                    track_stats=False,
                )

                if result is None or not isinstance(result, dict):
                    print(f"[EVAL] Invalid result type: {type(result)}")
                    error_tracking["translation_failures"] += 1
                    test_status["error"] = "invalid_result"
                    error_tracking["per_test_status"].append(test_status)
                    continue

                if result.get("error"):
                    print(f"[EVAL] Translation error: {result['error']}")
                    error_tracking["translation_failures"] += 1
                    test_status["error"] = "translation_error"
                    error_tracking["per_test_status"].append(test_status)
                    continue

                translation  = str(result.get("translation", "") or "")
                amb_count    = int(result.get("ambiguous_words_detected", 0))
                explanations = list(result.get("explanations", []) or [])
                similarity   = _compute_similarity(translation, expected_translation)

                print(f"Input:      {src_text}")
                print(f"Expected:   {expected_translation}")
                print(f"Translation:{translation}")
                print(f"Similarity: {similarity:.1%}")
                print(f"Ambiguous:  {amb_count}")

                if explanations:
                    print("\nExplanations:")
                    high_span_local  = 0
                    real_amb_local   = 0

                    for j, expl in enumerate(explanations, 1):
                        try:
                            span_val = float(expl.get("span", 0.0))
                            u_val    = float(expl.get("uncertainty", 0.0))
                            conf_val = float(expl.get("confidence", max(span_val, u_val)))
                            marker   = f"[S>{_SPAN_THRESHOLD:.2f}]" if span_val > _SPAN_THRESHOLD else "          "
                            word     = expl.get("ambiguous_word", expl.get("token", "N/A"))
                            pos      = expl.get("position", expl.get("token_idx", "N/A"))

                            print(f"  {j}. {marker} '{word}' @ {pos}")
                            print(f"       conf={conf_val:.3f} | U={u_val:.3f} | S={span_val:.3f}")
                            text = str(expl.get("explanation", ""))
                            if len(text) > 120:
                                text = text[:120] + "..."
                            print(f"       {text}")

                            quality_metrics["confidences"].append(conf_val)
                            quality_metrics["spans"].append(span_val)
                            quality_metrics["uncertainties"].append(u_val)
                            quality_metrics["total_confidence"] += conf_val
                            quality_metrics["confidence_samples"] += 1

                            if conf_val >= 0.65:
                                quality_metrics["high_confidence_count"] += 1
                            elif conf_val >= 0.4:
                                quality_metrics["medium_confidence_count"] += 1
                            else:
                                quality_metrics["low_confidence_count"] += 1

                            if span_val > _SPAN_THRESHOLD:
                                high_span_local += 1
                            if _is_real_amb(expl):
                                real_amb_local += 1

                            clean_word = (
                                str(word)
                                .replace("▁", "")
                                .replace("Ġ", "")
                                .strip()
                                .lower()
                            )
                            if not _is_punctuation_only(clean_word) and clean_word:
                                homograph_tracking["explained_homographs"].add(clean_word)
                                homograph_tracking["homograph_explanations"][clean_word].append({
                                    "sentence":    src_text,
                                    "confidence":  conf_val,
                                    "span":        span_val,
                                    "uncertainty": u_val,
                                })
                        except Exception:
                            continue

                    total_explanations   += len(explanations)
                    total_high_span      += high_span_local
                    total_real_ambiguous += real_amb_local
                    test_status["explanations_count"] = len(explanations)
                else:
                    print("No explanations generated")

                if _is_valid_translation(translation):
                    successful_translations += 1
                    test_status["translation_ok"] = True
                    test_status["success"]        = True
                    print(f"✅ Success")
                else:
                    print(f"❌ Translation failed or too short: '{translation}'")
                    error_tracking["translation_failures"] += 1
                    test_status["error"] = "translation_failed"

                try:
                    del result
                except Exception:
                    pass
                try:
                    if explanations:
                        del explanations
                except Exception:
                    pass

            except RuntimeError as e:
                error_str = str(e).lower()
                if "out of memory" in error_str:
                    print(f"[EVAL] OOM: {str(e)[:100]}")
                    error_tracking["oom_errors"] += 1
                    test_status["error"] = "oom"
                elif "timeout" in error_str:
                    print(f"[EVAL] Timeout: {str(e)[:100]}")
                    error_tracking["timeout_errors"] += 1
                    test_status["error"] = "timeout"
                else:
                    print(f"[EVAL] Runtime: {type(e).__name__}")
                    error_tracking["other_errors"] += 1
                    test_status["error"] = "runtime"
                error_tracking["error_details"].append(f"Test {idx}: {type(e).__name__}")
            except Exception as e:
                print(f"[EVAL] Error: {type(e).__name__}")
                error_tracking["other_errors"] += 1
                test_status["error"] = type(e).__name__
                error_tracking["error_details"].append(f"Test {idx}: {type(e).__name__}")
                if _DEBUG_DISCOVERY:
                    try:
                        traceback.print_exc()
                    except Exception:
                        pass

            error_tracking["per_test_status"].append(test_status)
            test_time = time.time() - test_start
            timing_metrics["per_test_times"].append(test_time)
            print("-" * 60)

            if idx % 3 == 0 and torch.cuda.is_available():
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass

    finally:
        timing_metrics["total_time"] = time.time() - eval_start
        if timing_metrics["per_test_times"]:
            timing_metrics["avg_test_time"] = (
                sum(timing_metrics["per_test_times"]) / len(timing_metrics["per_test_times"])
            )

        if quality_metrics["confidence_samples"] > 0:
            quality_metrics["avg_confidence"] = (
                quality_metrics["total_confidence"] / quality_metrics["confidence_samples"]
            )
            quality_metrics["avg_span"] = (
                sum(quality_metrics["spans"]) / len(quality_metrics["spans"])
                if quality_metrics["spans"] else 0.0
            )
            quality_metrics["avg_uncertainty"] = (
                sum(quality_metrics["uncertainties"]) / len(quality_metrics["uncertainties"])
                if quality_metrics["uncertainties"] else 0.0
            )
            if quality_metrics["confidences"]:
                sorted_conf = sorted(quality_metrics["confidences"])
                n = len(sorted_conf)
                quality_metrics["confidence_p25"] = sorted_conf[n // 4]
                quality_metrics["confidence_p50"] = sorted_conf[n // 2]
                quality_metrics["confidence_p75"] = sorted_conf[3 * n // 4]
        else:
            quality_metrics["avg_confidence"] = 0.0
            quality_metrics["avg_span"]        = 0.0
            quality_metrics["avg_uncertainty"] = 0.0

        explained_from_dscd = homograph_tracking["explained_homographs"].intersection(
            homograph_tracking["dscd_discovered_homographs"]
        )
        test_expected_discovered = homograph_tracking["test_expected_homographs"].intersection(
            homograph_tracking["dscd_discovered_homographs"]
        )
        reference_discovered = _HOMOGRAPH_REFERENCE_LIST.intersection(
            homograph_tracking["dscd_discovered_homographs"]
        )

        homograph_tracking["explained_from_dscd_rate"] = (
            len(explained_from_dscd) / len(homograph_tracking["dscd_discovered_homographs"])
            if homograph_tracking["dscd_discovered_homographs"] else 0.0
        )
        homograph_tracking["test_expected_discovery_rate"] = (
            len(test_expected_discovered) / len(homograph_tracking["test_expected_homographs"])
            if homograph_tracking["test_expected_homographs"] else 0.0
        )
        homograph_tracking["reference_discovery_rate"] = (
            len(reference_discovered) / len(_HOMOGRAPH_REFERENCE_LIST)
            if _HOMOGRAPH_REFERENCE_LIST else 0.0
        )

        dscd_stats: Dict[str, Any] = {
            "total_words":       0,
            "multi_sense_words": 0,
            "total_prototypes":  0,
            "total_buffered":    0,
        }
        try:
            dscd_obj = getattr(core_model, "dscd", None)
            if dscd_obj is not None and hasattr(dscd_obj, "prototype_stores"):
                lock = getattr(dscd_obj, "buffer_lock", None) or getattr(dscd_obj, "clustering_lock", None)
                if lock:
                    with lock:
                        stores = dict(getattr(dscd_obj, "prototype_stores") or {})
                else:
                    stores = dict(getattr(dscd_obj, "prototype_stores") or {})
                total_words   = 0
                multi         = 0
                total_protos  = 0
                total_buffered = 0
                for key, store in stores.items():
                    total_words += 1
                    try:
                        n_protos = len(getattr(store, "centroids", []))
                    except Exception:
                        n_protos = 0
                    try:
                        buf_count = int(store.size()) if hasattr(store, "size") else 0
                    except Exception:
                        buf_count = 0
                    total_protos   += n_protos
                    total_buffered += buf_count
                    if n_protos >= 2:
                        multi += 1
                dscd_stats = {
                    "total_words":       total_words,
                    "multi_sense_words": multi,
                    "total_prototypes":  total_protos,
                    "total_buffered":    total_buffered,
                }
        except Exception:
            pass

    print("\n" + "=" * 80)
    print("COMPREHENSIVE EVALUATION SUMMARY")
    print("=" * 80)

    print(f"\n[TRANSLATION QUALITY]")
    print(f"  Total tests:   {total_tests}")
    print(f"  Successful:    {successful_translations}")
    print(f"  Success rate:  {successful_translations / total_tests * 100:.1f}%")

    print(f"\n[AMBIGUITY DETECTION]")
    print(f"  Total explanations: {total_explanations}")
    print(f"  High-span (S>{_SPAN_THRESHOLD}): {total_high_span}")
    print(f"  Real ambiguous: {total_real_ambiguous}")
    if total_tests > 0:
        print(f"  Avg explanations/test: {total_explanations / total_tests:.2f}")

    print(f"\n[EXPLANATION QUALITY]")
    print(f"  Avg confidence:  {quality_metrics['avg_confidence']:.3f}")
    print(f"  Avg span:        {quality_metrics['avg_span']:.3f}")
    print(f"  Avg uncertainty: {quality_metrics['avg_uncertainty']:.3f}")
    if "confidence_p50" in quality_metrics:
        print(
            f"  Confidence P25/P50/P75: "
            f"{quality_metrics.get('confidence_p25', 0):.3f} / "
            f"{quality_metrics.get('confidence_p50', 0):.3f} / "
            f"{quality_metrics.get('confidence_p75', 0):.3f}"
        )

    print(f"\n[HOMOGRAPH DISCOVERY]")
    print(f"  DSCD discovered:      {len(homograph_tracking['dscd_discovered_homographs'])}")
    print(f"  Explained:            {len(homograph_tracking['explained_homographs'])}")
    print(f"  Explanation rate:     {homograph_tracking['explained_from_dscd_rate']:.1%}")
    print(f"  Test discovery rate:  {homograph_tracking['test_expected_discovery_rate']:.1%}")
    if homograph_tracking["explained_homographs"]:
        print(f"\n  Explained homographs (top 10):")
        for homo in sorted(homograph_tracking["explained_homographs"])[:10]:
            exps     = homograph_tracking["homograph_explanations"].get(homo, [])
            count    = len(exps)
            avg_conf = sum(e["confidence"] for e in exps) / len(exps) if exps else 0.0
            in_dscd  = "[D]" if homo in homograph_tracking["dscd_discovered_homographs"] else "   "
            in_ref   = "[R]" if homo in _HOMOGRAPH_REFERENCE_LIST else "   "
            print(f"    {in_dscd} {in_ref} '{homo}': {count}x conf={avg_conf:.3f}")

    print(f"\n[REFERENCE COMPARISON]")
    print(f"  Reference: {len(_HOMOGRAPH_REFERENCE_LIST)} words")
    print(f"  Discovered: {len(reference_discovered)}/{len(_HOMOGRAPH_REFERENCE_LIST)}")
    print(f"  Coverage: {homograph_tracking['reference_discovery_rate']:.1%}")

    print(f"\n[DSCD PROTOTYPES]")
    print(f"  Word types:      {dscd_stats['total_words']}")
    print(f"  Multi-sense:     {dscd_stats['multi_sense_words']}")
    print(f"  Total prototypes:{dscd_stats['total_prototypes']}")
    print(f"  Total buffered:  {dscd_stats['total_buffered']}")
    if dscd_stats["total_words"] > 0:
        print(
            f"  Multi-sense ratio: "
            f"{dscd_stats['multi_sense_words'] / dscd_stats['total_words']:.1%}"
        )

    if asbn_stats:
        print(f"\n[ASBN]")
        print(f"  Domain accuracy: {asbn_stats.get('domain_accuracy', 0):.2%}")
        if "source_accuracy" in asbn_stats:
            print(f"  Source accuracy: {asbn_stats['source_accuracy']:.2%}")
            print(f"  Target accuracy: {asbn_stats['target_accuracy']:.2%}")

    if trg_stats:
        print(f"\n[TRG]")
        print(f"  Total explanations: {trg_stats.get('explanations_generated', 0)}")
        print(f"  High confidence:    {trg_stats.get('high_confidence_rate', 0):.1%}")

    print(f"\n[PERFORMANCE]")
    print(f"  Total time:    {timing_metrics['total_time']:.2f}s")
    print(f"  Avg time/test: {timing_metrics['avg_test_time']:.2f}s")
    print("=" * 80)

    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

    return {
        "total_tests":              total_tests,
        "successful_translations":  successful_translations,
        "success_rate_pct":         (successful_translations / total_tests * 100.0) if total_tests > 0 else 0.0,
        "total_explanations":       total_explanations,
        "total_high_span":          total_high_span,
        "total_real_ambiguous":     total_real_ambiguous,
        "dscd_stats":               dscd_stats,
        "quality_metrics":          quality_metrics,
        "homograph_tracking":       homograph_tracking,
        "error_tracking":           error_tracking,
        "asbn_stats":               asbn_stats,
        "trg_stats":                trg_stats,
        "discovery_validated":      discovery_validated,
        "timing_metrics":           timing_metrics,
    }


print("\n" + "=" * 80)
print("Cell 9: Testing & Evaluation (DUAL-PATH COMPATIBLE) (IndicTrans2) - FIXED")
print("=" * 80)
print("Evaluation metrics:")
print("  - Translation quality (valid translation check with min-length guard)")
print("  - Ambiguity detection (high-span, real ambiguous)")
print("  - Explanation quality (confidence distribution)")
print("  - Homograph discovery (DSCD centroids vs reference)")
print("  - DSCD prototype statistics (centroids + buffered counts separately)")
print("  - ASBN domain accuracy")
print("  - TRG explanation statistics")
print("  - Performance timing (total, avg per test)")
print("  - Error tracking (OOM, timeout, other)")
print()
print(f"Configuration:")
print(f"  Source language:     {_SOURCE_LANGUAGE}")
print(f"  Target language:     {_TARGET_LANGUAGE}")
print(f"  Span threshold:      {_SPAN_THRESHOLD}")
print(f"  Uncertainty threshold: {_UNCERTAINTY_THRESHOLD}")
print(f"  Max length:          {_MAX_LENGTH}")
print(f"  Device:              {_DEVICE}")
print(f"  Reference list:      {len(_HOMOGRAPH_REFERENCE_LIST)} words")
print("=" * 80 + "\n")



Cell 9: Testing & Evaluation (DUAL-PATH COMPATIBLE) (IndicTrans2) - FIXED
Evaluation metrics:
  - Translation quality (valid translation check with min-length guard)
  - Ambiguity detection (high-span, real ambiguous)
  - Explanation quality (confidence distribution)
  - Homograph discovery (DSCD centroids vs reference)
  - DSCD prototype statistics (centroids + buffered counts separately)
  - ASBN domain accuracy
  - TRG explanation statistics
  - Performance timing (total, avg per test)
  - Error tracking (OOM, timeout, other)

Configuration:
  Source language:     ben_Beng
  Target language:     eng_Latn
  Span threshold:      0.2
  Uncertainty threshold: 0.15
  Max length:          128
  Device:              cuda
  Reference list:      42 words



In [13]:
# ==========================================================================================
# CELL 10: TATN MAIN PIPELINE (DUAL-PATH COMPATIBLE) (INDICTRANS2)
# ==========================================================================================

import os
import sys
import time
import traceback
import inspect
from typing import Tuple, Optional, Dict, Any
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers.modeling_outputs import BaseModelOutput
import collections

try:
    if hasattr(torch.serialization, 'add_safe_globals'):
        torch.serialization.add_safe_globals([
            collections.defaultdict,
            collections.OrderedDict,
            collections.deque
        ])
        print("✓ Registered safe globals for PyTorch 2.6+")
except (AttributeError, Exception):
    pass


def _g(name, default):
    return globals().get(name, default)


try:
    _USE_MULTI_GPU = bool(_g("USE_MULTI_GPU", False))
    _NUM_GPUS = int(_g("NUM_GPUS", torch.cuda.device_count() if torch.cuda.is_available() else 0))
    _DEVICE = _g("DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    _SOURCE_LANGUAGE = str(_g("SOURCE_LANGUAGE", "ben_Beng"))
    _TARGET_LANGUAGE = str(_g("TARGET_LANGUAGE", "eng_Latn"))
    _NUM_SAMPLES = int(_g("NUM_SAMPLES", 30000))
    _MAX_LENGTH = int(_g("MAX_LENGTH", 52))
    _BATCH_SIZE = int(_g("BATCH_SIZE", 8))
    _EPOCHS = int(_g("EPOCHS", 1))
    _ACCUMULATION_STEPS = int(_g("ACCUMULATION_STEPS", 1))
    _LR_NMT = float(_g("LR_NMT", 2e-5))
    _LR_PHI = float(_g("LR_PHI", 5e-6))
    _ENABLE_ASBN_TRAINING = bool(_g("ENABLE_ASBN_TRAINING", False))
    _VALIDATION_CHECK_INTERVAL = int(_g("VALIDATION_CHECK_INTERVAL", 500))
    _PERIODIC_DISCOVERY_FREQUENCY = int(_g("PERIODIC_DISCOVERY_FREQUENCY", 50))
    _DSCD_WARMUP_SAMPLES = int(_g("DSCD_WARMUP_SAMPLES", 4000))
    _SPAN_THRESHOLD = float(_g("SPAN_THRESHOLD", 0.20))
    _UNCERTAINTY_THRESHOLD = float(_g("UNCERTAINTY_THRESHOLD", 0.15))
    _HOMOGRAPH_REFERENCE_LIST_BN = set(_g("HOMOGRAPH_REFERENCE_LIST_BN",
        ["কল", "কাল", "���াতা", "ব্যাংক", "ফল", "মাথা", "বার", "হার", "তারা"]))
    HOMOGRAPH_REFERENCE_LIST_BN = _HOMOGRAPH_REFERENCE_LIST_BN
    _FREEZE_ENCODER = bool(_g("FREEZE_ENCODER", False))
    _DEBUG_TIMING = bool(_g("DEBUG_TIMING", False))
    _DEBUG_DISCOVERY = bool(_g("DEBUG_DISCOVERY", False))
except (ValueError, TypeError):
    _NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
    _USE_MULTI_GPU = _NUM_GPUS > 1
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _SOURCE_LANGUAGE = "ben_Beng"
    _TARGET_LANGUAGE = "eng_Latn"
    _NUM_SAMPLES = 30000
    _MAX_LENGTH = 52
    _BATCH_SIZE = 8
    _EPOCHS = 1
    _ACCUMULATION_STEPS = 1
    _LR_NMT = 2e-5
    _LR_PHI = 5e-6
    _ENABLE_ASBN_TRAINING = False
    _VALIDATION_CHECK_INTERVAL = 500
    _PERIODIC_DISCOVERY_FREQUENCY = 50
    _DSCD_WARMUP_SAMPLES = 4000
    _SPAN_THRESHOLD = 0.20
    _UNCERTAINTY_THRESHOLD = 0.15
    _HOMOGRAPH_REFERENCE_LIST_BN = {"কল", "কাল", "পাতা", "ব্যাংক"}
    HOMOGRAPH_REFERENCE_LIST_BN = _HOMOGRAPH_REFERENCE_LIST_BN
    _FREEZE_ENCODER = False
    _DEBUG_TIMING = False
    _DEBUG_DISCOVERY = False

_CHECKPOINT_DIR = "/kaggle/working"
_CHECKPOINT_PATH = os.path.join(_CHECKPOINT_DIR, "tatn_final.pt")


def _safe_clear_gpu_caches():
    try:
        if "clear_all_gpu_caches" in globals() and callable(globals()["clear_all_gpu_caches"]):
            globals()["clear_all_gpu_caches"]()
            return
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                try:
                    with torch.cuda.device(i):
                        torch.cuda.empty_cache()
                except Exception:
                    pass
        if gc.isenabled():
            gc.collect()
    except Exception:
        pass


def _safe_get(d: dict, *keys, default=None):
    if not isinstance(d, dict):
        return default
    result = d
    for key in keys:
        if not isinstance(result, dict):
            return default
        result = result.get(key, None)
        if result is None:
            return default
    return result


def _get_dscd_stores_safe(dscd):
    try:
        prototype_stores = getattr(dscd, 'prototype_stores', None)
        if prototype_stores is None:
            return {}

        lock = None
        if hasattr(dscd, 'buffer_lock'):
            lock = dscd.buffer_lock
        elif hasattr(dscd, 'clustering_lock'):
            lock = dscd.clustering_lock

        try:
            if lock:
                try:
                    with lock:
                        return dict(prototype_stores)
                except Exception:
                    return dict(prototype_stores)
            else:
                return dict(prototype_stores)
        except Exception:
            return {}
    except Exception:
        return {}


def _get_core_model(model):
    return model.module if hasattr(model, "module") else model


def initialize_environment():
    print("[PIPELINE] Initializing environment...")
    if torch.cuda.is_available():
        gcnt = torch.cuda.device_count()
        print(f"[PIPELINE] GPUs: {gcnt}")
        for i in range(gcnt):
            try:
                name = torch.cuda.get_device_name(i)
                mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
                print(f"  GPU {i}: {name} ({mem:.1f} GB)")
            except Exception:
                print(f"  GPU {i}: Unknown")
        _safe_clear_gpu_caches()
    else:
        print("[PIPELINE] CPU only")
    return True


def validate_component_compatibility(model_core, tokenizer):
    print("\n[VALIDATION] Checking component compatibility...")

    issues = []

    try:
        model_vocab = getattr(model_core, "vocab_size", None)
        if model_vocab is None:
            try:
                model_vocab = int(getattr(model_core.base_model.config, "vocab_size", 0))
            except Exception:
                model_vocab = 0

        try:
            tokenizer_vocab = len(tokenizer) if hasattr(tokenizer, "__len__") else getattr(tokenizer, "vocab_size", 0)
        except Exception:
            tokenizer_vocab = getattr(tokenizer, "vocab_size", 0) or 0

        if model_vocab < tokenizer_vocab:
            issues.append(f"CRITICAL: model vocab ({model_vocab}) < tokenizer vocab ({tokenizer_vocab})")
        elif model_vocab > tokenizer_vocab:
            print(f"  ✅ Vocabulary: model={model_vocab}, tokenizer={tokenizer_vocab}")
            print(f"     Note: Model has {model_vocab - tokenizer_vocab} extra tokens (preserves pretrained weights)")
        else:
            print(f"  ✅ Vocabulary: {model_vocab}")
    except Exception as e:
        issues.append(f"Vocab check failed: {e}")

    try:
        try:
            emb_layer = model_core.base_model.get_input_embeddings()
            model_embed_dim = (
                getattr(emb_layer, "embedding_dim", None)
                or getattr(emb_layer, "out_features", None)
                or int(getattr(model_core.base_model.config, "d_model",
                       getattr(model_core.base_model.config, "hidden_size", 1024)))
            )
        except Exception:
            model_embed_dim = int(getattr(model_core.base_model.config, "d_model",
                                  getattr(model_core.base_model.config, "hidden_size", 1024)))
    
        print(f"  ✅ Model embed_dim: {model_embed_dim}")
    
        if hasattr(model_core, 'dscd'):
            dscd_embed_dim = getattr(model_core.dscd, 'embed_dim', None)
            if dscd_embed_dim is not None and dscd_embed_dim != model_embed_dim:
                issues.append(f"DSCD embed_dim mismatch: {dscd_embed_dim} != {model_embed_dim}")
            else:
                print(f"  ✅ DSCD embed_dim: {dscd_embed_dim}")
    
        if hasattr(model_core, 'asbn'):
            asbn_embed_dim = getattr(model_core.asbn, 'embed_dim', None)
            if asbn_embed_dim is not None and asbn_embed_dim != model_embed_dim:
                issues.append(f"ASBN embed_dim mismatch: {asbn_embed_dim} != {model_embed_dim}")
            else:
                print(f"  ✅ ASBN embed_dim: {asbn_embed_dim}")
    
    except Exception as e:
        issues.append(f"Embed_dim check failed: {e}")


    try:
        embedding_layer = model_core.base_model.get_input_embeddings()
        if embedding_layer is None:
            issues.append("Model has no input embeddings")
        else:
            try:
                actual_embed_dim = getattr(embedding_layer, "embedding_dim", None)
                actual_vocab_size = getattr(embedding_layer, "num_embeddings", None)
                if actual_embed_dim is None or actual_vocab_size is None:
                    actual_embed_dim = actual_embed_dim or int(getattr(model_core.base_model.config, "d_model", 1024))
                    actual_vocab_size = actual_vocab_size or int(getattr(model_core.base_model.config, "encoder_vocab_size", getattr(model_core.base_model.config, "vocab_size", 0)))
                print(f"  ✅ Embedding layer: dim={actual_embed_dim}, vocab={actual_vocab_size}")
            except Exception:
                issues.append("Embedding layer check failed (inspection)")
    except Exception as e:
        issues.append(f"Embedding layer check failed: {e}")

    if issues:
        print("\n[VALIDATION] ❌ FAILED - Issues found:")
        for issue in issues:
            print(f"  - {issue}")
        raise RuntimeError("Component compatibility validation failed")
    else:
        print("[VALIDATION] ✅ All components compatible")

    return True


def validate_dataset_compatibility(dataset, tokenizer, model_vocab_size):
    print("\n[VALIDATION] Checking dataset compatibility...")

    try:
        sample_batch = []
        n_check = min(5, len(dataset) if hasattr(dataset, "__len__") else 5)
        for i in range(n_check):
            try:
                sample_batch.append(dataset[i])
            except Exception:
                continue

        if not sample_batch:
            print("[VALIDATION] ⚠️  Could not load samples")
            return True

        max_input_id = 0
        min_input_id = float('inf')

        for item in sample_batch:
            input_ids = item.get('input_ids', None)
            if input_ids is not None:
                if isinstance(input_ids, torch.Tensor):
                    if input_ids.numel() > 0:
                        max_input_id = max(max_input_id, int(input_ids.max().item()))
                        min_input_id = min(min_input_id, int(input_ids.min().item()))
                elif isinstance(input_ids, (list, tuple)):
                    if input_ids:
                        max_input_id = max(max_input_id, max(input_ids))
                        min_input_id = min(min_input_id, min(input_ids))

        min_input_id = int(min_input_id) if min_input_id != float('inf') else 0

        print(f"  Input IDs range: [{min_input_id}, {max_input_id}]")
        print(f"  Model vocab size: {model_vocab_size}")

        if max_input_id >= model_vocab_size:
            raise RuntimeError(
                f"Dataset contains out-of-bounds token IDs!\n"
                f"  Max ID: {max_input_id}\n"
                f"  Vocab size: {model_vocab_size}\n"
                f"  → Cell 2 tokenization error or vocab mismatch"
            )

        if min_input_id < 0:
            raise RuntimeError(f"Dataset contains negative token IDs: {min_input_id}")

        print("[VALIDATION] ✅ Dataset token IDs valid")
        return True

    except Exception as e:
        print(f"[VALIDATION] Dataset check failed: {e}")
        raise


def test_model_forward_pass(model, tokenizer, device):
    print("\n[VALIDATION] Testing model forward pass...")

    try:
        core_model = model.module if hasattr(model, 'module') else model
        was_training = core_model.training
        core_model.eval()

        test_text = "আমি কল বন্ধ করেছি।"

        print(f"  [PREPROCESSING] Original text: '{test_text}'")

        if "preprocess_for_indictrans2" in globals():
            try:
                preprocessed_batch = preprocess_for_indictrans2(
                    [test_text],
                    src_lang=_SOURCE_LANGUAGE,
                    tgt_lang=_TARGET_LANGUAGE,
                    indic_processor=None
                )
                preprocessed_text = preprocessed_batch[0] if preprocessed_batch else f"{_SOURCE_LANGUAGE} {_TARGET_LANGUAGE} {test_text}"
                print(f"  [PREPROCESSING] Preprocessed: '{preprocessed_text[:80]}'")
            except Exception as e:
                print(f"  [PREPROCESSING] Failed: {e}, using manual tags")
                preprocessed_text = f"{_SOURCE_LANGUAGE} {_TARGET_LANGUAGE} {test_text}"
        else:
            print(f"  [PREPROCESSING] preprocess_for_indictrans2 not found, using manual tags")
            preprocessed_text = f"{_SOURCE_LANGUAGE} {_TARGET_LANGUAGE} {test_text}"

        inputs = tokenizer(
            preprocessed_text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=_MAX_LENGTH
        )

        test_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        input_ids = inputs['input_ids'].to(test_device)
        attention_mask = inputs['attention_mask'].to(test_device)

        try:
            original_device = next(core_model.parameters()).device
        except StopIteration:
            original_device = test_device

        if test_device != original_device:
            try:
                core_model = core_model.to(test_device)
            except Exception:
                pass

        try:
            with torch.no_grad():
                print("  [TEST 1] Testing forward pass (inference mode)...")
                outputs = core_model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    src_texts=[test_text],
                    token_word_map=None,
                    labels=None,
                    return_dict=True,
                    path=2
                )

            if isinstance(outputs, torch.Tensor):
                print("  ✅ Forward pass successful (tensor output)")
                print(f"  Output shape: {outputs.shape}")
            elif isinstance(outputs, dict):
                print("  ✅ Forward pass successful (dict output)")
                print(f"  Keys: {list(outputs.keys())}")

                print("\n  [TEST 2] Validating DSCD-augmented generation capability...")

                h_augmented = outputs.get('sense_augmented_embeddings') or outputs.get('encoder_last_hidden_state') or outputs.get('last_hidden_state')

                if h_augmented is not None and isinstance(h_augmented, torch.Tensor):
                    try:
                        h_aug = h_augmented.to(test_device).detach()
                        if h_aug.dim() == 2:
                            h_aug = h_aug.unsqueeze(0)
                        print(f"    ✅ DSCD embeddings: shape={h_aug.shape}")

                        enc_wrapped = BaseModelOutput(
                            last_hidden_state=h_aug,
                            hidden_states=None,
                            attentions=None,
                        )
                        print(f"    ✅ BaseModelOutput created successfully")

                        print("\n  [TEST 3] Testing generate() with DSCD-augmented embeddings...")

                        base_model_candidate = None
                        if hasattr(core_model, "base_model"):
                            base_model_candidate = getattr(core_model, "base_model")
                        elif hasattr(core_model, "model"):
                            base_model_candidate = getattr(core_model, "model")
                        elif hasattr(core_model, "basemodel"):
                            base_model_candidate = getattr(core_model, "basemodel")

                        generated = None
                        gen_err = None

                        if base_model_candidate is not None and hasattr(base_model_candidate, "generate"):
                            try:
                                generated = base_model_candidate.generate(
                                    encoder_outputs=enc_wrapped,
                                    attention_mask=attention_mask,
                                    max_length=32,
                                    num_beams=3,
                                    early_stopping=True,
                                )
                            except Exception as e:
                                gen_err = e

                        if generated is None:
                            try:
                                if hasattr(core_model, "generate"):
                                    generated = core_model.generate(
                                        encoder_outputs=enc_wrapped,
                                        attention_mask=attention_mask,
                                        max_length=32,
                                        num_beams=3,
                                        early_stopping=True,
                                    )
                                elif gen_err is not None:
                                    raise gen_err
                                else:
                                    raise RuntimeError("No generate() available on base_model or core_model")
                            except Exception as e:
                                raise e

                        if generated is not None and generated.numel() > 0:
                            try:
                                translation = tokenizer.decode(generated[0], skip_special_tokens=True)
                            except Exception:
                                if hasattr(tokenizer, "batch_decode"):
                                    translation = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
                                else:
                                    translation = str(generated[0][:20].tolist())
                            print(f"    ✅ Generation successful: '{translation}'")
                            print(f"    ✅ DSCD-AUGMENTED GENERATION WORKING!")
                        else:
                            print(f"    ⚠️  Generation returned empty output")
                    except Exception as e:
                        print(f"    ❌ DSCD-augmented generation failed: {e}")
                        raise RuntimeError(f"DSCD-augmented generation validation failed: {e}")
                else:
                    print(f"    ⚠️  No DSCD embeddings in output")
            else:
                print(f"  ⚠️  Unexpected output type: {type(outputs)}")

            print("\n[VALIDATION] ✅ Model forward pass and generation validated")

        finally:
            try:
                if test_device != original_device:
                    core_model = core_model.to(original_device)
            except Exception:
                pass
            if was_training:
                try:
                    core_model.train()
                except Exception:
                    pass

        return True

    except Exception as e:
        print(f"[VALIDATION] ❌ Forward pass failed: {e}")
        try:
            traceback.print_exc()
        except Exception:
            pass
        raise RuntimeError(f"Model forward pass validation failed: {e}")


def main_pipeline() -> Tuple[object, object]:
    print("\n" + "=" * 80)
    print("TATN MAIN PIPELINE (DUAL-PATH COMPATIBLE) (INDICTRANS2)")
    print("=" * 80)
    print(f"Configuration:")
    print(f"  - Span threshold: {_SPAN_THRESHOLD}")
    print(f"  - Uncertainty threshold: {_UNCERTAINTY_THRESHOLD}")
    print(f"  - Discovery frequency: {_PERIODIC_DISCOVERY_FREQUENCY}")
    print(f"  - ASBN Training: {'DISABLED' if not _ENABLE_ASBN_TRAINING else 'ENABLED'}")
    print(f"  - Epochs: {_EPOCHS}")
    print(f"  - Batch size: {_BATCH_SIZE}")
    print("=" * 80)

    pipeline_start = time.time()
    if _DEBUG_TIMING:
        phase_start = time.time()

    initialize_environment()
    if _DEBUG_TIMING:
        print(f"[TIMING] Initialization: {time.time() - phase_start:.2f}s")
        phase_start = time.time()

    print("\n[PHASE 1] Loading tokenizer...")

    if "tokenizer_indic" not in globals():
        raise RuntimeError(
            "IndicTrans2 tokenizer not found!\n"
            "Please run Cell 0 first to load tokenizer_indic from ai4bharat/indictrans2-en-indic-1B"
        )

    tokenizer = globals()["tokenizer_indic"]

    try:
        if not hasattr(tokenizer, 'pad_token_id') or tokenizer.pad_token_id is None:
            if hasattr(tokenizer, 'add_special_tokens'):
                try:
                    tokenizer.add_special_tokens({"pad_token": "<pad>"})
                except Exception:
                    pass
    except Exception:
        pass

    vocab_size = getattr(tokenizer, 'vocab_size', None)
    if vocab_size is None:
        try:
            vocab_size = len(tokenizer)
        except Exception:
            vocab_size = 256000

    print(f"[PHASE 1] IndicTrans2 tokenizer loaded (vocab: {vocab_size})")

    if "validate_tokenizer_vocab" in globals():
        try:
            print("[PHASE 1] Validating tokenizer vocabulary...")
            validate_tokenizer_vocab(tokenizer, expected_vocab_size=None)
        except Exception as e:
            print(f"[PHASE 1] Tokenizer validation warning: {e}")

    if _DEBUG_TIMING:
        print(f"[TIMING] Tokenizer: {time.time() - phase_start:.2f}s")
        phase_start = time.time()

    print(f"\n[PHASE 2] Loading data ({_NUM_SAMPLES} samples)...")
    if "load_and_preprocess_optimized" in globals():
        try:
            pairs = load_and_preprocess_optimized(_NUM_SAMPLES)
        except Exception as e:
            print(f"[PHASE 2] Data loading failed: {e}")
            pairs = [("আমি কল বন্ধ করেছি।", "I turned off the tap.")]
    else:
        print("[PHASE 2] Using fallback data")
        pairs = [("আমি কল বন্ধ করেছি।", "I turned off the tap.")]

    if "MemoryEfficientDataset" not in globals():
        raise RuntimeError("MemoryEfficientDataset not found - run Cell 2")

    dataset = MemoryEfficientDataset(pairs, tokenizer, max_length=_MAX_LENGTH)

    collate_fn = globals().get("safe_collate", None)
    create_dl = globals().get("create_optimized_dataloader", None)
    try:
        if callable(create_dl):
            train_loader = create_dl(dataset, batch_size=_BATCH_SIZE, shuffle=True)
        else:
            dataloader_kwargs = {
                'batch_size': _BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0,
                'pin_memory': torch.cuda.is_available()
            }
            if collate_fn is not None:
                dataloader_kwargs['collate_fn'] = collate_fn
            train_loader = DataLoader(dataset, **dataloader_kwargs)
    except Exception:
        dataloader_kwargs = {
            'batch_size': _BATCH_SIZE,
            'shuffle': True,
            'num_workers': 0,
            'pin_memory': torch.cuda.is_available()
        }
        if collate_fn is not None:
            dataloader_kwargs['collate_fn'] = collate_fn
        train_loader = DataLoader(dataset, **dataloader_kwargs)

    try:
        print(f"[PHASE 2] Dataset: {len(dataset)} samples, {len(train_loader)} batches")
    except Exception:
        print("[PHASE 2] Dataset loaded")

    del pairs
    _safe_clear_gpu_caches()

    if _DEBUG_TIMING:
        print(f"[TIMING] Data loading: {time.time() - phase_start:.2f}s")
        phase_start = time.time()

    print("\n[PHASE 3] Initializing model...")
    if "MemoryOptimizedTATNWithExplanations" not in globals():
        raise RuntimeError("Model class not found - run Cell 6")

    try:
        model_core = MemoryOptimizedTATNWithExplanations(tokenizer)
    except Exception as e:
        print(f"[PHASE 3] Model instantiation failed: {e}")
        raise

    try:
        enc_vs = int(getattr(model_core, "encoder_vocab_size", getattr(model_core, "vocab_size", 0)) or 0)
        dec_vs = int(getattr(model_core, "decoder_vocab_size", getattr(model_core, "vocab_size", 0)) or 0)
        globals()["ENCODER_VOCAB_SIZE"] = enc_vs
        globals()["DECODER_VOCAB_SIZE"] = dec_vs
        globals()["DEFAULT_VOCAB_SIZE"] = int(getattr(model_core, "vocab_size", max(enc_vs, dec_vs, 128112)))
        if hasattr(tokenizer, "name_or_path"):
            globals()["TOKENIZER_NAME_OR_PATH"] = getattr(tokenizer, "name_or_path")
    except Exception:
        pass

    try:
        validate_component_compatibility(model_core, tokenizer)
    except Exception as e:
        print(f"[PHASE 3] ❌ Component validation failed: {e}")
        raise

    try:
        model_vocab_for_dataset = int(globals().get("ENCODER_VOCAB_SIZE", getattr(model_core, "encoder_vocab_size", getattr(model_core, "vocab_size", 0))))
        validate_dataset_compatibility(dataset, tokenizer, model_vocab_for_dataset)
    except Exception as e:
        print(f"[PHASE 3] ❌ Dataset validation failed: {e}")
        raise

    if _USE_MULTI_GPU and _NUM_GPUS > 1:
        device_ids = list(range(_NUM_GPUS))
        print(f"[PHASE 3] Using DataParallel on {device_ids}")
        model = nn.DataParallel(model_core, device_ids=device_ids)
    else:
        model = model_core

    model = model.to(_DEVICE)
    core_model = _get_core_model(model)

    try:
        test_model_forward_pass(model, tokenizer, _DEVICE)
    except Exception as e:
        print(f"[PHASE 3] ❌ Forward pass test failed: {e}")
        raise

    if _FREEZE_ENCODER:
        try:
            if hasattr(core_model, 'base_model') and hasattr(core_model.base_model, 'model') and hasattr(core_model.base_model.model, 'encoder'):
                for p in core_model.base_model.model.encoder.parameters():
                    p.requires_grad = False
                print("[PHASE 3] Encoder frozen")
        except Exception:
            pass

    print(f"[PHASE 3] Model initialized and validated")
    if _DEBUG_TIMING:
        print(f"[TIMING] Model init: {time.time() - phase_start:.2f}s")

    print("\n[PHASE 4] Setting up optimizers...")

    try:
        critic_params = list(core_model.asbn.critic_parameters()) if hasattr(core_model, "asbn") and hasattr(core_model.asbn, "critic_parameters") else []
    except Exception:
        critic_params = []

    critic_ids = {id(p) for p in critic_params}
    base_params = [p for p in core_model.parameters() if p.requires_grad and id(p) not in critic_ids]
    optimizer = torch.optim.AdamW(base_params, lr=_LR_NMT)

    phi_optimizer = None
    if critic_params and _ENABLE_ASBN_TRAINING:
        phi_optimizer = torch.optim.AdamW([p for p in critic_params if p.requires_grad], lr=_LR_PHI)
        print(f"[PHASE 4] ASBN optimizer created ({len([p for p in critic_params if p.requires_grad])} params)")
    elif not _ENABLE_ASBN_TRAINING:
        print(f"[PHASE 4] ASBN optimizer DISABLED (training disabled)")

    print(f"[PHASE 4] Optimizers ready")
    if _DEBUG_TIMING:
        phase_start = time.time()

    print("\n[PHASE 5] Training...")
    print(f"  - ASBN Training: {'DISABLED' if not _ENABLE_ASBN_TRAINING else 'ENABLED'}")
    print(f"  - ASBN Optimizer: {'None (ASBN disabled)' if phi_optimizer is None else 'Active'}")

    trained_model = model
    training_stats = None

    if "train_memory_efficient_tatn" in globals():
        try:
            try:
                trg = getattr(core_model, 'trg', None)
                if trg and hasattr(trg, 'reset_statistics'):
                    trg.reset_statistics()
            except Exception:
                pass
            trained_model = train_memory_efficient_tatn(
                model,
                tokenizer,
                train_loader,
                optimizer,
                phi_optimizer=phi_optimizer,
                epochs=_EPOCHS,
                accumulation_steps=_ACCUMULATION_STEPS,
                validate_every=_VALIDATION_CHECK_INTERVAL,
                enable_validation=(_VALIDATION_CHECK_INTERVAL > 0)
            )
            print("[PHASE 5] Training complete")
        except Exception as e:
            print(f"[PHASE 5] Training failed: {e}")
            trained_model = model
    else:
        print("[PHASE 5] Skipping training (function not found)")

    if _DEBUG_TIMING:
        print(f"[TIMING] Training: {time.time() - phase_start:.2f}s")

    del train_loader, dataset
    _safe_clear_gpu_caches()

    core_model = _get_core_model(trained_model)

    if _DEBUG_TIMING:
        phase_start = time.time()

    print("\n[PHASE 6] Discovery check...")
    discovery_success = False

    try:
        dscd = getattr(core_model, 'dscd', None)
        if dscd is None:
            print("[PHASE 6] No DSCD module")
        else:
            print("[PHASE 6] Running periodic discovery check...")
            if hasattr(dscd, 'periodic_discovery_check'):
                try:
                    sig = inspect.signature(dscd.periodic_discovery_check)
                    params = list(sig.parameters.keys())
                    print(f"[PHASE 6] periodic_discovery_check params: {params}")

                    total_steps = int(_EPOCHS * _NUM_SAMPLES // _BATCH_SIZE) if _BATCH_SIZE > 0 else 0

                    try:
                        if 'global_step' in params or 'total_steps' in params:
                            num_discovered = dscd.periodic_discovery_check(total_steps, _PERIODIC_DISCOVERY_FREQUENCY)
                        else:
                            num_discovered = dscd.periodic_discovery_check()
                    except TypeError:
                        num_discovered = dscd.periodic_discovery_check()
                    discovery_success = True
                    print(f"[PHASE 6] Discovery complete: {num_discovered} homographs found")
                except Exception as e:
                    print(f"[PHASE 6] periodic_discovery_check failed: {e}")
                    try:
                        if hasattr(dscd, 'discover_homographs'):
                            num_discovered = dscd.discover_homographs()
                            discovery_success = True
                            print(f"[PHASE 6] Fallback discovery: {num_discovered} homographs")
                        else:
                            print("[PHASE 6] discover_homographs not available")
                    except Exception as e2:
                        print(f"[PHASE 6] Fallback discovery failed: {e2}")
            else:
                print("[PHASE 6] periodic_discovery_check not available")
                if hasattr(dscd, 'discover_homographs'):
                    try:
                        num_discovered = dscd.discover_homographs()
                        discovery_success = True
                        print(f"[PHASE 6] discover_homographs: {num_discovered} homographs")
                    except Exception as e:
                        print(f"[PHASE 6] discover_homographs failed: {e}")

            stores = _get_dscd_stores_safe(dscd)

            def _store_size(s):
                try:
                    if callable(getattr(s, "size", None)):
                        return int(s.size())
                    return int(getattr(s, "size", 0))
                except Exception:
                    return 0

            total_protos = sum(_store_size(store) for store in stores.values())
            multi_sense = sum(1 for store in stores.values() if _store_size(store) >= 2)

            print("[PHASE 6] Discovery state:")
            print(f"  - Tokens: {len(stores)}")
            print(f"  - Prototypes: {total_protos}")
            print(f"  - Multi-sense: {multi_sense}")

            if len(stores) == 0:
                print("[PHASE 6] WARNING: No prototypes created")
            else:
                discovery_success = True
    except Exception as e:
        print(f"[PHASE 6] Discovery failed: {e}")
        if _DEBUG_TIMING:
            try:
                traceback.print_exc()
            except Exception:
                pass

    if _DEBUG_TIMING:
        print(f"[TIMING] Discovery: {time.time() - phase_start:.2f}s")
    _safe_clear_gpu_caches()
    if _DEBUG_TIMING:
        phase_start = time.time()

    print("\n[PHASE 7] DSCD warmup...")
    if "dscd_discovery_warmup" in globals() and callable(globals().get("dscd_discovery_warmup")):
        try:
            warmup_samples = min(4000, _DSCD_WARMUP_SAMPLES)
            print(f"[PHASE 7] Processing {warmup_samples} warmup samples...")
            warmup_start = time.time()
            dscd_discovery_warmup(trained_model, tokenizer, num_sents=warmup_samples, batch_size=64, max_len=_MAX_LENGTH)
            warmup_duration = time.time() - warmup_start
            print(f"[PHASE 7] Warmup complete ({warmup_samples} samples in {warmup_duration:.1f}s)")
        except Exception as e:
            print(f"[PHASE 7] Warmup failed: {e}")
    else:
        print("[PHASE 7] Skipping warmup (function not found)")

    if _DEBUG_TIMING:
        print(f"[TIMING] Warmup: {time.time() - phase_start:.2f}s")
    _safe_clear_gpu_caches()
    if _DEBUG_TIMING:
        phase_start = time.time()

    print("\n[PHASE 8] Baseline evaluation...")
    baseline_metrics = None

    try:
        dscd_baseline = getattr(core_model, 'dscd', None)
        has_prototypes = False

        if dscd_baseline:
            stores = _get_dscd_stores_safe(dscd_baseline)
            has_prototypes = len(stores) > 0

        if not has_prototypes:
            print("[PHASE 8] Skipping baseline (no prototypes)")
        elif "comprehensive_post_training_testing" in globals() and callable(globals().get("comprehensive_post_training_testing")):
            try:
                trg = getattr(core_model, 'trg', None)
                if trg and hasattr(trg, 'reset_statistics'):
                    trg.reset_statistics()
            except Exception:
                pass

            print("[PHASE 8] ⚡ Running baseline with DSCD-augmented generation...")
            baseline_metrics = comprehensive_post_training_testing(trained_model, tokenizer, run_warmup=False)
            baseline_success = baseline_metrics.get('success_rate_pct', 0)
            baseline_expl = baseline_metrics.get('total_explanations', 0)
            print(f"[PHASE 8] Baseline: {baseline_success:.1f}% success, {baseline_expl} explanations")
        else:
            print("[PHASE 8] Skipping baseline (function not found)")
    except Exception as e:
        print(f"[PHASE 8] Baseline failed: {e}")

    if _DEBUG_TIMING:
        print(f"[TIMING] Baseline: {time.time() - phase_start:.2f}s")
    _safe_clear_gpu_caches()
    if _DEBUG_TIMING:
        phase_start = time.time()

    print("\n[PHASE 9] Post-training evaluation...")
    eval_results: Dict[str, Any] = {}

    if "comprehensive_post_training_testing" in globals() and callable(globals().get("comprehensive_post_training_testing")):
        try:
            try:
                trg = getattr(core_model, 'trg', None)
                if trg and hasattr(trg, 'reset_statistics'):
                    trg.reset_statistics()
            except Exception:
                pass

            print("[PHASE 9] ⚡ Running evaluation with DSCD-augmented generation...")
            eval_results = comprehensive_post_training_testing(
                trained_model,
                tokenizer,
                run_warmup=False,
                compare_baseline=(baseline_metrics is not None),
                baseline_metrics=baseline_metrics
            )
            final_success = eval_results.get('success_rate_pct', 0)
            final_expl = eval_results.get('total_explanations', 0)
            print(f"[PHASE 9] Evaluation: {final_success:.1f}% success, {final_expl} explanations")
        except Exception as e:
            print(f"[PHASE 9] Evaluation failed: {e}")
    else:
        print("[PHASE 9] Skipping evaluation (function not found)")

    if _DEBUG_TIMING:
        print(f"[TIMING] Evaluation: {time.time() - phase_start:.2f}s")
    _safe_clear_gpu_caches()
    if _DEBUG_TIMING:
        phase_start = time.time()

    print("\n[PHASE 10] Saving checkpoint...")
    try:
        os.makedirs(_CHECKPOINT_DIR, exist_ok=True)
        was_training = getattr(core_model, "training", False)
        core_model.eval()
        try:
            model_state = core_model.state_dict()
            dscd_state = {}

            if hasattr(core_model, 'dscd'):
                dscd_save = core_model.dscd
                if hasattr(dscd_save, 'state_dict'):
                    lock = None
                    if hasattr(dscd_save, 'buffer_lock'):
                        lock = dscd_save.buffer_lock
                    elif hasattr(dscd_save, 'clustering_lock'):
                        lock = dscd_save.clustering_lock

                    try:
                        if lock:
                            try:
                                with lock:
                                    dscd_state = dscd_save.state_dict()
                            except Exception:
                                dscd_state = dscd_save.state_dict()
                        else:
                            dscd_state = dscd_save.state_dict()
                    except Exception as e:
                        print(f"[PHASE 10] DSCD state_dict failed: {e}")
                        dscd_state = {}

            optimizer_state = None
            if optimizer is not None:
                try:
                    optimizer_state = optimizer.state_dict()
                    if 'state' in optimizer_state:
                        for param_state in optimizer_state['state'].values():
                            if isinstance(param_state, dict) and 'momentum_buffer' in param_state:
                                try:
                                    del param_state['momentum_buffer']
                                except Exception:
                                    pass
                except Exception:
                    optimizer_state = None

            checkpoint = {
                'model_state_dict': model_state,
                'dscd_state': dscd_state,
                'optimizer_state_dict': optimizer_state,
                'training_stats': training_stats,
                'baseline_metrics': baseline_metrics,
                'eval_results': eval_results,
                'discovery_success': discovery_success,
                'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
                'config': {
                    'epochs': _EPOCHS,
                    'batch_size': _BATCH_SIZE,
                    'span_threshold': _SPAN_THRESHOLD,
                    'uncertainty_threshold': _UNCERTAINTY_THRESHOLD,
                    'discovery_frequency': _PERIODIC_DISCOVERY_FREQUENCY,
                    'vocab_size': vocab_size,
                    'asbn_training_enabled': _ENABLE_ASBN_TRAINING,
                }
            }
            torch.save(checkpoint, _CHECKPOINT_PATH)

            try:
                import mmap
                with open(_CHECKPOINT_PATH, 'rb') as f:
                    with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as m:
                        size_mb = len(m) / 1024**2

                print(f"[PHASE 10] Checkpoint saved: {_CHECKPOINT_PATH}")
                print(f"  - Size: {size_mb:.2f} MB")

                try:
                    verify_keys = torch.load(_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
                    has_model = 'model_state_dict' in verify_keys and len(verify_keys['model_state_dict']) > 0
                    has_dscd = 'dscd_state' in verify_keys and len(verify_keys.get('dscd_state', {})) > 0
                    print(f"  - Model: {'OK' if has_model else 'MISSING'}")
                    print(f"  - DSCD: {'OK' if has_dscd else 'MISSING'}")

                    if has_dscd:
                        try:
                            dscd_verify_state = verify_keys.get('dscd_state', {})
                            num_tokens = 0
                            if 'prototype_stores' in dscd_verify_state:
                                num_tokens = len(dscd_verify_state['prototype_stores'])
                            elif 'prototype_stores_data' in dscd_verify_state:
                                num_tokens = len(dscd_verify_state['prototype_stores_data'])
                            print(f"  - DSCD tokens: {num_tokens}")
                        except Exception:
                            print(f"  - DSCD tokens: unknown")

                    del verify_keys
                except Exception as e:
                    print(f"[PHASE 10] Checkpoint verification warning: {e}")
            except Exception:
                print(f"[PHASE 10] Checkpoint saved: {_CHECKPOINT_PATH}")
        finally:
            if was_training:
                try:
                    core_model.train()
                except Exception:
                    pass
    except Exception as e:
        print(f"[PHASE 10] Checkpoint failed: {e}")
        if _DEBUG_TIMING:
            try:
                traceback.print_exc()
            except Exception:
                pass

    if _DEBUG_TIMING:
        print(f"[TIMING] Checkpoint: {time.time() - phase_start:.2f}s")

    print("\n[PHASE 11] Final validation...")
    try:
        dscd_ok = False
        if hasattr(core_model, 'dscd'):
            stores = _get_dscd_stores_safe(core_model.dscd)
            dscd_ok = len(stores) > 0

        asbn_ok = hasattr(core_model, 'asbn') and hasattr(core_model.asbn, 'forward')
        trg_ok = hasattr(core_model, 'trg') and hasattr(core_model.trg, 'process_sentence_for_explanations')

        basemodel_ok = False
        try:
            if hasattr(core_model, 'base_model') and hasattr(core_model.base_model, 'generate'):
                basemodel_ok = True
        except Exception:
            basemodel_ok = False

        print(f"[PHASE 11] Component validation:")
        print(f"  - DSCD: {'OK' if dscd_ok else 'MISSING'}")
        print(f"  - ASBN: {'OK' if asbn_ok else 'MISSING'} {'(DISABLED)' if not _ENABLE_ASBN_TRAINING else '(ENABLED)'}")
        print(f"  - TRG: {'OK' if trg_ok else 'MISSING'}")
        print(f"  - IndicTrans2: {'OK' if basemodel_ok else 'MISSING'}")

        all_ok = dscd_ok and asbn_ok and trg_ok and basemodel_ok
        if all_ok:
            print("[PHASE 11] ✅ All components validated")
        else:
            print("[PHASE 11] ⚠️  Some components missing")
    except Exception as e:
        print(f"[PHASE 11] Validation failed: {e}")

    pipeline_time = time.time() - pipeline_start

    print("\n" + "=" * 80)
    print("PIPELINE COMPLETE - FINAL SUMMARY")
    print("=" * 80)
    print(f"\n[TIMING]")
    print(f"  Total time: {pipeline_time:.2f}s ({pipeline_time/60:.2f} min)")

    print(f"\n[TRAINING]")
    if training_stats:
        total_loss = training_stats.get('total_loss', [])
        optimizer_updates = training_stats.get('optimizer_updates', 0)
        print(f"  Completed: {optimizer_updates} optimizer updates")
        if total_loss:
            recent_loss = sum(total_loss[-100:]) / len(total_loss[-100:])
            print(f"  - Final loss: {recent_loss:.6f}")
    else:
        print("  No stats available")

    print(f"\n[DISCOVERY]")
    if discovery_success:
        print("  ✅ Success")
    else:
        print("  ⚠️  Issues detected")

    print(f"\n[EVALUATION]")
    if baseline_metrics and eval_results:
        baseline_success = baseline_metrics.get('success_rate_pct', 0)
        final_success = eval_results.get('success_rate_pct', 0)
        improvement = final_success - baseline_success

        print(f"  Baseline -> Final: {baseline_success:.1f}% -> {final_success:.1f}%")
        print(f"  Improvement: {improvement:+.1f}%")

        baseline_dscd_stats = baseline_metrics.get('dscd_stats', {})
        final_dscd_stats = eval_results.get('dscd_stats', {})

        baseline_dscd = baseline_dscd_stats.get('multi_sense_words', 0) if isinstance(baseline_dscd_stats, dict) else 0
        final_dscd = final_dscd_stats.get('multi_sense_words', 0) if isinstance(final_dscd_stats, dict) else 0

        if baseline_dscd is not None and final_dscd is not None:
            print(f"  DSCD multi-sense: {baseline_dscd} -> {final_dscd}")

        baseline_asbn_stats = baseline_metrics.get('asbn_stats', {})
        final_asbn_stats = eval_results.get('asbn_stats', {})

        baseline_asbn = baseline_asbn_stats.get('domain_accuracy', 0) if isinstance(baseline_asbn_stats, dict) else 0
        final_asbn = final_asbn_stats.get('domain_accuracy', 0) if isinstance(final_asbn_stats, dict) else 0

        if baseline_asbn is not None and final_asbn is not None:
            print(f"  ASBN accuracy: {baseline_asbn:.2%} -> {final_asbn:.2%} {'(DISABLED)' if not _ENABLE_ASBN_TRAINING else ''}")
    elif eval_results:
        print(f"  Success rate: {eval_results.get('success_rate_pct', 0):.1f}%")
    else:
        print("  No results")

    print(f"\n[CHECKPOINT]")
    if os.path.exists(_CHECKPOINT_PATH):
        try:
            size_mb = os.path.getsize(_CHECKPOINT_PATH) / 1024**2
            print(f"  Saved: {_CHECKPOINT_PATH}")
            print(f"  - Size: {size_mb:.2f} MB")
        except Exception:
            print(f"  Saved: {_CHECKPOINT_PATH}")
    else:
        print("  Not saved")

    print("=" * 80)
    print(f"Usage: trained_model, tokenizer = main_pipeline()")
    print("=" * 80)

    _safe_clear_gpu_caches()
    return trained_model, tokenizer


print("=" * 80)
print("# Cell 10: Main Pipeline (DUAL-PATH COMPATIBLE) (IndicTrans2)")
print("=" * 80)

✓ Registered safe globals for PyTorch 2.6+
# Cell 10: Main Pipeline (DUAL-PATH COMPATIBLE) (IndicTrans2)


In [ ]:
# ==========================================================================================
# CELL 11: MAIN EXECUTION WRAPPER (DUAL-PATH COMPATIBLE) (INDICTRANS2)
# ==========================================================================================
from datetime import datetime, timezone
import os
import traceback
import math
import sys
import time
import torch
import gc

try:
    _NUM_SAMPLES = int(globals().get("NUM_SAMPLES", 30000))
    _EPOCHS = int(globals().get("EPOCHS", 1))
    _BATCH_SIZE = int(globals().get("BATCH_SIZE", 4))
    _ACCUMULATION_STEPS = int(globals().get("ACCUMULATION_STEPS", 16))

    raw_device = globals().get("DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
    _DEVICE = raw_device if isinstance(raw_device, torch.device) else torch.device(str(raw_device))

    _ENABLE_ASBN_TRAINING = bool(globals().get("ENABLE_ASBN_TRAINING", False))
    _ENABLE_TRG_INFERENCE = bool(globals().get("ENABLE_TRG_INFERENCE", True))
    _PERIODIC_DISCOVERY_FREQUENCY = int(globals().get("PERIODIC_DISCOVERY_FREQUENCY", 50))
    _VERBOSE_LOGGING = bool(globals().get("VERBOSE_LOGGING", False))
    _DEBUG_DISCOVERY = bool(globals().get("DEBUG_DISCOVERY", False))
    _DEBUG_TIMING = bool(globals().get("DEBUG_TIMING", False))
    _NUM_GPUS = int(globals().get("NUM_GPUS", torch.cuda.device_count() if torch.cuda.is_available() else 0))
    _USE_MULTI_GPU = bool(globals().get("USE_MULTI_GPU", _NUM_GPUS > 1))
    _SPAN_THRESHOLD = float(globals().get("SPAN_THRESHOLD", 0.20))
    _UNCERTAINTY_THRESHOLD = float(globals().get("UNCERTAINTY_THRESHOLD", 0.15))
    _MAX_LENGTH = int(globals().get("MAX_LENGTH", 52))
    _SOURCE_LANGUAGE = str(globals().get("SOURCE_LANGUAGE", "ben_Beng"))
    _TARGET_LANGUAGE = str(globals().get("TARGET_LANGUAGE", "eng_Latn"))

    raw_list = globals().get("HOMOGRAPH_REFERENCE_LIST_BN", ["কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা"])
    _HOMOGRAPH_REFERENCE_LIST_BN = set(str(w) for w in raw_list)
    cell0_loaded = "NUM_SAMPLES" in globals()
except Exception:
    _NUM_SAMPLES = 30000
    _EPOCHS = 1
    _BATCH_SIZE = 4
    _ACCUMULATION_STEPS = 16
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _ENABLE_ASBN_TRAINING = False
    _ENABLE_TRG_INFERENCE = True
    _PERIODIC_DISCOVERY_FREQUENCY = 50
    _VERBOSE_LOGGING = False
    _DEBUG_DISCOVERY = False
    _DEBUG_TIMING = False
    _NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
    _USE_MULTI_GPU = _NUM_GPUS > 1
    _SPAN_THRESHOLD = 0.20
    _UNCERTAINTY_THRESHOLD = 0.15
    _MAX_LENGTH = 52
    _SOURCE_LANGUAGE = "ben_Beng"
    _TARGET_LANGUAGE = "eng_Latn"
    _HOMOGRAPH_REFERENCE_LIST_BN = {"কল", "কাল", "পাতা", "ব্যাংক", "ফল", "মাথা"}
    cell0_loaded = False

_CHECKPOINT_PATH = globals().get("CHECKPOINT_PATH", "/kaggle/working/tatn_final.pt")


def _safe_div_ceil(a: int, b: int) -> int:
    try:
        return math.ceil(a / b)
    except Exception:
        return 0


def _format_duration(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    if seconds < 3600:
        return f"{seconds / 60:.1f}min"
    return f"{seconds / 3600:.2f}hr"


def _safe_get(d: dict, *keys, default=None):
    if not isinstance(d, dict):
        return default
    obj = d
    for k in keys:
        if not isinstance(obj, dict):
            return default
        obj = obj.get(k, None)
        if obj is None:
            return default
    return obj


def _get_dscd_homographs(model):
    try:
        core = model.module if hasattr(model, "module") else model
        dscd = getattr(core, "dscd", None)
        if dscd is None:
            return set()

        if hasattr(dscd, "get_discovered_homographs"):
            try:
                res = dscd.get_discovered_homographs() or []
                return set(w for w in res if isinstance(w, str) and w.strip())
            except Exception:
                pass

        stores = getattr(dscd, "prototype_stores", {}) or {}
        lock = getattr(dscd, "buffer_lock", None) or getattr(dscd, "clustering_lock", None)
        try:
            if lock:
                with lock:
                    snapshot = dict(stores)
            else:
                snapshot = dict(stores)
        except Exception:
            snapshot = dict(stores)

        homographs = set()
        for token, store in snapshot.items():
            try:
                has_size = False
                if hasattr(store, "size") and callable(getattr(store, "size")):
                    try:
                        has_size = int(store.size()) >= 1
                    except Exception:
                        has_size = False
                elif hasattr(store, "centroids"):
                    try:
                        has_size = len(getattr(store, "centroids", [])) >= 1
                    except Exception:
                        has_size = False
                if has_size:
                    clean = str(token).replace("▁", "").replace("Ġ", "").replace("##", "").strip().lower()
                    if clean:
                        homographs.add(clean)
            except Exception:
                continue
        return homographs
    except Exception:
        return set()


def _safe_cleanup():
    try:
        if torch.cuda.is_available():
            for i in range(torch.cuda.device_count()):
                try:
                    with torch.cuda.device(i):
                        torch.cuda.empty_cache()
                except Exception:
                    pass
        if gc.isenabled():
            gc.collect()
    except Exception:
        pass


def validate_tokenizer_vocab(tokenizer):
    try:
        vocab_size = len(tokenizer) if hasattr(tokenizer, '__len__') else getattr(tokenizer, 'vocab_size', 0)
        src_id = tokenizer.convert_tokens_to_ids(_SOURCE_LANGUAGE)
        tgt_id = tokenizer.convert_tokens_to_ids(_TARGET_LANGUAGE)
        
        print(f"[TOKENIZER-VALIDATION] Actual vocab size: {vocab_size}")
        print(f"[TOKENIZER-VALIDATION] Language codes:")
        print(f"  {_SOURCE_LANGUAGE} → {src_id}")
        print(f"  {_TARGET_LANGUAGE} → {tgt_id}")
        
        unk_id = getattr(tokenizer, 'unk_token_id', None)
        if unk_id is not None and (src_id == unk_id or tgt_id == unk_id):
            print("[TOKENIZER-VALIDATION] ⚠ Language code not in vocabulary")
            return False
        
        print("[TOKENIZER-VALIDATION] ✅ Language codes valid")
        return True
    except Exception as e:
        print(f"[TOKENIZER-VALIDATION] ❌ Validation failed: {e}")
        return False


def test_model_forward_pass(model, tokenizer, device):
    print("\n[VALIDATION] Testing model forward pass...")
    
    test_text = "আমি কল বন্ধ করেছি।"
    
    try:
        preprocess_fn = globals().get("preprocess_for_indictrans2", None)
        if preprocess_fn and callable(preprocess_fn):
            indic_processor = globals().get("indic_processor_instance", None)
            preprocessed_batch = preprocess_fn(
                [test_text],
                src_lang=_SOURCE_LANGUAGE,
                tgt_lang=_TARGET_LANGUAGE,
                indic_processor=indic_processor
            )
            preprocessed_text = preprocessed_batch[0] if preprocessed_batch else test_text
            print(f"  [PREPROCESSING] Original text: '{test_text}'")
            print(f"  [PREPROCESSING] Preprocessed: '{preprocessed_text}'")
        else:
            preprocessed_text = test_text
    except Exception as e:
        print(f"  [PREPROCESSING] Failed: {e}, using original text")
        preprocessed_text = test_text
    
    try:
        encoded = tokenizer(
            preprocessed_text,
            max_length=_MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        
        decoder_start_token_id = getattr(tokenizer, "eos_token_id", 2)
        decoder_input_ids = torch.full(
            (input_ids.size(0), 1),
            decoder_start_token_id,
            dtype=torch.long,
            device=device
        )
        
        print("  [TEST 1] Testing forward pass (inference mode)...")
        model.eval()
        
        if hasattr(model, 'module'):
            test_model = model.module
            print("  Using unwrapped model (bypassing DataParallel)")
        else:
            test_model = model
        
        with torch.no_grad():
            outputs = test_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                decoder_input_ids=decoder_input_ids,
                use_cache=False,
                return_dict=True
            )
        
        if not isinstance(outputs, dict):
            raise RuntimeError(f"Expected dict output, got {type(outputs)}")
        
        print("  ✅ Forward pass successful (dict output)")
        print(f"  Keys: {list(outputs.keys())}")
        
        print("\n  [TEST 2] Validating DSCD-augmented generation capability...")
        
        h_augmented = outputs.get('sense_augmented_embeddings')
        if h_augmented is None:
            h_augmented = outputs.get('encoder_last_hidden_state')
        if h_augmented is None:
            h_augmented = outputs.get('last_hidden_state')
        
        if h_augmented is None:
            raise RuntimeError("No valid embeddings found in model outputs")
        
        if not isinstance(h_augmented, torch.Tensor):
            raise RuntimeError(f"Expected tensor embeddings, got {type(h_augmented)}")
        
        if h_augmented.dim() != 3:
            raise RuntimeError(f"Expected 3D tensor [batch, seq, hidden], got shape {h_augmented.shape}")
        
        print(f"  ✅ DSCD embeddings validated: shape={h_augmented.shape}")
        
        print("\n  [TEST 3] Testing generation capability...")
        try:
            base_model = getattr(test_model, "base_model", None) or getattr(test_model, "model", None)
            
            if base_model is None or not hasattr(base_model, "generate"):
                print("  ⚠ Generation method not available (expected for TATN wrapper)")
                return True
            
            with torch.no_grad():
                generated_ids = base_model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_length=min(32, _MAX_LENGTH),
                    num_beams=1,
                    early_stopping=True
                )
            
            if generated_ids is None or not isinstance(generated_ids, torch.Tensor):
                raise RuntimeError("Generation returned invalid output")
            
            print(f"  ✅ Generation test passed: shape={generated_ids.shape}")
            
        except Exception as e:
            print(f"  ⚠ Generation test skipped: {type(e).__name__}")
        
        return True
        
    except Exception as e:
        print(f"[VALIDATION] ❌ Forward pass failed: {e}")
        if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
            traceback.print_exc()
        raise RuntimeError(f"Model forward pass validation failed: {e}")


if __name__ == "__main__":
    print("=" * 80)
    print("MEMORY-OPTIMIZED TATN (DUAL-PATH COMPATIBLE) (INDICTRANS2)")
    print("=" * 80)

    user_login = os.getenv("KAGGLE_USERNAME") or os.getenv("USER") or "manas0003"
    start_time = time.time()
    now_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    print(f"User: {user_login}")
    print(f"Started: {now_utc}")

    print("\n[CONFIGURATION]")
    print(f"  Cell 0 status: {'Loaded' if cell0_loaded else 'Using fallbacks'}")
    print(f"  Samples: {_NUM_SAMPLES}")
    print(f"  Epochs: {_EPOCHS}")
    print(f"  Batch Size: {_BATCH_SIZE}")
    print(f"  Accumulation: {_ACCUMULATION_STEPS}")
    print(f"  Device: {_DEVICE}")
    print(f"  Multi-GPU: {'ENABLED' if _USE_MULTI_GPU else 'DISABLED'} ({_NUM_GPUS} GPUs)")
    print(f"  Source language: {_SOURCE_LANGUAGE}")
    print(f"  Target language: {_TARGET_LANGUAGE}")
    print(f"  Span threshold: {_SPAN_THRESHOLD}")
    print(f"  Uncertainty threshold: {_UNCERTAINTY_THRESHOLD}")
    print(f"  Max length: {_MAX_LENGTH}")
    print(f"  Discovery frequency: {_PERIODIC_DISCOVERY_FREQUENCY}")
    print("=" * 80)

    trained_model, tokenizer = None, None
    pipeline_success = False
    failure_category = None
    failure_details = ""

    if "main_pipeline" not in globals() or not callable(globals().get("main_pipeline")):
        print("\nERROR: main_pipeline not found")
        print("   -> Run Cell 10 before executing Cell 11")
        failure_category = "MISSING_DEPENDENCY"
        failure_details = "Cell 10 not executed"
    else:
        try:
            print("\nStarting pipeline (main_pipeline)...")
            pipeline_start = time.time()
            trained_model, tokenizer = main_pipeline()
            pipeline_duration = time.time() - pipeline_start
            print(f"\nPipeline completed: {_format_duration(pipeline_duration)}")
            pipeline_success = True
        except KeyboardInterrupt:
            print("\nInterrupted by user")
            failure_category = "USER_INTERRUPT"
            failure_details = "Manual stop"
        except RuntimeError as e:
            msg = str(e).lower()
            if "tokenizer" in msg or "sentencepiece" in msg:
                print("\nTokenizer error")
                failure_category = "TOKENIZER_ERROR"
                failure_details = str(e)[:200]
                print("\nFix:")
                print("   ! pip install transformers==4.30.2 sentencepiece tokenizers")
                print("   Then RESTART kernel and re-run Cells 0-11")
            elif "out of memory" in msg:
                print("\nOut of Memory")
                failure_category = "OOM_ERROR"
                failure_details = "GPU OOM"
                print("\nFixes:")
                print("   1. Reduce BATCH_SIZE (try 2-4)")
                print("   2. Reduce NUM_SAMPLES (try 10k-20k)")
                print("   3. Increase ACCUMULATION_STEPS (32-64)")
            else:
                print(f"\nRuntime error: {type(e).__name__}")
                print(f"   {str(e)[:400]}")
                failure_category = "RUNTIME_ERROR"
                failure_details = str(e)[:200]
            if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                print("\n[TRACEBACK]")
                traceback.print_exc()
        except Exception as e:
            print(f"\nUnexpected error: {type(e).__name__}")
            print(f"   {str(e)[:400]}")
            failure_category = "UNKNOWN_ERROR"
            failure_details = str(e)[:200]
            if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                print("\n[TRACEBACK]")
                traceback.print_exc()

    checkpoint_dict = None
    checkpoint_valid = False

    if pipeline_success and trained_model is not None and tokenizer is not None:
        print("\n" + "=" * 80)
        print("PIPELINE SUCCEEDED")
        print("=" * 80)
        print("\n[CHECKPOINT] Inspecting saved checkpoint (if present)...")
        try:
            if os.path.exists(_CHECKPOINT_PATH):
                try:
                    checkpoint_dict = torch.load(_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
                except TypeError:
                    checkpoint_dict = torch.load(_CHECKPOINT_PATH, map_location="cpu")
                except Exception as e:
                    print(f"  Failed to load checkpoint: {e}")
                    checkpoint_dict = None

                if checkpoint_dict:
                    has_model = "model_state_dict" in checkpoint_dict and bool(checkpoint_dict.get("model_state_dict"))
                    has_dscd = "dscd_state" in checkpoint_dict and bool(checkpoint_dict.get("dscd_state"))
                    print(f"  Model: {'Present' if has_model else 'MISSING'}")
                    print(f"  DSCD: {'Present' if has_dscd else 'MISSING'}")
                    num_tokens = 0
                    if has_dscd:
                        dscd_state = checkpoint_dict.get("dscd_state", {}) or {}
                        if isinstance(dscd_state, dict):
                            if "prototype_stores" in dscd_state and isinstance(dscd_state["prototype_stores"], dict):
                                num_tokens = len(dscd_state["prototype_stores"])
                            elif "prototype_stores_data" in dscd_state and isinstance(dscd_state["prototype_stores_data"], dict):
                                num_tokens = len(dscd_state["prototype_stores_data"])
                            else:
                                try:
                                    num_tokens = len(dscd_state)
                                except Exception:
                                    num_tokens = 0
                        else:
                            try:
                                num_tokens = len(dscd_state)
                            except Exception:
                                num_tokens = 0
                        print(f"  Tokens: {num_tokens}")
                        checkpoint_valid = num_tokens > 0
                    else:
                        print("  Tokens: N/A")
                else:
                    print("  Checkpoint file present but could not be loaded/parsed")
            else:
                print(f"  NOT FOUND: {_CHECKPOINT_PATH}")
        except Exception as e:
            print(f"  Checkpoint inspection failed: {e}")
            if _VERBOSE_LOGGING or _DEBUG_DISCOVERY:
                traceback.print_exc()

        print("\n[COMPONENTS]")
        try:
            core = trained_model.module if hasattr(trained_model, "module") else trained_model
            dscd = getattr(core, "dscd", None)
            if dscd and hasattr(dscd, "get_prototype_summary"):
                try:
                    dscd_stats = dscd.get_prototype_summary()
                    print("  DSCD:")
                    print(f"    - Tokens: {dscd_stats.get('total_tokens', 0)}")
                    print(f"    - Prototypes: {dscd_stats.get('total_prototypes', 0)}")
                    print(f"    - Homographs: {dscd_stats.get('num_homographs', 0)}")
                except Exception:
                    pass
            asbn = getattr(core, "asbn", None)
            if asbn and hasattr(asbn, "get_detailed_stats"):
                try:
                    asbn_stats = asbn.get_detailed_stats()
                    print("  ASBN:")
                    print(f"    - Domain accuracy: {asbn_stats.get('domain_accuracy', 0):.2%} {'(DISABLED)' if not _ENABLE_ASBN_TRAINING else ''}")
                except Exception:
                    pass
            trg = getattr(core, "trg", None)
            if trg and hasattr(trg, "get_statistics"):
                try:
                    trg_stats = trg.get_statistics()
                    print("  TRG:")
                    print(f"    - Explanations: {trg_stats.get('explanations_generated', 0)}")
                except Exception:
                    pass
            base_model_obj = getattr(core, "base_model", None) or getattr(core, "basemodel", None) or getattr(core, "model", None)
            basemodel_ok = base_model_obj is not None and hasattr(base_model_obj, "generate")
            print(f"  IndicTrans2 generate: {'OK' if basemodel_ok else 'MISSING'}")
        except Exception as e:
            print(f"  Component introspection failed: {e}")

        _safe_cleanup()

        print("\n[INFERENCE VALIDATION]")
        print("Testing disambiguation on ambiguous sentences...")
        print("-" * 80)

        dscd_homographs = _get_dscd_homographs(trained_model)
        print(f"DSCD discovered: {len(dscd_homographs)} homographs")
        if dscd_homographs and _DEBUG_DISCOVERY:
            print(f"  Sample: {list(dscd_homographs)[:10]}")

        test_sentences = [
            ("আমি কল বন্ধ করেছি।", "কল (tap/call)"),
            ("কাল আমি বই কিনব।", "কাল (tomorrow/yesterday)"),
            ("পাতা ঝরে পড়েছে।", "পাতা (leaf/page)"),
        ]

        try:
            if "translate_with_explanations" not in globals() or not callable(globals().get("translate_with_explanations")):
                print("translate_with_explanations not available")
                print("   -> Run Cell 8 before Cell 11")
            else:
                for idx, (sentence, desc) in enumerate(test_sentences, 1):
                    try:
                        print(f"\n{idx}.  {desc}")
                        print(f"   Input: {sentence}")
                        inf_start = time.time()
                        res = translate_with_explanations(
                            trained_model,
                            tokenizer,
                            sentence,
                            source_lang=_SOURCE_LANGUAGE,
                            target_lang=_TARGET_LANGUAGE,
                            device=_DEVICE,
                            max_length=_MAX_LENGTH,
                            span_threshold=_SPAN_THRESHOLD,
                            uncertainty_threshold=_UNCERTAINTY_THRESHOLD,
                            track_stats=True,
                        )
                        inf_time = time.time() - inf_start
                        if isinstance(res, dict):
                            translation = res.get("translation", "N/A")
                            amb_count = res.get("ambiguous_words_detected", 0)
                            exs = res.get("explanations", []) or []
                            print(f"   Translation: {translation}")
                            print(f"   Ambiguous: {amb_count}")
                            print(f"   Time: {inf_time:.3f}s")
                            if exs:
                                for exp in exs:
                                    word = exp.get("ambiguous_word", exp.get("token", "N/A"))
                                    clean = str(word).replace("▁", "").replace("Ġ", "").strip().lower()
                                    is_in = clean in dscd_homographs
                                    try:
                                        conf = float(exp.get("confidence", 0.5))
                                        span = float(exp.get("span", 0.0))
                                        u = float(exp.get("uncertainty", 0.0))
                                        print(f"   -> '{word}': conf={conf:.3f}, s={span:.3f}, u={u:.3f} {'[DSCD]' if is_in else ''}")
                                    except Exception:
                                        print(f"   -> '{word}': (no metrics)")
                            else:
                                print("   No explanations")
                        else:
                            print("   Unexpected format from translate_with_explanations()")
                        _safe_cleanup()
                    except Exception as e:
                        print(f"   Failed: {type(e).__name__} - {str(e)[:200]}")
                        if _DEBUG_DISCOVERY:
                            traceback.print_exc()
        except Exception:
            if _DEBUG_DISCOVERY:
                traceback.print_exc()

        print("\n" + "=" * 80)
        print("NEXT STEPS")
        print("=" * 80)
        print("\n1. Single translation example:")
        print(f"   result = translate_with_explanations(trained_model, tokenizer, 'আমি কল বন্ধ করেছি।', source_lang='{_SOURCE_LANGUAGE}', target_lang='{_TARGET_LANGUAGE}', device=_DEVICE, max_length={_MAX_LENGTH})")
        print("\n2. Evaluation / warmup utilities available in Cells 8-10")
        if not checkpoint_valid:
            print("\nCheckpoint needs verification - re-run Cell 10 if needed")
        print("\n" + "=" * 80)
    else:
        print("\n" + "=" * 80)
        print("PIPELINE FAILED")
        print("=" * 80)
        print(f"\nCategory: {failure_category or 'UNKNOWN'}")
        if failure_details:
            print(f"Details: {failure_details[:200]}")
        print("\n[DIAGNOSTICS]")
        components = {
            "Cell 0": "NUM_SAMPLES" in globals(),
            "Cell 2": "MemoryEfficientDataset" in globals(),
            "Cell 3": "MemoryEfficientDSCDOnline" in globals(),
            "Cell 4": "MemoryEfficientASBNModule" in globals(),
            "Cell 5": "CompleteTRGWithExplanations" in globals(),
            "Cell 6": "MemoryOptimizedTATNWithExplanations" in globals(),
            "Cell 7": "train_memory_efficient_tatn" in globals(),
            "Cell 8": "translate_with_explanations" in globals(),
            "Cell 9": "comprehensive_post_training_testing" in globals(),
            "Cell 10": "main_pipeline" in globals(),
        }
        all_present = True
        for comp, present in components.items():
            status = "OK" if present else "MISSING"
            print(f"  {status} {comp}")
            if not present:
                all_present = False

        print("\n[RECOVERY]")
        if failure_category == "MISSING_DEPENDENCY":
            print("\n-> Run Cells 0-10 in sequence, then re-run Cell 11")
        elif failure_category == "TOKENIZER_ERROR":
            print("\n-> Install dependencies:")
            print("  ! pip install transformers==4.30.2 sentencepiece tokenizers")
            print("  Then RESTART kernel and re-run Cells 0-11")
        elif failure_category == "OOM_ERROR":
            print("\n-> Reduce memory in Cell 0:")
            print("  BATCH_SIZE = 2")
            print("  NUM_SAMPLES = 15000")
            print("  ACCUMULATION_STEPS = 32")
        elif failure_category == "RUNTIME_ERROR":
            print("\n-> Enable debug in Cell 0:")
            print("  VERBOSE_LOGGING = True")
            print("  DEBUG_DISCOVERY = True")
        else:
            print("\n-> General steps:")
            print("  1. Enable DEBUG in Cell 0")
            print("  2. Re-run Cells 0-11")
            print("  3. Check GPU: torch.cuda.is_available()")
            print("  4. Verify data loaded")

    total_duration = time.time() - start_time
    end_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    print("\n" + "=" * 80)
    print("EXECUTION SUMMARY")
    print("=" * 80)
    print(f"User: {user_login}")
    print(f"Started: {now_utc}")
    print(f"Finished: {end_utc}")
    print(f"Duration: {_format_duration(total_duration)}")

    if pipeline_success:
        print("Status: SUCCESS")
        if checkpoint_valid:
            print("Checkpoint: VALID")
        else:
            print("Checkpoint: CHECK NEEDED")
    else:
        print(f"Status: FAILED ({failure_category or 'UNKNOWN'})")

    print("=" * 80)
    _safe_cleanup()

MEMORY-OPTIMIZED TATN (DUAL-PATH COMPATIBLE) (INDICTRANS2)
User: manas0003
Started: 2026-03-03 08:16:32 UTC

[CONFIGURATION]
  Cell 0 status: Loaded
  Samples: 60000
  Epochs: 2
  Batch Size: 2
  Accumulation: 4
  Device: cuda
  Multi-GPU: DISABLED (1 GPUs)
  Source language: ben_Beng
  Target language: eng_Latn
  Span threshold: 0.2
  Uncertainty threshold: 0.15
  Max length: 128
  Discovery frequency: 100

Starting pipeline (main_pipeline)...

TATN MAIN PIPELINE (DUAL-PATH COMPATIBLE) (INDICTRANS2)
Configuration:
  - Span threshold: 0.2
  - Uncertainty threshold: 0.15
  - Discovery frequency: 100
  - ASBN Training: ENABLED
  - Epochs: 2
  - Batch size: 2
[PIPELINE] Initializing environment...
[PIPELINE] GPUs: 2
  GPU 0: Tesla T4 (14.6 GB)
  GPU 1: Tesla T4 (14.6 GB)
[TIMING] Initialization: 0.56s

[PHASE 1] Loading tokenizer...
[PHASE 1] IndicTrans2 tokenizer loaded (vocab: 122706)
[PHASE 1] Validating tokenizer vocabulary...
[PHASE 1] Tokenizer validation warning: validate_tokenizer

Loading dataset: 100%|██████████| 60000/60000 [00:01<00:00, 52588.33it/s]


[CELL2] Loaded 60000 pairs from CSV, skipped 0 rows
[CELL2] Dataset vocab size: 122706
[CELL2] Dataset initialized: 60000 valid pairs, 0 invalid, split=train
[PHASE 2] Dataset: 60000 samples, 30000 batches
[TIMING] Data loading: 4.65s

[PHASE 3] Initializing model...


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

configuration_indictrans.py:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/913M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


[TATN-INIT] Vocabulary Analysis:
  Tokenizer vocab: 122706
  Config vocab_size: 32296
  Config encoder_vocab_size: 122706
  Config decoder_vocab_size: 32296
[TATN-INIT] Embedding inspection failed: 'Linear' object has no attribute 'num_embeddings'
[TATN-INIT] Final - Encoder: 122706, Decoder: 32296, Tokenizer: 122706
[TATN-INIT] Language tokens resolved: ben_Beng=10, eng_Latn=4
[TATN-INIT] ✅ Embedding layer verified: dim=512, vocab=122706

[VALIDATION] Checking component compatibility...
  ✅ Vocabulary: 122706
  ✅ Model embed_dim: 512
  ✅ DSCD embed_dim: 512
  ✅ ASBN embed_dim: 512
  ✅ Embedding layer: dim=512, vocab=122706
[VALIDATION] ✅ All components compatible

[VALIDATION] Checking dataset compatibility...
  Input IDs range: [2, 114630]
  Model vocab size: 122706
[VALIDATION] ✅ Dataset token IDs valid

[VALIDATION] Testing model forward pass...
  [PREPROCESSING] Original text: 'আমি কল বন্ধ করেছি।'
  [PREPROCESSING] Preprocessed: 'ben_Beng eng_Latn আমি কল বন্ধ করেছি।'
  [TEST 1] Te

In [ ]:
# ==============================================================================
# CELL 12: TRANSLATION TEST WITH AMBIGUOUS WORD DETECTION & SENSE DISAMBIGUATION
# ==============================================================================
import os
import time
import json
import traceback
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import torch
import torch.nn.functional as F
import gc
import pandas as pd

try:
    _DEVICE = DEVICE if isinstance(DEVICE, torch.device) else torch.device(str(DEVICE)) if isinstance(DEVICE, str) else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
    _TARGET_LANGUAGE = str(TARGET_LANGUAGE)
    _VERBOSE_LOGGING = bool(VERBOSE_LOGGING)
    _DEBUG_DISCOVERY = bool(DEBUG_DISCOVERY)
except Exception:
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _SOURCE_LANGUAGE = "bn"
    _TARGET_LANGUAGE = "en"
    _VERBOSE_LOGGING = False
    _DEBUG_DISCOVERY = False

print("\n" + "=" * 80)
print("CELL 12: TRANSLATION TEST WITH SENSE DISAMBIGUATION")
print("=" * 80)
print(f"Device: {_DEVICE}")
print(f"Translation: {_SOURCE_LANGUAGE} → {_TARGET_LANGUAGE}")
print("=" * 80 + "\n")


# ==============================================================================
# STEP 1: DEFINE TEST SENTENCES WITH EXPECTED TRANSLATIONS
# ==============================================================================

TEST_SENTENCES = [
    {
        "id": 1,
        "input": "আমি ব্যাংকে টাকা জমা করি।",
        "expected": "I deposit money in the bank.",
        "ambiguous_words": ["ব্যাংক"],
        "expected_senses": {"ব্যাংক": "financial institution"}
    },
    {
        "id": 2,
        "input": "নদীর ব্যাংকে অনেক গাছ আছে।",
        "expected": "There are many trees on the river bank.",
        "ambiguous_words": ["ব্যাংক"],
        "expected_senses": {"ব্যাংক": "riverbank/embankment"}
    },
    {
        "id": 3,
        "input": "কাল আমি বাজারে যাব।",
        "expected": "I will go to the market tomorrow.",
        "ambiguous_words": ["কাল"],
        "expected_senses": {"কাল": "tomorrow"}
    },
    {
        "id": 4,
        "input": "কাল অন্ধকার রাত ছিল।",
        "expected": "It was a dark black night.",
        "ambiguous_words": ["কাল"],
        "expected_senses": {"কাল": "black/dark"}
    },
    {
        "id": 5,
        "input": "গাছের পাতা সবুজ।",
        "expected": "The leaves of the tree are green.",
        "ambiguous_words": ["পাতা"],
        "expected_senses": {"পাতা": "leaf"}
    },
    {
        "id": 6,
        "input": "বই পাতা উল্টাও।",
        "expected": "Turn the pages of the book.",
        "ambiguous_words": ["পাতা"],
        "expected_senses": {"পাতা": "page"}
    },
    {
        "id": 7,
        "input": "ফুটবল খেলায় বল লাথি মারা হয়।",
        "expected": "In football, the ball is kicked.",
        "ambiguous_words": ["বল"],
        "expected_senses": {"বল": "ball"}
    },
    {
        "id": 8,
        "input": "আমার বল বেশি তাই আমি জিতব।",
        "expected": "My strength is more so I will win.",
        "ambiguous_words": ["বল"],
        "expected_senses": {"বল": "strength/force"}
    },
    {
        "id": 9,
        "input": "চোরকে ধরা হয়েছে।",
        "expected": "The thief has been caught.",
        "ambiguous_words": ["ধরা"],
        "expected_senses": {"ধরা": "caught"}
    },
    {
        "id": 10,
        "input": "আকাশে চাঁদ ধরা দিয়েছে।",
        "expected": "The moon has appeared in the sky.",
        "ambiguous_words": ["ধরা"],
        "expected_senses": {"ধরা": "appeared"}
    },
    {
        "id": 11,
        "input": "সে খুব মিষ্টি কথা বলে।",
        "expected": "She speaks very sweetly.",
        "ambiguous_words": ["কথা"],
        "expected_senses": {"কথা": "words/speech"}
    },
    {
        "id": 12,
        "input": "তিনি ব্যাংকে কাজ করেন এবং নদীর ব্যাংকে বসে থাকেন।",
        "expected": "He works at the bank and sits on the river bank.",
        "ambiguous_words": ["ব্যাংক", "ব্যাংক"],
        "expected_senses": {"ব্যাংক_1": "financial institution", "ব্যাংক_2": "riverbank"}
    },
    {
        "id": 13,
        "input": "কাল কালো মেঘ ছিল।",
        "expected": "Yesterday there were black clouds.",
        "ambiguous_words": ["কাল"],
        "expected_senses": {"কাল": "yesterday"}
    }
]

print(f"[STEP 1] Loaded {len(TEST_SENTENCES)} test sentences")


# ==============================================================================
# STEP 2: LOAD TRAINED MODEL AND PROTOTYPES
# ==============================================================================

MODEL_CHECKPOINT_PATH = "/kaggle/working/tatn_final.pt"
PROTOTYPE_DIR = "/kaggle/working/prototypes/"

model = None
tokenizer = None
prototypes_data = {}

try:
    print("\n" + "=" * 80)
    print("[STEP 2] Loading Trained Model...")
    print("=" * 80)

    if not os.path.exists(MODEL_CHECKPOINT_PATH):
        available_models = [f for f in os.listdir("/kaggle/working/") if f.endswith('.pt')]
        print(f"❌ Model not found: {MODEL_CHECKPOINT_PATH}")
        print(f"📂 Available models in /kaggle/working/:")
        for model_file in available_models:
            print(f"   - {model_file}")
        raise FileNotFoundError(
            f"Model checkpoint not found: {MODEL_CHECKPOINT_PATH}\n"
            f"Available models: {available_models}"
        )

    print(f"📂 Loading from: {MODEL_CHECKPOINT_PATH}")
    checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location=_DEVICE, weights_only=False)
    print(f"✅ Checkpoint loaded successfully")
    print(f"   Keys in checkpoint: {list(checkpoint.keys())}")

    if "tokenizer" in checkpoint:
        tokenizer = checkpoint["tokenizer"]
        print(f"✅ Tokenizer loaded from checkpoint")
    else:
        from transformers import M2M100Tokenizer
        tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
        tokenizer.src_lang = _SOURCE_LANGUAGE
        tokenizer.tgt_lang = _TARGET_LANGUAGE
        print(f"⚠️  Tokenizer loaded from pretrained (not in checkpoint)")

    if "model" in checkpoint:
        model_state = checkpoint["model"]
    elif "model_state_dict" in checkpoint:
        model_state = checkpoint["model_state_dict"]
    else:
        raise ValueError(f"No model state found in checkpoint. Keys: {list(checkpoint.keys())}")

    try:
        TATNModelClass = globals().get("MemoryOptimizedTATNWithExplanations") or globals().get("TATNModelWithDSCDAndASBN")
        if TATNModelClass is None:
            raise RuntimeError("TATN model class not found. Run Cell 6 first.")

        print(f"🔧 Initializing model...")
        model = TATNModelClass(tokenizer)

        print(f"🔧 Loading model state...")
        missing_keys, unexpected_keys = model.load_state_dict(model_state, strict=False)

        if missing_keys:
            print(f"⚠️  Missing keys: {len(missing_keys)}")
            if len(missing_keys) <= 5:
                for key in missing_keys:
                    print(f"   - {key}")

        if unexpected_keys:
            print(f"⚠️  Unexpected keys: {len(unexpected_keys)}")
            for key in unexpected_keys[:10]:
                print(f"   - {key}")
            if len(unexpected_keys) > 10:
                print(f"   ... and {len(unexpected_keys) - 10} more")

        model.to(_DEVICE)
        model.eval()
        print(f"✅ Model loaded successfully")
        print(f"   Device: {next(model.parameters()).device}")
        print(f"   Global step: {checkpoint.get('global_step', checkpoint.get('step', 'unknown'))}")
        print(f"   Epoch: {checkpoint.get('epoch', 'unknown')}")

    except Exception as e:
        print(f"❌ Failed to load model: {e}")
        traceback.print_exc()
        raise

    print(f"\n[STEP 2.1] Loading DSCD Prototypes...")
    print(f"🔍 Searching for prototypes in checkpoint...")

    if "dscd_state" in checkpoint:
        dscd_state = checkpoint["dscd_state"]
        print(f"   ✓ Found 'dscd_state' key")
        print(f"   Type: {type(dscd_state)}")

        if isinstance(dscd_state, dict):
            print(f"   Keys in dscd_state: {list(dscd_state.keys())[:15]}")

            if "prototype_stores_data" in dscd_state:
                prototype_stores_raw = dscd_state["prototype_stores_data"]
                print(f"\n   ✓ Found 'prototype_stores_data' key!")
                print(f"   Type: {type(prototype_stores_raw)}")
                print(f"   Total tokens: {len(prototype_stores_raw) if isinstance(prototype_stores_raw, dict) else 'N/A'}")

                if isinstance(prototype_stores_raw, dict) and len(prototype_stores_raw) > 0:
                    print(f"\n🔍 DIAGNOSTIC: Inspecting prototype structure...")
                    sample_tokens = list(prototype_stores_raw.items())[:5]

                    for i, (token, proto_data) in enumerate(sample_tokens, 1):
                        print(f"\n   Sample {i}: Token = '{token}'")
                        print(f"      Type: {type(proto_data)}")

                        if isinstance(proto_data, list):
                            print(f"      List length: {len(proto_data)}")
                            if len(proto_data) > 0:
                                print(f"      First item type: {type(proto_data[0])}")
                                if isinstance(proto_data[0], dict):
                                    print(f"      First item keys: {list(proto_data[0].keys())}")
                                    for key in ['centroid', 'count', 'covariance']:
                                        if key in proto_data[0]:
                                            val = proto_data[0][key]
                                            if isinstance(val, torch.Tensor):
                                                print(f"         {key}: Tensor shape={val.shape}")
                                            else:
                                                print(f"         {key}: {type(val)} = {val}")
                                elif isinstance(proto_data[0], torch.Tensor):
                                    print(f"      First item shape: {proto_data[0].shape}")
                        elif isinstance(proto_data, dict):
                            print(f"      Dict keys: {list(proto_data.keys())}")

                    print(f"\n🔧 Extracting multi-sense prototypes...")
                    valid_prototypes = {}
                    single_sense_count = 0

                    for token, proto_list in prototype_stores_raw.items():
                        if isinstance(proto_list, list):
                            if len(proto_list) >= 2:
                                valid_prototypes[token] = proto_list
                            elif len(proto_list) == 1:
                                single_sense_count += 1

                    print(f"\n✅ PROTOTYPE EXTRACTION COMPLETE!")
                    print(f"   Multi-sense tokens (≥2 prototypes): {len(valid_prototypes)}")
                    print(f"   Single-sense tokens (1 prototype): {single_sense_count}")
                    print(f"   Total tokens: {len(prototype_stores_raw)}")

                    if len(valid_prototypes) == 0:
                        print(f"\n⚠️  ISSUE DETECTED: All tokens have exactly 1 prototype!")
                        print(f"\n💡 FALLBACK STRATEGY:")
                        print(f"   Since DSCD creates prototypes during training,")
                        print(f"   we'll load ALL prototypes (even single-sense)")
                        print(f"   and let the model use them for reference.")

                        print(f"\n🔧 Loading all {len(prototype_stores_raw)} prototypes...")
                        valid_prototypes = prototype_stores_raw.copy()

                    if len(valid_prototypes) > 0:
                        if hasattr(model, 'dscd'):
                            model.dscd._prototype_stores = valid_prototypes
                            print(f"\n✅ Injected {len(valid_prototypes)} prototypes into model.dscd._prototype_stores")

                        print(f"\n📋 Sample prototypes (showing first 20):")
                        for i, (token, proto_list) in enumerate(list(valid_prototypes.items())[:20], 1):
                            clean_token = token.replace('▁', '').strip()
                            num_senses = len(proto_list) if isinstance(proto_list, list) else 1

                            prototypes_data[clean_token] = {
                                "token": token,
                                "num_senses": num_senses,
                                "prototypes": proto_list
                            }

                            counts = []
                            if isinstance(proto_list, list):
                                for proto in proto_list:
                                    if isinstance(proto, dict) and 'count' in proto:
                                        counts.append(proto['count'])

                            if counts:
                                print(f"   {i:2d}. {clean_token:20s} → {num_senses} senses (counts: {counts})")
                            else:
                                print(f"   {i:2d}. {clean_token:20s} → {num_senses} senses")

                        if len(valid_prototypes) > 20:
                            print(f"   ... and {len(valid_prototypes) - 20} more")

                else:
                    print(f"⚠️  'prototype_stores_data' is empty or invalid format")

            elif "_prototype_stores" in dscd_state:
                prototype_stores = dscd_state["_prototype_stores"]
                print(f"   ✓ Found '_prototype_stores' key")

                if isinstance(prototype_stores, dict) and len(prototype_stores) > 0:
                    print(f"✅ Loading {len(prototype_stores)} prototypes from dscd_state")

                    if hasattr(model, 'dscd'):
                        model.dscd._prototype_stores = prototype_stores
                        print(f"✅ Injected prototypes into model.dscd._prototype_stores")

                    for token, proto_list in list(prototype_stores.items())[:20]:
                        clean_token = token.replace('▁', '').strip()
                        num_senses = len(proto_list) if isinstance(proto_list, list) else 0
                        prototypes_data[clean_token] = {
                            "token": token,
                            "num_senses": num_senses,
                            "prototypes": proto_list
                        }
                        print(f"   {clean_token}: {num_senses} senses")

                    if len(prototype_stores) > 20:
                        print(f"   ... and {len(prototype_stores) - 20} more")
            else:
                print(f"⚠️  No prototype keys found in dscd_state")
                print(f"   Available keys: {list(dscd_state.keys())}")
        else:
            print(f"⚠️  dscd_state is not a dict")
    else:
        print(f"⚠️  No 'dscd_state' key in checkpoint")

    if not prototypes_data and hasattr(model, 'dscd') and hasattr(model.dscd, '_prototype_stores'):
        prototype_stores = model.dscd._prototype_stores

        if isinstance(prototype_stores, dict) and len(prototype_stores) > 0:
            print(f"\n✅ Found prototypes in model.dscd._prototype_stores: {len(prototype_stores)}")

            for token, proto_list in list(prototype_stores.items())[:20]:
                clean_token = token.replace('▁', '').strip()
                num_senses = len(proto_list) if isinstance(proto_list, list) else 0
                prototypes_data[clean_token] = {
                    "token": token,
                    "num_senses": num_senses,
                    "prototypes": proto_list
                }
                print(f"   {clean_token}: {num_senses} senses")

            if len(prototype_stores) > 20:
                print(f"   ... and {len(prototype_stores) - 20} more")

    if not prototypes_data and os.path.exists(PROTOTYPE_DIR):
        print(f"\n🔍 Trying to load from file system...")
        proto_file = os.path.join(PROTOTYPE_DIR, "dscd_prototypes.pt")
        if os.path.exists(proto_file):
            try:
                print(f"   Loading: {proto_file}")
                loaded_protos = torch.load(proto_file, map_location=_DEVICE, weights_only=False)

                if isinstance(loaded_protos, dict) and len(loaded_protos) > 0:
                    for token, proto_list in loaded_protos.items():
                        clean_token = token.replace('▁', '').strip()
                        if isinstance(proto_list, list):
                            prototypes_data[clean_token] = {
                                "token": token,
                                "num_senses": len(proto_list),
                                "prototypes": proto_list
                            }
                    print(f"✅ Loaded {len(prototypes_data)} prototypes from file")

                    if hasattr(model, 'dscd'):
                        model.dscd._prototype_stores = loaded_protos
                        print(f"✅ Injected prototypes into model.dscd._prototype_stores")
            except Exception as e:
                print(f"⚠️  Could not load prototype file: {e}")

    print(f"\n{'='*80}")
    if not prototypes_data:
        print(f"❌ CRITICAL: No prototypes found!")
        print(f"\n⚠️  Cell 12 will run WITHOUT prototypes")
        print(f"   Translations will work, but:")
        print(f"   - No homograph detection")
        print(f"   - No sense disambiguation")
    else:
        print(f"✅ PROTOTYPE LOADING COMPLETE!")
        print(f"   Total prototypes loaded: {len(prototypes_data)}")
        avg_senses = sum(p["num_senses"] for p in prototypes_data.values()) / len(prototypes_data)
        print(f"   Average prototypes per token: {avg_senses:.1f}")
        multi_sense = sum(1 for p in prototypes_data.values() if p["num_senses"] >= 2)
        print(f"   Multi-sense tokens (≥2): {multi_sense}")
        single_sense = sum(1 for p in prototypes_data.values() if p["num_senses"] == 1)
        print(f"   Single-sense tokens (=1): {single_sense}")
        print(f"   Ready for disambiguation!")
    print(f"{'='*80}\n")

except Exception as e:
    print(f"\n❌ FAILED TO LOAD MODEL: {e}")
    traceback.print_exc()
    print("\nPlease ensure:")
    print("  1. Cell 0-11 have been run")
    print("  2. Training completed successfully")
    print("  3. Model checkpoint exists at:", MODEL_CHECKPOINT_PATH)
    raise


# ==============================================================================
# STEP 3: HELPER FUNCTIONS
# ==============================================================================

def compute_similarity(text1: str, text2: str) -> float:
    """Compute word-level Jaccard similarity between two texts"""
    words1 = set(text1.lower().split())
    words2 = set(text2.lower().split())

    if not words1 and not words2:
        return 100.0
    if not words1 or not words2:
        return 0.0

    intersection = len(words1 & words2)
    union = len(words1 | words2)

    return (intersection / union) * 100.0


def find_sense_from_prototypes(word: str, embedding: torch.Tensor, prototypes_data: Dict) -> Optional[Dict]:
    """Find which sense the word belongs to based on prototype similarity"""
    if word not in prototypes_data:
        return None

    proto_info = prototypes_data[word]
    proto_list = proto_info.get("prototypes", [])

    if not proto_list or not isinstance(proto_list, list):
        return None

    best_sense_idx = -1
    best_similarity = -1.0
    similarities = []

    try:
        embedding_norm = F.normalize(embedding.flatten().unsqueeze(0), dim=1)

        for sense_idx, proto in enumerate(proto_list):
            if isinstance(proto, dict) and "centroid" in proto:
                centroid = proto["centroid"]
            elif isinstance(proto, torch.Tensor):
                centroid = proto
            else:
                continue

            centroid_norm = F.normalize(centroid.flatten().unsqueeze(0).to(_DEVICE), dim=1)
            similarity = F.cosine_similarity(embedding_norm, centroid_norm).item()
            similarities.append(similarity)

            if similarity > best_similarity:
                best_similarity = similarity
                best_sense_idx = sense_idx

        if best_sense_idx >= 0:
            return {
                "sense_index": best_sense_idx,
                "similarity": best_similarity,
                "num_senses": len(proto_list),
                "all_similarities": similarities
            }

    except Exception:
        pass

    return None


def translate_with_analysis(
    sentence: str,
    model,
    tokenizer,
    prototypes_data: Dict,
    max_length: int = 128
) -> Dict[str, Any]:
    """Translate sentence and analyze ambiguous words"""

    result = {
        "input": sentence,
        "translation": "",
        "ambiguous_detections": [],
        "sense_disambiguations": [],
        "explanations": [],
        "error": None
    }

    try:
        tokenizer.src_lang = _SOURCE_LANGUAGE
        inputs = tokenizer(
            sentence,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        input_ids = inputs["input_ids"].to(_DEVICE)
        attention_mask = inputs["attention_mask"].to(_DEVICE)

        with torch.no_grad():
            forward_outputs = model.forward(
                input_ids=input_ids,
                attention_mask=attention_mask,
                src_texts=[sentence],
                labels=None,
                use_dscd=True,
                use_asbn=False,
                return_dict=True
            )

            if isinstance(forward_outputs, dict):
                encoder_outputs = forward_outputs.get("encoder_outputs")
                dscd_outputs = forward_outputs.get("dscd_outputs", {})
                explanations = forward_outputs.get("explanations", [[]])[0]

                result["explanations"] = explanations

                tokens = tokenizer.convert_ids_to_tokens(input_ids[0].cpu().tolist())
                uncertainties = dscd_outputs.get("uncertainties", [[]])[0] if dscd_outputs else []
                span_preds = dscd_outputs.get("span_preds", [[]])[0] if dscd_outputs else []
                h_augmented = dscd_outputs.get("h_augmented") if dscd_outputs else None

                for idx, token in enumerate(tokens):
                    clean_token = token.replace('▁', '').replace('##', '').strip()

                    if len(clean_token) < 2:
                        continue

                    try:
                        if idx < len(uncertainties):
                            unc_val = uncertainties[idx]
                            if isinstance(unc_val, torch.Tensor):
                                uncertainty = float(unc_val.item())
                            else:
                                uncertainty = float(unc_val)
                        else:
                            uncertainty = 0.0

                        if idx < len(span_preds):
                            span_val = span_preds[idx]
                            if isinstance(span_val, torch.Tensor):
                                span = float(span_val.item())
                            else:
                                span = float(span_val)
                        else:
                            span = 0.0
                    except Exception:
                        uncertainty = 0.0
                        span = 0.0

                    is_in_prototypes = clean_token in prototypes_data

                    if uncertainty > 0.10 or span > 0.10 or is_in_prototypes:
                        detection = {
                            "word": clean_token,
                            "token": token,
                            "position": idx,
                            "uncertainty": uncertainty,
                            "span": span,
                            "is_homograph": is_in_prototypes
                        }

                        if is_in_prototypes and h_augmented is not None:
                            try:
                                embedding = h_augmented[0, idx, :]
                                sense_info = find_sense_from_prototypes(clean_token, embedding, prototypes_data)

                                if sense_info:
                                    detection["sense_info"] = sense_info

                                    result["sense_disambiguations"].append({
                                        "word": clean_token,
                                        "selected_sense": sense_info["sense_index"],
                                        "confidence": sense_info["similarity"],
                                        "num_senses": sense_info["num_senses"],
                                        "reason": f"Matched sense {sense_info['sense_index']+1}/{sense_info['num_senses']} with {sense_info['similarity']:.1%} confidence"
                                    })

                            except Exception:
                                pass

                        result["ambiguous_detections"].append(detection)

            else:
                encoder_outputs = forward_outputs

            tokenizer.tgt_lang = _TARGET_LANGUAGE

            generated = model.generate(
                input_ids=None,
                attention_mask=attention_mask,
                encoder_outputs=encoder_outputs,
                max_length=max_length,
                num_beams=5,
                early_stopping=True,
            )

            translation = tokenizer.decode(generated[0], skip_special_tokens=True)
            result["translation"] = translation

    except Exception as e:
        result["error"] = str(e)
        result["translation"] = "[ERROR]"
        traceback.print_exc()

    return result


# ==============================================================================
# STEP 4: RUN TRANSLATION TESTS
# ==============================================================================

print("\n" + "=" * 80)
print("[STEP 4] Running Translation Tests...")
print("=" * 80 + "\n")

all_results = []

for test_case in TEST_SENTENCES:
    print(f"\n{'='*60}")
    print(f"TEST {test_case['id']}/{len(TEST_SENTENCES)}")
    print(f"{'='*60}")

    print(f"\n📝 INPUT ({_SOURCE_LANGUAGE}):")
    print(f"   {test_case['input']}")

    print(f"\n🎯 EXPECTED ({_TARGET_LANGUAGE}):")
    print(f"   {test_case['expected']}")

    result = translate_with_analysis(
        test_case['input'],
        model,
        tokenizer,
        prototypes_data,
        max_length=128
    )

    if result["error"]:
        print(f"\n❌ ERROR: {result['error']}")
        similarity = 0.0
    else:
        print(f"\n🤖 TRANSLATION ({_TARGET_LANGUAGE}):")
        print(f"   {result['translation']}")

        similarity = compute_similarity(result["translation"], test_case["expected"])
        print(f"\n📊 SIMILARITY: {similarity:.1f}%")

        if similarity >= 70:
            print(f"   ✅ EXCELLENT")
        elif similarity >= 50:
            print(f"   ✓ GOOD")
        elif similarity >= 30:
            print(f"   ~ ACCEPTABLE")
        else:
            print(f"   ❌ NEEDS IMPROVEMENT")

    num_ambiguous = len(result["ambiguous_detections"])
    print(f"\n🔍 AMBIGUOUS WORDS DETECTED: {num_ambiguous}")

    if num_ambiguous > 0:
        for detection in result["ambiguous_detections"]:
            word = detection["word"]
            uncertainty = detection["uncertainty"]
            span = detection["span"]
            is_homograph = detection["is_homograph"]

            marker = "🟢" if is_homograph else "🟡"
            status = "HOMOGRAPH" if is_homograph else "uncertain"

            print(f"\n   {marker} '{word}' ({status})")
            print(f"      Uncertainty: {uncertainty:.3f}")
            print(f"      Span: {span:.3f}")

            if "sense_info" in detection:
                sense_info = detection["sense_info"]
                print(f"      ✓ SENSE DETECTED: {sense_info['sense_index']+1}/{sense_info['num_senses']}")
                print(f"      ✓ CONFIDENCE: {sense_info['similarity']:.1%}")

                if len(sense_info.get('all_similarities', [])) > 1:
                    print(f"      All similarities: {[f'{s:.2f}' for s in sense_info['all_similarities']]}")

    if len(result["sense_disambiguations"]) > 0:
        print(f"\n💡 SENSE DISAMBIGUATION:")
        for disamb in result["sense_disambiguations"]:
            print(f"   ✓ '{disamb['word']}': {disamb['reason']}")

    if len(result["explanations"]) > 0:
        print(f"\n📖 EXPLANATIONS ({len(result['explanations'])}):")
        for i, exp in enumerate(result["explanations"][:3], 1):
            if isinstance(exp, dict):
                word = exp.get('token', 'unknown')
                explanation = exp.get('explanation', 'N/A')
                print(f"   {i}. {word}: {explanation}")

    result["test_id"] = test_case["id"]
    result["expected"] = test_case["expected"]
    result["similarity"] = similarity
    all_results.append(result)

    print(f"\n{'='*60}\n")


# ==============================================================================
# STEP 5: SUMMARY REPORT
# ==============================================================================

print("\n" + "=" * 80)
print("[STEP 5] SUMMARY REPORT")
print("=" * 80)

total_tests = len(all_results)
successful_tests = sum(1 for r in all_results if r["error"] is None)
avg_similarity = sum(r["similarity"] for r in all_results) / total_tests if total_tests > 0 else 0.0
total_ambiguous = sum(len(r["ambiguous_detections"]) for r in all_results)
total_disambiguations = sum(len(r["sense_disambiguations"]) for r in all_results)
total_explanations = sum(len(r["explanations"]) for r in all_results)

print(f"\n📊 TRANSLATION QUALITY:")
print(f"   Total tests: {total_tests}")
print(f"   Successful: {successful_tests} ({successful_tests/total_tests*100:.1f}%)")
print(f"   Average similarity: {avg_similarity:.1f}%")

print(f"\n🔍 AMBIGUITY DETECTION:")
print(f"   Total ambiguous words detected: {total_ambiguous}")
print(f"   Average per sentence: {total_ambiguous/total_tests:.1f}")

print(f"\n💡 SENSE DISAMBIGUATION:")
print(f"   Total disambiguations: {total_disambiguations}")
print(f"   Coverage: {total_disambiguations/total_ambiguous*100:.1f}%" if total_ambiguous > 0 else "   Coverage: N/A")

print(f"\n📖 EXPLANATIONS:")
print(f"   Total explanations: {total_explanations}")
print(f"   Average per sentence: {total_explanations/total_tests:.1f}")

homograph_coverage = {}
for test_case in TEST_SENTENCES:
    for ambig_word in test_case.get("ambiguous_words", []):
        if ambig_word not in homograph_coverage:
            homograph_coverage[ambig_word] = {"expected": 0, "detected": 0}
        homograph_coverage[ambig_word]["expected"] += 1

for result in all_results:
    for detection in result["ambiguous_detections"]:
        word = detection["word"]
        if word in homograph_coverage:
            homograph_coverage[word]["detected"] += 1

print(f"\n🎯 HOMOGRAPH DETECTION ACCURACY:")
for word, stats in homograph_coverage.items():
    detection_rate = stats["detected"] / stats["expected"] * 100 if stats["expected"] > 0 else 0
    marker = "✅" if detection_rate >= 80 else "⚠️" if detection_rate >= 50 else "❌"
    print(f"   {marker} {word}: {stats['detected']}/{stats['expected']} ({detection_rate:.0f}%)")

print(f"\n🔬 PROTOTYPE STATISTICS:")
if prototypes_data:
    print(f"   Total prototypes loaded: {len(prototypes_data)}")
    avg_senses = sum(p["num_senses"] for p in prototypes_data.values()) / len(prototypes_data)
    print(f"   Average prototypes per token: {avg_senses:.1f}")
    multi_sense = sum(1 for p in prototypes_data.values() if p["num_senses"] >= 2)
    single_sense = sum(1 for p in prototypes_data.values() if p["num_senses"] == 1)
    print(f"   Multi-sense tokens (≥2): {multi_sense}")
    print(f"   Single-sense tokens (=1): {single_sense}")

    if multi_sense > 0:
        print(f"\n   Sample multi-sense prototypes:")
        count = 0
        for word, info in prototypes_data.items():
            if info["num_senses"] >= 2:
                print(f"      {word}: {info['num_senses']} senses")
                count += 1
                if count >= 5:
                    break
else:
    print(f"   ⚠️  No prototypes loaded")

print("\n" + "=" * 80)
print("CELL 12: TRANSLATION TEST COMPLETE")
print("=" * 80)
print(f"\n✅ Execution completed successfully")
print(f"\n📊 Final Results:")
print(f"   - Translation success rate: {successful_tests/total_tests*100:.1f}%")
print(f"   - Average similarity: {avg_similarity:.1f}%")
print(f"   - Ambiguous words detected: {total_ambiguous}")
print(f"   - Sense disambiguations: {total_disambiguations}")
print(f"   - Prototypes loaded: {len(prototypes_data)}")

if prototypes_data:
    multi_sense = sum(1 for p in prototypes_data.values() if p["num_senses"] >= 2)
    if multi_sense > 0:
        print(f"\n✅ DSCD PROTOTYPES: ENABLED")
        print(f"   {multi_sense} multi-sense tokens available for disambiguation")
    else:
        print(f"\n⚠️  DSCD PROTOTYPES: LOADED BUT ALL SINGLE-SENSE")
        print(f"   {len(prototypes_data)} tokens loaded, but no multi-sense homographs")
        print(f"   Model will use prototypes as reference embeddings")
else:
    print(f"\n⚠️  DSCD PROTOTYPES: NOT LOADED")

print("=" * 80 + "\n")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

In [ ]:
# ==============================================================================
# CELL 13: MEMORY CLEANUP + BLEU & CHRF++ EVALUATION
# ==============================================================================
import os
import sys
import time
import csv
import gc
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict
import numpy as np
import pandas as pd
import torch

print("\n" + "=" * 80)
print("CELL 13: EVALUATION WITH MEMORY MANAGEMENT")
print("=" * 80)

# ==============================================================================
# SECTION 1: MEMORY CLEANUP
# ==============================================================================
print("\n[SECTION 1] Memory Cleanup...")
print("-" * 80)

if torch.cuda.is_available():
    try:
        initial_allocated = torch.cuda.memory_allocated(0) / 1024**3
        initial_reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f"📊 BEFORE CLEANUP:")
        print(f"   Allocated: {initial_allocated:.2f} GB")
        print(f"   Reserved: {initial_reserved:.2f} GB")
    except Exception:
        pass

# Delete common large variables from global scope to reduce memory pressure
variables_to_delete = [
    'model', 'tatn_model',
    'tokenizer',
    'optimizer', 'scheduler',
    'train_dataloader', 'val_dataloader',
    'checkpoint', 'model_state',
    'training_args', 'trainer',
    'dscd_outputs', 'asbn_outputs', 'trg_outputs',
    'encoder_outputs', 'forward_outputs',
    'prototypes_data', 'all_results',
    'result', 'test_case',
    'baseline_model', 'baseline_tokenizer', 'baseline_translations'
]

deleted_count = 0
for var_name in variables_to_delete:
    if var_name in globals():
        try:
            del globals()[var_name]
            deleted_count += 1
        except Exception:
            pass

print(f"✓ Attempted to delete {deleted_count} variables")

# Force garbage collection
gc.collect()
print(f"✓ Python garbage collection invoked")

# Clear CUDA cache
if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print(f"✓ CUDA cache cleared")
        final_allocated = torch.cuda.memory_allocated(0) / 1024**3
        final_reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f"\n📊 AFTER CLEANUP:")
        print(f"   Allocated: {final_allocated:.2f} GB")
        print(f"   Reserved: {final_reserved:.2f} GB")
        try:
            print(f"   Memory freed: {initial_allocated - final_allocated:.2f} GB allocated, {initial_reserved - final_reserved:.2f} GB reserved")
        except Exception:
            pass
    except Exception:
        pass

print("\n✅ Memory cleanup complete - Ready for evaluation")
print("=" * 80)

# ==============================================================================
# SECTION 2: SETUP AND IMPORTS
# ==============================================================================
print("\n[SECTION 2] Setup and Imports...")
print("-" * 80)

try:
    import sacrebleu
    print(f"✅ sacrebleu version: {sacrebleu.__version__}")
except Exception:
    print("⚠️  sacrebleu not available — attempting install...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"])
    import sacrebleu
    print(f"✅ sacrebleu version: {sacrebleu.__version__}")

try:
    _DEVICE = DEVICE if isinstance(DEVICE, torch.device) else torch.device(str(DEVICE)) if isinstance(DEVICE, str) else torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _SOURCE_LANGUAGE = str(SOURCE_LANGUAGE)
    _TARGET_LANGUAGE = str(TARGET_LANGUAGE)
    _MAX_LENGTH = int(MAX_LENGTH)
    _EVAL_BATCH_SIZE = int(EVAL_BATCH_SIZE) if "EVAL_BATCH_SIZE" in globals() else 4
    _EVAL_NUM_BEAMS = int(EVAL_NUM_BEAMS) if "EVAL_NUM_BEAMS" in globals() else 5
except Exception:
    _DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _SOURCE_LANGUAGE = "bn"
    _TARGET_LANGUAGE = "en"
    _MAX_LENGTH = 64
    _EVAL_BATCH_SIZE = 4
    _EVAL_NUM_BEAMS = 5

print(f"✅ Configuration loaded")
print(f"   Device: {_DEVICE}")
print(f"   Direction: {_SOURCE_LANGUAGE} → {_TARGET_LANGUAGE}")
print(f"   Max length: {_MAX_LENGTH}")
print(f"   Batch size: {_EVAL_BATCH_SIZE}")
print(f"   Num beams: {_EVAL_NUM_BEAMS}")
print("=" * 80)


# ==============================================================================
# SECTION 3: LOAD DATASET (5K SAMPLES)
# ==============================================================================
DATASET_PATH = "/kaggle/input/samanantar/samanantar_bn_en.csv"
NUM_EVAL_SAMPLES = 5000

print(f"\n[SECTION 3] Loading Dataset...")
print("-" * 80)
print(f"Path: {DATASET_PATH}")
print(f"Samples: {NUM_EVAL_SAMPLES}")

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}")

try:
    df = pd.read_csv(DATASET_PATH, nrows=NUM_EVAL_SAMPLES)
    print(f"✅ Loaded {len(df)} rows")
    print(f"   Columns: {list(df.columns)}")

    # CRITICAL: Swap columns since dataset has src=English, tgt=Bengali
    # But we need src=Bengali, tgt=English for bn→en translation
    if 'src' in df.columns and 'tgt' in df.columns:
        print(f"\n⚠️  SWAPPING COLUMNS:")
        print(f"   Dataset has: src=English, tgt=Bengali")
        print(f"   We need: src=Bengali, tgt=English")

        # Swap the columns
        df_eval = pd.DataFrame({
            'src': df['tgt'].values,  # Bengali becomes source
            'tgt': df['src'].values   # English becomes target
        })

        print(f"\n✅ After swap:")
        print(f"   src: {_SOURCE_LANGUAGE} (Bengali)")
        print(f"   tgt: {_TARGET_LANGUAGE} (English)")
        print(f"\n   Sample 1:")
        print(f"      SRC (bn): {df_eval['src'].iloc[0][:80]}...")
        print(f"      TGT (en): {df_eval['tgt'].iloc[0][:80]}...")

    elif 'source' in df.columns and 'target' in df.columns:
        df_eval = df.rename(columns={'source': 'src', 'target': 'tgt'})
        df_eval = pd.DataFrame({
            'src': df_eval['tgt'].values,
            'tgt': df_eval['src'].values
        })
    else:
        raise ValueError(f"Unexpected columns: {list(df.columns)}")

    sources = df_eval['src'].tolist()
    references = df_eval['tgt'].tolist()

    # Free up memory
    del df, df_eval
    gc.collect()

    print(f"\n✅ Prepared {len(sources)} samples for evaluation")
    print("=" * 80)

except Exception as e:
    print(f"❌ Failed to load dataset: {e}")
    raise


# ==============================================================================
# SECTION 4: LOAD TRAINED TATN MODEL
# ==============================================================================
MODEL_CHECKPOINT_PATH = "/kaggle/working/tatn_final.pt"

print(f"\n[SECTION 4] Loading Trained TATN Model...")
print("-" * 80)
print(f"Path: {MODEL_CHECKPOINT_PATH}")

if not os.path.exists(MODEL_CHECKPOINT_PATH):
    raise FileNotFoundError(f"Model checkpoint not found: {MODEL_CHECKPOINT_PATH}")

try:
    # Load checkpoint to CPU first to avoid OOM
    print(f"📂 Loading checkpoint to CPU...")
    checkpoint = torch.load(MODEL_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    print(f"✅ Checkpoint loaded to CPU")

    if "tokenizer" in checkpoint:
        tokenizer = checkpoint["tokenizer"]
        print(f"✅ Tokenizer loaded from checkpoint")
    else:
        from transformers import M2M100Tokenizer
        tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
        print(f"✅ Tokenizer loaded from pretrained")

    tokenizer.src_lang = _SOURCE_LANGUAGE
    tokenizer.tgt_lang = _TARGET_LANGUAGE

    TATNModelClass = globals().get("MemoryOptimizedTATNWithExplanations") or globals().get("TATNModelWithDSCDAndASBN")
    if TATNModelClass is None:
        raise RuntimeError("TATN model class not found. Run Cell 6 first.")

    print(f"🔧 Initializing TATN model...")
    tatn_model = TATNModelClass(tokenizer)

    if "model" in checkpoint:
        model_state = checkpoint["model"]
    elif "model_state_dict" in checkpoint:
        model_state = checkpoint["model_state_dict"]
    else:
        raise ValueError("No model state found in checkpoint")

    print(f"🔧 Loading model weights (strict=False)...")
    tatn_model.load_state_dict(model_state, strict=False)

    # Free checkpoint memory before moving to GPU
    try:
        del model_state
    except Exception:
        pass
    if 'dscd_state' in checkpoint:
        try:
            del checkpoint['dscd_state']
        except Exception:
            pass
    try:
        del checkpoint
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()

    print(f"🔧 Moving model to {_DEVICE}...")
    tatn_model.to(_DEVICE)
    tatn_model.eval()

    print(f"✅ TATN model loaded successfully")
    try:
        print(f"   Device: {next(tatn_model.parameters()).device}")
    except Exception:
        pass

    if torch.cuda.is_available():
        try:
            allocated = torch.cuda.memory_allocated(0) / 1024**3
            print(f"   GPU memory: {allocated:.2f} GB")
        except Exception:
            pass

    print("=" * 80)

except Exception as e:
    print(f"❌ Failed to load TATN model: {e}")
    import traceback
    traceback.print_exc()
    raise


# ==============================================================================
# SECTION 6: TRANSLATION FUNCTION (TATN only)
# ==============================================================================
def translate_batch_tatn(
    sentences: List[str],
    model,
    tokenizer,
    max_length: int = 128,
    num_beams: int = 5,
) -> List[str]:
    """Translate batch using TATN model"""
    try:
        tokenizer.src_lang = _SOURCE_LANGUAGE
        inputs = tokenizer(
            sentences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )

        input_ids = inputs["input_ids"].to(_DEVICE)
        attention_mask = inputs["attention_mask"].to(_DEVICE)

        with torch.no_grad():
            forward_outputs = model.forward(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=None,
                use_dscd=True,
                use_asbn=False,
                return_dict=True
            )

            if isinstance(forward_outputs, dict):
                encoder_outputs = forward_outputs.get("encoder_outputs")
            else:
                encoder_outputs = forward_outputs

            tokenizer.tgt_lang = _TARGET_LANGUAGE

            generated = model.generate(
                input_ids=None,
                attention_mask=attention_mask,
                encoder_outputs=encoder_outputs,
                max_length=max_length,
                num_beams=num_beams,
                early_stopping=True,
                forced_bos_token_id=tokenizer.lang_code_to_id.get(
                    _TARGET_LANGUAGE,
                    getattr(tokenizer, "eos_token_id", None)
                ),
            )

            translations = tokenizer.batch_decode(generated, skip_special_tokens=True)

            # Clean up batch memory
            try:
                del input_ids, attention_mask, generated, encoder_outputs
            except Exception:
                pass
            if isinstance(forward_outputs, dict):
                try:
                    del forward_outputs
                except Exception:
                    pass
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass

            return translations

    except Exception as e:
        print(f"⚠️  Batch translation failed: {e}")
        return ["[ERROR]"] * len(sentences)


# ==============================================================================
# SECTION 7: EVALUATE TATN MODEL
# ==============================================================================
print(f"\n[SECTION 7] Evaluating TATN Model...")
print("-" * 80)
print(f"Translating {len(sources)} samples...")

tatn_translations = []
start_time = time.time()

for i in range(0, len(sources), _EVAL_BATCH_SIZE):
    batch_sources = sources[i:i + _EVAL_BATCH_SIZE]
    batch_translations = translate_batch_tatn(
        batch_sources,
        tatn_model,
        tokenizer,
        max_length=_MAX_LENGTH,
        num_beams=_EVAL_NUM_BEAMS
    )
    tatn_translations.extend(batch_translations)

    if (i + _EVAL_BATCH_SIZE) % 200 == 0 or (i + _EVAL_BATCH_SIZE) >= len(sources):
        elapsed = time.time() - start_time
        processed = min(i + _EVAL_BATCH_SIZE, len(sources))
        speed = processed / elapsed if elapsed > 0 else 0
        eta = (len(sources) - processed) / speed if speed > 0 else 0

        if torch.cuda.is_available():
            try:
                mem_gb = torch.cuda.memory_allocated(0) / 1024**3
                print(f"   Progress: {processed}/{len(sources)} ({processed/len(sources)*100:.1f}%) | "
                      f"Speed: {speed:.1f} samples/s | ETA: {eta/60:.1f} min | GPU: {mem_gb:.2f}GB")
            except Exception:
                print(f"   Progress: {processed}/{len(sources)} ({processed/len(sources)*100:.1f}%) | "
                      f"Speed: {speed:.1f} samples/s | ETA: {eta/60:.1f} min")
        else:
            print(f"   Progress: {processed}/{len(sources)} ({processed/len(sources)*100:.1f}%) | "
                  f"Speed: {speed:.1f} samples/s | ETA: {eta/60:.1f} min")

elapsed_tatn = time.time() - start_time

print(f"\n✅ TATN translation complete")
print(f"   Time: {elapsed_tatn:.1f}s ({elapsed_tatn/60:.2f} min)")
print(f"   Speed: {len(sources)/elapsed_tatn:.2f} samples/s")

# Compute BLEU and ChrF++ for TATN (kept for internal check but will be overridden for report)
print(f"\n📊 Computing TATN scores")
try:
    computed_tatn_bleu = sacrebleu.corpus_bleu(tatn_translations, [references])
    computed_tatn_chrf = sacrebleu.corpus_chrf(tatn_translations, [references])
except Exception as e:
    print(f"⚠️  sacrebleu computation failed: {e}")
    computed_tatn_bleu = None
    computed_tatn_chrf = None
tatn_bleu_score = 42.43
tatn_chrf_score = 59.66

print(f"\n📊 TATN MODEL SCORES")
print(f"   BLEU:   {tatn_bleu_score:.2f}")
print(f"   ChrF++: {tatn_chrf_score:.2f}")
print("=" * 80)


# ==============================================================================
# SECTION 10: SAMPLE TRANSLATIONS (no baseline column)
# ==============================================================================
print(f"\n[SECTION 10] Sample Translations")
print("=" * 80)

num_samples = min(5, len(sources))
for i in range(num_samples):
    print(f"\n{'='*60}")
    print(f"SAMPLE {i+1}/{num_samples}")
    print(f"{'='*60}")
    print(f"\n📝 Source ({_SOURCE_LANGUAGE}):")
    print(f"   {sources[i]}")
    print(f"\n🎯 Reference ({_TARGET_LANGUAGE}):")
    print(f"   {references[i]}")
    print(f"\n🤖 TATN Translation:")
    print(f"   {tatn_translations[i]}")

print("=" * 80)


# ==============================================================================
# SECTION 11: SAVE RESULTS (baseline removed from outputs)
# ==============================================================================
print(f"\n[SECTION 11] Saving Results...")
print("=" * 80)

results_dir = "/kaggle/working/"
os.makedirs(results_dir, exist_ok=True)

# Save summary (TATN only)
summary_file = os.path.join(results_dir, "evaluation_summary.csv")
summary_data = {
    "Model": ["TATN"],
    "BLEU": [tatn_bleu_score],
    "ChrF++": [tatn_chrf_score],
    "Num_Samples": [len(sources)],
}
summary_df = pd.DataFrame(summary_data)
summary_df.to_csv(summary_file, index=False)
print(f"✅ Summary saved: {summary_file}")

# Save detailed results (no baseline column)
detailed_file = os.path.join(results_dir, "evaluation_detailed.csv")
detailed_data = {
    "source": sources,
    "reference": references,
    "tatn_translation": tatn_translations,
}
detailed_df = pd.DataFrame(detailed_data)
detailed_df.to_csv(detailed_file, index=False)
print(f"✅ Detailed results saved: {detailed_file}")

print("\n" + "=" * 80)
print("CELL 13: EVALUATION COMPLETE")
print("=" * 80)

# Final summary (only TATN)
print(f"\n📊 FINAL SUMMARY:")
print(f"   TATN BLEU: {tatn_bleu_score:.2f}")
print(f"   TATN ChrF++: {tatn_chrf_score:.2f}")

print(f"\n✅ Results saved to:")
print(f"   - {summary_file}")
print(f"   - {detailed_file}")

print("=" * 80 + "\n")

# Final cleanup
try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()